# 无监督聚类分析

## 项目概述

本notebook实现了基于容量配置的无监督聚类分析，主要包括以下几个步骤：

1. **数据预处理** - 加载甲醇生产数据，剔除无效样本
2. **地理区域映射** - 将区域编号映射到主要大洲
3. **分大洲聚类分析** - 使用K-means对每个大洲的数据进行聚类
4. **结果排序** - 根据风电占比对聚类标签重新排序
5. **中位数点提取** - 提取各聚类的甲醇成本中位数代表点
6. **时序数据提取** - 获取代表点的风/光容量因子时间序列

## 数据来源
- 输入：综合区域可再生能源-低温电解-甲醇生产结果
- 输出：按大洲聚类的结果文件和代表点时序数据

---

## 1. 数据预处理与聚类分析

### 库导入与环境设置
首先导入必要的库并设置并行线程限制以优化性能。

### 数据加载与清洗
- 读取甲醇生产综合结果数据
- 剔除电解槽容量全为零的无效样本
- 进行大洲区域映射，保留主要大洲数据

### K-means聚类分析
对每个大洲分别进行聚类分析：
- 计算各技术容量占比并标准化
- 使用K-means算法分为4个聚类
- 根据风电占比对聚类标签重新排序

In [ ]:
# -*- coding: utf-8 -*-
"""
用途：测试不同聚类数量 K 对聚类效果的影响，并按“Nature Energy 风格”绘图（中文）
输出：对于每个匹配的 CSV，生成 4 张中文美化图（PNG+PDF）与 1 个指标表 CSV
说明：仅使用 Matplotlib（不依赖 seaborn）。标题/坐标轴/注释均为中文。
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

# =========================
# 路径（按需修改）
# =========================
input_dir  = r"D:\OneDrive\POTSED\DataOutput\Used_output"
out_dir    = r"D:\OneDrive\POTSED\03 Cluster"
os.makedirs(out_dir, exist_ok=True)

# 匹配文件名：combined_<scenario>_<prod>_results.csv
fname_re = re.compile(
    r"^combined_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)_results\.csv$",
    flags=re.IGNORECASE
)

# =========================
# 列名映射（与你原脚本一致）
# =========================
COLMAP = {
    "methanol": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "hp": "heat_pump_capacity_kW_th",   # 可选
        "dac": "dac_capacity_kgCO2_per_h",  # 可选
        "cost": "levelized_methanol_cost_per_ton"
    },
    "ftl": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "hp": "heat_pump_capacity_kW_th",
        "dac": "dac_capacity_kgCO2_per_h",
        "cost": "levelized_FTL_cost_per_ton"
    },
    "ammonia": {
        "wind": "wind_capacity",
        "pv": "pv_capacity",
        "bat_en": "battery_energy_capacity",
        "bat_st": "battery_storage_capacity",
        "ae": "electrolyzer_capacity_AE",
        "pem": "electrolyzer_capacity_PEM",
        "hp": "heat_pump_capacity_kW_th",
        "dac": "dac_capacity_kgCO2_per_h",
        "cost": "levelized_ammonia_cost_per_ton"
    }
}

# =========================
# Nature Energy 风格（中文）设置
# =========================
def set_nature_energy_style():
    # ---- 中文字体 ----
    preferred_fonts = ["Microsoft YaHei", "Sarasa UI SC", "Noto Sans CJK SC", "SimHei", "Source Han Sans CN", "DejaVu Sans"]
    mpl.rcParams["font.sans-serif"] = preferred_fonts
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["axes.unicode_minus"] = False  # 负号正常显示

    # ---- 字号集中管理 ----
    BASE_FONT = 14          # 全局基础
    TITLE_FONT = 13         # 标题
    LABEL_FONT = 15         # 坐标轴标签
    TICK_FONT  = 14         # 刻度
    LEGEND_FONT= 14         # 图例
    # （需要更大就整体+1/+2；Nature 文章版面一般 10–12pt）

    # ---- 画布/导出 ----
    mpl.rcParams["figure.figsize"] = (6.8, 4.6)  # 略放大
    mpl.rcParams["figure.dpi"] = 160
    mpl.rcParams["savefig.dpi"] = 300
    mpl.rcParams["savefig.bbox"] = "tight"
    mpl.rcParams["savefig.pad_inches"] = 0.04

    # ---- 轴&刻度 ----
    mpl.rcParams["axes.linewidth"] = 1.0
    mpl.rcParams["axes.edgecolor"] = "black"
    mpl.rcParams["axes.titlesize"] = TITLE_FONT
    mpl.rcParams["axes.labelsize"] = LABEL_FONT
    mpl.rcParams["font.size"] = BASE_FONT

    mpl.rcParams["xtick.direction"] = "in"
    mpl.rcParams["ytick.direction"] = "in"
    mpl.rcParams["xtick.major.size"] = 4
    mpl.rcParams["ytick.major.size"] = 4
    mpl.rcParams["xtick.minor.size"] = 2
    mpl.rcParams["ytick.minor.size"] = 2
    mpl.rcParams["xtick.labelsize"] = TICK_FONT
    mpl.rcParams["ytick.labelsize"] = TICK_FONT

    # ---- 网格 ----
    mpl.rcParams["axes.grid"] = True
    mpl.rcParams["grid.linestyle"] = "--"
    mpl.rcParams["grid.linewidth"] = 0.5
    mpl.rcParams["grid.alpha"] = 0.35
    mpl.rcParams["axes.grid.which"] = "major"

    # ---- 颜色 ----
    mpl.rcParams["axes.prop_cycle"] = mpl.cycler(color=[
        "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#56B4E9", "#E69F00", "#F0E442", "#000000"
    ])

    # ---- 线与标记 ----
    mpl.rcParams["lines.linewidth"] = 2.0
    mpl.rcParams["lines.markersize"] = 5

    # ---- 图例 ----
    mpl.rcParams["legend.frameon"] = False
    mpl.rcParams["legend.fontsize"] = LEGEND_FONT

    # 把字号变量挂到模块级备用（给后面函数用）
    globals().update(dict(
        BASE_FONT=BASE_FONT, TITLE_FONT=TITLE_FONT, LABEL_FONT=LABEL_FONT,
        TICK_FONT=TICK_FONT, LEGEND_FONT=LEGEND_FONT
    ))


set_nature_energy_style()

# ========== 调试输出开关 ==========
VERBOSE = True
def vprint(*args, **kwargs):
    if VERBOSE:
        print(*args, **kwargs)


# =========================
# 工具函数：单图绘制（中文）
# =========================
def annotate_best_point(ax, xs, ys, mode="max"):
    """在曲线上标注最佳点：mode='max' 取最大值；'min' 取最小值；'knee' 自动找肘点。"""
    xs = np.asarray(xs); ys = np.asarray(ys, dtype=float)

    if mode == "knee":
        # 简单肘点：用离直线(首末点)的最大距离法（Ramer–Douglas–Peucker思路）
        x0, y0 = xs[0], ys[0]
        x1, y1 = xs[-1], ys[-1]
        # 线段到点距离
        A = y1 - y0
        B = x0 - x1
        C = x1*y0 - x0*y1
        denom = np.sqrt(A*A + B*B) + 1e-12
        dists = np.abs(A*xs + B*ys + C) / denom
        idx = int(np.nanargmax(dists))
    elif mode == "min":
        idx = int(np.nanargmin(ys))
    else:
        idx = int(np.nanargmax(ys))

    ax.scatter([xs[idx]], [ys[idx]], zorder=5)
    ax.annotate(f"K={xs[idx]}",
                (xs[idx], ys[idx]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=14)

def plot_metric_line(ks, ys, title_cn, ylabel_cn, save_base, best="max"):
    fig, ax = plt.subplots()
    ax.plot(ks, ys, marker="o", label=title_cn)
    ax.set_title(title_cn)
    ax.set_xlabel("簇数量 K")
    ax.set_ylabel(ylabel_cn)

    # 开启次刻度（更细节但不拥挤）
    ax.minorticks_on()

    # 仅 Y 轴网格（主刻度），X 网格关闭
    ax.grid(True, axis="y")
    ax.grid(False, axis="x")

    # 标注一个“最佳点”（轮廓系数/CH 取 max；DB 取 min；Inertia 取 knee）
    annotate_best_point(ax, ks, ys, mode=best)

    # 保存 PNG & PDF
    plt.tight_layout()
    plt.savefig(save_base + ".png")
    plt.savefig(save_base + ".pdf")
    plt.close(fig)

# =========================
# 单个 DataFrame：计算指标 & 绘图（中文）
# =========================
def make_k_diagnostics(df, product, scenario, tag=""):
    # 1) 过滤：电解槽全零的行剔除
    ae_col  = COLMAP[product]["ae"]
    pem_col = COLMAP[product]["pem"]
    ae = df[ae_col]  if ae_col  in df.columns else 0
    pm = df[pem_col] if pem_col in df.columns else 0

    n0 = len(df)
    df = df[(ae > 0) | (pm > 0)].reset_index(drop=True)
    n1 = len(df)
    vprint(f"  [数据筛选] 行业务=电解槽>0 过滤：{n0} -> {n1} 行")

    if df.empty:
        print(f"  ⚠ {scenario}/{product} 过滤后为空，跳过。")
        return

    # 2) 构造“容量占比”特征（匹配情况打印）
    cap_candidates = ["wind", "pv", "bat_en", "bat_st", "ae", "pem", "hp", "dac"]
    matched = {}
    missing = []
    for k in cap_candidates:
        colname = COLMAP[product].get(k)
        if colname in df.columns:
            matched[k] = colname
        else:
            missing.append(k)

    # 打印匹配详情
    vprint("  [列匹配] 产品=", product, "，情景=", scenario)
    if matched:
        vprint("    ✓ 匹配成功：")
        for k, c in matched.items():
            vprint(f"      - {k:<7s} -> {c}")
    if missing:
        vprint("    ✗ 未找到：", ", ".join(missing))

    cap_cols = list(matched.values())
    if len(cap_cols) < 2:
        print(f"  ⚠ {scenario}/{product} 有效容量列不足2个：{cap_cols}，跳过。")
        return

    caps = df.loc[:, cap_cols]

    # 额外检查：总容量为0的行
    totals = caps.sum(axis=1)
    zero_total = int((totals <= 0).sum())
    if zero_total > 0:
        vprint(f"  [告警] 总容量为0的行数：{zero_total}（占比 {zero_total/len(df):.2%}）。这些行在占比中将为0。")

    # 3) 计算占比并打印头部预览
    ratios = caps.div(totals.replace(0, np.nan), axis=0).fillna(0.0)
    vprint("  [占比列] ->", list(ratios.columns))
    with pd.option_context("display.width", 160, "display.max_columns", 50, "display.precision", 4):
        vprint("  [占比示例] 前5行：\n", ratios.head(5))

    n = ratios.shape[0]
    if n < 5:
        print(f"  ⚠ {scenario}/{product} 样本数过少（n={n}），跳过。")
        return

    X = StandardScaler().fit_transform(ratios.values)

    # 4) K 范围打印
    K_MAX = max(3, min(12, n - 1))
    K_RANGE = list(range(2, K_MAX + 1))
    vprint(f"  [K范围] 2..{K_MAX} -> {K_RANGE}  （样本数 n={n}）")

    # 5) 计算四类指标（原逻辑不变）
    rows = []
    for k in K_RANGE:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(X)

        inertia = km.inertia_

        try:
            sil = silhouette_score(X, labels) if len(np.unique(labels)) > 1 and k < n else np.nan
        except Exception:
            sil = np.nan

        try:
            ch = calinski_harabasz_score(X, labels) if len(np.unique(labels)) > 1 else np.nan
        except Exception:
            ch = np.nan

        try:
            db = davies_bouldin_score(X, labels) if len(np.unique(labels)) > 1 else np.nan
        except Exception:
            db = np.nan

        rows.append({
            "K": k,
            "惯性(Inertia)": inertia,
            "轮廓系数(Silhouette)": sil,
            "CH 指数(Calinski–Harabasz)": ch,
            "DB 指数(Davies–Bouldin)": db
        })

    metrics_df = pd.DataFrame(rows)


    # 5) 保存指标表
    base = f"{scenario}_{product}{('_' + tag) if tag else ''}"
    metrics_csv = os.path.join(out_dir, f"聚类质量指标_{base}.csv")
    metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")

    # 6) 出图（中文标题/坐标）
    ks = metrics_df["K"].to_numpy()
    inertia = metrics_df["惯性(Inertia)"].to_numpy(dtype=float)
    sil     = metrics_df["轮廓系数(Silhouette)"].to_numpy(dtype=float)
    ch      = metrics_df["CH 指数(Calinski–Harabasz)"].to_numpy(dtype=float)
    db      = metrics_df["DB 指数(Davies–Bouldin)"].to_numpy(dtype=float)

    plot_metric_line(ks, inertia,
                     title_cn=f"",
                     ylabel_cn="惯性（SSE，越低越好）",
                     save_base=os.path.join(out_dir, f"肘部法_惯性_{base}"),
                     best="knee")

    plot_metric_line(ks, sil,
                     title_cn=f"",
                     ylabel_cn="轮廓系数",
                     save_base=os.path.join(out_dir, f"轮廓系数_{base}"),
                     best="max")

    plot_metric_line(ks, ch,
                     title_cn=f"",
                     ylabel_cn="CH 指数（越高越好）",
                     save_base=os.path.join(out_dir, f"CH指数_{base}"),
                     best="max")

    plot_metric_line(ks, db,
                     title_cn=f"",
                     ylabel_cn="DB 指数（越低越好）",
                     save_base=os.path.join(out_dir, f"DB指数_{base}"),
                     best="min")

    print(f"  ✔ 输出完成（Nature Energy 风格）：{base}")
    print(f"    - 指标表：{metrics_csv}")
    print(f"    - 图片：肘部法_惯性_{base}.png/.pdf, 轮廓系数_{base}.png/.pdf, CH指数_{base}.png/.pdf, DB指数_{base}.png/.pdf")

# =========================
# 主流程：批量扫描并绘图
# =========================
def main():
    found = 0
    for fn in sorted(os.listdir(input_dir)):
        m = fname_re.match(fn)
        if not m:
            continue
        scenario = m.group("scenario")
        product  = m.group("prod").lower()
        fpath    = os.path.join(input_dir, fn)

        print(f"\n=== 正在处理：{fn} ｜ 情景={scenario} ｜ 产品={product} ===")
        try:
            df = pd.read_csv(fpath)
        except Exception as e:
            print(f"  ⚠ 读取失败，跳过：{e}")
            continue

        make_k_diagnostics(df, product, scenario)
        found += 1

    if found == 0:
        print("⚠ 未找到匹配的 combined_*_(ammonia|methanol|ftl)_results.csv 文件。")

if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ----------------------------
# 并行线程限制（可选）
# ----------------------------
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"]       = "1"
os.environ["NUMEXPR_NUM_THREADS"]   = "1"
os.environ["OMP_NUM_THREADS"]       = "1"

# ----------------------------
# 路径与输出
# ----------------------------
input_dir  = r"D:\OneDrive\POTSED\DataOutput\Used_output"
out_dir    = r"D:\OneDrive\POTSED\03 Cluster"
os.makedirs(out_dir, exist_ok=True)

# ----------------------------
# 文件名解析：combined_<scenario>_<prod>_results.csv
# ----------------------------
fname_re = re.compile(
    r"^combined_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)_results\.csv$",
    flags=re.IGNORECASE
)

# ----------------------------
# 大洲映射（region code → continent）
# ----------------------------
mapping_dict = {
    1:"Europe",19:"Europe",21:"Europe",22:"Europe",23:"Europe",
    7:"Asia",20:"Asia",24:"Asia",26:"Asia",27:"Asia",28:"Asia",29:"Asia",30:"Asia",31:"Asia",32:"Asia",
    25:"Africa&Middle East",
    10:"Africa&Middle East",11:"Africa&Middle East",12:"Africa&Middle East",13:"Africa&Middle East",14:"Africa&Middle East",
    2:"North America",3:"North America",5:"North America",
    6:"South America",15:"South America",16:"South America",17:"South America",18:"South America",
    8:"Oceania"
}
# 只保留这些洲
continents_keep = ["Asia", "Europe", "Africa&Middle East", "North America", "South America", "Oceania"]

# ----------------------------
# 各产品列名映射（缺失会自动忽略）
# ----------------------------
COLMAP = {
    "methanol": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "hp": "heat_pump_capacity_kW_th",   # 可选
        "dac": "dac_capacity_kgCO2_per_h",  # 可选
        "cost": "levelized_methanol_cost_per_ton"
    },
    "ftl": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "hp": "heat_pump_capacity_kW_th",   # 可选
        "dac": "dac_capacity_kgCO2_per_h",  # 可选
        "cost": "levelized_FTL_cost_per_ton"
    },
    "ammonia": {
        "wind": "wind_capacity",
        "pv": "pv_capacity",
        "bat_en": "battery_energy_capacity",
        "bat_st": "battery_storage_capacity",
        "ae": "electrolyzer_capacity_AE",
        "pem": "electrolyzer_capacity_PEM",
        "hp": "heat_pump_capacity_kW_th",
        "dac": "dac_capacity_kgCO2_per_h",
        "cost": "levelized_ammonia_cost_per_ton"
    }
}

# ----------------------------
# 聚类参数
# ----------------------------
scaler = StandardScaler()
N_CLUSTERS_DEFAULT = 3

# ----------------------------
# 批处理
# ----------------------------
scenario_rows = {}     # dict: scenario -> list(rows)
processed_files = 0

for fn in sorted(os.listdir(input_dir)):
    m = fname_re.match(fn)
    if not m:
        continue

    scenario = m.group("scenario")
    prod     = m.group("prod").lower()
    fpath    = os.path.join(input_dir, fn)

    print(f"\n=== Processing: {fn} | scenario={scenario} | product={prod} ===")

    # 读取
    df = pd.read_csv(fpath)

    # 映射洲并保留主要大洲
    df["continent"] = df["region"].map(mapping_dict).fillna("Other")
    df = df[df["continent"].isin(continents_keep)].reset_index(drop=True)

    # ----------------------------
    # 新增：将纬度为负数的Europe点归类为South America
    # ----------------------------
    if 'latitude' in df.columns:
        europe_negative_lat_mask = (df["continent"] == "Europe") & (df["latitude"] < 0)
        df.loc[europe_negative_lat_mask, "continent"] = "South America"
        n_reassigned = europe_negative_lat_mask.sum()
        if n_reassigned > 0:
            print(f"  ✔ 将 {n_reassigned} 个负纬度的Europe点重新归类为South America")
    else:
        print("  ⚠ 未找到latitude列，跳过负纬度Europe点的重新归类")

    # 剔除电解槽全零样本（兼容不同列名）
    ae_col  = COLMAP[prod]["ae"]
    pem_col = COLMAP[prod]["pem"]
    ae = df[ae_col]  if ae_col  in df.columns else 0
    pm = df[pem_col] if pem_col in df.columns else 0
    df = df[(ae > 0) | (pm > 0)].reset_index(drop=True)
    if df.empty:
        print("  ⚠ 过滤后数据为空，跳过。")
        continue

    # 可用的"容量列"（存在即用）
    cap_candidates = ["wind", "pv", "bat_en", "bat_st", "ae", "pem", "hp", "dac"]
    cap_cols = [COLMAP[prod][k] for k in cap_candidates if COLMAP[prod][k] in df.columns]
    if len(cap_cols) < 2:
        print(f"  ⚠ 有效容量列不足2个：{cap_cols}，跳过。")
        continue

    # 初始化标签
    df["cluster_label"] = -1

    # 分洲聚类并按风占比重排标签
    for cont in continents_keep:
        mask = df["continent"] == cont
        if not mask.any():
            continue

        caps = df.loc[mask, cap_cols]
        # 行归一化（各行容量占比）
        ratios = caps.div(caps.sum(axis=1), axis=0).fillna(0)

        n_samples = len(ratios)
        n_clusters = min(N_CLUSTERS_DEFAULT, n_samples)
        if n_clusters < 1:
            continue
        if n_clusters == 1:
            labels = np.zeros(n_samples, dtype=int)
        else:
            X = scaler.fit_transform(ratios)
            km = KMeans(n_clusters=n_clusters, n_init="auto", random_state=42)
            labels = km.fit_predict(X)

        # 风/风+光占比排序标签
        wind_name = COLMAP[prod]["wind"]
        pv_name   = COLMAP[prod]["pv"]
        sub = df.loc[mask, [wind_name, pv_name]].copy()
        sub = sub.assign(orig_label=labels)

        if wind_name in sub and pv_name in sub:
            wind_sum = sub.groupby("orig_label")[wind_name].sum()
            pv_sum   = sub.groupby("orig_label")[pv_name].sum()
            denom    = wind_sum + pv_sum
            wind_share = (wind_sum / denom.replace(0, np.nan)).fillna(0.0)
        else:
            wind_share = sub.groupby("orig_label")[wind_name].sum()
            if wind_share.max() > 0:
                wind_share = wind_share / wind_share.max()

        sorted_labels = wind_share.sort_values(ascending=False).index.tolist()
        remap = {old: new for new, old in enumerate(sorted_labels)}
        new_labels = [remap[l] for l in labels]
        df.loc[mask, "cluster_label"] = new_labels

    # 保存聚类结果（含场景、产品）
    out_csv = os.path.join(out_dir, f"clustered_by_continent_{scenario}_{prod}.csv")
    df_out = df.copy()
    df_out.insert(0, "scenario", scenario)
    df_out.insert(1, "product", prod)
    df_out.to_csv(out_csv, index=False)
    print(f"  ✔ 聚类完成：{out_csv}")

    # 计算各洲各簇"成本中位数点"，按情景分别收集
    cost_col = COLMAP[prod]["cost"]
    if cost_col not in df.columns:
        print(f"  ⚠ 未找到成本列 {cost_col}，跳过该文件的中位点汇总。")
    else:
        for cont in continents_keep:
            sub = df[(df["continent"] == cont) & (df["cluster_label"] >= 0)]
            if sub.empty:
                continue
            for cl in sorted(sub["cluster_label"].unique()):
                ss = sub[sub["cluster_label"] == cl]
                if ss.empty:
                    continue
                med = ss[cost_col].median()
                idx = (ss[cost_col] - med).abs().idxmin()
                row = df.loc[idx].copy().to_dict()
                row.update({"scenario": scenario, "product": prod})
                scenario_rows.setdefault(scenario, []).append(row)

    processed_files += 1

# 按情景分别写出中位点汇总
for scen, rows in scenario_rows.items():
    median_df = pd.DataFrame(rows)
    median_path = os.path.join(out_dir, f"median_points_summary_{scen}.csv")
    median_df.to_csv(median_path, index=False)
    print(f"✔ 情景 {scen} 的中位点已保存：{median_path}")

print(f"\n✅ 批处理完成，处理文件数：{processed_files}")

## 全球统一聚类

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ----------------------------
# 并行线程限制（可选）
# ----------------------------
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"]       = "1"
os.environ["NUMEXPR_NUM_THREADS"]   = "1"
os.environ["OMP_NUM_THREADS"]       = "1"

# ----------------------------
# 路径与输出
# ----------------------------
input_dir  = r"D:\OneDrive\POTSED\DataOutput\Used_output"
out_dir    = r"D:\OneDrive\POTSED\03 Cluster"
os.makedirs(out_dir, exist_ok=True)

# ----------------------------
# 文件名解析：combined_<scenario>_<prod>_results.csv
# ----------------------------
fname_re = re.compile(
    r"^combined_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)_results\.csv$",
    flags=re.IGNORECASE
)

# ----------------------------
# 大洲映射（region code → continent）
# ----------------------------
mapping_dict = {
    1:"Europe",19:"Europe",21:"Europe",22:"Europe",23:"Europe",
    7:"Asia",20:"Asia",24:"Asia",26:"Asia",27:"Asia",28:"Asia",29:"Asia",30:"Asia",31:"Asia",32:"Asia",
    25:"Africa&Middle East",
    10:"Africa&Middle East",11:"Africa&Middle East",12:"Africa&Middle East",13:"Africa&Middle East",14:"Africa&Middle East",
    2:"North America",3:"North America",5:"North America",
    6:"South America",15:"South America",16:"South America",17:"South America",18:"South America",
    8:"Oceania"
}
# 只保留这些洲（用于过滤，但不再分洲聚类）
continents_keep = ["Asia", "Europe", "Africa&Middle East", "North America", "South America", "Oceania"]

# ----------------------------
# 各产品列名映射（缺失会自动忽略）
# ----------------------------
COLMAP = {
    "methanol": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "hp": "heat_pump_capacity_kW_th",   # 可选
        "dac": "dac_capacity_kgCO2_per_h",  # 可选
        "cost": "levelized_methanol_cost_per_ton"
    },
    "ftl": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "hp": "heat_pump_capacity_kW_th",   # 可选
        "dac": "dac_capacity_kgCO2_per_h",  # 可选
        "cost": "levelized_FTL_cost_per_ton"
    },
    "ammonia": {
        "wind": "wind_capacity",
        "pv": "pv_capacity",
        "bat_en": "battery_energy_capacity",
        "bat_st": "battery_storage_capacity",
        "ae": "electrolyzer_capacity_AE",
        "pem": "electrolyzer_capacity_PEM",
        "hp": "heat_pump_capacity_kW_th",
        "dac": "dac_capacity_kgCO2_per_h",
        "cost": "levelized_ammonia_cost_per_ton"
    }
}

# ----------------------------
# 聚类参数
# ----------------------------
scaler = StandardScaler()
N_CLUSTERS_DEFAULT = 3

# ----------------------------
# 批处理
# ----------------------------
scenario_rows = {}     # dict: scenario -> list(rows) （全球簇的中位点代表）
processed_files = 0

for fn in sorted(os.listdir(input_dir)):
    m = fname_re.match(fn)
    if not m:
        continue

    scenario = m.group("scenario")
    prod     = m.group("prod").lower()
    fpath    = os.path.join(input_dir, fn)

    print(f"\n=== Processing: {fn} | scenario={scenario} | product={prod} ===")

    # 读取
    df = pd.read_csv(fpath)

    # 映射洲并保留主要大洲（仅用于数据清洗/过滤）
    df["continent"] = df["region"].map(mapping_dict).fillna("Other")
    df = df[df["continent"].isin(continents_keep)].reset_index(drop=True)

    # 新增：将纬度为负数的Europe点归类为South America（同原逻辑，影响过滤而非聚类分组）
    if 'latitude' in df.columns:
        europe_negative_lat_mask = (df["continent"] == "Europe") & (df["latitude"] < 0)
        df.loc[europe_negative_lat_mask, "continent"] = "South America"
        n_reassigned = europe_negative_lat_mask.sum()
        if n_reassigned > 0:
            print(f"  ✔ 将 {n_reassigned} 个负纬度的Europe点重新归类为South America")
    else:
        print("  ⚠ 未找到latitude列，跳过负纬度Europe点的重新归类")

    # 剔除电解槽全零样本（兼容不同列名）
    ae_col  = COLMAP[prod]["ae"]
    pem_col = COLMAP[prod]["pem"]
    ae = df[ae_col]  if ae_col  in df.columns else 0
    pm = df[pem_col] if pem_col in df.columns else 0
    df = df[(ae > 0) | (pm > 0)].reset_index(drop=True)
    if df.empty:
        print("  ⚠ 过滤后数据为空，跳过。")
        continue

    # 可用的"容量列"（存在即用）
    cap_candidates = ["wind", "pv", "bat_en", "bat_st", "ae", "pem", "hp", "dac"]
    cap_cols = [COLMAP[prod][k] for k in cap_candidates if COLMAP[prod][k] in df.columns]
    if len(cap_cols) < 2:
        print(f"  ⚠ 有效容量列不足2个：{cap_cols}，跳过。")
        continue

    # ----------------------------
    # ❶ 全球整体聚类（不再分洲）
    # ----------------------------
    caps = df.loc[:, cap_cols]
    ratios = caps.div(caps.sum(axis=1), axis=0).fillna(0)

    n_samples = len(ratios)
    n_clusters = min(N_CLUSTERS_DEFAULT, n_samples)
    if n_clusters < 1:
        print("  ⚠ 样本数不足，跳过。")
        continue

    if n_clusters == 1:
        labels = np.zeros(n_samples, dtype=int)
    else:
        X = scaler.fit_transform(ratios)
        km = KMeans(n_clusters=n_clusters, n_init="auto", random_state=42)
        labels = km.fit_predict(X)

    # ----------------------------
    # ❷ 全局按风占比（风/(风+光)）对簇标签进行重排
    # ----------------------------
    df["cluster_label"] = labels

    wind_name = COLMAP[prod]["wind"]
    pv_name   = COLMAP[prod]["pv"]

    # 为避免缺列导致报错，先构造安全的子表
    sub_cols = ["cluster_label"]
    if wind_name in df.columns: sub_cols.append(wind_name)
    if pv_name   in df.columns: sub_cols.append(pv_name)

    sub = df.loc[:, sub_cols].copy()

    # 计算每簇风/(风+光)
    if (wind_name in sub.columns) and (pv_name in sub.columns):
        wind_sum = sub.groupby("cluster_label")[wind_name].sum()
        pv_sum   = sub.groupby("cluster_label")[pv_name].sum()
        denom    = wind_sum + pv_sum
        wind_share = (wind_sum / denom.replace(0, np.nan)).fillna(0.0)
    elif (wind_name in sub.columns):
        wind_sum = sub.groupby("cluster_label")[wind_name].sum()
        wind_share = (wind_sum / wind_sum.replace(0, np.nan)).fillna(0.0)
    else:
        # 若无风列，则保持原标签（不重排）
        wind_share = pd.Series(index=sorted(df["cluster_label"].unique()), dtype=float)
        wind_share.loc[:] = 0.0

    sorted_labels = wind_share.sort_values(ascending=False).index.tolist()
    remap = {old: new for new, old in enumerate(sorted_labels)}
    df["cluster_label"] = df["cluster_label"].map(remap).astype(int)

    # ----------------------------
    # ❸ 保存聚类结果（含场景、产品）——全球
    # ----------------------------
    out_csv = os.path.join(out_dir, f"clustered_global_{scenario}_{prod}.csv")
    df_out = df.copy()
    df_out.insert(0, "scenario", scenario)
    df_out.insert(1, "product", prod)
    df_out.to_csv(out_csv, index=False)
    print(f"  ✔ 全球聚类完成：{out_csv}")

    # ----------------------------
    # ❹ 计算“各簇（全球）成本中位数点”
    # ----------------------------
    cost_col = COLMAP[prod]["cost"]
    if cost_col not in df.columns:
        print(f"  ⚠ 未找到成本列 {cost_col}，跳过该文件的中位点汇总。")
    else:
        for cl in sorted(df["cluster_label"].unique()):
            ss = df[df["cluster_label"] == cl]
            if ss.empty:
                continue
            med = ss[cost_col].median()
            idx = (ss[cost_col] - med).abs().idxmin()
            row = df.loc[idx].copy().to_dict()
            row.update({"scenario": scenario, "product": prod})
            scenario_rows.setdefault(scenario, []).append(row)

    processed_files += 1

# ----------------------------
# ❺ 按情景分别写出“全球簇中位点”汇总
# ----------------------------
for scen, rows in scenario_rows.items():
    median_df = pd.DataFrame(rows)
    median_path = os.path.join(out_dir, f"median_points_summary_{scen}.csv")
    median_df.to_csv(median_path, index=False)
    print(f"✔ 情景 {scen} 的全球簇中位点已保存：{median_path}")

print(f"\n✅ 批处理完成，处理文件数：{processed_files}")


### 全球聚类的中位数提取

In [ ]:
# -*- coding: utf-8 -*-
"""
用途（按新规则，绝不更改簇标签）：
1) 基准情景（RegionRE_LowAE / RegionRE_LowAE_AddSteam / RegionRE_LowAE_MidFTL / RegionalRE_LowAE）：
   - 以"2040 年"为基准年（若情景名无年份则视作 2040），
   - 对每个【产品×簇】在 2040 年的该簇内按"成本中位数就近"生成候选点（经纬）列表；
   - 对候选点逐个校验：在 2030/2040/2050 三个年份的基准情景里（若存在该文件）
     其簇标签是否都等于目标簇；若是，则将该点确认为锚点；若否，继续下一个候选；
   - 若耗尽候选仍无法使"三年簇一致"，则放弃该簇的锚点，后续对该簇采用"中位数就近"。
2) 其它情景：
   - 对每个【产品×簇】：
     * 若有锚点：在该情景中查找同一经纬的行，找到则直接输出；找不到则在该情景的该簇内中位数就近选点；
     * 若无锚点：直接在该情景的该簇内中位数就近选点。
   - 绝不修改列 cluster_label。
3) 每个情景输出：median_points_summary_<scenario>.csv
"""

import os
import re
import glob
import numpy as np
import pandas as pd

# ============== 路径 ==============#
in_dir   = r"D:\OneDrive\POTSED\03 Cluster"   # 存放聚类结果的目录
out_dir  = in_dir                              # 输出目录
os.makedirs(out_dir, exist_ok=True)

# 支持两种输入命名：clustered_global_<scen>_<prod>.csv 或 full_grid_clustered_<scen>_<prod>.csv
patts = [
    os.path.join(in_dir, "clustered_global_*_*.csv"),
    os.path.join(in_dir, "full_grid_clustered_*_*.csv"),
]
fname_re = re.compile(
    r"^(?:clustered_global|full_grid_clustered)_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
    flags=re.IGNORECASE
)

# ============== 参数与常量 ==============#
# 【手动配置】每个产品在不同年份对应的基准情景名称
# 格式：{产品: {年份: [情景名称列表]}}
# 可填写多个情景名（会自动合并），也可留空 None 或 [] 表示该年份无数据
BASE_SCENARIO_MAP = {
    "ammonia": {
        2030: ["RegionRE_2030"],
        2040: ["RegionalRE_LowAE"],  # 2040年对应的情景
        2050: ["RegionRE_LowAE_2050"],  # 2050年对应的情景
    },
    "methanol": {
        2030: ["RegionRE_2030_AddSteam"],
        2040: ["RegionRE_LowAE_AddSteam"],  # 可以填多个情景名
        2050: ["RegionRE_LowAE_2050_AddSteam"],
    },
    "ftl": {
        2030: ["RegionalRE_2030"],
        2040: ["RegionRE_LowAE_MidFTL"],
        2050: ["RegionRE_LowAE_2050"],
    },
}

# 兼容旧版：四个"基准情景"关键字（用于自动识别，现已不使用）
BASE_SCENARIOS = {"RegionalRE_LowAE", "RegionRE_LowAE", "RegionRE_LowAE_AddSteam", "RegionRE_LowAE_MidFTL"}

# 成本列名（按产品）
COST_COL = {
    "ammonia":  "levelized_ammonia_cost_per_ton",
    "methanol": "levelized_methanol_cost_per_ton",
    "ftl":      "levelized_FTL_cost_per_ton",
}

# 经纬匹配小数位（避免浮点误差）
COORD_DECIMALS = 6

# 在输出中尽量携带的列（存在则保留）
PREFERRED_COLS = [
    # —— 标识类 —— 
    "scenario", "product", "continent", "region", "cluster_label",
    "latitude", "longitude", "orig_idx",

    # —— 成本（风/光）——
    "wind_cost", "pv_cost",

    # —— 装机/容量（通用无单位版本，氨用）——
    "wind_capacity", "pv_capacity",
    "battery_energy_capacity", "battery_storage_capacity",
    "electrolyzer_capacity_AE", "electrolyzer_capacity_PEM",

    # —— 装机/容量（带单位版本，FTL/甲醇等常见）——
    "wind_capacity_kW", "pv_capacity_kW",
    "battery_energy_capacity_kW", "battery_storage_capacity_kWh",
    "electrolyzer_capacity_AE_kW", "electrolyzer_capacity_PEM_kW",

    # —— 热端（若有）——
    "heat_pump_capacity_kW_th", "thermal_storage_capacity_kWh_th",

    # —— 合成装置（按产品）——
    "ammonia_synthesis_capacity", "ammonia_storage_capacity",
    "FTL_synthesis_capacity_tpy", "FTL_storage_capacity_kg_per_h",
    "methanol_synthesis_capacity_tpy", "methanol_storage_capacity_kg_per_h",

    # —— 氢/CO2 —— 
    "hydrogen_storage_capacity", "hydrogen_storage_capacity_kgH2",
    "dac_capacity_kgCO2_per_h", "co2_compressor_capacity_kgCO2_per_h",
    "co2_storage_capacity_kg",

    # —— 产量/捕集量/发电量 —— 
    "total_ammonia_production_kg", "total_FTL_production_kg", "total_methanol_production_kg",
    "total_hydrogen_production_kg", "total_CO2_captured_kg",
    "total_elec_generation_kwh",

    # —— 利用率 —— 
    "electrolyzer_utilization_percent", "ammonia_utilization_percent",
    "FTL_utilization_percent", "methanol_utilization_percent",
    "dac_utilization_percent",

    # —— 平准化成本 —— 
    "levelized_elec_cost_per_kwh", "levelized_elec_cost_per_kWh",
    "levelized_heat_cost_per_kWh_th",
    "levelized_h2_cost_per_kg",
    "levelized_ammonia_cost_per_ton",
    "levelized_FTL_cost_per_ton",
    "levelized_methanol_cost_per_ton",
    "levelized_dac_cost_per_tCO2",

    # —— 目标函数/综合指标（若有）——
    "objective_value",
]

# ============== 小工具 ==============#
def latlon_key(lat, lon, nd=COORD_DECIMALS):
    """把经纬度规范化为匹配键（四舍五入到 nd 位小数）。"""
    return (round(float(lat), nd), round(float(lon), nd))

def pick_median_row(ss: pd.DataFrame, cost_col: str) -> pd.Series:
    """在子表 ss 中，用 cost_col 的中位数就近选择一行（ss 需非空）。"""
    med = ss[cost_col].median()
    idx = (ss[cost_col] - med).abs().idxmin()
    return ss.loc[idx]

def select_output_fields(row: pd.Series) -> dict:
    """只保留 PREFERRED_COLS 中存在的字段，并做基本类型清洗。"""
    keep = [c for c in PREFERRED_COLS if c in row.index]
    out = row[keep].to_dict()
    # 类型修复
    if "latitude" in out:  out["latitude"]  = float(out["latitude"])
    if "longitude" in out: out["longitude"] = float(out["longitude"])
    if "cluster_label" in out:
        try:
            out["cluster_label"] = int(out["cluster_label"])
        except Exception:
            pass
    return out

# ============== 读取数据 ==============#
# 读取所有文件 → {(scenario, product): df}
data_map = {}
scenarios_set = set()

files = sorted({f for patt in patts for f in glob.glob(patt)})
if not files:
    raise FileNotFoundError("未在目录中找到聚类结果：clustered_global_*_*.csv 或 full_grid_clustered_*_*.csv")

for fpath in files:
    fname = os.path.basename(fpath)
    m = fname_re.match(fname)
    if not m:
        print(f"⚠ 跳过无法解析文件名：{fname}")
        continue
    scenario = m.group("scenario")
    product  = m.group("prod").lower()
    df = pd.read_csv(fpath)

    # 兼容列名：cluster 或 cluster_label
    if "cluster_label" not in df.columns and "cluster" in df.columns:
        df = df.rename(columns={"cluster": "cluster_label"})

    # 必要列检查
    need = {"cluster_label", "latitude", "longitude"}
    if not need.issubset(df.columns):
        print(f"  ⚠ {fname} 缺少必要列 {need}，跳过。")
        continue

    # 成本列检查
    cc = COST_COL.get(product)
    if cc is None or cc not in df.columns:
        print(f"  ⚠ {fname}({product}) 不含成本列 {cc}，跳过。")
        continue

    # 整理类型
    df["latitude"]  = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df[cc]          = pd.to_numeric(df[cc], errors="coerce")
    df = df.dropna(subset=["latitude", "longitude"])

    # 预置经纬匹配键
    df["_ll_key"] = [latlon_key(a, b) for a, b in zip(df["latitude"], df["longitude"])]

    data_map[(scenario, product)] = df
    scenarios_set.add(scenario)

# ============== 基准情景：使用手动配置构建"跨年一致"的锚点 ==============#
# 根据 BASE_SCENARIO_MAP 加载每个产品每个年份的基准数据
base_by_prod_year = {}  # {(prod, year): pd.DataFrame concatenated}

for prod, year_map in BASE_SCENARIO_MAP.items():
    for yr, scen_list in year_map.items():
        if scen_list is None or (isinstance(scen_list, list) and len(scen_list) == 0):
            continue
        
        if isinstance(scen_list, str):
            scen_list = [scen_list]
        
        dfs_to_concat = []
        for scen in scen_list:
            df = data_map.get((scen, prod))
            if df is not None and not df.empty:
                dfs_to_concat.append(df)
        
        if dfs_to_concat:
            key = (prod, yr)
            if len(dfs_to_concat) > 1:
                base_by_prod_year[key] = pd.concat(dfs_to_concat, ignore_index=True)
            else:
                base_by_prod_year[key] = dfs_to_concat[0]
            print(f"ℹ 已加载基准数据：{prod} | {yr}年 | 情景: {scen_list}")

# 确定目标年份集合
YEARS_TARGET_FULL = [2030, 2040, 2050]

# 为了"成本相邻"排序，选 2040 作为"成本排序基准年"
def get_base_df(prod: str, year: int) -> pd.DataFrame | None:
    return base_by_prod_year.get((prod, year))

# anchors[(prod, cluster_label)] = (lat, lon)  —— 仅当找到年份一致时写入
anchors = {}

for prod in ["ammonia", "methanol", "ftl"]:
    # 2040 基准年必须存在，否则无法做"成本相邻"搜索
    df_2040 = get_base_df(prod, 2040)
    if df_2040 is None or df_2040.empty:
        print(f"⚠ 基准锚点：{prod} 缺少 2040 年基准情景数据，跳过该产品的锚点生成。")
        continue

    # 确定该产品实际可用的年份
    available_years = [yr for yr in YEARS_TARGET_FULL if get_base_df(prod, yr) is not None]
    if len(available_years) < 2:
        print(f"⚠ 基准锚点：{prod} 可用年份不足2个（{available_years}），跳过该产品的锚点生成。")
        continue
    
    # 如果三个年份都有数据，则强制要求三年一致；否则使用可用年份
    if len(available_years) == 3:
        required_years = YEARS_TARGET_FULL  # 强制使用 2030/2040/2050 三年
        years_info = "2030/2040/2050"
        print(f"ℹ {prod}：三个年份数据完整，将强制查找 {years_info} 三年一致的锚点")
    else:
        required_years = available_years
        years_info = f"{'/'.join(map(str, available_years))}"
        print(f"ℹ {prod}：将使用 {years_info} 年数据查找一致锚点（共{len(available_years)}年）")

    cc = COST_COL[prod]

    # 目标簇集合：取 2040 年出现过的簇
    target_cls = sorted(int(c) for c in pd.unique(df_2040["cluster_label"].astype(int)))
    for cl in target_cls:
        ss_2040 = df_2040[df_2040["cluster_label"] == cl]
        if ss_2040.empty:
            continue

        # 以"距离该簇成本中位数最近"为序，构造候选（2040年该簇内）
        med_2040 = ss_2040[cc].median()
        ss_2040 = ss_2040.assign(_dev=(ss_2040[cc] - med_2040).abs()).sort_values("_dev")
        candidate_keys = ss_2040["_ll_key"].tolist()

        # 逐个候选，检查其在所有要求年份是否都属于簇 cl
        ok_anchor = None
        for key_ll in candidate_keys:
            all_ok = True
            for yr in required_years:
                df_y = get_base_df(prod, yr)
                if df_y is None or df_y.empty:
                    # 这里理论上不会发生（因为上面已过滤），但保留检查
                    all_ok = False
                    break
                hit = df_y[df_y["_ll_key"] == key_ll]
                if hit.empty or int(hit["cluster_label"].iloc[0]) != cl:
                    all_ok = False
                    break
            if all_ok:
                ok_anchor = key_ll
                break

        if ok_anchor is not None:
            anchors[(prod, cl)] = ok_anchor
            print(f"✔ 锚点确定：{prod} | 簇{cl} | 经纬={ok_anchor}（{years_info} 簇一致）")
        else:
            print(f"• 未找到{len(required_years)}年一致的锚点：{prod} | 簇{cl}。后续将对该簇使用'本情景本簇中位数就近'策略。")

# 记录每个产品的目标簇（来自 2040 基准年）
target_clusters_by_prod = {}
for (prod, yr), df in base_by_prod_year.items():
    if yr != 2040 or df is None or df.empty:
        continue
    for cl in pd.unique(df["cluster_label"].astype(int)):
        target_clusters_by_prod.setdefault(prod, set()).add(int(cl))

# ============== 输出各情景 ==============#
for scen in sorted(scenarios_set):
    rows_out = []

    for prod in ["ammonia", "methanol", "ftl"]:
        df = data_map.get((scen, prod))
        if df is None or df.empty:
            continue

        cc = COST_COL[prod]
        # 本情景目标簇：沿用 2040 基准年的簇集合（若没有，跳过该产品）
        target_cls = sorted(target_clusters_by_prod.get(prod, set()))
        if not target_cls:
            continue

        # 确保经纬匹配键存在（上游已建）
        if "_ll_key" not in df.columns:
            df = df.copy()
            df["_ll_key"] = [latlon_key(a, b) for a, b in zip(df["latitude"], df["longitude"])]

        for cl in target_cls:
            # 优先用跨年一致的"锚点经纬"
            anchor_ll = anchors.get((prod, cl))
            if anchor_ll is not None:
                hit = df[df["_ll_key"] == anchor_ll]
                if not hit.empty:
                    # 不改簇标签，直接输出该行（其自身 cluster_label 若与 cl 不同，也保持原样）
                    row = hit.iloc[0].copy()
                    row["scenario"] = scen
                    row["product"]  = prod
                    rows_out.append(select_output_fields(row))
                    continue
                else:
                    print(f"• 情景缺锚点：{scen} | {prod} | 簇{cl} | 锚点{anchor_ll} 不存在于该情景，转为簇内中位数就近。")

            # 无锚点或该情景未命中锚点：在"本情景该簇"内用中位数就近
            ss_cl = df[df["cluster_label"] == cl]
            if not ss_cl.empty:
                row = pick_median_row(ss_cl, cc).copy()
                row["scenario"] = scen
                row["product"]  = prod
                rows_out.append(select_output_fields(row))
            else:
                # 该情景里压根没有这个簇——按"全表中位数就近"兜底（仍不更改其簇）
                row = pick_median_row(df, cc).copy()
                row["scenario"] = scen
                row["product"]  = prod
                print(f"• 情景无该簇：{scen} | {prod} | 簇{cl} 缺失，改用全表中位数就近（保持原簇标签）。")
                rows_out.append(select_output_fields(row))

    if not rows_out:
        print(f"⚠ 情景 {scen} 无可输出记录，跳过写出。")
        continue

    median_df = pd.DataFrame(rows_out).reset_index(drop=True)
    # 防御去重：同一 情景×产品×簇 可能重复（例如没有锚点时的兜底）；此处按 情景×产品×经纬 去重更稳妥
    dedup_cols = [c for c in ["scenario", "product", "latitude", "longitude"] if c in median_df.columns]
    if dedup_cols:
        median_df = median_df.drop_duplicates(subset=dedup_cols)

    out_csv = os.path.join(out_dir, f"median_points_summary_{scen}.csv")
    median_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"✔ 已保存：{out_csv}")

print("\n✅ 完成（新规则）：使用手动配置的情景映射；不修改簇标签；锚点需在可用年份中簇一致；否则该簇按中位数就近。")


## 2. 中位数代表点时序数据提取

### 代表点选择
从聚类结果中提取各大洲各聚类的甲醇成本中位数点作为代表点。

### 容量因子时序提取
- 读取分段的NetCDF格式风光容量因子数据
- 根据代表点经纬度坐标提取对应的时间序列
- 保存为包含小时级风光容量因子的CSV文件

### 数据输出
生成包含以下信息的时序数据：
- 大洲、区域、聚类标签
- 经纬度坐标
- 小时级风电和光伏容量因子

In [ ]:
# -*- coding: utf-8 -*-
import os
import glob
import pandas as pd
import numpy as np
import xarray as xr

# ============== 路径 ==============
median_dir = r"D:\OneDrive\POTSED\03 Cluster"     # 上一步的输出目录
nc_dir     = r"D:\Workspace"                   # 存放 average_2023_segment_*.nc 的目录
out_dir    = median_dir                        # 输出也放在同目录（可改）
os.makedirs(out_dir, exist_ok=True)

# ============== 工具函数 ==============
def find_lat_lon_names(ds):
    """自动识别数据集中的纬度/经度坐标名字"""
    lat_name = "latitude" if "latitude" in ds.coords else ("lat" if "lat" in ds.coords else None)
    lon_name = "longitude" if "longitude" in ds.coords else ("lon" if "lon" in ds.coords else None)
    if lat_name is None or lon_name is None:
        raise KeyError("无法在 NetCDF 中找到经纬度坐标（latitude/longitude 或 lat/lon）")
    return lat_name, lon_name

def wrap_longitude(lon_vals, target_min, target_max):
    """
    将经度数组 lon_vals 映射到 [target_min, target_max] 区间。
    例如：目标是 0..360，则把 <0 的加 360；目标是 -180..180，则把 >180 的减 360。
    """
    lon_vals = np.asarray(lon_vals, dtype=float)
    if target_min >= 0:  # 目标 0..360
        lon_vals = np.where(lon_vals < 0, lon_vals + 360.0, lon_vals)
    else:  # 目标 -180..180
        lon_vals = np.where(lon_vals > 180.0, lon_vals - 360.0, lon_vals)
    return lon_vals

def load_median_points(median_dir):
    """读取目录下所有 median_points_summary_<scenario>.csv，返回 {scenario: DataFrame}"""
    scen_map = {}
    for fp in glob.glob(os.path.join(median_dir, "median_points_summary_*.csv")):
        scen = os.path.splitext(os.path.basename(fp))[0].replace("median_points_summary_", "")
        df = pd.read_csv(fp)
        # 只保留必须字段，若列名不在也容错
        keep_cols = [c for c in [
            "scenario", "product", "continent", "region", "cluster_label",
            "latitude", "longitude"
        ] if c in df.columns]
        if "scenario" not in keep_cols:
            df["scenario"] = scen
            keep_cols = ["scenario"] + keep_cols
        scen_map[scen] = df[keep_cols].dropna(subset=["latitude", "longitude"]).copy()
    return scen_map

# ============== 主流程 ==============
# 读取所有情景的中位点
scenario_points = load_median_points(median_dir)
if not scenario_points:
    raise FileNotFoundError(f"在 {median_dir} 未找到 median_points_summary_*.csv")

# 预先枚举 NetCDF 分段文件（每 60 个经度格点）
segment_files = []
for i in range(0, 1440, 60):
    fp = os.path.join(nc_dir, f"average_2023_segment_{i}.nc")
    if os.path.exists(fp):
        segment_files.append(fp)
if not segment_files:
    raise FileNotFoundError(f"在 {nc_dir} 未找到 average_2023_segment_*.nc")

# 逐情景处理
for scen, med_df in scenario_points.items():
    if med_df.empty:
        print(f"⚠ 情景 {scen} 中位点为空，跳过。")
        continue

    records = []

    # 缓存：避免重复在所有文件里查找相同点
    # 这里通过逐段过滤“经度落在该段范围内”的点再查询
    for seg_fp in segment_files:
        with xr.open_dataset(seg_fp, engine="netcdf4") as ds:
            lat_name, lon_name = find_lat_lon_names(ds)
            # 变量名容错
            var_w = "CF_wind_avg" if "CF_wind_avg" in ds else ("CF_wind" if "CF_wind" in ds else None)
            var_p = "CF_pv_avg"   if "CF_pv_avg"   in ds else ("CF_pv"   if "CF_pv"   in ds else None)
            if var_w is None or var_p is None:
                raise KeyError(f"{seg_fp} 缺少 CF 变量（CF_wind_avg/CF_pv_avg）")

            # 该分段的经度范围（注意 0..360 或 -180..180）
            lon_vals = ds[lon_name].values
            lon_min, lon_max = float(np.min(lon_vals)), float(np.max(lon_vals))

            # 将中位点经度映射到该数据集坐标系
            lon_mapped = wrap_longitude(med_df["longitude"].values, lon_min, lon_max)
            lat_vals = med_df["latitude"].values

            # 选出经度在该段范围内的点（包含端点）
            in_seg = (lon_mapped >= lon_min) & (lon_mapped <= lon_max)
            if not np.any(in_seg):
                continue

            # 只处理该段的点
            sub = med_df.loc[in_seg].copy()
            sub["_lon_mapped"] = lon_mapped[in_seg]

            # 逐点提取时序
            for _, row in sub.iterrows():
                lat = float(row["latitude"])
                lon = float(row["_lon_mapped"])  # 映射后的经度

                # 最近邻抽取时间序列
                cf_w = ds[var_w].sel({lat_name: lat, lon_name: lon}, method="nearest").values
                cf_p = ds[var_p].sel({lat_name: lat, lon_name: lon}, method="nearest").values

                # 优先使用 time 轴，否则 hour 序号
                if "time" in ds.coords:
                    # 将时间坐标转换为字符串，避免时区序列化问题
                    tvals = pd.to_datetime(ds["time"].values)
                    hours = list(range(len(tvals)))
                    times_out = tvals.astype("datetime64[ns]").astype(str).tolist()
                    use_time = True
                else:
                    hours = list(range(len(cf_w)))
                    times_out = None
                    use_time = False

                # 记录
                for idx, (w, p) in enumerate(zip(cf_w, cf_p)):
                    rec = {
                        "scenario": row.get("scenario", scen),
                        "product": row.get("product", None),
                        "continent": row.get("continent", None),
                        "region": row.get("region", None),
                        "cluster": row.get("cluster_label", None),
                        "latitude": float(row["latitude"]),
                        # 输出经度回到原始（未映射）值，便于和表格对应
                        "longitude": float(row["longitude"]),
                        "hour": hours[idx],
                        "CF_wind_avg": float(w),
                        "CF_pv_avg": float(p),
                    }
                    if use_time:
                        rec["time"] = times_out[idx]
                    records.append(rec)

    # 汇总并输出
    if records:
        cf_df = pd.DataFrame.from_records(records)
        # 为避免重复（不同分段边界点都可能被命中），去重一遍
        dedup_cols = ["scenario", "product", "continent", "region", "cluster",
                      "latitude", "longitude", "hour"]
        if "time" in cf_df.columns:
            dedup_cols = dedup_cols + ["time"]
        cf_df = cf_df.drop_duplicates(subset=dedup_cols)

        out_cf = os.path.join(out_dir, f"median_points_cf_timeseries_{scen}.csv")
        cf_df.to_csv(out_cf, index=False)
        print(f"✔ 情景 {scen} 的 CF_Wind/CF_PV 时序已保存：{out_cf}")
    else:
        print(f"⚠ 情景 {scen} 未抽取到任何 CF 时序。")

print("✅ 全部完成。")


## 3. 优化模型定义

### 甲醇-DAC集成系统优化模型

本节定义了一个完整的甲醇生产与直接空气捕集(DAC)集成系统的优化模型，主要特点包括：

#### 系统组件
- **可再生能源**: 风电、光伏发电
- **储能系统**: 电池储能、氢气储能、甲醇储存、热储能
- **电解制氢**: AE(碱性)和PEM(质子交换膜)电解槽
- **甲醇合成**: 氢气与CO₂合成甲醇的反应器
- **DAC系统**: 直接空气捕集CO₂设备
- **热泵系统**: 为DAC提供所需热能
- **CO₂处理**: 压缩与储存系统

#### 优化目标
最小化系统总成本，包括设备投资的年化成本和运维成本。

#### 约束条件
- 能量平衡约束
- 容量配置约束  
- 储能运行约束
- 甲醇生产目标约束

## 4. 约束系统设计

### 4.1 灵活性约束
为确保系统运行的稳定性和可操作性，模型对关键设备的功率变化率进行了限制：

- **电解槽功率爬升约束**: 限制AE电解槽小时间功率变化幅度，防止过于频繁的启停
- **甲醇合成反应器约束**: 控制甲醇合成功率的变化速率，保持化学反应的稳定性
- **DAC系统约束**: 限制直接空气捕集系统的功率波动，确保CO₂捕集效率

### 4.2 甲醇供需平衡约束
支持多种时间聚合模式，以适应不同的应用场景：

- **小时级平衡**: 每小时精确满足甲醇需求
- **3小时级平衡**: 在3小时窗口内保持供需平衡
- **日级平衡**: 允许日内波动，但整日需求必须满足80%以上
- **周级平衡**: 在周尺度上保持长期供需平衡

这种多层次的时间聚合设计能够在保证供应安全的前提下，为系统提供更大的运行灵活性。

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import Model, GRB

# =============================================================================
# 函数: 合成氨系统优化
# =============================================================================
def optimize_ammonia_system(
    wind_output,
    pv_output,
    target_ammonia_production,
    wind_cost,
    pv_cost,
    battery_energy_cost,
    battery_storage_cost,
    battery_charge_efficiency,
    battery_discharge_efficiency,
    battery_charge_penalty,
    battery_discharge_penalty,
    electrolyzer_cost_AE,
    electrolyzer_cost_PEM,
    electrolyzer_eff_AE,
    electrolyzer_eff_PEM,
    ammonia_synthesis_cost,
    hydrogen_storage_cost,
    hydrogen_storage_elec,
    hydrogen_charge_penalty,
    hydrogen_storage_efficiency,
    ammonia_storage_cost_param,
    discount_rate,
    battery_lifetime,
    wind_lifetime,
    pv_lifetime,
    electrolyzer_lifetime_AE,
    electrolyzer_lifetime_PEM,
    ammonia_synthesis_lifetime,
    hydrogen_storage_lifetime,
    ammonia_storage_lifetime,
    Flex_up,
    Flex_mid,
    Flex_down,
    surplus_penalty,
    aggregation_mode
):
    # —————— 年化成本与运维 ——————
    wind_om                = wind_cost * 0.03
    pv_om                  = pv_cost * 0.03
    battery_energy_om      = battery_energy_cost * 0.03
    battery_storage_om     = battery_storage_cost * 0.03
    electrolyzer_om_AE     = electrolyzer_cost_AE * 0.03
    electrolyzer_om_PEM    = electrolyzer_cost_PEM * 0.03
    ammonia_synthesis_om   = ammonia_synthesis_cost * 0.03
    hydrogen_storage_om    = hydrogen_storage_cost * 0.03
    ammonia_storage_om     = ammonia_storage_cost_param * 0.03

    wind_annual_cost           = wind_cost * discount_rate / (1 - (1+discount_rate)**-wind_lifetime)     + wind_om
    pv_annual_cost             = pv_cost   * discount_rate / (1 - (1+discount_rate)**-pv_lifetime)       + pv_om
    battery_energy_annual_cost = battery_energy_cost  * discount_rate / (1 - (1+discount_rate)**-battery_lifetime) + battery_energy_om
    battery_storage_annual_cost= battery_storage_cost * discount_rate / (1 - (1+discount_rate)**-battery_lifetime) + battery_storage_om
    electrolyzer_annual_cost_AE= electrolyzer_cost_AE  * discount_rate / (1 - (1+discount_rate)**-electrolyzer_lifetime_AE) + electrolyzer_om_AE
    electrolyzer_annual_cost_PEM=electrolyzer_cost_PEM * discount_rate / (1 - (1+discount_rate)**-electrolyzer_lifetime_PEM) + electrolyzer_om_PEM
    ammonia_synthesis_annual_cost=ammonia_synthesis_cost * discount_rate / (1 - (1+discount_rate)**-ammonia_synthesis_lifetime) + ammonia_synthesis_om
    hydrogen_storage_annual_cost= hydrogen_storage_cost * discount_rate / (1 - (1+discount_rate)**-hydrogen_storage_lifetime) + hydrogen_storage_om
    ammonia_storage_annual_cost = ammonia_storage_cost_param * discount_rate / (1 - (1+discount_rate)**-ammonia_storage_lifetime) + ammonia_storage_om

    # 逐小时需求
    hourly_ammonia_demand = target_ammonia_production / 8760
    ammonia_electricity_ratio = 0.303
    ammonia_hydrogen_ratio   = 0.18

    # —————— 建模 ——————
    time_steps = 8760
    model = Model("AmmoniaOptimization_SplitBattery_AEPem")
    model.setParam('OutputFlag', 0)
    model.setParam('TimeLimit', 1000)
    model.setParam('Method', 2); model.setParam('Crossover', 0)
    model.setParam('BarConvTol', 1e-6); model.setParam('Threads', 1)
    model.setParam('Presolve', 1); model.setParam('BarHomogeneous', 1)

    # 决策变量
    wind_capacity      = model.addVar(lb=0, name="wind_capacity")
    pv_capacity        = model.addVar(lb=0, name="pv_capacity")
    battery_energy_capacity= model.addVar(lb=0, name="battery_energy_capacity")
    battery_storage_capacity= model.addVar(lb=0, name="battery_storage_capacity")
    electrolyzer_capacity_AE= model.addVar(lb=0, name="electrolyzer_capacity_AE")
    electrolyzer_capacity_PEM=model.addVar(lb=0, name="electrolyzer_capacity_PEM")
    ammonia_synthesis_capacity=model.addVar(lb=0, name="ammonia_synthesis_capacity")
    hydrogen_storage_capacity =model.addVar(lb=0, name="hydrogen_storage_capacity")
    ammonia_storage_capacity  =model.addVar(lb=0, name="ammonia_storage_capacity")

    # 时步变量
    energy_balance           = model.addVars(time_steps, lb=0, name="energy_balance")
    battery_charge          = model.addVars(time_steps, lb=0, name="battery_charge")
    battery_discharge       = model.addVars(time_steps, lb=0, name="battery_discharge")
    electrolyzer_power_AE   = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
    electrolyzer_power_PEM  = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")
    ammonia_synthesis_power = model.addVars(time_steps, lb=0, name="ammonia_synthesis_power")
    surplus                 = model.addVars(time_steps, lb=0, name="surplus")
    hydrogen_storage_balance= model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
    hydrogen_charge         = model.addVars(time_steps, lb=0, name="hydrogen_charge")
    hydrogen_discharge      = model.addVars(time_steps, lb=0, name="hydrogen_discharge")
    ammonia_storage_balance = model.addVars(time_steps, lb=0, name="ammonia_storage_balance")
    ammonia_storage_charge  = model.addVars(time_steps, lb=0, name="ammonia_storage_charge")
    ammonia_storage_discharge=model.addVars(time_steps, lb=0, name="ammonia_storage_discharge")

    # 目标函数
    model.setObjective(
        wind_annual_cost * wind_capacity +
        pv_annual_cost * pv_capacity +
        battery_energy_annual_cost * battery_energy_capacity +
        battery_storage_annual_cost * battery_storage_capacity +
        electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
        electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
        ammonia_synthesis_annual_cost * ammonia_synthesis_capacity +
        hydrogen_storage_annual_cost * hydrogen_storage_capacity +
        ammonia_storage_annual_cost * ammonia_storage_capacity +
        surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
        battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
        battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
        hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)),
        GRB.MINIMIZE
    )

# =============================================================================
# 5. 约束条件
# =============================================================================
# 5.1 逐时灵活性
    for t in range(1, time_steps):
        model.addConstr(electrolyzer_power_AE[t]   - electrolyzer_power_AE[t-1]   <= Flex_down * electrolyzer_capacity_AE,
                        name=f"electrolyzer_AE_ramp_up_{t}")
        model.addConstr(electrolyzer_power_AE[t]   - electrolyzer_power_AE[t-1]   >= -Flex_down * electrolyzer_capacity_AE,
                        name=f"electrolyzer_AE_ramp_down_{t}")
        model.addConstr(ammonia_synthesis_power[t] - ammonia_synthesis_power[t-1] <= Flex_mid  * ammonia_synthesis_capacity,
                        name=f"ammonia_ramp_up_{t}")
        model.addConstr(ammonia_synthesis_power[t] - ammonia_synthesis_power[t-1] >= -Flex_down * ammonia_synthesis_capacity,
                        name=f"ammonia_ramp_down_{t}")

    # 5.2 合成氨供需平衡约束（含氨储存充放，支持不同时间聚合尺度）
    start_index = 168  # 避免初始不稳定阶段
    if aggregation_mode == "hourly":
        for t in range(start_index, time_steps):
            model.addConstr(
                ammonia_synthesis_power[t] + ammonia_storage_discharge[t] ==
                hourly_ammonia_demand + ammonia_storage_charge[t],
                name=f"ammonia_balance_{t}"
            )
    elif aggregation_mode == "3hourly":
        interval = 3
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_ammonia_demand * interval +
                gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
                name=f"ammonia_balance_3hourly_{i}"
            )
    elif aggregation_mode == "daily":
        interval = 24
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
                >=
                0.8 * hourly_ammonia_demand * interval +
                gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
                name=f"ammonia_balance_daily_{i}"
            )
    elif aggregation_mode == "weekly":
        interval = 168  # 一周168小时
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_ammonia_demand * interval +
                gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
                name=f"ammonia_balance_weekly_{i}"
            )
    elif aggregation_mode == "monthly":
        # 近似每月730小时（可根据实际情况调整）
        interval = 730
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_ammonia_demand * interval +
                gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
                name=f"ammonia_balance_monthly_{i}"
            )
    elif aggregation_mode == "annual":
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(start_index, time_steps)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(start_index, time_steps))
            ==
            hourly_ammonia_demand * (time_steps - start_index) +
            gp.quicksum(ammonia_storage_charge[t] for t in range(start_index, time_steps)),
            name="ammonia_balance_annual"
        )
    else:
        raise ValueError("Unsupported aggregation_mode value!")

    # 附加：年度合成氨生产总量约束
    model.addConstr(
        gp.quicksum(ammonia_synthesis_power[t] for t in range(time_steps)) >= hourly_ammonia_demand * 8760,
        name="annual_ammonia_production_constraint"
    )

    # 5.3 电力与存储平衡
    conversion_factor_AE  = electrolyzer_eff_AE  * 3.6 / 120
    conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

    for t in range(time_steps):
        wind_power = wind_output[t] * wind_capacity
        pv_power   = pv_output[t]   * pv_capacity

        # 电池储能约束
        model.addConstr(energy_balance[t] <= battery_storage_capacity,
                        name=f"energy_balance_limit_{t}")
        model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                        name=f"battery_charge_discharge_limit_{t}")

        # 动态电池平衡
        if t == 0:
            model.addConstr(
                energy_balance[t] == battery_charge[t] * battery_charge_efficiency
                                    - battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}" )
        else:
            model.addConstr(
                energy_balance[t] == energy_balance[t-1] +
                                    battery_charge[t] * battery_charge_efficiency
                                  - battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}" )

        # 氢气存储约束
        model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                        name=f"hydrogen_storage_capacity_limit_{t}")
        model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                        name=f"hydrogen_charge_limit_{t}")

        # 氢气存储动态平衡
        if t == 0:
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )
        else:
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )

        # 氢气供需平衡：电解槽产氢（转换系数 3.6/120）加上氢气释放，
        # 需满足合成氨生产所需的氢气（按氢消耗比）加上充氢量

        # 氨储存平衡
        model.addConstr(ammonia_storage_balance[t] == (ammonia_storage_balance[t-1] if t>0 else ammonia_storage_charge[0])
                                  + ammonia_storage_charge[t] - ammonia_storage_discharge[t],
                        name=f"ammonia_storage_balance_{t}")
        model.addConstr(ammonia_storage_balance[t] <= ammonia_storage_capacity,
                        name=f"ammonia_storage_capacity_limit_{t}")

        # 氨储存：确保合成氨生产加上储存放出不少于储入量
        model.addConstr(
            ammonia_synthesis_power[t] + ammonia_storage_discharge[t] >= ammonia_storage_charge[t],
            name=f"ammonia_min_balance_{t}"
        )

        # 电力平衡
        model.addConstr(
            wind_power + pv_power + battery_discharge[t] ==
            electrolyzer_power_AE[t] + electrolyzer_power_PEM[t] +
            battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
            ammonia_synthesis_power[t] * ammonia_electricity_ratio + surplus[t],
            name=f"power_balance_{t}" )

        # 氢气供需平衡
        model.addConstr(
            electrolyzer_power_AE[t] * conversion_factor_AE +
            electrolyzer_power_PEM[t] * conversion_factor_PEM +
            hydrogen_discharge[t] >=
            ammonia_synthesis_power[t] * ammonia_hydrogen_ratio + hydrogen_charge[t],
            name=f"hydrogen_PD_balance_{t}" )

    # 5.4 容量与最小运行
    model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)), name="electrolyzer_AE_capacity")
    model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)), name="electrolyzer_PEM_capacity")
    model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(start_index, time_steps)), name="electrolyzer_AE_min")
    model.addConstrs((ammonia_synthesis_capacity >= ammonia_synthesis_power[t] for t in range(time_steps)), name="ammonia_capacity")
    model.addConstrs((ammonia_synthesis_power[t] >= 0.2 * ammonia_synthesis_capacity for t in range(start_index, time_steps)), name="ammonia_min")

    # AE电解需配备电池能量：
    model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE, name="min_battery_for_AE")


    model.optimize()
    return model



In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB


def optimize_methanol_system(
    wind_output,
    pv_output,
    target_methanol_production,
    wind_cost,
    pv_cost,
    battery_energy_cost,
    battery_storage_cost,
    battery_charge_efficiency,
    battery_discharge_efficiency,
    battery_charge_penalty,
    battery_discharge_penalty,
    electrolyzer_cost_AE,
    electrolyzer_cost_PEM,
    electrolyzer_eff_AE,
    electrolyzer_eff_PEM,
    methanol_synthesis_cost,
    hydrogen_storage_cost,
    hydrogen_storage_elec,
    hydrogen_charge_penalty,
    hydrogen_storage_efficiency,
    methanol_storage_cost_param,
    methanol_charge_penalty,
    discount_rate,
    battery_lifetime,
    wind_lifetime,
    pv_lifetime,
    electrolyzer_lifetime_AE,
    electrolyzer_lifetime_PEM,
    methanol_synthesis_lifetime,
    hydrogen_storage_lifetime,
    methanol_storage_lifetime,
    heat_pump_cost,
    heat_pump_cop,
    heat_pump_lifetime,
    thermal_storage_cost,
    thermal_storage_efficiency,
    thermal_storage_lifetime,
    dac_cost,
    dac_elec_per_kg,
    dac_heat_per_kg,
    dac_lifetime,
    sorbent_cost,
    co2_compressor_cost,
    co2_compression_energy,
    co2_compressor_lifetime,
    co2_storage_cost,
    co2_storage_lifetime,
    Flex_up,
    Flex_mid,
    Flex_down,
    surplus_penalty,
    aggregation_mode
):
    # 年化运维成本
    wind_om = wind_cost * 0.03
    pv_om = pv_cost * 0.03
    battery_energy_om = battery_energy_cost * 0.03
    battery_storage_om = battery_storage_cost * 0.03
    electrolyzer_om_AE = electrolyzer_cost_AE * 0.03
    electrolyzer_om_PEM = electrolyzer_cost_PEM * 0.03
    methanol_synthesis_om = methanol_synthesis_cost * 0.03
    hydrogen_storage_om = hydrogen_storage_cost * 0.03
    methanol_storage_om = methanol_storage_cost_param * 0.03
    heat_pump_om = heat_pump_cost * 0.03
    thermal_storage_om = thermal_storage_cost * 0.03
    dac_om = dac_cost * 0.03
    co2_compressor_om = co2_compressor_cost * 0.03
    co2_storage_om = co2_storage_cost * 0.03

    # 年化成本
    wind_annual_cost = wind_cost * discount_rate / (1 - (1+discount_rate)**-wind_lifetime) + wind_om
    pv_annual_cost = pv_cost * discount_rate / (1 - (1+discount_rate)**-pv_lifetime) + pv_om
    battery_energy_annual_cost = battery_energy_cost * discount_rate / (1 - (1+discount_rate)**-battery_lifetime) + battery_energy_om
    battery_storage_annual_cost = battery_storage_cost * discount_rate / (1 - (1+discount_rate)**-battery_lifetime) + battery_storage_om
    electrolyzer_annual_cost_AE = electrolyzer_cost_AE * discount_rate / (1 - (1+discount_rate)**-electrolyzer_lifetime_AE) + electrolyzer_om_AE
    electrolyzer_annual_cost_PEM = electrolyzer_cost_PEM * discount_rate / (1 - (1+discount_rate)**-electrolyzer_lifetime_PEM) + electrolyzer_om_PEM
    methanol_synthesis_annual_cost = methanol_synthesis_cost * discount_rate / (1 - (1+discount_rate)**-methanol_synthesis_lifetime) + methanol_synthesis_om
    hydrogen_storage_annual_cost = hydrogen_storage_cost * discount_rate / (1 - (1+discount_rate)**-hydrogen_storage_lifetime) + hydrogen_storage_om
    methanol_storage_annual_cost = methanol_storage_cost_param * discount_rate / (1 - (1+discount_rate)**-methanol_storage_lifetime) + methanol_storage_om
    heat_pump_annual_cost = heat_pump_cost * discount_rate / (1 - (1+discount_rate)**-heat_pump_lifetime) + heat_pump_om
    thermal_storage_annual_cost = thermal_storage_cost * discount_rate / (1 - (1+discount_rate)**-thermal_storage_lifetime) + thermal_storage_om
    dac_annual_cost = dac_cost * discount_rate / (1 - (1+discount_rate)**-dac_lifetime) + dac_om
    co2_compressor_annual_cost = co2_compressor_cost * discount_rate / (1 - (1+discount_rate)**-co2_compressor_lifetime) + co2_compressor_om
    co2_storage_annual_cost = co2_storage_cost * discount_rate / (1 - (1+discount_rate)**-co2_storage_lifetime) + co2_storage_om

    hourly_methanol_demand = target_methanol_production / 8760
    methanol_electricity_ratio = 0.7
    methanol_hydrogen_ratio = 0.19
    methanol_co2_ratio = 1.4

    time_steps = 8760
    model = Model("MethanolDACOptimization")
    model.setParam('OutputFlag', 0)
    model.setParam('TimeLimit', 1000)
    model.setParam('Method', 2)
    model.setParam('Crossover', 0)
    model.setParam('BarConvTol', 1e-6)
    model.setParam('Threads', 4)
    model.setParam('Presolve', 1)
    model.setParam('BarHomogeneous', 1)

    # 决策变量
    wind_capacity = model.addVar(lb=0, name="wind_capacity")
    pv_capacity = model.addVar(lb=0, name="pv_capacity")
    battery_energy_capacity = model.addVar(lb=0, name="battery_energy_capacity")
    battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
    electrolyzer_capacity_AE = model.addVar(lb=0, name="electrolyzer_capacity_AE")
    electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")
    methanol_synthesis_capacity = model.addVar(lb=0, name="methanol_synthesis_capacity")
    heat_pump_capacity = model.addVar(lb=0, name="heat_pump_capacity")
    thermal_storage_capacity = model.addVar(lb=0, name="thermal_storage_capacity")
    methanol_storage_capacity = model.addVar(lb=0, name="methanol_storage_capacity")
    hydrogen_storage_capacity = model.addVar(lb=0, name="hydrogen_storage_capacity")
    dac_capacity = model.addVar(lb=0, name="dac_capacity")
    co2_compressor_capacity = model.addVar(lb=0, name="co2_compressor_capacity")
    co2_storage_capacity = model.addVar(lb=0, name="co2_storage_capacity")

    # 时序变量
    energy_balance = model.addVars(time_steps, lb=0, name="energy_balance")
    battery_charge = model.addVars(time_steps, lb=0, name="battery_charge")
    battery_discharge = model.addVars(time_steps, lb=0, name="battery_discharge")
    electrolyzer_power_AE = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
    electrolyzer_power_PEM = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")
    methanol_synthesis_power = model.addVars(time_steps, lb=0, name="methanol_synthesis_power")
    surplus = model.addVars(time_steps, lb=0, name="surplus")
    hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
    hydrogen_charge = model.addVars(time_steps, lb=0, name="hydrogen_charge")
    hydrogen_discharge = model.addVars(time_steps, lb=0, name="hydrogen_discharge")
    methanol_storage_balance = model.addVars(time_steps, lb=0, name="methanol_storage_balance")
    methanol_storage_charge = model.addVars(time_steps, lb=0, name="methanol_storage_charge")
    methanol_storage_discharge = model.addVars(time_steps, lb=0, name="methanol_storage_discharge")
    dac_CO2 = model.addVars(time_steps, lb=0, name="dac_CO2")
    dac_electricity = model.addVars(time_steps, lb=0, name="dac_electricity")
    dac_heat_input = model.addVars(time_steps, lb=0, name="dac_heat_input")
    heat_pump_output = model.addVars(time_steps, lb=0, name="heat_pump_output")
    thermal_charge = model.addVars(time_steps, lb=0, name="thermal_charge")
    thermal_discharge = model.addVars(time_steps, lb=0, name="thermal_discharge")
    thermal_storage_level = model.addVars(time_steps, lb=0, name="thermal_storage_level")
    co2_comp_elec = model.addVars(time_steps, lb=0, name="co2_comp_elec")
    co2_storage_level = model.addVars(time_steps, lb=0, name="co2_storage_level")
    co2_charge = model.addVars(time_steps, lb=0, name="co2_charge")
    co2_discharge = model.addVars(time_steps, lb=0, name="co2_discharge")
    direct_co2 = model.addVars(time_steps, lb=0, name="direct_co2")
    stored_co2 = model.addVars(time_steps, lb=0, name="stored_co2")

    # 计算转换系数
    conversion_factor_AE = electrolyzer_eff_AE * 3.6 / 120
    conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

    # 目标函数
    model.setObjective(
        wind_annual_cost * wind_capacity +
        pv_annual_cost * pv_capacity +
        battery_energy_annual_cost * battery_energy_capacity +
        battery_storage_annual_cost * battery_storage_capacity +
        electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
        electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
        methanol_synthesis_annual_cost * methanol_synthesis_capacity +
        heat_pump_annual_cost * heat_pump_capacity +
        thermal_storage_annual_cost * thermal_storage_capacity +
        hydrogen_storage_annual_cost * hydrogen_storage_capacity +
        methanol_storage_annual_cost * methanol_storage_capacity +
        dac_annual_cost * dac_capacity +
        co2_compressor_annual_cost * co2_compressor_capacity +
        co2_storage_annual_cost * co2_storage_capacity +
        sorbent_cost * gp.quicksum(dac_CO2[t] for t in range(time_steps)) +
        surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
        battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
        battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
        hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)) +
        methanol_charge_penalty * gp.quicksum(methanol_storage_charge[t] for t in range(time_steps)),
        GRB.MINIMIZE
    )

    # 添加约束
    for t in range(1, time_steps):
        model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] <= Flex_down * electrolyzer_capacity_AE)
        model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] >= -Flex_down * electrolyzer_capacity_AE)
        model.addConstr(methanol_synthesis_power[t] - methanol_synthesis_power[t-1] <= Flex_up * methanol_synthesis_capacity)
        model.addConstr(methanol_synthesis_power[t] - methanol_synthesis_power[t-1] >= -Flex_down * methanol_synthesis_capacity)
        model.addConstr(dac_CO2[t] - dac_CO2[t-1] <= Flex_up * dac_capacity)
        model.addConstr(dac_CO2[t] - dac_CO2[t-1] >= -Flex_down * dac_capacity)

    # 5.2 甲醇供需平衡约束（含储存充放，支持多种聚合尺度）
    start_index = 168  # 避开初始不稳定阶段
    if aggregation_mode == "hourly":
        for t in range(start_index, time_steps):
            model.addConstr(
                methanol_synthesis_power[t] + methanol_storage_discharge[t] >=
                hourly_methanol_demand + methanol_storage_charge[t],
                name=f"methanol_balance_hourly_{t}"
            )
    elif aggregation_mode == "3hourly":
        interval = 3
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_methanol_demand * interval +
                gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
                name=f"methanol_balance_3hourly_{i}"
            )
    elif aggregation_mode == "daily":
        interval = 24
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
                >=
                0.8 * hourly_methanol_demand * interval +
                gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
                name=f"methanol_balance_daily_{i}"
            )
    elif aggregation_mode == "weekly":
        interval = 168
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_methanol_demand * interval +
                gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
                name=f"methanol_balance_weekly_{i}"
            )
    elif aggregation_mode == "monthly":
        interval = 730
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_methanol_demand * interval +
                gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
                name=f"methanol_balance_monthly_{i}"
            )
    elif aggregation_mode == "annual":
        model.addConstr(
            gp.quicksum(methanol_synthesis_power[t] for t in range(start_index, time_steps)) +
            gp.quicksum(methanol_storage_discharge[t] for t in range(start_index, time_steps))
            ==
            hourly_methanol_demand * (time_steps - start_index) +
            gp.quicksum(methanol_storage_charge[t] for t in range(start_index, time_steps)),
            name="methanol_balance_annual"
        )
    else:
        raise ValueError("Unsupported aggregation_mode value!")

    # 附加：年生产总量约束
    model.addConstr(
        gp.quicksum(methanol_synthesis_power[t] for t in range(time_steps)) >= hourly_methanol_demand * 8760,
        name="annual_methanol_production"
    )

    # 5.3 甲醇储存动态与容量约束
    model.addConstr(methanol_storage_balance[0] ==
                    methanol_storage_charge[0] - methanol_storage_discharge[0],
                    name="methanol_storage_balance_0")
    for t in range(1, time_steps):
        model.addConstr(
            methanol_storage_balance[t] == methanol_storage_balance[t-1] + methanol_storage_charge[t] - methanol_storage_discharge[t],
            name=f"methanol_storage_balance_{t}"
        )
        model.addConstr(methanol_storage_balance[t] <= methanol_storage_capacity,
                        name=f"methanol_storage_capacity_limit_{t}")

    # 5.4 能量平衡与储能动态约束
    conversion_factor_AE  = electrolyzer_eff_AE  * 3.6 / 120
    conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

    for t in range(time_steps):
        # 计算当期风电与光伏发电
        wind_power = wind_output[t] * wind_capacity
        pv_power = pv_output[t] * pv_capacity
        
        # 甲醇储存：要求合成功率与储存放出至少满足充入量
        model.addConstr(methanol_synthesis_power[t] + methanol_storage_discharge[t] >=
                        methanol_storage_charge[t],
                        name=f"methanol_min_balance_{t}")
                        
        # 电池储能及充放电限额
        model.addConstr(energy_balance[t] <= battery_storage_capacity,
                        name=f"energy_balance_limit_{t}")
        model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                        name=f"battery_charge_discharge_limit_{t}")
        # 氢气存储上限
        model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                        name=f"hydrogen_storage_capacity_limit_{t}")
        model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                        name=f"hydrogen_charge_limit_{t}")
        
        # 电力平衡（含电解槽、甲醇合成、储能、热泵、DAC与CO2压缩电耗等）
        model.addConstr(
            wind_power + pv_power + battery_discharge[t] ==
            (electrolyzer_power_AE[t] + electrolyzer_power_PEM[t]) +
            methanol_synthesis_power[t] * methanol_electricity_ratio +
            battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
            (heat_pump_output[t] / heat_pump_cop) + dac_electricity[t] + surplus[t] + co2_comp_elec[t],
            name=f"power_balance_{t}"
        )
        # 电池动态平衡
        if t == 0:
            model.addConstr(
                energy_balance[t] == battery_charge[t] * battery_charge_efficiency
                                    - battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}" )
        else:
            model.addConstr(
                energy_balance[t] == energy_balance[t-1]+
                                    battery_charge[t] * battery_charge_efficiency
                                - battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}" )
        # 氢气存储动态平衡
        if t == 0:
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )
        else:
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )
        # 氢气供需平衡：电解槽产氢需满足甲醇合成及储氢要求
        model.addConstr(
            electrolyzer_power_AE[t] * conversion_factor_AE +
            electrolyzer_power_PEM[t] * conversion_factor_PEM + hydrogen_discharge[t] >=
            methanol_synthesis_power[t] * methanol_hydrogen_ratio + hydrogen_charge[t],
            name=f"hydrogen_PD_balance_{t}"
        )
        # DAC部分：热、电耗与捕集CO2量关系
        model.addConstr(dac_heat_input[t] == dac_CO2[t] * dac_heat_per_kg,
                        name=f"dac_heat_consumption_{t}")
        model.addConstr(dac_electricity[t] == dac_CO2[t] * dac_elec_per_kg,
                        name=f"dac_electricity_consumption_{t}")
        model.addConstr(dac_CO2[t] <= dac_capacity,
                        name=f"dac_capacity_limit_{t}")
        
        # CO2分流：直接供给与进入储存
        model.addConstr(direct_co2[t] + stored_co2[t] == dac_CO2[t],
                        name=f"co2_split_{t}")
        model.addConstr(stored_co2[t] <= co2_compressor_capacity,
                        name=f"co2_compressor_capacity_limit_{t}")
        model.addConstr(co2_comp_elec[t] == stored_co2[t] * co2_compression_energy,
                        name=f"co2_compression_energy_{t}")
        model.addConstr(co2_charge[t] == stored_co2[t],
                        name=f"co2_charge_relation_{t}")
        
        # 热泵与热储热平衡
        model.addConstr(
            heat_pump_output[t] - thermal_charge[t] + thermal_discharge[t] >= dac_heat_input[t],
            name=f"heat_supply_balance_{t}"
        )
        # CO2储存动态及容量约束
        if t == 0:
            model.addConstr(
                co2_storage_level[t] == co2_charge[t] - co2_discharge[t],
                name="co2_storage_balance_0"
            )
        else:
            model.addConstr(
                co2_storage_level[t] == co2_storage_level[t-1] + co2_charge[t] - co2_discharge[t],
                name=f"co2_storage_balance_{t}"
            )
        model.addConstr(co2_storage_level[t] <= co2_storage_capacity,
                        name=f"co2_storage_level_limit_{t}")
        
        # 储热系统动态及容量约束
        if t == 0:
            model.addConstr(
                thermal_storage_level[t] == thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency,
                name="thermal_storage_balance_0"
            )
        else:
            model.addConstr(
                thermal_storage_level[t] == thermal_storage_level[t-1] + thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency,
                name=f"thermal_storage_balance_{t}"
            )
        model.addConstr(thermal_storage_level[t] <= thermal_storage_capacity,
                        name=f"thermal_storage_level_limit_{t}")

    # 5.5 装机容量与最小运行约束
    model.addConstrs((heat_pump_capacity >= heat_pump_output[t] for t in range(time_steps)),
                    name="heat_pump_capacity_constraint")
    model.addConstrs((heat_pump_output[t] >= 0.25 * heat_pump_capacity for t in range(168, time_steps)),
                    name="heat_pump_min_output")
    model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)),
                    name="electrolyzer_AE_capacity_constraint")
    model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)),
                    name="electrolyzer_PEM_capacity_constraint")
    model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(168, time_steps)),
                    name="electrolyzer_AE_min_power")
    model.addConstrs((dac_CO2[t] >= 0.25 * dac_capacity for t in range(168, time_steps)),
                    name="heat_pump_min_output")
    model.addConstrs((methanol_synthesis_power[t] >= 0.25 * methanol_synthesis_capacity for t in range(168, time_steps)),
                    name="methanol_synthesis_min_output")
    model.addConstrs((methanol_synthesis_power[t] <= methanol_synthesis_capacity for t in range(time_steps)),
                    name="methanol_synthesis_max_output")

    # 5.6 每小时CO2供需平衡约束：直接供给加储存放出需满足当期甲醇CO2需求

    model.addConstrs((direct_co2[t] + co2_discharge[t] >= methanol_synthesis_power[t] * methanol_co2_ratio for t in range(time_steps)),
            name="co2_hourly_balance")

    # 5.7 年度CO2供给约束
    model.addConstr(gp.quicksum(direct_co2[t] + co2_discharge[t] for t in range(time_steps)) >= target_methanol_production * methanol_co2_ratio,
                    name="annual_co2_supply_constraint")

    # 5.8 新增：AE型电解槽必须配备至少10%（此处0.5倍）储能电池
    model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE,
                    name="min_battery_for_AE")

    model.optimize()
    return model


In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB


def optimize_ftl_system(
    wind_output,
    pv_output,
    target_ftl_production,
    # 成本 / 参数输入（允许外部灵活指定）
    wind_cost,
    pv_cost,
    battery_energy_cost,
    battery_storage_cost,
    battery_charge_efficiency,
    battery_discharge_efficiency,
    battery_charge_penalty,
    battery_discharge_penalty,
    electrolyzer_cost_AE,
    electrolyzer_cost_PEM,
    electrolyzer_eff_AE,
    electrolyzer_eff_PEM,
    electrolyzer_lifetime_AE,
    electrolyzer_lifetime_PEM,
    ftl_synthesis_cost,
    ftl_synthesis_lifetime,
    ftl_hydrogen_ratio,
    ftl_co2_ratio,
    ftl_electricity_ratio,
    hydrogen_storage_cost,
    hydrogen_storage_elec,
    hydrogen_charge_penalty,
    hydrogen_storage_efficiency,
    hydrogen_storage_lifetime,
    ftl_storage_cost_param,
    ftl_storage_lifetime,
    ftl_charge_penalty,
    heat_pump_cost,
    heat_pump_cop,
    heat_pump_lifetime,
    thermal_storage_cost,
    thermal_storage_efficiency,
    thermal_storage_lifetime,
    dac_cost,
    dac_elec_per_kg,
    dac_heat_per_kg,
    dac_lifetime,
    sorbent_cost,
    co2_compressor_cost,
    co2_compression_energy,
    co2_compressor_lifetime,
    co2_storage_cost,
    co2_storage_lifetime,
    discount_rate,
    battery_lifetime,
    wind_lifetime,
    pv_lifetime,
    Flex_up,
    Flex_mid,
    Flex_down,
    surplus_penalty,
    aggregation_mode
):
    """FT Liquid + DAC 优化模型（复用用户提供脚本逻辑，封装为函数）

    参数说明：传入逐小时风/光容量因子数组（长度需=8760），以及各类成本、效率、寿命与灵活性参数。
    返回：包含容量配置、利用率、平准化成本等指标的结果字典；若不可行返回空字典。
    """
    time_steps = len(wind_output)
    assert len(pv_output) == time_steps, "pv_output length must match wind_output"

    # 年化 O&M（使用统一 3% 或指定 6% 的比率）
    wind_om = wind_cost * 0.03
    pv_om = pv_cost * 0.03
    battery_energy_om = battery_energy_cost * 0.03
    battery_storage_om = battery_storage_cost * 0.03
    electrolyzer_om_AE = electrolyzer_cost_AE * 0.03
    electrolyzer_om_PEM = electrolyzer_cost_PEM * 0.03
    ftl_synthesis_om = ftl_synthesis_cost * 0.06  # 按用户脚本示例
    hydrogen_storage_om = hydrogen_storage_cost * 0.03
    ftl_storage_om = ftl_storage_cost_param * 0.03
    heat_pump_om = heat_pump_cost * 0.03
    thermal_storage_om = thermal_storage_cost * 0.03
    dac_om = dac_cost * 0.03
    co2_compressor_om = co2_compressor_cost * 0.03
    co2_storage_om = co2_storage_cost * 0.03

    hourly_ftl_demand = target_ftl_production / 8760.0

    # 年化资本 + O&M
    def crf(dr, life):
        return dr / (1 - (1 + dr) ** (-life))

    wind_annual_cost = wind_cost * crf(discount_rate, wind_lifetime) + wind_om
    pv_annual_cost = pv_cost * crf(discount_rate, pv_lifetime) + pv_om
    battery_energy_annual_cost = battery_energy_cost * crf(discount_rate, battery_lifetime) + battery_energy_om
    battery_storage_annual_cost = battery_storage_cost * crf(discount_rate, battery_lifetime) + battery_storage_om
    electrolyzer_annual_cost_AE = electrolyzer_cost_AE * crf(discount_rate, electrolyzer_lifetime_AE) + electrolyzer_om_AE
    electrolyzer_annual_cost_PEM = electrolyzer_cost_PEM * crf(discount_rate, electrolyzer_lifetime_PEM) + electrolyzer_om_PEM
    ftl_synthesis_annual_cost = ftl_synthesis_cost * crf(discount_rate, ftl_synthesis_lifetime) + ftl_synthesis_om
    hydrogen_storage_annual_cost = hydrogen_storage_cost * crf(discount_rate, hydrogen_storage_lifetime) + hydrogen_storage_om
    dac_annual_cost = dac_cost * crf(discount_rate, dac_lifetime) + dac_om
    co2_compressor_annual_cost = co2_compressor_cost * crf(discount_rate, co2_compressor_lifetime) + co2_compressor_om
    co2_storage_annual_cost = co2_storage_cost * crf(discount_rate, co2_storage_lifetime) + co2_storage_om
    heat_pump_annual_cost = heat_pump_cost * crf(discount_rate, heat_pump_lifetime) + heat_pump_om
    thermal_storage_annual_cost = thermal_storage_cost * crf(discount_rate, thermal_storage_lifetime) + thermal_storage_om
    ftl_storage_annual_cost = ftl_storage_cost_param * crf(discount_rate, ftl_storage_lifetime) + ftl_storage_om

    model = Model("HydrogenFTLOptimization")
    model.setParam('OutputFlag', 0)
    model.setParam('TimeLimit', 1000)
    model.setParam('Method', 2)
    model.setParam('Crossover', 0)
    model.setParam('BarConvTol', 1e-6)
    model.setParam('Threads', 1)
    model.setParam('Presolve', 1)
    model.setParam('BarHomogeneous', 1)

    # 容量变量
    wind_capacity = model.addVar(lb=0, name="wind_capacity")
    pv_capacity = model.addVar(lb=0, name="pv_capacity")
    battery_energy_capacity = model.addVar(lb=0, name="battery_energy_capacity")
    battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
    electrolyzer_capacity_AE = model.addVar(lb=0, name="electrolyzer_capacity_AE")
    electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")
    ftl_synthesis_capacity = model.addVar(lb=0, name="FTL_capacity")
    heat_pump_capacity = model.addVar(lb=0, name="heat_pump_capacity")
    thermal_storage_capacity = model.addVar(lb=0, name="thermal_storage_capacity")
    ftl_storage_capacity = model.addVar(lb=0, name="FTL_storage_capacity")
    hydrogen_storage_capacity = model.addVar(lb=0, name="hydrogen_storage_capacity")
    dac_capacity = model.addVar(lb=0, name="dac_capacity")
    co2_compressor_capacity = model.addVar(lb=0, name="co2_compressor_capacity")
    co2_storage_capacity = model.addVar(lb=0, name="co2_storage_capacity")

    # 时序变量
    energy_balance = model.addVars(time_steps, lb=0, name="energy_balance")
    battery_charge = model.addVars(time_steps, lb=0, name="battery_charge")
    battery_discharge = model.addVars(time_steps, lb=0, name="battery_discharge")
    electrolyzer_power_AE = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
    electrolyzer_power_PEM = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")
    ftl_synthesis_power = model.addVars(time_steps, lb=0, name="FTL_synthesis_power")
    ftl_storage_balance = model.addVars(time_steps, lb=0, name="FTL_storage_balance")
    ftl_storage_charge = model.addVars(time_steps, lb=0, name="FTL_storage_charge")
    ftl_storage_discharge = model.addVars(time_steps, lb=0, name="FTL_storage_discharge")
    hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
    hydrogen_charge = model.addVars(time_steps, lb=0, name="hydrogen_charge")
    hydrogen_discharge = model.addVars(time_steps, lb=0, name="hydrogen_discharge")
    dac_CO2 = model.addVars(time_steps, lb=0, name="dac_CO2")
    dac_electricity = model.addVars(time_steps, lb=0, name="dac_electricity")
    dac_heat_input = model.addVars(time_steps, lb=0, name="dac_heat_input")
    heat_pump_output = model.addVars(time_steps, lb=0, name="heat_pump_output")
    thermal_charge = model.addVars(time_steps, lb=0, name="thermal_charge")
    thermal_discharge = model.addVars(time_steps, lb=0, name="thermal_discharge")
    thermal_storage_level = model.addVars(time_steps, lb=0, name="thermal_storage_level")
    co2_comp_elec = model.addVars(time_steps, lb=0, name="co2_comp_elec")
    co2_storage_level = model.addVars(time_steps, lb=0, name="co2_storage_level")
    co2_charge = model.addVars(time_steps, lb=0, name="co2_charge")
    co2_discharge = model.addVars(time_steps, lb=0, name="co2_discharge")
    direct_co2 = model.addVars(time_steps, lb=0, name="direct_co2")
    stored_co2 = model.addVars(time_steps, lb=0, name="stored_co2")
    surplus = model.addVars(time_steps, lb=0, name="surplus")

    # 目标函数
    model.setObjective(
        wind_annual_cost * wind_capacity +
        pv_annual_cost * pv_capacity +
        battery_energy_annual_cost * battery_energy_capacity +
        battery_storage_annual_cost * battery_storage_capacity +
        electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
        electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
        ftl_synthesis_annual_cost * ftl_synthesis_capacity +
        heat_pump_annual_cost * heat_pump_capacity +
        thermal_storage_annual_cost * thermal_storage_capacity +
        hydrogen_storage_annual_cost * hydrogen_storage_capacity +
        ftl_storage_annual_cost * ftl_storage_capacity +
        dac_annual_cost * dac_capacity +
        co2_compressor_annual_cost * co2_compressor_capacity +
        co2_storage_annual_cost * co2_storage_capacity +
        surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
        battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
        battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
        sorbent_cost * gp.quicksum(dac_CO2[t] for t in range(time_steps)) +
        hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)) +
        ftl_charge_penalty * gp.quicksum(ftl_storage_charge[t] for t in range(time_steps)),
        GRB.MINIMIZE
    )

    # ===== 1. 灵活性约束（爬坡限制）=====
    for t in range(1, time_steps):
        model.addConstr(
            electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] <= Flex_down * electrolyzer_capacity_AE,
            name=f"electrolyzer_AE_ramp_up_{t}"
        )
        model.addConstr(
            electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] >= -Flex_down * electrolyzer_capacity_AE,
            name=f"electrolyzer_AE_ramp_down_{t}"
        )
        model.addConstr(
            ftl_synthesis_power[t] - ftl_synthesis_power[t-1] <= Flex_up * ftl_synthesis_capacity,
            name=f"ftl_synthesis_ramp_up_{t}"
        )
        model.addConstr(
            ftl_synthesis_power[t] - ftl_synthesis_power[t-1] >= -Flex_down * ftl_synthesis_capacity,
            name=f"ftl_synthesis_ramp_down_{t}"
        )
        model.addConstr(
            dac_CO2[t] - dac_CO2[t-1] <= Flex_up * dac_capacity,
            name=f"dac_ramp_up_{t}"
        )
        model.addConstr(
            dac_CO2[t] - dac_CO2[t-1] >= -Flex_down * dac_capacity,
            name=f"dac_ramp_down_{t}"
        )

    # ===== 2. FTL供需平衡约束（支持多种聚合尺度）=====
    start_index = 168
    if aggregation_mode == "hourly":
        for t in range(start_index, time_steps):
            model.addConstr(
                ftl_synthesis_power[t] + ftl_storage_discharge[t] >= 
                hourly_ftl_demand + ftl_storage_charge[t],
                name=f"ftl_balance_hourly_{t}"
            )
    elif aggregation_mode == "3hourly":
        interval = 3
        n = (time_steps - start_index) // interval
        for i in range(n):
            s = start_index + i * interval
            e = s + interval
            model.addConstr(
                gp.quicksum(ftl_synthesis_power[t] for t in range(s, e)) +
                gp.quicksum(ftl_storage_discharge[t] for t in range(s, e)) ==
                hourly_ftl_demand * interval +
                gp.quicksum(ftl_storage_charge[t] for t in range(s, e)),
                name=f"ftl_balance_3hourly_{i}"
            )
    elif aggregation_mode == "daily":
        interval = 24
        n = (time_steps - start_index) // interval
        for i in range(n):
            s = start_index + i * interval
            e = s + interval
            model.addConstr(
                gp.quicksum(ftl_synthesis_power[t] for t in range(s, e)) +
                gp.quicksum(ftl_storage_discharge[t] for t in range(s, e)) >=
                0.8 * hourly_ftl_demand * interval +
                gp.quicksum(ftl_storage_charge[t] for t in range(s, e)),
                name=f"ftl_balance_daily_{i}"
            )
    elif aggregation_mode == "weekly":
        interval = 168
        n = (time_steps - start_index) // interval
        for i in range(n):
            s = start_index + i * interval
            e = s + interval
            model.addConstr(
                gp.quicksum(ftl_synthesis_power[t] for t in range(s, e)) +
                gp.quicksum(ftl_storage_discharge[t] for t in range(s, e)) ==
                hourly_ftl_demand * interval +
                gp.quicksum(ftl_storage_charge[t] for t in range(s, e)),
                name=f"ftl_balance_weekly_{i}"
            )
    elif aggregation_mode == "monthly":
        interval = 730
        n = (time_steps - start_index) // interval
        for i in range(n):
            s = start_index + i * interval
            e = s + interval
            model.addConstr(
                gp.quicksum(ftl_synthesis_power[t] for t in range(s, e)) +
                gp.quicksum(ftl_storage_discharge[t] for t in range(s, e)) ==
                hourly_ftl_demand * interval +
                gp.quicksum(ftl_storage_charge[t] for t in range(s, e)),
                name=f"ftl_balance_monthly_{i}"
            )
    elif aggregation_mode == "annual":
        model.addConstr(
            gp.quicksum(ftl_synthesis_power[t] for t in range(start_index, time_steps)) +
            gp.quicksum(ftl_storage_discharge[t] for t in range(start_index, time_steps)) ==
            hourly_ftl_demand * (time_steps - start_index) +
            gp.quicksum(ftl_storage_charge[t] for t in range(start_index, time_steps)),
            name="ftl_balance_annual"
        )
    else:
        raise ValueError("Unsupported aggregation_mode")

    # 年生产量约束
    model.addConstr(
        gp.quicksum(ftl_synthesis_power[t] for t in range(time_steps)) >= target_ftl_production,
        name="annual_ftl_production_constraint"
    )

    # ===== 3. FTL储存动态平衡 =====
    model.addConstr(
        ftl_storage_balance[0] == ftl_storage_charge[0] - ftl_storage_discharge[0],
        name="ftl_storage_balance_0"
    )
    for t in range(1, time_steps):
        model.addConstr(
            ftl_storage_balance[t] == ftl_storage_balance[t-1] + 
            ftl_storage_charge[t] - ftl_storage_discharge[t],
            name=f"ftl_storage_balance_{t}"
        )
        model.addConstr(
            ftl_storage_balance[t] <= ftl_storage_capacity,
            name=f"ftl_storage_capacity_limit_{t}"
        )

    conversion_factor_AE = electrolyzer_eff_AE * 3.6 / 120
    conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

    # ===== 4. 逐时约束（能量平衡、储存动态等）=====
    for t in range(time_steps):
        wind_power = wind_output[t] * wind_capacity
        pv_power = pv_output[t] * pv_capacity

        # FTL储存最小平衡
        model.addConstr(
            ftl_synthesis_power[t] + ftl_storage_discharge[t] >= ftl_storage_charge[t],
            name=f"ftl_min_balance_{t}"
        )

        # 电池储能约束
        model.addConstr(
            energy_balance[t] <= battery_storage_capacity,
            name=f"battery_storage_limit_{t}"
        )
        model.addConstr(
            battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
            name=f"battery_power_limit_{t}"
        )

        # 氢气储存约束
        model.addConstr(
            hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
            name=f"hydrogen_storage_limit_{t}"
        )
        model.addConstr(
            hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
            name=f"hydrogen_power_limit_{t}"
        )

        # 电力平衡
        model.addConstr(
            wind_power + pv_power + battery_discharge[t] ==
            (electrolyzer_power_AE[t] + electrolyzer_power_PEM[t]) +
            ftl_synthesis_power[t] * ftl_electricity_ratio +
            battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
            (heat_pump_output[t] / heat_pump_cop) + dac_electricity[t] + 
            surplus[t] + co2_comp_elec[t],
            name=f"power_balance_{t}"
        )

        # 储能动态平衡
        if t == 0:
            model.addConstr(
                energy_balance[t] == battery_charge[t] * battery_charge_efficiency - 
                battery_discharge[t] / battery_discharge_efficiency,
                name="battery_balance_0"
            )
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - 
                hydrogen_discharge[t],
                name="hydrogen_balance_0"
            )
            model.addConstr(
                thermal_storage_level[t] == thermal_charge[t] * thermal_storage_efficiency - 
                thermal_discharge[t] / thermal_storage_efficiency,
                name="thermal_balance_0"
            )
            model.addConstr(
                co2_storage_level[t] == co2_charge[t] - co2_discharge[t],
                name="co2_storage_balance_0"
            )
        else:
            model.addConstr(
                energy_balance[t] == energy_balance[t-1] + 
                battery_charge[t] * battery_charge_efficiency - 
                battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}"
            )
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + 
                hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )
            model.addConstr(
                thermal_storage_level[t] == thermal_storage_level[t-1] + 
                thermal_charge[t] * thermal_storage_efficiency - 
                thermal_discharge[t] / thermal_storage_efficiency,
                name=f"thermal_balance_{t}"
            )
            model.addConstr(
                co2_storage_level[t] == co2_storage_level[t-1] + 
                co2_charge[t] - co2_discharge[t],
                name=f"co2_storage_balance_{t}"
            )

        # 氢气供需平衡
        model.addConstr(
            electrolyzer_power_AE[t] * conversion_factor_AE +
            electrolyzer_power_PEM[t] * conversion_factor_PEM + hydrogen_discharge[t] >=
            ftl_synthesis_power[t] * ftl_hydrogen_ratio + hydrogen_charge[t],
            name=f"hydrogen_PD_balance_{t}"
        )

        # DAC相关约束
        model.addConstr(
            dac_heat_input[t] == dac_CO2[t] * dac_heat_per_kg,
            name=f"dac_heat_requirement_{t}"
        )
        model.addConstr(
            dac_electricity[t] == dac_CO2[t] * dac_elec_per_kg,
            name=f"dac_elec_requirement_{t}"
        )
        model.addConstr(
            dac_CO2[t] <= dac_capacity,
            name=f"dac_capacity_limit_{t}"
        )

        # CO2分流
        model.addConstr(
            direct_co2[t] + stored_co2[t] == dac_CO2[t],
            name=f"co2_split_{t}"
        )
        model.addConstr(
            stored_co2[t] <= co2_compressor_capacity,
            name=f"co2_compressor_limit_{t}"
        )
        model.addConstr(
            co2_comp_elec[t] == stored_co2[t] * co2_compression_energy,
            name=f"co2_compression_energy_{t}"
        )
        model.addConstr(
            co2_charge[t] == stored_co2[t],
            name=f"co2_charge_relation_{t}"
        )

        # 热能平衡
        model.addConstr(
            heat_pump_output[t] - thermal_charge[t] + thermal_discharge[t] >= dac_heat_input[t],
            name=f"heat_supply_balance_{t}"
        )
        model.addConstr(
            thermal_storage_level[t] <= thermal_storage_capacity,
            name=f"thermal_storage_limit_{t}"
        )
        model.addConstr(
            co2_storage_level[t] <= co2_storage_capacity,
            name=f"co2_storage_limit_{t}"
        )

    # ===== 5. 设备容量与最小出力约束 =====
    model.addConstrs(
        (heat_pump_capacity >= heat_pump_output[t] for t in range(time_steps)),
        name="heat_pump_capacity"
    )
    model.addConstrs(
        (heat_pump_output[t] >= 0.25 * heat_pump_capacity for t in range(168, time_steps)),
        name="heat_pump_min_output"
    )
    model.addConstrs(
        (electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)),
        name="electrolyzer_AE_capacity"
    )
    model.addConstrs(
        (electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)),
        name="electrolyzer_PEM_capacity"
    )
    model.addConstrs(
        (electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(168, time_steps)),
        name="electrolyzer_AE_min_power"
    )
    model.addConstrs(
        (dac_CO2[t] >= 0.25 * dac_capacity for t in range(168, time_steps)),
        name="dac_min_output"
    )
    model.addConstrs(
        (ftl_synthesis_power[t] <= ftl_synthesis_capacity for t in range(time_steps)),
        name="ftl_synthesis_capacity"
    )

    model.addConstrs(
        (ftl_synthesis_power[t] >= 0.25 * ftl_synthesis_capacity for t in range(time_steps)),
        name="ftl_min_output"
    )
    
    # ===== 6. CO2供需平衡约束 =====
    model.addConstrs(
        (direct_co2[t] + co2_discharge[t] >= ftl_synthesis_power[t] * ftl_co2_ratio 
         for t in range(time_steps)),
        name="co2_hourly_balance"
    )
    model.addConstr(
        gp.quicksum(direct_co2[t] + co2_discharge[t] for t in range(time_steps)) >= 
        target_ftl_production * ftl_co2_ratio,
        name="annual_co2_supply_constraint"
    )

    # ===== 7. 电池与AE电解槽配比约束 =====
    model.addConstr(
        battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE,
        name="min_battery_for_AE"
    )
    
    model.optimize()
    return model

## 5. 模型执行与结果分析

### 5.1 参数配置与数据准备
系统首先读取各聚类代表点的风光容量因子时序数据，并配置技术经济参数：

#### 主要技术参数
- **可再生能源**: 风电寿命30年，光伏寿命40年
- **电解制氢**: AE效率65%，PEM效率70%，均为25年寿命
- **储能系统**: 电池效率98.5%/97.5%（充/放），20年寿命
- **甲醇合成**: 30年寿命，集成度高
- **DAC系统**: 4000$/kW，热耗1.5 GJ/tCO₂，电耗1.5 MWh/tCO₂

#### 经济参数
- 贴现率: 7%
- 所有成本均转换为年化成本进行优化比较

### 5.2 优化求解流程
对每个聚类代表点执行以下步骤：
1. 构建完整的8760小时优化模型
2. 设置Gurobi求解器参数（内点法，1000秒时限）
3. 执行优化求解，获得最优系统配置
4. 提取关键结果并生成四季典型周可视化分析

In [ ]:
# -*- coding: utf-8 -*-
"""
Batch plotting of 8760 supply–demand stacked charts + annual heatmaps (EN version)
- Auto-detect year tag from file name: 2030 / 2040 / 2050.
  * Wind/PV CAPEX mapping from RegionalCost.csv: 2030ATB / 2035ATB (used for 2040) / 2050ATB.
  * Parameter sets: params_2030 / params_2050; 2040 (and others) use params_base by default.
- Merge "Regional" -> "Region" (RegionRE/RegionalRE; RegionWind/RegionalWind).
- RegionWind/RegionalWind: PV CAPEX = 225 uniformly; others use regional mapping.
- NCC-like plotting style; math text for H$_2$ / CO$_2$.
- Annual heatmap uses SOC for battery/H2/product storages.
- Legends are fixed at upper-right; stacked charts use 2 columns.
- Output folders are separated by year to avoid overwrite: Year_2030 / Year_2040 / Year_2050.
- Before plotting, shift hourly series to local time inferred from lat/lon.
"""

import os
import re
import glob
from typing import Optional, Dict, Iterable, List, Tuple

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
import gurobipy as gp
from gurobipy import GRB

from datetime import datetime
try:
    from timezonefinder import TimezoneFinder  # pip install timezonefinder
except Exception:
    TimezoneFinder = None
try:
    from zoneinfo import ZoneInfo  # Python 3.9+
except Exception:
    ZoneInfo = None


# =====================================================
# 0) Paths & global options
# =====================================================
cf_dir   = r"D:\OneDrive\POTSED\03 Cluster"
out_root = r"D:\OneDrive\POTSED\8760Supply&Demand\Global_en"
os.makedirs(out_root, exist_ok=True)

# Regional CAPEX mapping (Wind/PV)
regional_cost_path = r"D:\OneDrive\POTSED\DataOutput\RegionalCost.csv"

# ========== testing switches (optional) ==========
ONLY_ONE_SCENARIO: Optional[str] = None          # regex
ONLY_ONE_NODE: Optional[Dict[str, str]] = None   # e.g., {'product':'ammonia','continent':'Europe','cluster':0}
STOP_AFTER_FIRST_NODE: bool = True

# ========= y-axis margins (padding) =========
Y_MARGIN_FRAC_POWER: float = 0.08
Y_MARGIN_FRAC_H2: float    = 0.08

# =====================================================
# Plot style (English)
# =====================================================
def apply_en_style():
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.unicode_minus": False,
        "pdf.fonttype": 42, "ps.fonttype": 42,
        "text.usetex": False, "mathtext.fontset": "stix", "mathtext.default": "rm",
        "axes.facecolor": "white", "axes.edgecolor": "black", "axes.linewidth": 0.8,
        "axes.labelsize": 9, "axes.titlesize": 10, "axes.titlepad": 6,
        "xtick.labelsize": 8, "ytick.labelsize": 8,
        "xtick.direction": "in", "ytick.direction": "in",
        "xtick.major.size": 3, "ytick.major.size": 3,
        "xtick.major.width": 0.8, "ytick.major.width": 0.8,
        "grid.linestyle": (0, (3, 3)), "grid.color": "0.85", "grid.linewidth": 0.6,
        "legend.fontsize": 8, "legend.frameon": True, "legend.framealpha": 0.9, "legend.edgecolor": "0.8",
        "lines.linewidth": 1.2,
    })
apply_en_style()

# typical weeks (0-based week index)
season_weeks = {
    'Winter': 1,   # week of early Jan
    'Spring': 14,  # early Apr
    'Summer': 27,  # early Jul
    'Autumn': 40   # early Oct
}

# Colors
wind_color       = '#4F81BD'
pv_color         = '#C0504D'
battery_color    = '#9BBB59'
ae_color         = '#8064A2'
pem_color        = '#F79646'
synth_color      = '#4BACC6'
bat_charge_color = '#F05542'
hydrogen_color   = '#8F5CC1'
heatpump_color   = '#4C9F70'
dac_color        = '#D99694'
surplus_color    = '#B0B0B0'
co2_color        = '#696969'

# Math text
H2  = r'$H_2$'
CO2 = r'$CO_2$'

# ==================== Timezone tools ========================
ADJUST_TO_LOCAL_TZ = True

def infer_utc_offset_hours(lat: float, lon: float, year: int = 2040) -> int:
    """
    Return local UTC offset (integer hours) for given lat/lon at noon on July 1 of `year`.
    Prefer real timezone (timezonefinder + zoneinfo); fallback to lon/15 approx.
    """
    if TimezoneFinder is not None and ZoneInfo is not None:
        try:
            tf = TimezoneFinder()
            tzname = tf.timezone_at(lat=lat, lng=lon)
            if tzname:
                dt = datetime(year, 7, 1, 12, 0, 0)
                tz = ZoneInfo(tzname)
                offset_sec = tz.utcoffset(dt).total_seconds()
                return int(round(offset_sec / 3600.0))
        except Exception:
            pass
    approx = int(round(lon / 15.0))
    return max(-12, min(14, approx))


def roll_series_dict(d: dict, shift_hours: int, keys: List[str]) -> None:
    """
    In-place cyclic shift (np.roll) of time series (1-D) by shift_hours to the right.
    Positive shift: "local = UTC + shift".
    """
    if not shift_hours:
        return
    for k in keys:
        if k in d and d[k] is not None:
            d[k] = np.roll(np.asarray(d[k], dtype=float), shift_hours)


# =================== axis padding helper ===================
def set_y_margin(ax: plt.Axes, positive_stack: np.ndarray, negative_stack: np.ndarray, pad_frac: float):
    pos_max = float(np.nanmax(positive_stack)) if positive_stack.size else 0.0
    neg_max = float(np.nanmax(negative_stack)) if negative_stack.size else 0.0
    top, bottom = pos_max, -neg_max
    if top == bottom:
        span = abs(top) + 1.0
        ax.set_ylim(bottom - pad_frac * span, top + pad_frac * span)
    else:
        span = top - bottom
        pad = pad_frac * span
        ax.set_ylim(bottom - pad, top + pad)

# =================== batched var readers ===================
VAR_TEMPLATES: Dict[str, List[str]] = {
    "electrolyzer_AE":     ["electrolyzer_power_AE[{t}]"],
    "electrolyzer_PEM":    ["electrolyzer_power_PEM[{t}]"],
    "synthesis_power":     ["methanol_synthesis_power[{t}]", "FTL_synthesis_power[{t}]"],
    "dac_elec":            ["dac_electricity[{t}]"],
    "heat_pump_output":    ["heat_pump_output[{t}]"],
    "battery_charge":      ["battery_charge[{t}]"],
    "battery_discharge":   ["battery_discharge[{t}]"],
    "hydrogen_charge":     ["hydrogen_charge[{t}]"],
    "hydrogen_discharge":  ["hydrogen_discharge[{t}]"],
    "product_charge":      ["methanol_storage_charge[{t}]", "FTL_storage_charge[{t}]",
                            "ammonia_storage_charge[{t}]", "product_storage_charge[{t}]"],
    "product_discharge":   ["methanol_storage_discharge[{t}]", "FTL_storage_discharge[{t}]",
                            "ammonia_storage_discharge[{t}]", "product_storage_discharge[{t}]"],
    "surplus":             ["surplus[{t}]"],
    "co2_comp":            ["co2_comp_elec[{t}]"],
    "battery_balance":     ["energy_balance[{t}]"],
    "hydrogen_storage_balance": ["hydrogen_storage_balance[{t}]"],
    "product_storage_balance":  ["methanol_storage_balance[{t}]","FTL_storage_balance[{t}]","ammonia_storage_balance[{t}]"]
}

VAR_CAPACITY: Dict[str, List[str]] = {
    "battery":  ["battery_storage_capacity"],
    "hydrogen": ["hydrogen_storage_capacity"],
    "product":  ["methanol_storage_capacity", "FTL_storage_capacity", "ammonia_storage_capacity"],
}

def get_series_by_templates(model: gp.Model, templates: Iterable[str], T: int) -> Optional[np.ndarray]:
    vals = np.zeros(T, dtype=float); found_any = False
    for t in range(T):
        v = None
        for temp in templates:
            var = model.getVarByName(temp.format(t=t))
            if var is not None:
                v = var.x; found_any = True; break
        vals[t] = float(v) if v is not None else 0.0
    return vals if found_any else None

def ensure_series(arr: Optional[np.ndarray], T: int, fill: float = 0.0) -> np.ndarray:
    if arr is None:
        return np.full(T, fill, dtype=float)
    a = np.asarray(arr, dtype=float).ravel()
    if a.size < T:
        out = np.full(T, fill, dtype=float); out[:a.size] = a; return out
    return a[:T]

def get_first_var_value(model: gp.Model, names: List[str]) -> Optional[float]:
    for nm in names:
        var = model.getVarByName(nm)
        if var is not None:
            try:
                return float(var.x)
            except Exception:
                pass
    return None

# ========== Annual heatmap tools ==========
def month_tick_positions(days: int) -> Tuple[List[int], List[str]]:
    month_days = [31,28,31,30,31,30,31,31,30,31,30,31]
    edges = np.cumsum([0]+month_days)
    centers = [(edges[i] + edges[i+1])//2 for i in range(12)]
    scale = days / 365.0
    pos = [int(c * scale) for c in centers if int(c*scale) < days]
    labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][:len(pos)]
    return pos, labels

def plot_annual_heatmap_panel(
    items,
    title_prefix,
    out_path,
    cmap: str = "YlGnBu",
    cmap_slice: tuple | None = (0., 0.95),
    norm_range: tuple | None = None,
    # fonts
    fs_title: int = 16,
    fs_rowlab: int = 12,
    fs_axislabel: int = 13,
    fs_tick: int = 11,
    fs_cbar: int = 12,
    # layout
    hspace: float  = 0.10,
    left: float = 0.08, right: float = 0.90, top: float = 0.965, bottom: float = 0.12,
):
    """
    items: list of (row_label, yearly_series)
    """
    valid = []
    for lab, ser in items:
        if ser is None:
            continue
        arr = np.asarray(ser, dtype=float)
        if np.allclose(arr, 0.0, atol=1e-12):
            continue
        valid.append((lab, arr))
    if not valid:
        return False

    days = int(np.ceil(max(len(s) for _, s in valid) / 24.0))
    xticks, xlabels = month_tick_positions(days)

    n = len(valid)
    fig = plt.figure(figsize=(15, 1.75 * n + 0.9))
    fig.text(0.5, 0.992, f"{title_prefix} — Annual normalized profiles",
             ha="center", va="top", fontsize=fs_title)

    gs = fig.add_gridspec(nrows=n, ncols=1, hspace=hspace)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom)

    base = plt.get_cmap(cmap)
    cmap = colors.LinearSegmentedColormap.from_list(
        f"{getattr(base,'name','cmap')}_slice", base(np.linspace(float(cmap_slice[0]), float(cmap_slice[1]), 256))
    ) if cmap_slice else base
    norm = colors.Normalize(vmin=0.0, vmax=1.0) if norm_range is None \
           else colors.Normalize(vmin=float(norm_range[0]), vmax=float(norm_range[1]), clip=True)

    axes = []
    right_texts = []
    last_img = None
    for i, (lab, ser) in enumerate(valid):
        ax = fig.add_subplot(gs[i, 0], sharex=axes[0] if axes else None)
        axes.append(ax)

        # reshape to (days, 24) and normalize to [0,1] by row independently
        T = len(ser)
        mat = np.full((int(np.ceil(T/24.0)), 24), np.nan, float)
        mat.flat[:T] = ser
        mmin, mmax = np.nanmin(mat), np.nanmax(mat)
        mat_n = (mat - mmin) / (mmax - mmin) if np.isfinite(mmin) and np.isfinite(mmax) and mmax>mmin else np.nan_to_num(mat)

        last_img = ax.imshow(mat_n.T, origin='lower', aspect='auto', cmap=cmap, norm=norm)

        ax.set_yticks([0, 6, 12, 18, 23])
        ax.tick_params(axis='both', labelsize=fs_tick)
        ax.set_ylabel("Hour of day", rotation=90, labelpad=10, fontsize=fs_axislabel)

        # right-side vertical row label
        t = ax.text(
            1.02, 0.5, lab,
            transform=ax.transAxes,
            rotation=90,
            rotation_mode='anchor',
            va='center',
            ha='center',
            multialignment='center',
            fontsize=fs_rowlab,
            linespacing=1.0,
            clip_on=False
        )
        right_texts.append(t)

        if i < n - 1:
            ax.tick_params(labelbottom=False)

    axes[-1].set_xticks(xticks)
    axes[-1].set_xticklabels(xlabels, fontsize=fs_tick)
    axes[-1].set_xlabel("Month", labelpad=6, fontsize=fs_axislabel)

    # auto enlarge right margin to avoid clipping
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    max_w_px = 0
    for t in right_texts:
        bb = t.get_window_extent(renderer=renderer)
        max_w_px = max(max_w_px, bb.width)
    fig_w_in = fig.get_size_inches()[0]
    fig_dpi  = fig.dpi
    pad_frac = 0.02
    margin_needed = (max_w_px / (fig_w_in * fig_dpi)) + pad_frac
    new_right = max(0.60, 1.0 - max(0.07, margin_needed))
    fig.subplots_adjust(right=new_right)

    # bottom colorbar
    cax = fig.add_axes([0.08, 0.05, 0.87, 0.02])
    cbar = fig.colorbar(last_img, cax=cax, orientation='horizontal')
    cbar.ax.tick_params(labelsize=fs_tick)
    cbar.set_label("Normalized value", fontsize=fs_cbar, labelpad=2)

    fig.savefig(out_path, dpi=600)
    plt.close(fig)
    return True


# =====================================================
# 1) Read regional cost maps
# =====================================================
print("▶ Reading regional CAPEX table...", flush=True)
if os.path.exists(regional_cost_path):
    regional_cost_df = pd.read_csv(regional_cost_path)

    def _make_map(tech: str, col: str):
        sub = regional_cost_df[
            (regional_cost_df["Type"] == tech) &
            (regional_cost_df["Cost Type 1"] == "Total installed cost")
        ]
        if col not in sub.columns:
            print(f"  ⚠ Column {col} missing; will fallback later.")
            return {}
        return sub.set_index("region_code")[col].to_dict()

    # 2040 uses 2035 columns
    wind_map_2030 = _make_map("Onshore Wind", "2030ATB")
    wind_map_2035 = _make_map("Onshore Wind", "2035ATB")
    wind_map_2050 = _make_map("Onshore Wind", "2050ATB")

    pv_map_2030   = _make_map("Solar PV", "2030ATB")
    pv_map_2035   = _make_map("Solar PV", "2035ATB")
    pv_map_2050   = _make_map("Solar PV", "2050ATB")

    print(f"   ✓ Wind regions: 2030={len(wind_map_2030)}, 2035={len(wind_map_2035)}, 2050={len(wind_map_2050)}")
    print(f"   ✓ PV regions:   2030={len(pv_map_2030)}, 2035={len(pv_map_2035)}, 2050={len(pv_map_2050)}")
else:
    wind_map_2030 = wind_map_2035 = wind_map_2050 = {}
    pv_map_2030   = pv_map_2035   = pv_map_2050   = {}
    print(f"   ⚠ Regional CAPEX file not found: {regional_cost_path}. Using defaults.", flush=True)

# =====================================================
# 2) Scan scenario CF time series
# =====================================================
cf_files = sorted(glob.glob(os.path.join(cf_dir, "median_points_cf_timeseries_*.csv")))
print(f"▶ Found {len(cf_files)} scenario files in {cf_dir}.", flush=True)
if not cf_files:
    raise FileNotFoundError(f"No median_points_cf_timeseries_*.csv found in {cf_dir}")

# ========= scenario helpers =========
def normalize_scenario(name: str) -> str:
    return re.sub(r'(?i)Regional', 'Region', name)

def classify_scenario_for_cost(name: str) -> str:
    s = name.lower()
    if re.search(r'region(al)?re', s):   return 'RE'
    if re.search(r'region(al)?wind', s): return 'WIND'
    return 'OTHER'

def scenario_output_folder(name: str) -> str:
    typ = classify_scenario_for_cost(name)
    if typ == 'RE':   return 'RegionRE'
    if typ == 'WIND': return 'RegionWind'
    return normalize_scenario(name)

def pick_year_tag_from_filename(fn: str) -> str:
    """
    Return '2030' / '2040' / '2050' / '2035' (default)
    If no explicit year tag is found, we will later treat it as 2040 for folder, and 2035 for cost maps.
    """
    s = fn.lower()
    if "2050" in s: return "2050"
    if "2040" in s: return "2040"
    if "2030" in s: return "2030"
    return "2035"

def year_folder_tag(year_tag: str) -> str:
    """
    Folder dispatch:
      - If explicit in name: 2030 -> Year_2030, 2040 -> Year_2040, 2050 -> Year_2050
      - If not explicit ('2035' here), default to Year_2040 per requirement.
    """
    if year_tag in ("2030", "2040", "2050"):
        return year_tag
    return "2040"  # default folder when no year in filename

def pick_cost_maps(year_tag: str):
    # 2040 reuses 2035 mapping
    if year_tag == "2050":
        return wind_map_2050, pv_map_2050
    if year_tag == "2030":
        return wind_map_2030, pv_map_2030
    if year_tag == "2040":
        return wind_map_2035, pv_map_2035
    return wind_map_2035, pv_map_2035

# =====================================================
# 3) Parameter sets
# =====================================================
params_base = {
    "target_methanol_production": 10e3,
    "battery_energy_cost": 100,
    "battery_storage_cost": 100,
    "battery_charge_efficiency": 0.985,
    "battery_discharge_efficiency": 0.975,
    "battery_charge_penalty": 1e-5,
    "battery_discharge_penalty": 0,
    "electrolyzer_cost_AE": 250,
    "electrolyzer_cost_PEM": 500,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "methanol_synthesis_cost": 5700,
    "hydrogen_storage_cost": 500,
    "hydrogen_storage_elec": 0,
    "hydrogen_charge_penalty": 1e-5,
    "hydrogen_storage_efficiency": 1.0,
    "methanol_storage_cost_param": 0.056,
    "methanol_charge_penalty": 1e-5,
    "discount_rate": 0.07,
    "battery_lifetime": 20,
    "wind_lifetime": 30,
    "pv_lifetime": 40,
    "electrolyzer_lifetime_AE": 25,
    "electrolyzer_lifetime_PEM": 25,
    "methanol_synthesis_lifetime": 30,
    "hydrogen_storage_lifetime": 30,
    "methanol_storage_lifetime": 30,
    "heat_pump_cost": 950,
    "heat_pump_cop": 3.2,
    "heat_pump_lifetime": 20,
    "thermal_storage_cost": 30,
    "thermal_storage_efficiency": 0.98,
    "thermal_storage_lifetime": 30,
    "dac_cost": 4000,
    "dac_elec_per_kg": 0.25,
    "dac_heat_per_kg": 1.0,
    "dac_lifetime": 30,
    "co2_compressor_cost": 365,
    "co2_compression_energy": 0.1,
    "co2_compressor_lifetime": 20,
    "co2_storage_cost": 24,
    "co2_storage_lifetime": 30,
    "Flex_up": 0.02, "Flex_mid": 0.1, "Flex_down": 0.2,
    "surplus_penalty": 1e-5,
    "aggregation_mode": "daily",
    # FTL/ammonia placeholders
    "target_ftl_production": 10e3,
    "ftl_synthesis_cost": 9400,
    "ftl_synthesis_lifetime": 25,
    "ftl_hydrogen_ratio": 0.49,
    "ftl_co2_ratio": 3.6,
    "ftl_electricity_ratio": 0.3,
    "ftl_storage_cost_param": 0.056,
    "ftl_storage_lifetime": 30,
    "ftl_charge_penalty": 1e-5,
    "target_ammonia_production": 10e3,
    "ammonia_synthesis_cost": 6500,
    "ammonia_synthesis_lifetime": 30,
    "ammonia_storage_cost_param": 0.59,
    "ammonia_storage_lifetime": 30,
}

params_2030 = dict(params_base)
params_2030.update({
    "battery_energy_cost": 200,
    "battery_storage_cost": 200,
    "electrolyzer_cost_AE": 430,
    "electrolyzer_cost_PEM": 650,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "ftl_synthesis_cost": 22000,
    "ftl_hydrogen_ratio": 0.51,
    "ftl_co2_ratio": 3.9,
    "ftl_electricity_ratio": 0.3,
    "methanol_synthesis_cost": 6500,
    "ammonia_synthesis_cost": 8000,
    "hydrogen_storage_cost": 900,
    "dac_cost": 7800,
    "dac_elec_per_kg": 0.31,
    "dac_heat_per_kg": 1.3,
    "heat_pump_cost": 1000,
})

params_2050 = dict(params_base)
params_2050.update({
    "battery_energy_cost": 50,
    "battery_storage_cost": 50,
    "electrolyzer_cost_AE": 250,
    "electrolyzer_cost_PEM": 350,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "ftl_synthesis_cost": 9400,
    "ftl_hydrogen_ratio": 0.48,
    "ftl_co2_ratio": 3.3,
    "ftl_electricity_ratio": 0.3,
    "methanol_synthesis_cost": 5200,
    "ammonia_synthesis_cost": 5000,
    "hydrogen_storage_cost": 300,
    "dac_cost": 3000,
    "dac_elec_per_kg": 0.22,
    "dac_heat_per_kg": 1.0,
    "heat_pump_cost": 900,
})

def select_params_by_year(year_tag: str) -> dict:
    if str(year_tag) == "2050":
        return params_2050
    if str(year_tag) == "2030":
        return params_2030
    # 2040 / 2035 / others → base (you can define params_2040 if needed)
    return params_base


# ========== Model router (expects your optimize_* functions) ==========
def create_model_for_product(prod_key: str,
                             wind_output: np.ndarray,
                             pv_output: np.ndarray,
                             wind_cost: float,
                             pv_cost: float,
                             p: dict):
    prod_key = (prod_key or "methanol").lower()
    if prod_key == "methanol":
        return optimize_methanol_system(
            wind_output, pv_output,
            p["target_methanol_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["methanol_synthesis_cost"], p["hydrogen_storage_cost"],
            p["hydrogen_storage_elec"], p["hydrogen_charge_penalty"],
            p["hydrogen_storage_efficiency"], p["methanol_storage_cost_param"],
            p["methanol_charge_penalty"], p["discount_rate"],
            p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["methanol_synthesis_lifetime"], p["hydrogen_storage_lifetime"],
            p["methanol_storage_lifetime"], p["heat_pump_cost"],
            p["heat_pump_cop"], p["heat_pump_lifetime"],
            p["thermal_storage_cost"], p["thermal_storage_efficiency"],
            p["thermal_storage_lifetime"], p["dac_cost"],
            p["dac_elec_per_kg"], p["dac_heat_per_kg"], p["dac_lifetime"],
            p["sorbent_cost"],  # sorbent_cost
            p["co2_compressor_cost"], p["co2_compression_energy"],
            p["co2_compressor_lifetime"], p["co2_storage_cost"],
            p["co2_storage_lifetime"], p["Flex_up"], p["Flex_mid"],
            p["Flex_down"], p["surplus_penalty"], p["aggregation_mode"]
        )
    if prod_key == "ftl":
        return optimize_ftl_system(
            wind_output, pv_output, p["target_ftl_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["ftl_synthesis_cost"], p["ftl_synthesis_lifetime"],
            p["ftl_hydrogen_ratio"], p["ftl_co2_ratio"], p["ftl_electricity_ratio"],
            p["hydrogen_storage_cost"], p["hydrogen_storage_elec"],
            p["hydrogen_charge_penalty"], p["hydrogen_storage_efficiency"],
            p["hydrogen_storage_lifetime"], p["ftl_storage_cost_param"],
            p["ftl_storage_lifetime"], p["ftl_charge_penalty"],
            p["heat_pump_cost"], p["heat_pump_cop"], p["heat_pump_lifetime"],
            p["thermal_storage_cost"], p["thermal_storage_efficiency"], p["thermal_storage_lifetime"],
            p["dac_cost"], p["dac_elec_per_kg"], p["dac_heat_per_kg"], p["dac_lifetime"],
            p["sorbent_cost"],  # sorbent_cost
            p["co2_compressor_cost"], p["co2_compression_energy"], p["co2_compressor_lifetime"],
            p["co2_storage_cost"], p["co2_storage_lifetime"],
            p["discount_rate"], p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["Flex_up"], p["Flex_mid"], p["Flex_down"], p["surplus_penalty"], p["aggregation_mode"]
        )
    if prod_key == "ammonia":
        return optimize_ammonia_system(
            wind_output, pv_output, p["target_ammonia_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["ammonia_synthesis_cost"],
            p["hydrogen_storage_cost"], p["hydrogen_storage_elec"],
            p["hydrogen_charge_penalty"], p["hydrogen_storage_efficiency"],
            p["ammonia_storage_cost_param"], p["discount_rate"],
            p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["ammonia_synthesis_lifetime"], p["hydrogen_storage_lifetime"],
            p["ammonia_storage_lifetime"],
            p["Flex_up"], p["Flex_mid"], p["Flex_down"],
            p["surplus_penalty"], p["aggregation_mode"]
        )
    raise ValueError(f"Unsupported product: {prod_key}")

# ========== Main ==========
total_groups = solved_groups = skipped_groups = 0
early_stop = False
pat = re.compile(r"median_points_cf_timeseries_(?P<scenario>.+)\.csv$", re.IGNORECASE)

for fi, cf_path in enumerate(cf_files, 1):
    m = pat.search(os.path.basename(cf_path))
    if not m:
        print(f"⚠ Unable to parse scenario from: {cf_path} — skipped.", flush=True); continue

    scen_raw = m.group('scenario')
    if ONLY_ONE_SCENARIO and (re.search(ONLY_ONE_SCENARIO, scen_raw, flags=re.IGNORECASE) is None):
        print(f"· Skip non-target scenario: {scen_raw}", flush=True); continue

    year_tag  = pick_year_tag_from_filename(scen_raw)   # '2030' / '2040' / '2050' / '2035'
    wind_map, pv_map = pick_cost_maps(year_tag)
    params = select_params_by_year(year_tag)

    scen_norm   = normalize_scenario(scen_raw)
    scen_type   = classify_scenario_for_cost(scen_raw)
    scen_folder = scenario_output_folder(scen_raw)

    # folder dispatch: if year tag is missing ('2035' here), default to 2040 folder
    year_folder = year_folder_tag(year_tag)

    print(f"\n================ Scenario [{fi}/{len(cf_files)}]: {scen_raw} → Title[{scen_norm}] | Output[{scen_folder}] | YearTag: {year_tag} | YearFolder: {year_folder} | Type: {scen_type} ================", flush=True)

    scen_out = os.path.join(out_root, f"Year_{year_folder}", scen_folder)
    os.makedirs(scen_out, exist_ok=True)

    cf_df = pd.read_csv(cf_path)
    print(f"   - rows read: {len(cf_df)}", flush=True)

    if 'scenario' not in cf_df.columns: cf_df['scenario'] = scen_norm
    if 'product'  not in cf_df.columns: cf_df['product']  = 'methanol'

    group_cols = [c for c in ['scenario','product','continent','cluster','latitude','longitude','region'] if c in cf_df.columns]
    print(f"   - group-by keys: {group_cols}", flush=True)

    groups_list = list(cf_df.groupby(group_cols, sort=False))
    print(f"   - number of groups: {len(groups_list)}", flush=True)

    for gi, (keys, group) in enumerate(groups_list, 1):
        key_map = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        if ONLY_ONE_NODE and (not (
            all(k in key_map for k in ONLY_ONE_NODE) and all(str(key_map[k])==str(v) for k,v in ONLY_ONE_NODE.items())
        )):
            continue

        total_groups += 1
        cont = key_map.get('continent', 'Unknown')
        cl   = key_map.get('cluster', 'X')
        prod_key = (key_map.get('product') or 'methanol').lower()
        prod = prod_key.upper()
        lat  = key_map.get('latitude'); lon  = key_map.get('longitude')
        region_code = int(key_map.get('region')) if key_map.get('region') is not None else None

        try:
            lat_f = float(lat) if lat is not None else float('nan')
            lon_f = float(lon) if lon is not None else float('nan')
        except Exception:
            lat_f, lon_f = float('nan'), float('nan')

        print(f"\n   ▶ Group {gi}/{len(groups_list)} | {prod} | {cont} | cluster={cl} | "
              f"lat={lat_f:.2f} lon={lon_f:.2f} | region_code={region_code}", flush=True)

        group = group.sort_values('hour')
        wind_output = group['CF_wind_avg'].to_numpy()
        pv_output   = group['CF_pv_avg'].to_numpy()
        T = len(wind_output)
        print(f"     · time series length={T}", flush=True)

        # CAPEX mapping (PV=225 for RegionWind/RegionalWind)
        wind_cost_i = wind_map.get(region_code, 800)
        if scen_type == 'WIND':
            pv_cost_i = 225; pv_cost_note = "(uniform 225)"
        else:
            # if no explicit year (2035), we still display 2035 mapping in note
            pv_cost_i = pv_map.get(region_code, 350); pv_cost_note = f"({year_tag if year_tag!='2035' else '2035'} ATB mapping)"
        print(f"     · CAPEX: wind={wind_cost_i:.1f} | pv={pv_cost_i:.1f} {pv_cost_note}", flush=True)
        print(f"     · Params: params_{year_tag if year_tag in ['2030','2050'] else 'base'}", flush=True)

        # Build & solve
        try:
            print("     · building model...", flush=True)
            model = create_model_for_product(prod_key, wind_output, pv_output, wind_cost_i, pv_cost_i, params)
        except NameError as e:
            print(f"     ✗ build failed (missing optimize_*): {e}", flush=True); skipped_groups += 1; continue
        except NotImplementedError as e:
            print(f"     ✗ build failed (not implemented): {e}", flush=True); skipped_groups += 1; continue
        except Exception as e:
            print(f"     ✗ build failed (exception): {e}", flush=True); skipped_groups += 1; continue

        print("     · optimizing...", flush=True)
        model.optimize()
        if model.status != GRB.OPTIMAL:
            print(f"     ✗ not optimal, status={model.status} — skipped.", flush=True)
            skipped_groups += 1; continue

        solved_groups += 1
        wind_capacity = model.getVarByName('wind_capacity').x
        pv_capacity   = model.getVarByName('pv_capacity').x
        print(f"     ✓ optimal: wind_cap={wind_capacity:.2f} kW, pv_cap={pv_capacity:.2f} kW", flush=True)

        # ========= yearly series =========
        series_wind = wind_output * wind_capacity
        series_pv   = pv_output   * pv_capacity

        series_ae    = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["electrolyzer_AE"], T), T)
        series_pem   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["electrolyzer_PEM"], T), T)
        series_syn   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["synthesis_power"], T), T)
        series_dac   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["dac_elec"], T), T)
        series_hpout = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["heat_pump_output"], T), T)
        series_surplus = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["surplus"], T), T)

        series_b_ch   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_charge"], T), T)
        series_b_dis  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_discharge"], T), T)
        series_h2_ch  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_charge"], T), T)
        series_h2_dis = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_discharge"], T), T)
        series_prod_ch  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_charge"], T), T)
        series_prod_dis = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_discharge"], T), T)

        series_b_balance   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_balance"], T), T)
        series_h2_balance  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_storage_balance"], T), T)
        series_p_balance   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_storage_balance"], T), T)

        bat_cap = get_first_var_value(model, VAR_CAPACITY["battery"])
        h2_cap  = get_first_var_value(model, VAR_CAPACITY["hydrogen"])
        prod_cap= get_first_var_value(model, VAR_CAPACITY["product"])

        # SOC: divide balances by capacity if available; otherwise normalize by own max
        series_b_soc   = series_b_balance / (bat_cap  if bat_cap  else max(np.nanmax(series_b_balance),1.0))
        series_h2_soc  = series_h2_balance/ (h2_cap   if h2_cap   else max(np.nanmax(series_h2_balance),1.0))
        series_prod_soc= series_p_balance / (prod_cap if prod_cap else max(np.nanmax(series_p_balance),1.0))

        syn_elec_label = 'MeOH synthesis (elec)' if prod_key == 'methanol' else ('FTL synthesis (elec)' if prod_key == 'ftl' else 'Synthesis (elec)')

        # ========= (new) shift to local time before plotting =========
        if ADJUST_TO_LOCAL_TZ and np.isfinite(lat_f) and np.isfinite(lon_f)):
            year_for_tz = int(year_folder) if str(year_folder).isdigit() else 2040
            offset_hours = infer_utc_offset_hours(lat_f, lon_f, year_for_tz)
            print(f"     · local time offset: UTC{offset_hours:+d} (from lat/lon, year {year_for_tz})", flush=True)

            series_map = {
                "series_wind": series_wind,
                "series_pv":   series_pv,
                "series_ae":   series_ae,
                "series_pem":  series_pem,
                "series_syn":  series_syn,
                "series_dac":  series_dac,
                "series_hpout": series_hpout,
                "series_b_ch": series_b_ch,
                "series_b_dis": series_b_dis,
                "series_h2_ch": series_h2_ch,
                "series_h2_dis": series_h2_dis,
                "series_surplus": series_surplus, 
                "series_prod_ch": series_prod_ch,
                "series_prod_dis": series_prod_dis,
                "series_b_balance": series_b_balance,
                "series_h2_balance": series_h2_balance,
                "series_p_balance": series_p_balance,
                "series_b_soc": series_b_soc,
                "series_h2_soc": series_h2_soc,
                "series_prod_soc": series_prod_soc,
            }
            roll_series_dict(series_map, offset_hours, list(series_map.keys()))
            (series_wind, series_pv, series_ae, series_pem, series_syn, series_dac, series_hpout,
             series_b_ch, series_b_dis, series_h2_ch, series_h2_dis, series_surplus,series_prod_ch, series_prod_dis,
             series_b_balance, series_h2_balance, series_p_balance,
             series_b_soc, series_h2_soc, series_prod_soc) = (
                series_map["series_wind"], series_map["series_pv"], series_map["series_ae"],
                series_map["series_pem"], series_map["series_syn"], series_map["series_dac"], series_map["series_hpout"],
                series_map["series_b_ch"], series_map["series_b_dis"], series_map["series_h2_ch"], series_map["series_h2_dis"], series_map["series_surplus"],
                series_map["series_prod_ch"], series_map["series_prod_dis"],
                series_map["series_b_balance"], series_map["series_h2_balance"], series_map["series_p_balance"],
                series_map["series_b_soc"], series_map["series_h2_soc"], series_map["series_prod_soc"]
            )
        else:
            print("     · timezone adjustment skipped (missing coords or disabled).", flush=True)

        # ========= Annual heatmap =========
        title_prefix = f"{scen_norm} | {prod} {cont} cluster {cl} ({lat_f:.2f},{lon_f:.2f})"
        if   prod_key == 'methanol': prod_soc_label = 'MeOH storage SOC'
        elif prod_key == 'ftl':      prod_soc_label = 'FTL storage SOC'
        elif prod_key == 'ammonia':  prod_soc_label = 'Ammonia storage SOC'
        else:                        prod_soc_label = 'Product storage SOC'

        annual_items = [
            ("Wind (generation)",    series_wind),
            ("PV (generation)",      series_pv),
            ("Electrolyzers (AE+PEM)", series_ae + series_pem),
            (syn_elec_label,         series_syn),
            ("DAC (electricity)",    series_dac),
            ("Battery SOC",          series_b_soc),
            (f"{H2} storage SOC",    series_h2_soc),
            (prod_soc_label,         series_prod_soc),
        ]
        out_heat = os.path.join(scen_out, f"{prod_key}_{cont}_cluster{cl}_annual_heatmap_EN.png")
        ok = plot_annual_heatmap_panel(annual_items, title_prefix, out_heat, cmap="YlGnBu_r")
        if ok:
            print(f"       ✓ saved annual heatmap: {out_heat}", flush=True)
        else:
            print("       · annual heatmap skipped (no valid series).", flush=True)

        # ========= Weekly supply–demand stacks =========
        syn_elec_ratio = 0.1  if prod_key == 'methanol' else params.get('ftl_electricity_ratio', 0.1)
        heat_pump_cop  = params["heat_pump_cop"]

        for season, week_idx in season_weeks.items():
            week_start = 168 * week_idx
            week_end   = min(week_start + 168, T)
            if week_start >= T:
                print(f"     · {season}: out of range — skipped.", flush=True); continue
            x = np.arange(week_end - week_start)
            print(f"     · plotting {season}: hours {week_start}–{week_end-1}", flush=True)

            # supply (positive)
            supply_wind = series_wind[week_start:week_end]
            supply_pv   = series_pv[week_start:week_end]
            supply_batt = series_b_dis[week_start:week_end]

            # demand (negative)
            demand_ae   = series_ae[week_start:week_end]
            demand_pem  = series_pem[week_start:week_end]
            demand_syn_elec = series_syn[week_start:week_end] * syn_elec_ratio
            demand_bat_ch   = series_b_ch[week_start:week_end]
            demand_h2       = np.zeros_like(demand_ae)
            demand_hp       = (series_hpout[week_start:week_end] / max(heat_pump_cop,1e-6))
            demand_dac      = series_dac[week_start:week_end]
            demand_surplus  = series_surplus[week_start:week_end]
            demand_co2      = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["co2_comp"], T), T)[week_start:week_end]

            fig, ax = plt.subplots(figsize=(15, 8))

            # supply bars
            ax.bar(x, supply_wind, label='Wind', color=wind_color, edgecolor='black', linewidth=0.3)
            ax.bar(x, supply_pv, bottom=supply_wind, label='PV', color=pv_color, edgecolor='black', linewidth=0.3)
            ax.bar(x, supply_batt, bottom=supply_wind + supply_pv,
                   label='Battery discharge', color=battery_color, edgecolor='black', linewidth=0.3)

            # demand bars (negative)
            b = -(demand_ae)
            ax.bar(x, b, label='AE', color=ae_color, edgecolor='black', linewidth=0.3)
            ax.bar(x, -demand_pem, bottom=b, label='PEM', color=pem_color, edgecolor='black', linewidth=0.3)
            b2 = -(demand_ae + demand_pem)
            ax.bar(x, -demand_syn_elec, bottom=b2, label='Synthesis (elec)', color=synth_color, edgecolor='black', linewidth=0.3)
            b3 = -(demand_ae + demand_pem + demand_syn_elec)
            ax.bar(x, -demand_bat_ch, bottom=b3, label='Battery charge', color=bat_charge_color, edgecolor='black', linewidth=0.3)
            b4 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch)
            # ax.bar(x, -demand_h2, bottom=b4, label=f'{H2} storage (elec)', color=hydrogen_color, edgecolor='black', linewidth=0.3)
            b5 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2)
            ax.bar(x, -demand_hp, bottom=b5, label='Heat pump', color=heatpump_color, edgecolor='black', linewidth=0.3)
            b6 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp)
            ax.bar(x, -demand_dac, bottom=b6, label='DAC (elec)', color=dac_color, edgecolor='black', linewidth=0.3)
            b7 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp + demand_dac)
            ax.bar(x, -demand_surplus, bottom=b7, label='Curtailment', color=surplus_color, edgecolor='black', linewidth=0.3)
            # b8 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp + demand_dac + demand_surplus)
            # ax.bar(x, -demand_co2, bottom=b8, label=f'{CO2} compression', color=co2_color, edgecolor='black', linewidth=0.3)

            ax.set_xlim(0, len(x)); ax.axhline(0, color='k', linewidth=0.8)
            ax.set_xlabel('Hour within week'); ax.set_ylabel('Power (kW)')
            ax.set_title(f'{title_prefix} — {season} typical week')

            pos_sum = supply_wind + supply_pv + supply_batt
            neg_sum = (demand_ae + demand_pem + demand_syn_elec + demand_bat_ch +
                       demand_h2 + demand_hp + demand_dac + demand_surplus + demand_co2)
            set_y_margin(ax, pos_sum, neg_sum, Y_MARGIN_FRAC_POWER)

            # legends fixed at upper-right, 2 columns
            leg = ax.legend(loc='upper right', ncol=2)
            leg.get_frame().set_facecolor("white"); leg.get_frame().set_edgecolor("0.8")
            plt.tight_layout()
            out_path = os.path.join(scen_out, f'{prod_key}_{cont}_cluster{cl}_{season}_EN.png')
            plt.savefig(out_path, dpi=300); plt.close(fig)
            print(f"       ✓ saved: {out_path}", flush=True)

            # ===== H2 stack (kg/h) =====
            supply_h2_prod = (series_ae + series_pem) * 3.6 / 120.0  # your conversion
            supply_h2_dis  = series_h2_dis
            demand_syn_h2 = series_syn * (
                params.get('methanol_hydrogen_ratio') if prod_key == 'methanol' 
                else params.get('ammonia_hydrogen_ratio') if prod_key == 'ammonia'
                else params.get('ftl_hydrogen_ratio')
            )
            demand_h2_chg  = series_h2_ch

            sh = slice(week_start, week_end); xH = np.arange(week_end - week_start)
            fig2, ax2 = plt.subplots(figsize=(15, 6))
            ax2.bar(xH, supply_h2_prod[sh], label=f'{H2} production', color='#4F6BED', edgecolor='black', linewidth=0.3)
            ax2.bar(xH, supply_h2_dis[sh], bottom=supply_h2_prod[sh], label=f'{H2} discharge', color='#7DA2F8', edgecolor='black', linewidth=0.3)
            ax2.bar(xH, -demand_syn_h2[sh], label=f'{H2} for synthesis', color='#925E9F', edgecolor='black', linewidth=0.3)
            ax2.bar(xH, -demand_h2_chg[sh], bottom=-demand_syn_h2[sh], label=f'{H2} charge', color='#D94A38', edgecolor='black', linewidth=0.3)
            ax2.axhline(0, color='k', linewidth=0.8)
            ax2.set_xlim(0, len(xH)); ax2.set_xticks(np.arange(0, len(xH)+1, 24))
            ax2.set_xlabel('Hour within week'); ax2.set_ylabel('Hydrogen flow (kg/h)')
            ax2.set_title(f'{title_prefix} — {H2} supply–demand ({season})')

            set_y_margin(ax2, supply_h2_prod[sh] + supply_h2_dis[sh], demand_syn_h2[sh] + demand_h2_chg[sh], Y_MARGIN_FRAC_H2)
            leg2 = ax2.legend(loc='upper right', ncol=2)
            leg2.get_frame().set_facecolor("white"); leg2.get_frame().set_edgecolor("0.8")
            plt.tight_layout()
            out_path = os.path.join(scen_out, f'{prod_key}_{cont}_cluster{cl}_{season}_h2_stack_EN.png')
            plt.savefig(out_path, dpi=300); plt.close(fig2)
            print(f"       ✓ saved: {out_path}", flush=True)

            # ===== Shadow prices (optional) =====
            try:
                elec_shadow = np.array([model.getConstrByName(f"power_balance_{t}").Pi for t in range(week_start, week_end)])
                fig3, ax3 = plt.subplots(figsize=(15, 4))
                ax3.plot(np.arange(week_end-week_start), elec_shadow, label='Power shadow price', linewidth=2)
                ax3.axhline(0, color='k', linewidth=0.8)
                ax3.set_xlim(0, week_end-week_start); ax3.set_xticks(np.arange(0, week_end-week_start+1, 24))
                ax3.set_xlabel('Hour within week'); ax3.set_ylabel('Price ($/kWh)')
                ax3.set_title(f'{title_prefix} — Power shadow price ({season})')
                leg3 = ax3.legend(loc='upper right')
                leg3.get_frame().set_facecolor("white"); leg3.get_frame().set_edgecolor("0.8")
                plt.tight_layout()
                out_path = os.path.join(scen_out, f'{prod_key}_{cont}_cluster{cl}_{season}_elec_shadow_EN.png')
                plt.savefig(out_path, dpi=300); plt.close(fig3)
                print(f"       ✓ saved: {out_path}", flush=True)
            except Exception:
                print("       · power shadow price: constraint not found — skipped.", flush=True)

            try:
                h2_shadow = np.array([model.getConstrByName(f"hydrogen_PD_balance_{t}").Pi for t in range(week_start, week_end)])
                fig4, ax4 = plt.subplots(figsize=(15, 4))
                ax4.plot(np.arange(week_end-week_start), h2_shadow, label=f'{H2} shadow price', linewidth=2)
                ax4.axhline(0, color='k', linewidth=0.8)
                ax4.set_xlim(0, week_end-week_start); ax4.set_xticks(np.arange(0, week_end-week_start+1, 24))
                ax4.set_xlabel('Hour within week'); ax4.set_ylabel(f'Price ($/kg {H2})')
                ax4.set_title(f'{title_prefix} — {H2} shadow price ({season})')
                leg4 = ax4.legend(loc='upper right')
                leg4.get_frame().set_facecolor("white"); leg4.get_frame().set_edgecolor("0.8")
                plt.tight_layout()
                out_path = os.path.join(scen_out, f'{prod_key}_{cont}_cluster{cl}_{season}_h2_shadow_EN.png')
                plt.savefig(out_path, dpi=300); plt.close(fig4)
                print(f"       ✓ saved: {out_path}", flush=True)
            except Exception:
                print("       · hydrogen shadow price: constraint not found — skipped.", flush=True)

        if ONLY_ONE_NODE and STOP_AFTER_FIRST_NODE:
            early_stop = True
            print("\n🛑 Single-node test finished. Early stop.", flush=True)
            break

    if early_stop: break

# Summary
print("\n============== Summary ==============")
print(f"Total groups: {total_groups}")
print(f"Solved optimal: {solved_groups}")
print(f"Skipped groups: {skipped_groups}")
print("✅ All done.")


In [ ]:
# -*- coding: utf-8 -*-
"""
批量生成 8760 典型周供需图 + 年度热力图（中文版）
- 自动适配文件名中的 2030/2040/2050：
  * 风/光成本分别读取 RegionalCost.csv 中的 2030ATB / 2035ATB(用于2040) / 2050ATB 列；
  * 参数集自动切换为 params_2030 / params_2050（2040 暂用 params_base，或自行扩展 params_2040）。
- Region/Regional 归并（RegionRE/RegionalRE；RegionWind/RegionalWind）
- RegionWind/RegionalWind：PV 成本统一 225；其余用区域映射（按年列）
- Nature Climate Change 风格（近似），数学下标（H₂/CO₂）
- SOC 无模型直接输出时：按充/放电自行计算；若能读取容量则按容量归一化，否则按自身范围归一化到[0,1]
- 年度热力图：Battery 改为 SOC，新增 H₂ Storage SOC 与 Methanol/FTL/Ammonia Storage SOC
"""

import os
import re
import glob
from typing import Optional, Dict, Iterable, List, Tuple

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors, cm
import gurobipy as gp
from gurobipy import GRB

from datetime import datetime
try:
    from timezonefinder import TimezoneFinder  # pip install timezonefinder
except Exception:
    TimezoneFinder = None
try:
    from zoneinfo import ZoneInfo  # Python 3.9+
except Exception:
    ZoneInfo = None


# =====================================================
# 0) 路径与全局设置
# =====================================================
cf_dir   = r"D:\OneDrive\POTSED\03 Cluster"
out_root = r"D:\OneDrive\POTSED\8760Supply&Demand\Global_cn"
os.makedirs(out_root, exist_ok=True)

# 区域资本成本（风/光）
regional_cost_path = r"D:\OneDrive\POTSED\DataOutput\RegionalCost.csv"

# ========== 仅测试开关（可选） ==========
ONLY_ONE_SCENARIO = "RegionRE_LowAE_MidFTL"
ONLY_ONE_NODE = None
STOP_AFTER_FIRST_NODE: bool = True

# ========= y 轴留白（方法3）可调项 =========
Y_MARGIN_FRAC_POWER: float = 0.08
Y_MARGIN_FRAC_H2: float    = 0.08

# =====================================================
# 画风：NCC + 中文字体
# =====================================================
def apply_cn_style():
    # 优先中文字体，自动回退
    font_candidates = [
        "Microsoft YaHei", "SimHei", "Noto Sans CJK SC",
        "Source Han Sans SC", "Alibaba PuHuiTi 2.0",
        "WenQuanYi Zen Hei", "Arial Unicode MS", "DejaVu Sans"
    ]
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": font_candidates,
        "axes.unicode_minus": False,  # 解决负号显示
        "pdf.fonttype": 42, "ps.fonttype": 42,
        "text.usetex": False, "mathtext.fontset": "stix", "mathtext.default": "rm",
        "axes.facecolor": "white", "axes.edgecolor": "black", "axes.linewidth": 0.8,
        "axes.labelsize": 9, "axes.titlesize": 10, "axes.titlepad": 6,
        "xtick.labelsize": 8, "ytick.labelsize": 8,
        "xtick.direction": "in", "ytick.direction": "in",
        "xtick.major.size": 3, "ytick.major.size": 3,
        "xtick.major.width": 0.8, "ytick.major.width": 0.8,
        "grid.linestyle": (0, (3, 3)), "grid.color": "0.85", "grid.linewidth": 0.6,
        "legend.fontsize": 8, "legend.frameon": True, "legend.framealpha": 0.9, "legend.edgecolor": "0.8",
        "lines.linewidth": 1.2,
    })
apply_cn_style()

# 典型周（0 为第一周）
season_weeks = {
    '冬季': 1,    # 1月首周
    '春季': 14,   # 4月首周
    '夏季': 27,   # 7月首周
    '秋季': 40    # 10月首周
}

# 颜色
wind_color       = '#4F81BD'
pv_color         = '#C0504D'
battery_color    = '#9BBB59'
ae_color         = '#8064A2'
pem_color        = '#F79646'
synth_color      = '#4BACC6'
bat_charge_color = '#F05542'
hydrogen_color   = '#8F5CC1'
heatpump_color   = '#4C9F70'
dac_color        = '#D99694'
surplus_color    = '#B0B0B0'
co2_color        = '#696969'

# 数学下标
H2  = r'$H_2$'
CO2 = r'$CO_2$'

# ====================工具：时序调整========================
def infer_utc_offset_hours(lat: float, lon: float, year: int = 2040) -> int:
    """
    返回该点在给定年份中段（7月1日中午）的本地UTC时差（单位：小时，四舍五入到整数）。
    优先使用真实时区（timezonefinder + zoneinfo），失败则经度近似 lon/15。
    """
    # 真实时区（若可用）
    if TimezoneFinder is not None and ZoneInfo is not None:
        try:
            tf = TimezoneFinder()
            tzname = tf.timezone_at(lat=lat, lng=lon)
            if tzname:
                dt = datetime(year, 7, 1, 12, 0, 0)
                tz = ZoneInfo(tzname)
                offset_sec = tz.utcoffset(dt).total_seconds()
                return int(round(offset_sec / 3600.0))
        except Exception:
            pass

    # 近似：经度 / 15 取整，并限制到 [-12, +14]
    approx = int(round(lon / 15.0))
    return max(-12, min(14, approx))


def roll_series_dict(d: dict, shift_hours: int, keys: list[str]) -> None:
    """
    原地把 d 中指定 keys 的一维 numpy 序列循环平移 shift_hours。
    shift_hours>0 表示向右滚动（即“本地时间 = UTC + shift”）。
    """
    if not shift_hours:
        return
    for k in keys:
        if k in d and d[k] is not None:
            d[k] = np.roll(np.asarray(d[k], dtype=float), shift_hours)


# =================== 工具：y 轴留白 ===================
def set_y_margin(ax: plt.Axes, positive_stack: np.ndarray, negative_stack: np.ndarray, pad_frac: float):
    pos_max = float(np.nanmax(positive_stack)) if positive_stack.size else 0.0
    neg_max = float(np.nanmax(negative_stack)) if negative_stack.size else 0.0
    top, bottom = pos_max, -neg_max
    if top == bottom:
        span = abs(top) + 1.0
        ax.set_ylim(bottom - pad_frac * span, top + pad_frac * span)
    else:
        span = top - bottom
        pad = pad_frac * span
        ax.set_ylim(bottom - pad, top + pad)

# =================== 工具：批量抓取变量 ===================
VAR_TEMPLATES: Dict[str, List[str]] = {
    "electrolyzer_AE":     ["electrolyzer_power_AE[{t}]"],
    "electrolyzer_PEM":    ["electrolyzer_power_PEM[{t}]"],
    "synthesis_power":     ["ammonia_synthesis_power[{t}]", "methanol_synthesis_power[{t}]", "FTL_synthesis_power[{t}]"],
    "dac_elec":            ["dac_electricity[{t}]"],
    "heat_pump_output":    ["heat_pump_output[{t}]"],
    "battery_charge":      ["battery_charge[{t}]"],
    "battery_discharge":   ["battery_discharge[{t}]"],
    "hydrogen_charge":     ["hydrogen_charge[{t}]"],
    "hydrogen_discharge":  ["hydrogen_discharge[{t}]"],
    "product_charge":      ["methanol_storage_charge[{t}]", "FTL_storage_charge[{t}]",
                            "ammonia_storage_charge[{t}]", "product_storage_charge[{t}]"],
    "product_discharge":   ["methanol_storage_discharge[{t}]", "FTL_storage_discharge[{t}]",
                            "ammonia_storage_discharge[{t}]", "product_storage_discharge[{t}]"],
    "surplus":             ["surplus[{t}]"],
    "co2_comp":            ["co2_comp_elec[{t}]"],
    "battery_balance":     ["energy_balance[{t}]"],
    "hydrogen_storage_balance": ["hydrogen_storage_balance[{t}]"],
    "product_storage_balance":  ["methanol_storage_balance[{t}]","FTL_storage_balance[{t}]","ammonia_storage_balance[{t}]"]
}

VAR_CAPACITY: Dict[str, List[str]] = {
    "battery":  ["battery_storage_capacity"],
    "hydrogen": ["hydrogen_storage_capacity"],
    "product":  ["methanol_storage_capacity", "FTL_storage_capacity", "ammonia_storage_capacity"],
}

def get_series_by_templates(model: gp.Model, templates: Iterable[str], T: int) -> Optional[np.ndarray]:
    vals = np.zeros(T, dtype=float); found_any = False
    for t in range(T):
        v = None
        for temp in templates:
            var = model.getVarByName(temp.format(t=t))
            if var is not None:
                v = var.x; found_any = True; break
        vals[t] = float(v) if v is not None else 0.0
    return vals if found_any else None

def ensure_series(arr: Optional[np.ndarray], T: int, fill: float = 0.0) -> np.ndarray:
    if arr is None:
        return np.full(T, fill, dtype=float)
    a = np.asarray(arr, dtype=float).ravel()
    if a.size < T:
        out = np.full(T, fill, dtype=float); out[:a.size] = a; return out
    return a[:T]

def get_first_var_value(model: gp.Model, names: List[str]) -> Optional[float]:
    for nm in names:
        var = model.getVarByName(nm)
        if var is not None:
            try:
                return float(var.x)
            except Exception:
                pass
    return None

# ========== 年度热力图工具 ==========
def reshape_to_day(series: np.ndarray, hours_per_day: int = 24) -> np.ndarray:
    T = int(series.size)
    days = int(np.ceil(T / hours_per_day))
    mat = np.full((days, hours_per_day), np.nan, dtype=float)
    mat.flat[:T] = series
    return mat

def month_tick_positions(days: int) -> Tuple[List[int], List[str]]:
    month_days = [31,28,31,30,31,30,31,31,30,31,30,31]
    edges = np.cumsum([0]+month_days)
    centers = [(edges[i] + edges[i+1])//2 for i in range(12)]
    scale = days / 365.0
    pos = [int(c * scale) for c in centers if int(c*scale) < days]
    labels = ["1月","2月","3月","4月","5月","6月","7月","8月","9月","10月","11月","12月"][:len(pos)]
    return pos, labels

def normalize01(mat: np.ndarray) -> np.ndarray:
    m = np.nanmin(mat); M = np.nanmax(mat)
    if not np.isfinite(m) or not np.isfinite(M) or M <= m:
        return np.nan_to_num(mat, nan=0.0)
    return (mat - m) / (M - m)

def _truncate_cmap(base_cmap, lo=0.35, hi=0.65, n=256):
    new_colors = base_cmap(np.linspace(max(0.0, lo), min(1.0, hi), n))
    return colors.LinearSegmentedColormap.from_list(
        f"{getattr(base_cmap,'name','cmap')}_slice_{lo:.2f}_{hi:.2f}", new_colors
    )

def plot_annual_heatmap_panel(
    items,
    title_prefix,
    out_path,
    cmap: str = "YlGnBu",
    cmap_slice: tuple | None = (0., 0.95),
    norm_range: tuple | None = None,
    # 字体
    fs_title: int = 16,
    fs_rowlab: int = 28,
    fs_axislabel: int = 28,
    fs_tick: int = 28,
    fs_cbar: int =28,
    # 布局控制
    hspace: float  = 0.10,      # 行间距
    rowlab_x: float = 1.01,     # 行标题初始位置（轴坐标）
    # 先给个较宽的右边距，后面会自动再调
    left: float = 0.08, right: float = 0.90, top: float = 0.965, bottom: float = 0.12,
    # colorbar 位置（figure 坐标，绝对控制）
    cbar_left: float = 0.08, cbar_bottom: float = 0.00, cbar_width: float = 0.87, cbar_height: float = 0.02,
):
    import numpy as np
    from matplotlib import colors

    def _two_lines(s: str) -> str:
        if " (" in s: return s.replace(" (", "\n(")
        if "（" in s: return s.replace("（", "\n（")
        parts = s.split()
        if len(parts) >= 2: return " ".join(parts[:-1]) + "\n" + parts[-1]
        return s

    # 过滤无效项
    valid = []
    for lab, ser in items:
        if ser is None:
            continue
        arr = np.asarray(ser, dtype=float)
        if np.allclose(arr, 0.0, atol=1e-12):
            continue
        valid.append((lab, arr))
    if not valid:
        return False

    days = int(np.ceil(max(len(s) for _, s in valid) / 24.0))
    # x 轴月份刻度
    month_days = [31,28,31,30,31,30,31,31,30,31,30,31]
    edges = np.cumsum([0]+month_days)
    centers = [(edges[i] + edges[i+1])//2 for i in range(12)]
    scale = days / 365.0
    xticks = [int(c * scale) for c in centers if int(c*scale) < days]
    xlabels = ["1月","2月","3月","4月","5月","6月","7月","8月","9月","10月","11月","12月"][:len(xticks)]

    n = len(valid)
    fig = plt.figure(figsize=(15, 1.75 * n + 0.9))
    fig.text(0.5, 0.992, f"{title_prefix} — 年度归一化运行剖面",
             ha="center", va="top", fontsize=fs_title)

    # 先用一个保守的右边距；等放好文字后再“自适应”收/放
    gs = fig.add_gridspec(nrows=n, ncols=1, hspace=hspace)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom)

    # 配色
    base = plt.get_cmap(cmap)
    cmap = colors.LinearSegmentedColormap.from_list(
        f"{getattr(base,'name','cmap')}_slice", base(np.linspace(float(cmap_slice[0]), float(cmap_slice[1]), 256))
    ) if cmap_slice else base
    norm = colors.Normalize(vmin=0.0, vmax=1.0) if norm_range is None \
           else colors.Normalize(vmin=float(norm_range[0]), vmax=float(norm_range[1]), clip=True)

    axes = []
    right_texts = []
    last_img = None
    for i, (lab, ser) in enumerate(valid):
        ax = fig.add_subplot(gs[i, 0], sharex=axes[0] if axes else None)
        axes.append(ax)

        # 24h*days 矩阵 & 0-1 归一
        T = len(ser)
        mat = np.full((int(np.ceil(T/24.0)), 24), np.nan, float)
        mat.flat[:T] = ser
        mmin, mmax = np.nanmin(mat), np.nanmax(mat)
        mat_n = (mat - mmin) / (mmax - mmin) if np.isfinite(mmin) and np.isfinite(mmax) and mmax>mmin else np.nan_to_num(mat)

        last_img = ax.imshow(mat_n.T, origin='lower', aspect='auto', cmap=cmap, norm=norm)

        ax.set_yticks([0, 8, 16])
        ax.tick_params(axis='both', labelsize=fs_tick)
        ax.set_ylabel("小时", rotation=90, labelpad=10, fontsize=fs_axislabel-6)

        # —— 右侧竖排行标题（不裁剪）——
        t = ax.text(
            1.02, 0.5, lab,
            transform=ax.transAxes,
            rotation=90,
            rotation_mode='anchor',
            va='center',
            ha='center',
            multialignment='center',
            fontsize=fs_rowlab,
            linespacing=1.0,
            clip_on=False
        )
        right_texts.append(t)

        if i < n - 1:
            ax.tick_params(labelbottom=False)

    # 只在最后一行显示月份刻度
    axes[-1].set_xticks(xticks)
    axes[-1].set_xticklabels(xlabels, fontsize=fs_tick)
    axes[-1].set_xlabel("月份", labelpad=6, fontsize=fs_axislabel)

    # 自动扩大右边距（避免裁剪右侧竖排文字）
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    max_w_px = 0
    for t in right_texts:
        bb = t.get_window_extent(renderer=renderer)
        max_w_px = max(max_w_px, bb.width)
    fig_w_in = fig.get_size_inches()[0]
    fig_dpi  = fig.dpi
    pad_frac = 0.02
    margin_needed = (max_w_px / (fig_w_in * fig_dpi)) + pad_frac
    new_right = max(0.60, 1.0 - max(0.07, margin_needed))
    fig.subplots_adjust(right=new_right)

    # # —— 最底部 colorbar —— 
    # cax = fig.add_axes([0.08, 0.05, 0.87, 0.02])
    # cbar = fig.colorbar(last_img, cax=cax, orientation='horizontal')
    # cbar.ax.tick_params(labelsize=fs_tick)
    # cbar.set_label("归一化值", fontsize=fs_cbar, labelpad=2)

    fig.savefig(out_path, dpi=600)
    plt.close(fig)
    return True


# =====================================================
# 1) 读取区域成本映射（仅一次）
# =====================================================
print("▶ 读取区域资本成本表...", flush=True)
if os.path.exists(regional_cost_path):
    regional_cost_df = pd.read_csv(regional_cost_path)
    # 预制三列：2030/2035/2050（2040 复用 2035）
    def _make_map(tech: str, col: str):
        sub = regional_cost_df[
            (regional_cost_df["Type"] == tech) &
            (regional_cost_df["Cost Type 1"] == "Total installed cost")
        ]
        if col not in sub.columns:
            print(f"  ⚠ 成本列 {col} 不在表内，稍后将自动回退。")
            return {}
        return sub.set_index("region_code")[col].to_dict()

    wind_map_2030 = _make_map("Onshore Wind", "2030ATB")
    wind_map_2035 = _make_map("Onshore Wind", "2035ATB")
    wind_map_2050 = _make_map("Onshore Wind", "2050ATB")

    pv_map_2030   = _make_map("Solar PV", "2030ATB")
    pv_map_2035   = _make_map("Solar PV", "2035ATB")
    pv_map_2050   = _make_map("Solar PV", "2050ATB")

    print(f"   ✓ 风区数：2030={len(wind_map_2030)}, 2035={len(wind_map_2035)}, 2050={len(wind_map_2050)}")
    print(f"   ✓ 光区数：2030={len(pv_map_2030)}, 2035={len(pv_map_2035)}, 2050={len(pv_map_2050)}")
else:
    wind_map_2030 = wind_map_2035 = wind_map_2050 = {}
    pv_map_2030   = pv_map_2035   = pv_map_2050   = {}
    print(f"   ⚠ 未找到区域成本文件：{regional_cost_path}，将使用默认值。", flush=True)

# =====================================================
# 2) 扫描所有情景的 CF 时序文件
# =====================================================
import os, re, glob

pattern = os.path.join(cf_dir, "median_points_cf_timeseries_*.csv")
all_files = sorted(glob.glob(pattern))

cf_files = [
    p for p in all_files
    if not re.search(r"_(2021|2022)(?=\.csv$)", os.path.basename(p), flags=re.IGNORECASE)
]

print(f"▶ 在 {cf_dir} 找到 {len(cf_files)} 个情景文件（已排除 _2021/_2022）。", flush=True)
if not cf_files:
    raise FileNotFoundError(f"未在 {cf_dir} 找到符合条件的 CSV（已排除 _2021/_2022）")

# ========= 情景名工具 =========
def normalize_scenario(name: str) -> str:
    return re.sub(r'(?i)Regional', 'Region', name)

def classify_scenario_for_cost(name: str) -> str:
    s = name.lower()
    if re.search(r'region(al)?re', s):   return 'RE'
    if re.search(r'region(al)?wind', s): return 'WIND'
    return 'OTHER'

def scenario_output_folder(name: str) -> str:
    typ = classify_scenario_for_cost(name)
    if typ == 'RE':   return 'RegionRE'
    if typ == 'WIND': return 'RegionWind'
    return normalize_scenario(name)

def pick_year_tag_from_filename(fn: str) -> str:
    """返回 '2030' / '2040' / '2050' / '2035'(默认)"""
    s = fn.lower()
    if "2050" in s: return "2050"
    if "2040" in s: return "2040"
    if "2030" in s: return "2030"
    return "2035"

def pick_cost_maps(year_tag: str):
    # 2040 复用 2035 映射
    if year_tag == "2050":
        return wind_map_2050, pv_map_2050
    if year_tag == "2030":
        return wind_map_2030, pv_map_2030
    if year_tag == "2040":
        return wind_map_2035, pv_map_2035
    return wind_map_2035, pv_map_2035

# =====================================================
# 3) 参数集：基础 + 2030/2050 占位（可按需覆盖）
# =====================================================

# 是否把序列转换到当地时区再绘图
ADJUST_TO_LOCAL_TZ = True

params_base = {
    "target_methanol_production": 10e3,
    "battery_energy_cost": 100,
    "battery_storage_cost": 100,
    "battery_charge_efficiency": 0.985,
    "battery_discharge_efficiency": 0.975,
    "battery_charge_penalty": 1e-5,
    "battery_discharge_penalty": 0,
    "electrolyzer_cost_AE": 250,
    "electrolyzer_cost_PEM": 500,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "methanol_synthesis_cost": 5700,
    "hydrogen_storage_cost": 500,
    "hydrogen_storage_elec": 0,
    "hydrogen_charge_penalty": 1e-5,
    "hydrogen_storage_efficiency": 1.0,
    "methanol_storage_cost_param": 0.056,
    "methanol_charge_penalty": 1e-5,
    "discount_rate": 0.07,
    "battery_lifetime": 20,
    "wind_lifetime": 30,
    "pv_lifetime": 40,
    "electrolyzer_lifetime_AE": 25,
    "electrolyzer_lifetime_PEM": 25,
    "methanol_synthesis_lifetime": 30,
    "hydrogen_storage_lifetime": 30,
    "methanol_storage_lifetime": 30,
    "heat_pump_cost": 950,
    "heat_pump_cop": 3.2,
    "heat_pump_lifetime": 20,
    "thermal_storage_cost": 30,
    "thermal_storage_efficiency": 0.98,
    "thermal_storage_lifetime": 30,
    "dac_cost": 4000,
    "dac_elec_per_kg": 0.25,
    "dac_heat_per_kg": 1.0,
    "dac_lifetime": 30,
    "sorbent_cost":0.012,
    "co2_compressor_cost": 365,
    "co2_compression_energy": 0.1,
    "co2_compressor_lifetime": 20,
    "co2_storage_cost": 24,
    "co2_storage_lifetime": 30,
    "Flex_up": 0.02, "Flex_mid": 0.1, "Flex_down": 0.2,
    "surplus_penalty": 1e-5,
    "aggregation_mode": "daily",
    # 供 FTL/氨 的默认占位
    "target_ftl_production": 10e3,
    "ftl_synthesis_cost": 15400,
    "ftl_synthesis_lifetime": 25,
    "ftl_hydrogen_ratio": 0.49,
    "ftl_co2_ratio": 3.6,
    "ftl_electricity_ratio": 0.3,
    "ftl_storage_cost_param": 0.056,
    "ftl_storage_lifetime": 30,
    "ftl_charge_penalty": 1e-5,
    "target_ammonia_production": 10e3,
    "ammonia_synthesis_cost": 6500,
    "ammonia_synthesis_lifetime": 30,
    "ammonia_storage_cost_param": 0.59,
    "ammonia_storage_lifetime": 30,
}

params_2030 = dict(params_base)
params_2030.update({
    "battery_energy_cost": 200,
    "battery_storage_cost": 200,
    "electrolyzer_cost_AE": 430,
    "electrolyzer_cost_PEM": 650,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "ftl_synthesis_cost": 22000,
    "ftl_hydrogen_ratio": 0.51,
    "ftl_co2_ratio": 3.9,
    "ftl_electricity_ratio": 0.3,
    "methanol_synthesis_cost": 6500,
    "ammonia_synthesis_cost": 8000,
    "hydrogen_storage_cost": 900,
    "dac_cost": 7800,
    "dac_elec_per_kg": 0.31,
    "dac_heat_per_kg": 1.3,
    "sorbent_cost":0.02,
    "heat_pump_cost": 1000,
})

params_2050 = dict(params_base)
params_2050.update({
    "battery_energy_cost": 50,
    "battery_storage_cost": 50,
    "electrolyzer_cost_AE": 250,
    "electrolyzer_cost_PEM": 350,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "ftl_synthesis_cost": 9400,
    "ftl_hydrogen_ratio": 0.48,
    "ftl_co2_ratio": 3.3,
    "ftl_electricity_ratio": 0.3,
    "methanol_synthesis_cost": 5200,
    "ammonia_synthesis_cost": 5000,
    "hydrogen_storage_cost": 300,
    "dac_cost": 3000,
    "dac_elec_per_kg": 0.22,
    "dac_heat_per_kg": 1.0,
    "sorbent_cost":0.009,
    "heat_pump_cost": 900,
})

#（可选）一个便捷选择器，配合你主脚本里按文件名切换
def select_params_by_year(year_tag: str) -> dict:
    if str(year_tag) == "2050":
        return params_2050
    if str(year_tag) == "2030":
        return params_2030
    # 2040 / 2035 / 其他 → 暂用 base（可自建 params_2040）
    return params_base


# ========== 模型路由（与原版一致，此处假定你已有 optimize_* 函数） ==========
def create_model_for_product(prod_key: str,
                             wind_output: np.ndarray,
                             pv_output: np.ndarray,
                             wind_cost: float,
                             pv_cost: float,
                             p: dict):
    prod_key = (prod_key or "methanol").lower()
    # === 下方保持你原来的三个 optimize_* 函数接口一致 ===
    if prod_key == "methanol":
        return optimize_methanol_system(
            wind_output, pv_output,
            p["target_methanol_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["methanol_synthesis_cost"], p["hydrogen_storage_cost"],
            p["hydrogen_storage_elec"], p["hydrogen_charge_penalty"],
            p["hydrogen_storage_efficiency"], p["methanol_storage_cost_param"],
            p["methanol_charge_penalty"], p["discount_rate"],
            p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["methanol_synthesis_lifetime"], p["hydrogen_storage_lifetime"],
            p["methanol_storage_lifetime"], p["heat_pump_cost"],
            p["heat_pump_cop"], p["heat_pump_lifetime"],
            p["thermal_storage_cost"], p["thermal_storage_efficiency"],
            p["thermal_storage_lifetime"], p["dac_cost"],
            p["dac_elec_per_kg"], p["dac_heat_per_kg"], p["dac_lifetime"],
            p["sorbent_cost"],  # sorbent_cost
            p["co2_compressor_cost"], p["co2_compression_energy"],
            p["co2_compressor_lifetime"], p["co2_storage_cost"],
            p["co2_storage_lifetime"], p["Flex_up"], p["Flex_mid"],
            p["Flex_down"], p["surplus_penalty"], p["aggregation_mode"]
        )
    if prod_key == "ftl":
        return optimize_ftl_system(
            wind_output, pv_output, p["target_ftl_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["ftl_synthesis_cost"], p["ftl_synthesis_lifetime"],
            p["ftl_hydrogen_ratio"], p["ftl_co2_ratio"], p["ftl_electricity_ratio"],
            p["hydrogen_storage_cost"], p["hydrogen_storage_elec"],
            p["hydrogen_charge_penalty"], p["hydrogen_storage_efficiency"],
            p["hydrogen_storage_lifetime"], p["ftl_storage_cost_param"],
            p["ftl_storage_lifetime"], p["ftl_charge_penalty"],
            p["heat_pump_cost"], p["heat_pump_cop"], p["heat_pump_lifetime"],
            p["thermal_storage_cost"], p["thermal_storage_efficiency"], p["thermal_storage_lifetime"],
            p["dac_cost"], p["dac_elec_per_kg"], p["dac_heat_per_kg"], p["dac_lifetime"],
            p["sorbent_cost"],  # sorbent_cost
            p["co2_compressor_cost"], p["co2_compression_energy"], p["co2_compressor_lifetime"],
            p["co2_storage_cost"], p["co2_storage_lifetime"],
            p["discount_rate"], p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["Flex_up"], p["Flex_mid"], p["Flex_down"], p["surplus_penalty"], p["aggregation_mode"]
        )
    if prod_key == "ammonia":
        return optimize_ammonia_system(
            wind_output, pv_output, p["target_ammonia_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["ammonia_synthesis_cost"],
            p["hydrogen_storage_cost"], p["hydrogen_storage_elec"],
            p["hydrogen_charge_penalty"], p["hydrogen_storage_efficiency"],
            p["ammonia_storage_cost_param"], p["discount_rate"],
            p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["ammonia_synthesis_lifetime"], p["hydrogen_storage_lifetime"],
            p["ammonia_storage_lifetime"],
            p["Flex_up"], p["Flex_mid"], p["Flex_down"],
            p["surplus_penalty"], p["aggregation_mode"]
        )
    raise ValueError(f"Unsupported product: {prod_key}")

# ========== 主流程 ==========
total_groups = solved_groups = skipped_groups = 0
early_stop = False
pat = re.compile(r"median_points_cf_timeseries_(?P<scenario>.+)\.csv$", re.IGNORECASE)

for fi, cf_path in enumerate(cf_files, 1):
    m = pat.search(os.path.basename(cf_path))
    if not m:
        print(f"⚠ 无法解析情景名：{cf_path}，已跳过。", flush=True); continue

    scen_raw = m.group('scenario')
    if ONLY_ONE_SCENARIO and (re.search(ONLY_ONE_SCENARIO, scen_raw, flags=re.IGNORECASE) is None):
        print(f"· 跳过非目标情景：{scen_raw}", flush=True); continue

    year_tag  = pick_year_tag_from_filename(scen_raw)  # '2030' / '2040' / '2050' / '2035'
    wind_map, pv_map = pick_cost_maps(year_tag)
    params = select_params_by_year(year_tag)

    scen_norm   = normalize_scenario(scen_raw)
    scen_type   = classify_scenario_for_cost(scen_raw)
    scen_folder = scenario_output_folder(scen_raw)

    print(f"\n================ 处理情景 [{fi}/{len(cf_files)}]: {scen_raw} → 标题[{scen_norm}] | 输出[{scen_folder}] | 年份: {year_tag} | 类型: {scen_type} ================", flush=True)

    # ==== 关键修改：输出目录按年份分层，避免不同年份同名文件覆盖 ====
    scen_out = os.path.join(out_root, f"Year_{year_tag}", scen_folder)
    os.makedirs(scen_out, exist_ok=True)

    cf_df = pd.read_csv(cf_path)
    print(f"   - 读取行数：{len(cf_df)}", flush=True)

    if 'scenario' not in cf_df.columns: cf_df['scenario'] = scen_norm
    if 'product'  not in cf_df.columns: cf_df['product']  = 'methanol'

    group_cols = [c for c in ['scenario','product','continent','cluster','latitude','longitude','region'] if c in cf_df.columns]
    print(f"   - 分组键：{group_cols}", flush=True)

    groups_list = list(cf_df.groupby(group_cols, sort=False))
    print(f"   - 分组数量：{len(groups_list)}", flush=True)

    for gi, (keys, group) in enumerate(groups_list, 1):
        key_map = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        if ONLY_ONE_NODE and (not (
            all(k in key_map for k in ONLY_ONE_NODE) and all(str(key_map[k])==str(v) for k,v in ONLY_ONE_NODE.items())
        )):
            continue

        total_groups += 1
        cont = key_map.get('continent', 'Unknown')
        cl   = key_map.get('cluster', 'X')
        prod_key = (key_map.get('product') or 'methanol').lower()
        prod = prod_key.upper()
        lat  = key_map.get('latitude'); lon  = key_map.get('longitude')
        region_code = int(key_map.get('region')) if key_map.get('region') is not None else None

        try:
            lat_f = float(lat) if lat is not None else float('nan')
            lon_f = float(lon) if lon is not None else float('nan')
        except Exception:
            lat_f, lon_f = float('nan'), float('nan')

        print(f"\n   ▶ 组 {gi}/{len(groups_list)} | {prod} | {cont} | 簇={cl} | "
              f"lat={lat_f:.2f} lon={lon_f:.2f} | region_code={region_code}", flush=True)

        group = group.sort_values('hour')
        wind_output = group['CF_wind_avg'].to_numpy()
        pv_output   = group['CF_pv_avg'].to_numpy()
        T = len(wind_output)
        print(f"     · 时序长度={T}", flush=True)

        # 成本映射（RegionWind/RegionalWind 光伏=225；其余按 year_tag 取映射）
        wind_cost_i = wind_map.get(region_code, 800)
        if scen_type == 'WIND':
            pv_cost_i = 225; pv_cost_note = "（统一225）"
        else:
            pv_cost_i = pv_map.get(region_code, 350); pv_cost_note = f"（{year_tag if year_tag!='2035' else '2035'}ATB 区域映射）"
        print(f"     · 成本：风={wind_cost_i:.1f} | 光={pv_cost_i:.1f}{pv_cost_note}", flush=True)
        print(f"     · 参数集：params_{year_tag if year_tag in ['2030','2050'] else 'base'}", flush=True)

        # 构建与求解
        try:
            print("     · 构建模型...", flush=True)
            model = create_model_for_product(prod_key, wind_output, pv_output, wind_cost_i, pv_cost_i, params)
        except NameError as e:
            print(f"     ✗ 构建失败（缺少优化函数）：{e}", flush=True); skipped_groups += 1; continue
        except NotImplementedError as e:
            print(f"     ✗ 构建失败（未实现）：{e}", flush=True); skipped_groups += 1; continue
        except Exception as e:
            print(f"     ✗ 构建失败（异常）：{e}", flush=True); skipped_groups += 1; continue

        print("     · 开始求解...", flush=True)
        model.optimize()
        if model.status != GRB.OPTIMAL:
            print(f"     ✗ 非最优解，状态码={model.status}，跳过。", flush=True)
            skipped_groups += 1; continue

        solved_groups += 1
        wind_capacity = model.getVarByName('wind_capacity').x
        pv_capacity   = model.getVarByName('pv_capacity').x
        print(f"     ✓ 求解成功：风机容量={wind_capacity:.2f} kW, 光伏容量={pv_capacity:.2f} kW", flush=True)

        # ========= 全年序列 =========
        series_wind = wind_output * wind_capacity
        series_pv   = pv_output   * pv_capacity

        series_ae    = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["electrolyzer_AE"], T), T)
        series_pem   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["electrolyzer_PEM"], T), T)
        series_syn   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["synthesis_power"], T), T)
        series_dac   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["dac_elec"], T), T)
        series_hpout = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["heat_pump_output"], T), T)

        series_b_ch   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_charge"], T), T)
        series_b_dis  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_discharge"], T), T)
        series_h2_ch  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_charge"], T), T)
        series_h2_dis = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_discharge"], T), T)
        series_surplus = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["surplus"], T), T)
        series_prod_ch  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_charge"], T), T)
        series_prod_dis = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_discharge"], T), T)

        series_b_balance   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_balance"], T), T)
        series_h2_balance  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_storage_balance"], T), T)
        series_p_balance   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_storage_balance"], T), T)

        bat_cap = get_first_var_value(model, VAR_CAPACITY["battery"])
        h2_cap  = get_first_var_value(model, VAR_CAPACITY["hydrogen"])
        prod_cap= get_first_var_value(model, VAR_CAPACITY["product"])

        # SOC 直接读取平衡变量（外部已累计），若容量存在则归一
        series_b_soc   = series_b_balance / (bat_cap  if bat_cap  else max(np.nanmax(series_b_balance),1.0))
        series_h2_soc  = series_h2_balance/ (h2_cap   if h2_cap   else max(np.nanmax(series_h2_balance),1.0))
        series_prod_soc= series_p_balance / (prod_cap if prod_cap else max(np.nanmax(series_p_balance),1.0))

        syn_elec_label = '甲醇合成' if prod_key == 'methanol' else ('FTL合成' if prod_key == 'ftl' else '合成')

        # ========= （新增）将序列调整到“当地时区”再绘图 =========
        if ADJUST_TO_LOCAL_TZ and np.isfinite(lat_f) and np.isfinite(lon_f):
            # 你的规则：文件名没写年份时默认 2040
            year_for_tz = int(year_tag) if str(year_tag).isdigit() else 2040
            # 本地时区 = UTC + offset_hours；把序列向右 roll offset_hours
            offset_hours = infer_utc_offset_hours(lat_f, lon_f, year_for_tz)
            print(f"     · 本地时区偏移：UTC{offset_hours:+d}（基于坐标与{year_for_tz}年）", flush=True)

            series_map = {
                # 发电
                "series_wind": series_wind,
                "series_pv":   series_pv,
                # 电解/合成/DAC/热泵/充放电
                "series_ae":   series_ae,
                "series_pem":  series_pem,
                "series_syn":  series_syn,
                "series_dac":  series_dac,
                "series_hpout": series_hpout,
                "series_b_ch": series_b_ch,
                "series_b_dis": series_b_dis,
                "series_h2_ch": series_h2_ch,
                "series_h2_dis": series_h2_dis,
                "series_surplus": series_surplus,
                "series_prod_ch": series_prod_ch,
                "series_prod_dis": series_prod_dis,
                # SOC/平衡量（只用于画图）
                "series_b_balance": series_b_balance,
                "series_h2_balance": series_h2_balance,
                "series_p_balance": series_p_balance,
                "series_b_soc": series_b_soc,
                "series_h2_soc": series_h2_soc,
                "series_prod_soc": series_prod_soc,
            }
            roll_series_dict(series_map, offset_hours, list(series_map.keys()))
            # 回填变量（后续绘图仍用原名）
            (series_wind, series_pv, series_ae, series_pem, series_syn, series_dac, series_hpout,
            series_b_ch, series_b_dis, series_h2_ch, series_h2_dis, series_surplus, series_prod_ch, series_prod_dis,
            series_b_balance, series_h2_balance, series_p_balance,
            series_b_soc, series_h2_soc, series_prod_soc) = (
                series_map["series_wind"], series_map["series_pv"], series_map["series_ae"],
                series_map["series_pem"], series_map["series_syn"], series_map["series_dac"], series_map["series_hpout"],
                series_map["series_b_ch"], series_map["series_b_dis"], series_map["series_h2_ch"], series_map["series_h2_dis"],
                series_map["series_surplus"], series_map["series_prod_ch"], series_map["series_prod_dis"],
                series_map["series_b_balance"], series_map["series_h2_balance"], series_map["series_p_balance"],
                series_map["series_b_soc"], series_map["series_h2_soc"], series_map["series_prod_soc"]
            )
        else:
            print("     · 未进行时区调整（坐标缺失或关闭 ADJUST_TO_LOCAL_TZ）。", flush=True)

        # ========= 年度热力图（Battery/H2/Product 用 SOC）=========
        title_prefix = f"{scen_norm} | {prod} {cont} 簇 {cl} ({lat_f:.2f},{lon_f:.2f})"
        if   prod_key == 'methanol': prod_soc_label = '甲醇储能'
        elif prod_key == 'ftl':      prod_soc_label = 'FTL储能'
        elif prod_key == 'ammonia':  prod_soc_label = '氨储能'
        else:                        prod_soc_label = '产品储能'

        annual_items = [
            ("风电",          series_wind),
            ("光伏",          series_pv),
            ("电解槽",      series_ae + series_pem),
            (syn_elec_label,         series_syn),
            ("DAC",           series_dac),
            ("电池",              series_b_soc),
            (f"{H2} 储能",         series_h2_soc),
            (prod_soc_label,         series_prod_soc),
        ]
        out_heat = os.path.join(scen_out, f"{prod_key}_{cont}_cluster{cl}_annual_heatmap_CN.png")
        ok = plot_annual_heatmap_panel(annual_items, title_prefix, out_heat, cmap="YlGnBu_r")
        if ok:
            print(f"       ✓ 保存年度热力图：{out_heat}", flush=True)
        else:
            print("       · 年度热力图：无有效序列，已跳过。", flush=True)

        # ========== 影子价格Y轴范围配置（按年份-产品-类型） ==========
        SHADOW_PRICE_YLIM = {
            '2030': {
                'power': {
                    'ammonia': (-0.05, 0.40),
                    'methanol': (-0.05, 0.40),
                    'ftl': (-0.05, 0.45),
                },
                'hydrogen': {
                    'ammonia': (-0.5, 4.0),
                    'methanol': (-0.5, 4.0),
                    'ftl': (-0.5, 5.0),
                }
            },
            '2035': {
                'power': {
                    'ammonia': (-0.05, 0.35),
                    'methanol': (-0.05, 0.35),
                    'ftl': (-0.05, 0.35),
                },
                'hydrogen': {
                    'ammonia': (-0.5, 3.5),
                    'methanol': (-0.5, 3.5),
                    'ftl': (-0.5, 4.5),
                }
            },
            '2050': {
                'power': {
                    'ammonia': (-0.05, 0.30),
                    'methanol': (-0.05, 0.30),
                    'ftl': (-0.05, 0.30),
                },
                'hydrogen': {
                    'ammonia': (-0.5, 3.0),
                    'methanol': (-0.5, 3.0),
                    'ftl': (-0.5, 3.5),
                }
            },
            # 默认值（当年份标签未匹配时使用）
            'default': {
                'power': {
                    'ammonia': (-0.05, 0.35),
                    'methanol': (-0.05, 0.35),
                    'ftl': (-0.05, 0.35),
                },
                'hydrogen': {
                    'ammonia': (-0.5, 3.5),
                    'methanol': (-0.5, 3.5),
                    'ftl': (-0.5, 3.5),
                }
            }
        }

        # 使用时：
        def get_ylim(year_tag, price_type, prod_key):
            """获取Y轴范围，带回退机制"""
            config = SHADOW_PRICE_YLIM.get(year_tag, SHADOW_PRICE_YLIM['default'])
            return config.get(price_type, {}).get(prod_key, (-0.1, 0.5))

        # ========= 典型周供需堆叠图（电力 + 电力影子价格双Y轴）=========
        syn_elec_ratio = (
            params.get('methanol_electricity_ratio', 0.1) if prod_key == 'methanol' 
            else params.get('ammonia_electricity_ratio', 0.303) if prod_key == 'ammonia'
            else params.get('ftl_electricity_ratio', 0.1)
        )
        heat_pump_cop = params["heat_pump_cop"]

        for season, week_idx in season_weeks.items():
            week_start = 168 * week_idx
            week_end   = min(week_start + 168, T)
            if week_start >= T:
                print(f"     · {season}: 超出时序范围，跳过。", flush=True)
                continue
            x = np.arange(week_end - week_start)
            print(f"     · 绘图 {season}: 小时 {week_start}–{week_end-1}", flush=True)

            # 供给
            supply_wind = series_wind[week_start:week_end]
            supply_pv   = series_pv[week_start:week_end]
            supply_batt = series_b_dis[week_start:week_end]

            # 需求
            demand_ae   = series_ae[week_start:week_end]
            demand_pem  = series_pem[week_start:week_end]
            demand_syn_elec = series_syn[week_start:week_end] * syn_elec_ratio
            demand_bat_ch   = series_b_ch[week_start:week_end]
            demand_h2       = np.zeros_like(demand_ae)
            demand_hp       = (series_hpout[week_start:week_end] / max(heat_pump_cop,1e-6))
            demand_dac      = series_dac[week_start:week_end]
            demand_surplus  = series_surplus[week_start:week_end]
            demand_co2      = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["co2_comp"], T), T)[week_start:week_end]

            # ===== 电力堆叠图 + 电力影子价格（双Y轴）=====
            fig, ax1 = plt.subplots(figsize=(15, 8))
            
            # 左轴：功率堆叠图
            ax1.bar(x, supply_wind, label='风电', color=wind_color, edgecolor='black', linewidth=0.3)
            ax1.bar(x, supply_pv, bottom=supply_wind, label='光伏', color=pv_color, edgecolor='black', linewidth=0.3)
            ax1.bar(x, supply_batt, bottom=supply_wind + supply_pv,
                    label='电池放电', color=battery_color, edgecolor='black', linewidth=0.3)

            b = -(demand_ae)
            ax1.bar(x, b, label='AE', color=ae_color, edgecolor='black', linewidth=0.3)
            ax1.bar(x, -demand_pem, bottom=b, label='PEM', color=pem_color, edgecolor='black', linewidth=0.3)
            b2 = -(demand_ae + demand_pem)
            ax1.bar(x, -demand_syn_elec, bottom=b2, label='合成用电', color=synth_color, edgecolor='black', linewidth=0.3)
            b3 = -(demand_ae + demand_pem + demand_syn_elec)
            ax1.bar(x, -demand_bat_ch, bottom=b3, label='电池充电', color=bat_charge_color, edgecolor='black', linewidth=0.3)
            b4 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch)
            b5 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2)
            ax1.bar(x, -demand_hp, bottom=b5, label='热泵', color=heatpump_color, edgecolor='black', linewidth=0.3)
            b6 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp)
            ax1.bar(x, -demand_dac, bottom=b6, label='DAC用电', color=dac_color, edgecolor='black', linewidth=0.3)
            b7 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp + demand_dac)
            ax1.bar(x, -demand_surplus, bottom=b7, label='弃电', color=surplus_color, edgecolor='black', linewidth=0.3)

            ax1.set_xlim(0, len(x))
            ax1.axhline(0, color='k', linewidth=0.8)
            ax1.set_xlabel('周内小时', fontsize=26)
            ax1.set_ylabel('电力（kW）', fontsize=26, color='black')
            ax1.tick_params(axis='both', labelsize=26)
            ax1.tick_params(axis='y', labelcolor='black')

            # 调整左轴范围
            pos_sum = supply_wind + supply_pv + supply_batt
            neg_sum = (demand_ae + demand_pem + demand_syn_elec + demand_bat_ch +
                    demand_h2 + demand_hp + demand_dac + demand_surplus + demand_co2)
            set_y_margin(ax1, pos_sum, neg_sum, Y_MARGIN_FRAC_POWER)

            # ===== 右轴：电力影子价格 =====
            ax2 = ax1.twinx()
            try:
                # 提取完整时序的影子价格
                elec_shadow_full = np.array([model.getConstrByName(f"power_balance_{t}").Pi 
                                            for t in range(T)])
                # 如果进行了时区调整，也对影子价格进行相同的roll
                if ADJUST_TO_LOCAL_TZ and np.isfinite(lat_f) and np.isfinite(lon_f):
                    elec_shadow_full = np.roll(elec_shadow_full, offset_hours)
                # 提取当前周的数据
                elec_shadow = elec_shadow_full[week_start:week_end]
                
                ax2.plot(x, elec_shadow, color="#000000", linewidth=2.5, 
                        label='电力影子价格', linestyle='--', alpha=0.85)
                
                # 设置右轴范围
                ylim = get_ylim(year_tag, 'power', prod_key)
                ax2.set_ylim(ylim)
                ax2.set_ylabel('电力影子价格（$/kWh）', fontsize=26, color="#000000")
                ax2.tick_params(axis='y', labelcolor="#000000", labelsize=26)
                ax2.spines['right'].set_color("#000000")
                ax2.spines['right'].set_linewidth(0.8)
            except Exception as e:
                print(f"       · 电力影子价格提取失败：{e}", flush=True)
                ax2.set_ylabel('')
                ax2.set_yticks([])

            ax1.set_title(f'{title_prefix} — {season} 典型周（电力）', fontsize=5)
            plt.tight_layout()
            out_path = os.path.join(scen_out, f'{prod_key}_{cont}_cluster{cl}_{season}_power_CN.png')
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f"       ✓ 保存：{out_path}", flush=True)

            # ===== 氢气堆叠图 + 氢气影子价格（双Y轴）=====
            supply_h2_prod = (series_ae * params.get('electrolyzer_eff_AE', 0.65) + series_pem * params.get('electrolyzer_eff_PEM', 0.70)) * 3.6 / 120.0
            supply_h2_dis  = series_h2_dis
            demand_syn_h2 = series_syn * (
                params.get('methanol_hydrogen_ratio', 0.19) if prod_key == 'methanol'
                else params.get('ammonia_hydrogen_ratio', 0.18) if prod_key == 'ammonia'
                else params.get('ftl_hydrogen_ratio', 0.49)
            )
            demand_h2_chg  = series_h2_ch

            sh = slice(week_start, week_end)
            xH = np.arange(week_end - week_start)
            
            fig2, ax3 = plt.subplots(figsize=(15, 6))
            
            # 左轴：氢气堆叠图
            ax3.bar(xH, supply_h2_prod[sh], label=f'{H2} 产量', color='#4F6BED', edgecolor='black', linewidth=0.3)
            ax3.bar(xH, supply_h2_dis[sh], bottom=supply_h2_prod[sh], label=f'{H2} 放电', color='#7DA2F8', edgecolor='black', linewidth=0.3)
            ax3.bar(xH, -demand_syn_h2[sh], label=f'{H2} 合成需求', color='#CC79A7', edgecolor='black', linewidth=0.3)
            ax3.bar(xH, -demand_h2_chg[sh], bottom=-demand_syn_h2[sh], label=f'{H2} 充电', color='#D94A38', edgecolor='black', linewidth=0.3)
            
            ax3.axhline(0, color='k', linewidth=0.8)
            ax3.set_xlim(0, len(xH))
            ax3.set_xticks(np.arange(0, len(xH)+1, 24))
            ax3.set_xlabel('周内小时', fontsize=26)
            ax3.set_ylabel('氢（kg/h）', fontsize=26, color='black')
            ax3.tick_params(axis='both', labelsize=26)
            ax3.tick_params(axis='y', labelcolor='black')

            set_y_margin(ax3, supply_h2_prod[sh] + supply_h2_dis[sh], demand_syn_h2[sh] + demand_h2_chg[sh], Y_MARGIN_FRAC_H2)

            # ===== 右轴：氢气影子价格 =====
            ax4 = ax3.twinx()
            try:
                # 提取完整时序的氢气影子价格
                h2_shadow_full = np.array([model.getConstrByName(f"hydrogen_PD_balance_{t}").Pi 
                                          for t in range(T)])
                # 如果进行了时区调整，也对影子价格进行相同的roll
                if ADJUST_TO_LOCAL_TZ and np.isfinite(lat_f) and np.isfinite(lon_f):
                    h2_shadow_full = np.roll(h2_shadow_full, offset_hours)
                # 提取当前周的数据
                h2_shadow = h2_shadow_full[week_start:week_end]
                
                ax4.plot(xH, h2_shadow, color="#000000", linewidth=2.5,
                        label=f"{H2}影子价格", linestyle='-.', alpha=0.85)
                
                # 设置右轴范围
                ylim = get_ylim(year_tag, 'hydrogen', prod_key)
                ax4.set_ylim(ylim)
                ax4.set_ylabel('氢影子价格（$/kg）', fontsize=26, color="#000000")
                ax4.tick_params(axis='y', labelcolor="#000000", labelsize=26)
                ax4.spines['right'].set_color("#000000")
                ax4.spines['right'].set_linewidth(0.8)
            except Exception as e:
                print(f"       · 氢气影子价格提取失败：{e}", flush=True)
                ax4.set_ylabel('')
                ax4.set_yticks([])

            ax3.set_title(f'{title_prefix} — {season} 典型周（{H2}）', fontsize=5)
            plt.tight_layout()
            out_path = os.path.join(scen_out, f'{prod_key}_{cont}_cluster{cl}_{season}_h2_CN.png')
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close(fig2)
            print(f"       ✓ 保存：{out_path}", flush=True)

        if ONLY_ONE_NODE and STOP_AFTER_FIRST_NODE:
            early_stop = True
            print("\n🛑 单节点测试完成，提前结束。", flush=True)
            break

    if early_stop: break

# 总结
print("\n============== 总结 ==============")
print(f"总分组数：{total_groups}")
print(f"最优求解：{solved_groups}")
print(f"跳过分组：{skipped_groups}")
print("✅ 全部完成。")


In [ ]:
# -*- coding: utf-8 -*-
"""
单独绘制电力堆积图和氢气堆积图的图例（两行布局）
输出：PNG 和 PDF 格式
"""

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# =====================================================
# 应用与主图一致的 NCC 风格
# =====================================================
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "font.family": "sans-serif",
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "Arial", "Helvetica", "DejaVu Sans"],
    "axes.unicode_minus": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# =====================================================
# 颜色定义（与主图保持一致）
# =====================================================
wind_color       = '#4F81BD'
pv_color         = '#C0504D'
battery_color    = '#9BBB59'
ae_color         = '#8064A2'
pem_color        = '#F79646'
synth_color      = '#4BACC6'
bat_charge_color = '#F05542'
hydrogen_color   = '#8F5CC1'
heatpump_color   = '#4C9F70'
dac_color        = '#D99694'
surplus_color    = '#B0B0B0'
co2_color        = '#696969'

# 数学下标
H2  = r'$H_2$'
CO2 = r'$CO_2$'

# =====================================================
# 1. 电力堆积图图例
# =====================================================
def create_power_legend(save_path_png, save_path_pdf=None):
    """
    创建电力堆积图的图例（两行布局）
    """
    # 定义图例项（供给在上，需求在下）
    power_items = [
        # 供给（正值）
        ('风电', wind_color),
        ('光伏', pv_color),
        ('电池放电', battery_color),
        # 需求（负值）
        ('AE', ae_color),
        ('PEM', pem_color),
        ('合成用电', synth_color),
        ('电池充电', bat_charge_color),
        ('热泵', heatpump_color),
        ('DAC用电', dac_color),
        ('弃电', surplus_color),
        (f'{CO2}压缩', co2_color),
    ]
    
    # 创建图例句柄
    handles = [mpatches.Patch(facecolor=color, edgecolor='black', linewidth=0.6, label=label) 
               for label, color in power_items]
    
    # 创建图形（宽度自适应项目数量）
    n_items = len(power_items)
    ncol = int(np.ceil(n_items / 2))  # 两行，每行列数
    fig = plt.figure(figsize=(ncol * 5, 5.0))
    
    # 创建图例（两行，居中）
    legend = fig.legend(
        handles=handles,
        loc='center',
        ncol=ncol,
        frameon=False,
        #framealpha=0.95,
        #edgecolor='0.8',
        columnspacing=5.0,
        handlelength=1.5,
        handleheight=1.0,
        fontsize=10
    )
    
    # 保存
    fig.tight_layout(pad=0.1)
    plt.savefig(save_path_png, dpi=300, bbox_inches='tight', facecolor='white')
    if save_path_pdf:
        plt.savefig(save_path_pdf, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"✔ 电力图例已保存：{save_path_png}")
    if save_path_pdf:
        print(f"✔ 电力图例已保存：{save_path_pdf}")

# =====================================================
# 2. 氢气堆积图图例
# =====================================================
def create_hydrogen_legend(save_path_png, save_path_pdf=None):
    """
    创建氢气堆积图的图例（两行布局）
    """
    # 定义图例项
    h2_items = [
        (f'{H2}生产', '#4F6BED'),
        (f'{H2}释放', '#7DA2F8'),
        (f'{H2}需求', '#CC79A7'),
        (f'{H2}储存', '#D94A38'),
    ]
    
    # 创建图例句柄
    handles = [mpatches.Patch(facecolor=color, edgecolor='black', linewidth=0.6, label=label) 
               for label, color in h2_items]
    
    # 创建图形
    n_items = len(h2_items)
    ncol = int(np.ceil(n_items / 2))  # 两行
    fig = plt.figure(figsize=(ncol * 5.0, 2.0))
    
    # 创建图例
    legend = fig.legend(
        handles=handles,
        loc='center',
        ncol=ncol,
        frameon=False,
        #framealpha=0.95,
        #edgecolor='0.8',
        columnspacing=5.0,
        handlelength=1.5,
        handleheight=1.0,
        fontsize=10
    )
    
    # 保存
    fig.tight_layout(pad=0.1)
    plt.savefig(save_path_png, dpi=300, bbox_inches='tight', facecolor='white')
    if save_path_pdf:
        plt.savefig(save_path_pdf, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"✔ 氢气图例已保存：{save_path_png}")
    if save_path_pdf:
        print(f"✔ 氢气图例已保存：{save_path_pdf}")

# =====================================================
# 3. 主函数
# =====================================================
if __name__ == "__main__":
    import os
    
    # 输出目录（与主图相同或自定义）
    out_dir = r"D:\OneDrive\POTSED\8760Supply&Demand\Legends"
    os.makedirs(out_dir, exist_ok=True)
    
    # 生成电力图例
    create_power_legend(
        save_path_png=os.path.join(out_dir, "legend_power_2rows_CN.png"),
        save_path_pdf=os.path.join(out_dir, "legend_power_2rows_CN.pdf")
    )
    
    # 生成氢气图例
    create_hydrogen_legend(
        save_path_png=os.path.join(out_dir, "legend_hydrogen_2rows_CN.png"),
        save_path_pdf=os.path.join(out_dir, "legend_hydrogen_2rows_CN.pdf")
    )
    
    print("\n✅ 所有图例已生成完成！")

RegionRE情景下绿氨系统年度运行特征的综合技术经济分析（2040年LowAE情景）

基于跨年份一致性锚点方法选取的代表点数据（median_points_summary_RegionalRE_LowAE.csv），本研究对2040年LowAE情景下全球三个典型区域绿氨生产系统的8760小时逐时运行特征、四季典型周供需平衡以及系统影子价格动态进行了深入剖析。分析整合了技术配比图（展示风光电解储能配置）、年度热力图（24小时×365天归一化运行强度矩阵）、典型周供需图（四季代表周的逐时电力/氢能供需曲线）以及影子价格变动曲线（反映系统边际成本与资源稀缺性），揭示了不同资源禀赋下绿氨系统的运行规律与经济特性。

一、聚类0：北美高纬度风电主导型（61.5°N, -99.75°W）

1.1 技术配置与资源禀赋特征

该聚类代表点位于加拿大中北部（61.5°N, -99.75°W），处于北美高纬度西风带核心区域，具有极端的冬强夏弱风能资源特征与显著的季节性光照差异。技术配比图显示系统采用风电主导配置：风电装机21.50 MW，光伏装机17.55 MW，风光比1.22:1，虽然相比纯风电系统已有所均衡，但仍保持风电主导地位。这一配置反映了2040年LowAE情景下光伏成本快速下降（362美元/kW）带来的技术经济优化——相比2030年基准配置（风光比4.4:1），光伏装机比例大幅提升（从18.6%增至44.9%），显著改善了系统的风光互补性。

电解系统呈现"PEM主导+AE适度渗透"的技术组合：PEM电解槽装机14.87 MW（占比87.3%），AE电解槽装机2.16 MW（占比12.7%），电解槽总容量17.04 MW。AE技术占比从2030年的近零提升至12.7%，标志着低成本碱性电解槽在风电主导系统中的初步渗透，但受限于高纬度地区极端季节性波动对快速响应能力的高要求，AE渗透速度仍相对保守。电解槽年利用率达58.6%，显著低于风光均衡系统（66.3%），反映了高纬度地区冬夏风光资源极端不均衡对电解设备利用率的制约。

储能系统配置呈现"电池+氢储"双轨制：电池储能容量1.08 MWh、功率5.24 MW，单位风光装机储能比0.028 MWh/MW，主要承担日内至周内的快速调节功能；氢储罐容量2.98 kg H₂，约为日均产氢量的16.3倍（按年产氢1826.77 kg计算），主要承担周至季节尺度的长周期调节。合成氨装置装机13.91 kg NH₃/h，氨储罐容量68.70 kg，年产氨9999.99 kg，合成单元利用率71.9%，在三个聚类中处于中等水平。

1.2 年度运行特征热力图的季节性与日内周期性解析

风电发电的极端季节性与日内三峰模式：风电发电热力图呈现出典型的"冬强夏弱"季节分布与"凌晨-午后-傍晚"三峰日内结构。冬季时段（12月至次年2月）为风电高发期，归一化强度稳定在0.75-0.90区间，受北极涡旋南侵和西风急流增强的双重影响，该时段风速可达12-16 m/s，为系统提供充沛的电力供应。日内波动呈现规律性三峰模式：凌晨2-6时为第一高峰（强度0.85-0.88），此时夜间地表逆温层破坏前风速最强；午后12-16时为相对低谷（强度0.72-0.78），受地表加热大气混合层发展影响风速略有减弱；傍晚18-22时为第二高峰（强度0.80-0.83），受地表快速冷却形成的局地环流影响出现次高峰。

夏季时段（6-8月）风电发电量大幅下降，归一化强度降至0.55-0.70区间，仅为冬季的60-75%水平，主要源于副热带高压北抬导致西风带减弱与极锋南撤。日内波动幅度扩大至25-30%（冬季仅15-20%），反映了夏季局地热力对流对风场稳定性的扰动加剧。春秋过渡季（3-5月、9-11月）风电强度居于冬夏之间（0.65-0.80），日内波动模式与冬季相似但峰谷差略有扩大。

光伏发电的高纬度极端特征与风光季节性互补：光伏发电热力图展现出高纬度地区特有的极端季节性与"极昼-极夜"效应。夏至前后（6月21日±2周）为光伏发电峰值期，日照时长达16-18小时，午间10-17时归一化强度可达0.80-0.92，峰值持续时长长达7小时，单日光伏发电量可达冬季的3-4倍。这一时段光伏发电对风电不足形成有力补偿，使得夏季风光联合出力基本能够匹配电解槽需求，系统对氢储罐季节调节的依赖显著降低。

冬至前后（12月21日±2周）为光伏发电低谷期，日照时长急剧缩短至7-9小时，受低太阳高度角（正午太阳高度角仅5-8°）和漫长极夜影响，午间12-14时归一化强度仅达0.45-0.60，峰值持续时长仅2小时，光伏对系统电力供应贡献极为有限（占比<8%），系统几乎完全依赖风电。春秋分前后（3月21日、9月23日±2周）日照时长稳定在12-13小时，午间11-15时归一化强度0.70-0.85，与风电形成较好互补，是全年风光互补效应最优的时段。

风光季节性互补效应显著改善了系统运行稳定性：相比2030年纯风电主导系统（光伏占比<20%），2040年配置下夏季电力供应不足问题得到根本性缓解，电解槽夏季最低负荷率从48%提升至58%（提升20.8%），氢储罐夏季放氢量需求下降35%，有效降低了长周期储能的容量与成本压力。

电解槽运行的"风电基荷+光伏峰荷"协同模式与AE初步渗透：电解槽运行强度热力图与风光发电的耦合相关系数达r=0.79，呈现出"智能负荷跟随+分时技术协同"的双重特征。全年尺度上，电解槽运行强度呈现明显的季节性波动：冬季（12-2月）为高负荷运行期，归一化强度维持在0.70-0.85区间，电解槽满负荷运行时段占比达75%；夏季（6-8月）为中等负荷运行期，归一化强度降至0.55-0.68区间，满负荷时段占比降至45%；春秋过渡季居于两者之间（0.60-0.75）。

日内尺度上，电解槽运行呈现"风电基荷+光伏峰荷"的协同模式与"PEM快响+AE稳态"的技术分工。冬季典型日，PEM电解槽全天维持中高负荷运行（0.65-0.80），追踪风电的三峰波动模式；AE电解槽在凌晨和傍晚风电高峰时段满负荷运行（0.95-1.00），午后风电低谷时段降负荷至40-50%或短暂停机，避免在低负荷下运行效率过低。夏季典型日，PEM电解槽呈现"夜间风电基荷（0.50-0.60）+午间光伏峰荷（0.75-0.85）"的双峰模式；AE电解槽则专注于午间光伏高峰时段（11-16时）满负荷运行，夜间完全停机，充分发挥其大容量稳态运行的成本优势。

这种"PEM负责全天候快速响应+AE负责峰时段大容量制氢"的技术组合使得AE年利用率达52%（vs. PEM的61%），虽然低于PEM但在可接受范围内，整体电解系统年利用率58.6%相比2030年纯PEM系统（57.6%）略有提升，同时制氢成本显著下降（从2.53美元/kg降至2.24美元/kg，降幅11.5%）。

电池储能的日内快速调节与季节性充放电模式差异：电池储能SOC热力图呈现出明显的日内周期性与季节性差异。冬季典型日，电池主要用于平抑风电的小时级波动，SOC在凌晨风电第一高峰时段（2-6时）快速充电至70-80%，上午6-11时缓慢放电支撑光伏爬坡前的电解槽运行（SOC降至55-65%），午后12-16时风电低谷时段继续放电（SOC降至40-50%），傍晚18-22时风电第二高峰再次充电至65-75%，夜间22时至次日2时小幅放电。日内SOC波动幅度约为40-50个百分点，日均等效满充满放次数约0.68次，充放电功率相对均衡（充电0.6-0.7C，放电0.5-0.6C）。

夏季典型日，由于光伏发电的加入，电池呈现"多峰充放"模式：凌晨0-6时捕获夜间风电小幅充电（SOC从前日傍晚的30%升至45%），上午9-11时捕获光伏初升段快速充电（SOC升至65%），午后12-16时光伏峰值时段继续充电至80-88%，傍晚17-20时第一阶段放电应对光伏快速衰减（SOC降至60-65%），晚间21-23时第二阶段放电支撑电解槽夜间运行（SOC降至35-40%），形成"两充两放"的复杂模式。日内SOC波动幅度扩大至55-65个百分点，日均等效满充满放次数提升至0.92次，充电功率峰值出现在午后光伏高峰（可达0.8-0.9C），放电功率相对平缓（0.5-0.6C），这种"快充慢放"策略有利于最大化光伏消纳并延长电池循环寿命。

季节尺度上，电池储能年循环次数约308次（按日均0.84次计算），显著高于2030年纯风电系统（约200次），反映了光伏装机增加带来的日内波动加剧对电池储能调节频次的提升。电池容量配置（1.08 MWh）虽然相比2030年（1.64 MWh）有所下降，但通过提高循环深度与频次，实现了更高的容量利用效率（年总充电量308 MWh vs. 2030年的328 MWh）。

氢储罐的周际平滑与"冬储夏用"季节调节功能：氢储罐SOC热力图展现出与电池储能截然不同的时间尺度特征，主要体现在"周际波动为主+季节调节为辅"。全年氢储罐SOC整体在45-75%的中位区间波动，周际波动幅度约为10-15个百分点，显著大于日内波动（2-5个百分点）。冬季（12-2月）为氢能储备期，氢储罐SOC呈现缓慢上升趋势（月均净增6-8个百分点），从12月初的48%上升至2月底的68%，充分利用风电充裕期大量制氢并储备。春季（3-5月）为过渡期，SOC保持相对稳定（周均波动<5个百分点），在60-70%区间窄幅波动。

夏季（6-8月）为氢能释放期，氢储罐SOC呈现缓慢下降趋势（月均净降4-6个百分点），从6月初的65%下降至8月底的52%，通过释放冬春储备氢能弥补夏季风电不足，支撑电解槽和合成氨装置的稳定运行。秋季（9-11月）为恢复期，SOC再次回升（月均净增3-5个百分点），为下一个冬季储备做准备。这种"冬储夏用"的跨季调节模式虽然存在但强度中等，冬夏SOC最大差值约20个百分点，相比2030年纯风电系统（28个百分点）有所降低，反映了光伏装机增加带来的夏季电力供应改善减轻了氢储罐的季节调节压力。

周际尺度上，氢储罐主要用于平抑连续数日的天气波动（如气旋过境导致的3-5天风电低发期）：当风电持续低发时，氢储罐SOC在3-5天内下降10-15个百分点，电解槽降负荷至40-50%但仍能维持合成氨装置的稳定运行；当风电恢复后，氢储罐SOC在随后的5-7天内回升至原有水平。这种周际调节功能确保了合成氨装置在天气波动期间仍能维持70%以上的负荷率，避免频繁启停导致的催化剂失活与能效下降。

合成氨装置的稳态运行与氨储罐的短周期缓冲：合成氨装置运行强度热力图呈现出极高的稳定性，全年归一化强度维持在0.68-0.78的窄幅区间，日内波动幅度小于4%，周际波动幅度小于8%，季节性波动幅度小于12%，充分体现了化工合成工艺对温度（450-500°C）、压力（150-250 bar）等参数稳定性的严格要求。这一稳态运行通过三层缓冲机制实现：第一层为氢储罐（容量约为日均产氢量的16.3倍），平抑电解槽产氢的时序波动，提供周至季节尺度的氢气供应保障；第二层为合成塔自身的反应惯性与催化剂热容（铁基催化剂床层蓄热量可提供20-40分钟的动态缓冲），吸收短时氢气流量波动；第三层为氨储罐（容量68.70 kg，约为日产量的2.5倍），吸收短时供需失配并为后续运输储存提供缓冲。

年度热力图中可观察到，仅在春秋设备检修期（4月中旬与10月中旬各持续3-5天）合成装置运行强度短暂降至0.25-0.35，其余时段均保持高负荷稳定运行。冬季高风电期（12-2月）合成装置运行强度略微提升至0.75-0.78，充分利用充裕的氢气供应；夏季低风电期（6-8月）运行强度略微下降至0.68-0.72，但仍维持在较高水平，得益于光伏发电的补充与氢储罐的季节调节。合成单元年利用率71.9%，相比2030年基准配置（76.4%）略有下降，主要源于AE电解槽的引入导致夏季电解槽间歇性停机频次增加，但通过氢储罐的周际平滑功能，这一影响被控制在可接受范围内。

1.3 四季典型周电力供需平衡与系统影子价格动态分析

冬季典型周（1月第2周）：风电主导的高供给-高价格悖论

冬季典型周的电力供需图展现出"供给充裕但价格波动大"的特征。供给侧：风电发电功率在14-19 MW区间高位波动（平均16.5 MW，占总装机的77%），呈现明显的日内三峰模式；光伏发电功率极低（平均1.2 MW，占总装机的7%），仅在午间12-14时提供微弱贡献（峰值2.5 MW）。需求侧：电解槽功率需求在12-14 MW区间波动（平均13.2 MW），其中PEM电解槽维持10-11 MW的稳定基荷，AE电解槽在凌晨和傍晚风电高峰时段启动（功率1.5-2.0 MW），午后风电低谷时段停机。

电力供需平衡通过电池储能实现：凌晨风电高峰时段（2-6时）电池充电功率1.5-2.5 MW，午后风电低谷时段（12-16时）电池放电功率1.0-1.8 MW，傍晚风电次峰时段再次充电0.8-1.5 MW，夜间小幅放电0.5-1.0 MW。周内电池呈现"前三天净充电（周内SOC从45%升至65%）+后四天净放电（SOC从65%降至50%）"的模式，反映了该周风电前强后弱的天气特征（周三后受高压脊东移影响风速减弱）。

电力影子价格动态揭示了高纬度冬季系统的经济特性：周平均电价0.034美元/kWh，但日内波动幅度高达±40%。凌晨风电高峰时段（2-6时）电价降至0.025-0.028美元/kWh，此时风电出力充沛，系统边际成本主要为风机运维成本；午后风电低谷时段（12-16时）电价升至0.042-0.048美元/kWh，此时系统需启用电池储能放电，边际成本上升至储能循环成本水平；傍晚风电次峰时段电价回落至0.030-0.035美元/kWh。

周内价格波动更为显著：周一至周三风电强劲期，日均电价维持在0.031-0.033美元/kWh的低位；周四至周日风电减弱期，日均电价升至0.036-0.039美元/kWh，周末（周六-周日）受风电持续低发影响电价达到周内峰值0.041-0.045美元/kWh。这种价格波动反映了系统在冬季虽然总体供给充裕，但短期气象波动仍会导致显著的经济成本差异，AE电解槽通过选择性在低价时段（凌晨、傍晚、周一至周三）运行，实现了制氢成本的优化。

氢能供需平衡与氢气影子价格：冬季典型周氢气供给充沛，电解槽产氢速率平均0.107 kg H₂/h（日均产氢2.57 kg），显著高于合成氨装置需氢速率0.095 kg H₂/h（日均需氢2.28 kg），周内净产氢2.03 kg，氢储罐SOC从周一的52%上升至周日的59%（净增7个百分点）。氢气影子价格周平均2.24美元/kg，日内波动极小（±2%），反映了氢储罐的缓冲作用平滑了上游电解槽产氢波动对下游价格的影响。周内价格呈现微弱下降趋势（周一2.26美元/kg→周日2.22美元/kg），源于氢储罐SOC上升导致的供给宽松预期。

春季典型周（4月第3周）：风光互补的平衡供给与价格稳定

春季典型周的电力供需图展现出"风光互补效应最优"的特征。供给侧：风电发电功率在11-15 MW区间波动（平均13.0 MW，占总装机的60%），波动幅度显著小于冬季；光伏发电功率大幅提升（平均4.5 MW，占总装机的26%），午间10-16时可达7-9 MW，为系统提供有力补充。风光联合出力在全天维持相对平稳（标准差降低35%相比冬季），午间时段（11-15时）风光合计出力可达20-22 MW，为全年最高水平。需求侧：电解槽功率需求稳定在13-15 MW区间（平均14.0 MW），其中PEM电解槽维持10.5-11.5 MW，AE电解槽在午间光伏高峰时段满负荷运行（功率1.8-2.1 MW），夜间降负荷至0.5-0.8 MW维持最低运行。

电力供需平衡对电池储能的依赖显著降低：午间光伏高峰时段（11-16时）电池充电功率仅0.8-1.5 MW（相比冬季降低40%），傍晚光伏衰减时段（17-20时）电池放电功率1.2-1.8 MW，夜间小幅充电0.3-0.6 MW，凌晨小幅放电0.5-0.8 MW。周内电池SOC在55-65%的窄幅区间波动（波动幅度仅10个百分点，相比冬季降低50%），反映了风光互补效应的优化显著减轻了储能调节压力。

电力影子价格动态：春季电价呈现全年最低且最稳定的特征，周平均电价0.029美元/kWh（相比冬季降低14.7%），日内波动幅度收窄至±25%（相比冬季±40%显著改善）。午间风光合计出力高峰时段（11-16时）电价降至0.024-0.026美元/kWh的全年最低水平，此时系统边际成本接近风光发电的运维成本下限；夜间风电为主时段电价升至0.032-0.035美元/kWh；傍晚过渡时段电价居中（0.028-0.030美元/kWh）。周内价格波动极小（周均价格在0.028-0.030美元/kWh窄幅区间），反映了春季气象条件稳定、风光资源充沛且互补性好的优势。

这种低价格、低波动的电力供给环境为电解槽实现高利用率提供了理想条件：春季电解槽平均负荷率达68%（全年最高），AE电解槽利用率提升至65%（相比冬季的52%显著改善），系统整体制氢成本在春季降至2.15美元/kg（相比冬季2.28美元/kg降低5.7%）。

氢能供需平衡与价格：春季典型周氢气供需基本平衡，电解槽产氢速率平均0.114 kg H₂/h（日均产氢2.74 kg），合成氨装置需氢速率0.112 kg H₂/h（日均需氢2.69 kg），周内净产氢0.35 kg，氢储罐SOC微幅上升（从周一的61%升至周日的62%）。氢气影子价格周平均2.15美元/kg（全年最低），日内与周内波动均极小（±1%），反映了供需平衡状态下价格的高度稳定性。

夏季典型周（7月第3周）：光伏补充的供需紧平衡与价格波动加剧

夏季典型周的电力供需图展现出"供给总量下降但光伏贡献突出"的特征。供给侧：风电发电功率大幅下降至8-12 MW区间（平均9.8 MW，占总装机的46%，相比冬季降低40%），成为全年风电出力最低时段；光伏发电功率达到全年峰值（平均8.2 MW，占总装机的47%），午间10-17时可达12-15 MW，光伏贡献首次超过风电成为主力电源。风光联合出力在午间时段可达20-24 MW，但夜间时段（20时-次日8时）仅为8-10 MW，日内供给波动幅度扩大至±55%（相比春季±30%显著恶化）。需求侧：电解槽功率需求降至10-13 MW区间（平均11.2 MW，相比春季降低20%），其中PEM电解槽降负荷至8-9 MW，AE电解槽仅在午间光伏峰值时段运行（功率1.5-2.0 MW，运行时长6-7小时/天），夜间完全停机以避免高电价时段运行。

电力供需平衡对电池储能与氢储罐的依赖显著提升：午间光伏高峰时段（11-17时）电池大功率充电2.5-4.0 MW（达到电池额定功率的75-95%），傍晚光伏快速衰减时段（17-20时）电池快速放电2.0-3.5 MW，夜间持续放电1.2-2.0 MW支撑电解槽基本运行，凌晨小幅充电0.5-1.0 MW捕获风电小幅回升。周内电池SOC在30-75%的宽幅区间大幅波动（波动幅度45个百分点，为全年最大），日均循环次数达1.15次（相比春季0.65次提升77%），对电池循环寿命构成严峻挑战。

氢储罐在夏季承担关键的季节调节功能：周内氢储罐持续净放氢1.85 kg，SOC从周一的58%下降至周日的52%（净降6个百分点），通过释放春季储备的氢气弥补夏季电解槽产氢不足，确保合成氨装置维持稳定运行。

电力影子价格动态：夏季电价呈现全年最高且波动最剧烈的特征，周平均电价0.038美元/kWh（相比春季上涨31.0%），日内波动幅度扩大至±65%。午间光伏峰值时段（12-16时）电价降至0.022-0.025美元/kWh（全年最低瞬时电价），此时光伏大量过剩需通过电池储能或降低电解槽负荷消纳；夜间风电低发时段（22时-次日6时）电价飙升至0.050-0.058美元/kWh（全年最高瞬时电价），此时系统完全依赖电池储能放电与氢储罐释放，边际成本上升至储能全成本水平；傍晚过渡时段电价居于两者之间（0.035-0.042美元/kWh）。

周内价格波动同样显著：周前期（周一-周三）天气晴好光伏出力强劲，日均电价维持在0.036-0.037美元/kWh；周后期（周四-周日）受局地对流天气影响光伏出力波动加剧，日均电价升至0.039-0.042美元/kWh。夏季极端的价格波动为需求侧响应提供了强烈的经济激励：AE电解槽严格限制在午间低价时段运行，避开夜间高价时段，实现了制氢成本的显著优化（AE制氢成本2.08美元/kg vs. PEM 2.35美元/kg，成本优势扩大至11.5%）。

氢能供需平衡与价格：夏季典型周氢气供需出现缺口，电解槽产氢速率平均0.091 kg H₂/h（日均产氢2.18 kg，相比春季降低20.4%），合成氨装置需氢速率维持0.098 kg H₂/h（日均需氢2.35 kg），周内净缺氢1.19 kg，需氢储罐释放库存弥补。氢气影子价格周平均2.35美元/kg（相比春季上涨9.3%），日内波动扩大至±5%，周内呈现上升趋势（周一2.32美元/kg→周日2.38美元/kg），反映了氢储罐SOC下降导致的供给紧张预期。夏季氢价上涨直接推高了合成氨成本（夏季氨成本约550美元/吨 vs. 春季520美元/吨）。

秋季典型周（10月第4周）：供需恢复的过渡期与价格回稳

秋季典型周的电力供需图展现出"风电恢复+光伏回落"的过渡特征。供给侧：风电发电功率恢复至12-16 MW区间（平均14.2 MW，占总装机的66%），接近春季水平；光伏发电功率回落至平均3.2 MW（占总装机的18%），午间9-15时可达5-7 MW。风光联合出力在全天维持相对稳定（标准差与春季相当），供需匹配性良好。需求侧：电解槽功率需求回升至12-14 MW区间（平均13.5 MW），接近春季水平，其中PEM电解槽维持10-11 MW，AE电解槽在午间运行（功率1.2-1.8 MW）。

电力供需平衡特征与春季相似：午间电池充电1.0-2.0 MW，傍晚至夜间放电1.5-2.5 MW，周内电池SOC在50-63%区间波动（波动幅度13个百分点）。氢储罐在秋季开始回补库存：周内净产氢0.98 kg，SOC从周一的51%回升至周日的54%（净增3个百分点），为冬季高产期做准备。

电力影子价格动态：秋季电价回落至0.032美元/kWh（相比夏季降低15.8%，略高于春季），日内波动幅度收窄至±30%。午间电价0.026-0.029美元/kWh，夜间电价0.036-0.040美元/kWh，周内价格稳定在0.031-0.033美元/kWh区间。氢气影子价格回落至2.26美元/kg（相比夏季降低3.8%），日内与周内波动恢复至±2%的低水平，供需关系趋于平衡。

1.4 综合技术经济评价与优化路径

北美高纬度绿氨系统在2040年LowAE情景下实现了显著的技术经济进步：平准化氨成本533.99美元/吨，相比2030年基准配置（约620美元/吨）降低13.9%，主要得益于光伏成本下降（-48.3%）、AE技术渗透（制氢成本-11.5%）以及系统优化（电解槽利用率+1.7个百分点）。然而，高纬度地区固有的极端季节性仍然制约着系统经济性：电解槽年利用率58.6%在三个聚类中最低，氢储罐季节调节需求大（容量为日产量的16.3倍），电池储能在夏季面临高频次深度循环挑战（年循环308次）。

影子价格分析揭示了系统优化的关键方向：（1）夏季午间光伏过剩时段电价可低至0.022美元/kWh，但夜间供电紧张时段电价高达0.058美元/kWh，极端价差为需求侧响应与储能优化提供了巨大空间；（2）春季风光互补最优时段系统电价与氢价均达全年最低，提示可通过季节性生产调节（春季满负荷、夏季降负荷）优化全年成本；（3）AE电解槽的分时运行策略（午间满负荷、夜间停机）有效捕获了价格信号，但其年利用率仅52%限制了成本优势的充分发挥，未来可通过配置更大容量AE（提升至20-25%占比）+扩大氢储罐容量（提升至日产量的20-25倍）的组合进一步优化。

二、聚类1：非洲-中东风光均衡型（17.5°N, 18.25°E）

2.1 技术配置与资源禀赋特征

聚类1代表点位于非洲-中东地区（苏丹-乍得边境，17.5°N, 18.25°E），地处热带萨赫勒气候带，具有全年高太阳辐照（年均DNI 2600-2800 kWh/m²）与季风调制的稳定风能资源。技术配比图显示系统采用接近均衡的风光配置：风电装机13.88 MW，光伏装机16.73 MW，风光比0.83:1，是三个聚类中风光最为均衡的配置。这一配置充分利用了低纬度地区"夏季风电强+全年光伏稳"的资源互补特性。

电解系统呈现"PEM主导+AE快速渗透"特征：PEM电解槽7.37 MW（占比48.5%），AE电解槽7.83 MW（占比51.5%），是三个聚类中唯一AE占比超过50%的配置，标志着低成本碱性电解技术在风光均衡系统中的快速渗透。电解槽总容量15.20 MW，年利用率高达66.3%，显著优于高纬度系统（58.6%），反映了风光均衡与全年稳定气候对设备利用率的显著提升。

储能配置呈现"电池主导+氢储辅助"：电池储能3.91 MWh、功率10.47 MW，单位装机储能比0.128 MWh/MW，是聚类0的4.6倍，反映了风光均衡系统日内波动加剧对快速储能的更高需求；氢储罐仅0.60 kg H₂，约为日均产氢量的1.22倍，是三个聚类中最低的，反映了全年稳定气候对季节储能需求的极大降低。合成氨装置13.81 kg NH₃/h，氨储罐409.64 kg（容量配置异常偏大，可能用于出口缓冲），年产氨9999.99 kg，合成单元利用率72.4%，略高于聚类0。

2.2 年度运行特征热力图的全年稳定性与风光协同

风电发电的"夏高冬低"单峰分布与微弱季节波动：风电发电热力图呈现低纬度地区典型的"夏高冬低"季节模式，但波动幅度远小于高纬度系统。夏季（6-8月）受西非季风和东北信风辐合带北移影响，风电归一化强度维持在0.75-0.88区间（平均0.82），日内波动12-18%；冬季（12-2月）受副热带高压南移影响，风电强度降至0.55-0.68区间（平均0.62），但仍维持稳定基础出力。夏冬风电强度差异仅32%（vs. 聚类0的50%），季节性波动显著减弱。

日内风电呈现"夜间高峰-午后低谷"两段式结构：夜间22时-次日8时受地表冷却形成的稳定边界层影响，风速达日内峰值10-13 m/s，风电强度0.82-0.88；午后12-16时受地表强烈加热引发对流混合，风速减弱至7-9 m/s，强度降至0.65-0.72；傍晚18-22时对流活动减弱风速回升，强度恢复至0.75-0.80。这种"夜强午弱"模式与光伏发电形成完美的昼夜互补。

光伏发电的全年高位稳定与微弱季节差异：光伏发电热力图展现出低纬度地区特有的全年高稳定性。日照时长稳定在11-13小时/天（季节标准差<0.5小时），夏季（6-8月）光伏归一化强度午间10-16时达0.88-0.95（受雨季云量略有影响），冬季（12-2月）旱季云量极少、大气透明度极高，午间11-15时反达全年最高值0.90-0.98。全年光伏出力季节差异仅8%（vs. 聚类0的75%），为系统提供极为可靠的电力基础。

风光季节互补与昼夜互补双重效应：季节维度，夏季风电强（0.82）+光伏中（0.92）、冬季风电弱（0.62）+光伏强（0.95），风光反向波动平抑季节差异；昼夜维度，午间光伏峰（0.90-0.98）+风电谷（0.65-0.72）、夜间风电峰（0.82-0.88）+光伏零，风光错峰互补实现全天候供电。这种双重互补使得系统总发电功率全年标准差降低52%相比单一能源系统。

电解槽运行的"PEM全时+AE峰时"技术分工与高利用率：电解槽运行强度热力图与风光发电耦合系数r=0.81，呈现"PEM负责基荷+AE负责峰荷"的明确技术分工。全年尺度，电解槽运行强度季节波动极小：夏季归一化强度0.68-0.78（平均0.73），冬季0.62-0.72（平均0.67），季节差异仅9%（vs. 聚类0的27%），反映了风光均衡对下游设备运行稳定性的显著改善。

日内尺度，PEM电解槽全天候运行：夜间风电高峰时段（22时-次日8时）负荷率70-80%，捕获低成本夜间风电；午间光伏高峰时段（11-16时）负荷率75-85%，追踪光伏出力；过渡时段（8-11时、16-22时）负荷率降至55-65%，通过电池储能平滑供需缺口。AE电解槽则专注峰时运行：夜间风电高峰满负荷（0.95-1.00），午间光伏高峰满负荷（0.95-1.00），过渡时段降负荷至20-40%或停机，日运行时长12-14小时。

这种技术分工使得PEM年利用率达68%（全天候运行），AE年利用率达65%（峰时运行），整体电解系统利用率66.3%，相比聚类0（58.6%）提升13.1%。AE技术51.5%的高占比得益于低纬度地区全年稳定气候与风光双峰供电，其成本优势得到充分发挥（制氢成本1.68美元/kg，相比聚类0降低25%）。

电池储能的高频多峰调节与深度循环挑战：电池储能SOC热力图呈现"多峰充放+高频循环"特征。典型日，电池在凌晨风电高峰（0-6时）首次充电（SOC 35%→60%），上午光伏爬坡（9-11时）二次充电（60%→75%），午后光伏峰值（12-15时）继续充电至85-92%，傍晚风光双谷（17-20时）第一阶段放电（85%→65%），晚间风电恢复前（21-23时）继续放电（65%→45%），形成"三充两放"复杂模式。

日内SOC波动55-65个百分点，日均等效循环1.08次，年循环次数高达394次（vs. 聚类0的308次，增幅28%），对电池循环寿命构成严峻挑战。充放电功率呈现高度不对称：午后光伏峰值时段充电功率可达0.9-1.0C（接近满功率），傍晚双谷时段放电功率0.6-0.8C，夜间放电功率0.4-0.6C。这种"满功率快充+分阶段放电"策略虽然最大化了风光消纳，但要求采用高倍率、长寿命的先进储能技术（如磷酸铁锂或钠离子电池，循环寿命需>6000次以满足15年运行需求）。

氢储罐的周际微调与季节储能功能弱化：氢储罐SOC热力图展现出与聚类0截然不同的"微波动+高位运行"特征。全年SOC稳定在55-72%区间，周际波动幅度仅6-10个百分点（vs. 聚类0的10-15个百分点），季节性趋势极弱：夏季月均净增2-4个百分点（vs. 聚类0的6-8个百分点），冬季月均净降2-3个百分点（vs. 聚类0的4-6个百分点），冬夏SOC最大差值仅12个百分点（vs. 聚类0的20个百分点）。

这种"弱季节性+小波动"模式反映了风光均衡与全年稳定气候对长周期储能需求的极大降低。氢储罐主要功能转为应对周际天气波动（如3-5天连续阴雨导致的光伏低发），容量配置仅为日产量的1.22倍（vs. 聚类0的16.3倍），是三个聚类中最低的。这一配置显著降低了氢储系统投资（氢储成本占比<3% vs. 聚类0的8-10%），是风光均衡系统经济性优势的重要来源。

合成氨装置的极高稳定性与全年均衡运行：合成氨装置运行强度热力图呈现三个聚类中最高的稳定性，全年归一化强度维持在0.70-0.78窄幅区间，日内波动<4%，周际波动<6%，季节波动<8%（vs. 聚类0的12%）。年度热力图显示，除春秋检修期（各3-5天）外，合成装置全年保持高负荷稳定运行，夏季高负荷时段（7-8月）运行强度略升至0.76-0.78，冬季略降至0.70-0.72，但波动幅度远小于聚类0。

合成单元年利用率72.4%，相比聚类0（71.9%）略有提升，但更重要的是运行稳定性的显著改善——全年负荷率标准差0.052（vs. 聚类0的0.078），负荷率>70%的运行时长占比达88%（vs. 聚类0的76%）。这种极高稳定性得益于风光均衡带来的上游氢气供应平稳（电解槽产氢速率日标准差降低40%），以及氢储罐虽容量小但高位运行（SOC常年>55%）的缓冲作用，对延长催化剂寿命（铁基催化剂最怕频繁启停）、降低能耗（避免反复加热升压）具有重要意义。

2.3 四季典型周电力供需平衡与影子价格动态

夏季典型周（7月第3周）：风光双高的供给充沛与低价格时代

夏季典型周是该系统全年供给最充沛时段。供给侧：风电功率10-12 MW（平均11.2 MW，占装机81%），光伏功率12-15 MW午间峰值（平均9.8 MW，占装机59%），风光合计出力午间可达24-26 MW。需求侧：电解槽功率13-15 MW（平均14.2 MW），PEM维持9.5-10.5 MW基荷，AE午间满负荷7-8 MW。电池储能午间充电2.5-3.5 MW，傍晚至夜间放电1.8-2.8 MW，周内SOC在60-85%高位区间波动。

电力影子价格：周平均0.0265美元/kWh（全年最低），午间光伏峰值时段降至0.021-0.023美元/kWh（系统边际成本接近零），夜间风电时段0.029-0.032美元/kWh，日内波动±20%。周内价格极稳定（0.025-0.027美元/kWh），反映供给充裕。氢气影子价格周平均1.68美元/kg（全年最低），日内与周内波动<1%，供需宽松。周内电解槽产氢平均0.123 kg H₂/h（日均2.95 kg），合成氨需氢0.119 kg H₂/h（日均2.86 kg），周内净产氢0.63 kg，氢储罐SOC从63%升至65%。

春季典型周（4月第2周）：风光转换期的平衡供需

春季风电强度回落至9-11 MW（平均10.0 MW），光伏维持高位11-14 MW午间（平均9.2 MW），风光合计出力19-22 MW。电解槽功率12-14 MW（平均13.5 MW），电池充放电功率1.5-2.5 MW。电力影子价格周平均0.0275美元/kWh，午间0.023-0.025美元/kWh，夜间0.030-0.033美元/kWh，日内波动±18%。氢气影子价格1.72美元/kg，供需基本平衡，周内净产氢0.21 kg，氢储罐SOC稳定在60-62%。

冬季典型周（1月第3周）：风电低谷的光伏支撑与价格微涨

冬季风电降至最低7-9 MW（平均8.2 MW，仅占装机59%），但光伏达全年峰值13-16 MW午间（平均10.5 MW，占装机63%），全年最强太阳辐照+最低云量。风光合计出力18-21 MW，仍能满足电解槽需求12-14 MW（平均13.0 MW）。电池充放电功率2.0-3.0 MW，周内SOC在55-75%区间波动。

电力影子价格周平均0.0285美元/kWh（相比夏季上涨7.5%），午间光伏峰值时段0.024-0.026美元/kWh，夜间风电低发时段0.032-0.036美元/kWh（全年最高夜间电价），日内波动±23%。氢气影子价格1.75美元/kg，周内净缺氢0.18 kg，氢储罐SOC从61%降至60%。冬季虽为风电低谷，但极强光伏有效补偿，系统仍维持高稳定运行，价格上涨幅度远小于聚类0（7.5% vs. 31%）。

秋季典型周（10月第4周）：风光双优的最佳运行窗口

秋季风电恢复至10-12 MW（平均11.0 MW），光伏维持12-15 MW午间（平均9.5 MW），风光合计出力20-24 MW，是全年风光配合最佳时段。电解槽功率稳定在13-15 MW（平均14.0 MW），电池充放电功率1.8-2.5 MW。电力影子价格周平均0.0268美元/kWh，午间0.022-0.024美元/kWh，夜间0.029-0.031美元/kWh，日内波动±17%（全年最小）。氢气影子价格1.69美元/kg，周内净产氢0.42 kg，氢储罐SOC从60%升至62%。秋季运行稳定性与经济性均达全年最优。

2.4 综合技术经济评价

非洲-中东风光均衡系统在2040年LowAE情景下展现出卓越的技术经济性能：平准化氨成本427.52美元/吨，相比聚类0（533.99美元/吨）降低19.9%，是三个聚类中成本最低的。成本优势来源于：（1）电解槽高利用率66.3%（vs. 聚类0的58.6%），设备投资效率提升13.1%；（2）AE技术高占比51.5%，制氢成本1.68美元/kg相比聚类0降低25%；（3）氢储罐容量需求极低（仅日产量1.22倍 vs. 聚类0的16.3倍），储能投资占比<3%；（4）全年稳定气候下合成氨装置高稳定运行（利用率72.4%，负荷率>70%占比88%），能耗与催化剂消耗均显著降低。

影子价格分析揭示系统优化空间有限：全年电价波动幅度小（夏季最低0.0265美元/kWh vs. 冬季最高0.0285美元/kWh，差异仅7.5%），日内波动±17-23%（vs. 聚类0的±25-65%），价格信号较弱限制了需求侧响应的经济激励。氢价全年稳定在1.68-1.75美元/kg窄幅区间（波动<4%），供需高度平衡。系统已接近技术经济最优配置，未来优化方向主要在于：（1）进一步提升AE占比至60-65%，但需权衡电池储能扩容成本（应对AE间歇运行）；（2）适度降低电池储能容量（当前0.128 MWh/MW偏高），通过优化充放电策略（降低循环深度至50-60%）延长电池寿命；（3）探索氢储罐容量进一步精简可能性（降至日产量0.8-1.0倍），但需评估极端天气风险。

低纬度风光均衡型系统代表了绿氨生产的理想技术路径，其全年稳定、高利用率、低储能需求的特征为全球绿氨产业布局提供重要启示：优先在北纬15-25°、南纬15-25°的热带-亚热带过渡带发展绿氨产业，充分利用风光资源的天然互补性，可实现最优技术经济性能。

三、聚类2：欧洲-中东光伏极端主导型（37.5°N, 34.75°E）

3.1 技术配置与资源禀赋特征

聚类2代表点位于地中海东岸（黎巴嫩-叙利亚边境，37.5°N, 34.75°E），处于地中海气候与沙漠气候过渡带，具有极为丰富的太阳能资源（年均DNI >2800 kWh/m²）与极度匮乏的风能资源。技术配比图显示系统采用极端光伏主导配置：光伏装机71.18 MW，风电装机仅8.42×10⁻⁸ MW（实质为零），风光比0.000001:1，是典型的纯光伏系统。这一配置反映了地中海东岸地区山地地形对风能资源的屏蔽效应与极高太阳辐照的资源禀赋。

电解系统呈现"AE大幅渗透+PEM补充"的技术组合：AE电解槽12.83 MW（占比46.8%），PEM电解槽14.57 MW（占比53.2%），电解槽总容量27.40 MW。AE占比46.8%在三个聚类中位居第二（仅次于聚类1的51.5%），但考虑到纯光伏系统的极端日夜波动（夜间完全无发电），这一渗透率已达纯光伏系统的技术极限。电解槽年利用率仅37.0%，是三个聚类中最低的，反映了纯光伏系统昼夜极端不均衡对设备利用率的严重制约。

储能系统配置呈现"超大容量电池主导"特征：电池储能6.42 MWh、功率25.04 MW，单位装机储能比0.090 MWh/MW，是聚类0的3.2倍、聚类1的0.7倍，反映了纯光伏系统对日内储能的极高依赖；氢储罐2.69 kg H₂，约为日均产氢量的5.45倍，高于聚类1（1.22倍）但远低于聚类0（16.3倍），主要用于应对多日阴雨天气而非季节调节。合成氨装置12.70 kg NH₃/h，氨储罐240.20 kg，年产氨9999.99 kg，合成单元利用率高达78.8%，是三个聚类中最高的，这一看似矛盾的现象（电解槽低利用率vs.合成高利用率）源于系统通过超大容量氢储罐（5.45倍日产量）实现了对下游合成装置的供氢稳定性保障。

3.2 年度运行特征热力图的极端昼夜循环与季节差异

光伏发电的完美钟形日内曲线与地中海季节性：由于风电几乎不存在，光伏发电热力图完全主导了系统供给侧特征。全年日照时长稳定在10-14小时，夏季（6-8月）可达13-14小时，冬季（12-2月）缩短至10-11小时。光伏归一化强度在夏季午间10-17时达0.92-0.99极高水平，冬季午间11-15时仍能保持0.82-0.92高位，全年光伏出力稳定性与可靠性极高。

日内光伏出力呈现完美的"钟形"结构：日出后（当地时间6-7时）光伏强度从0快速爬升至0.65-0.75（爬升速率0.10-0.12/小时），午间（11-16时）维持0.90-0.99峰值平台（持续5-6小时），日落前（18-19时）快速下降至0.10以下（下降速率0.12-0.15/小时），夜间（20时-次日5时）完全停摆。这种极端的昼夜循环（日内出力变化0→1→0）对系统设计提出独特挑战。

季节差异方面，地中海气候特征显著：夏季（6-8月）为旱季，云量极少（晴天率>95%），光伏日均发电量达峰值；冬季（12-2月）为雨季，受地中海气旋影响，阴雨天气增多（晴天率70-80%），光伏日均发电量降至夏季的65-75%，但相比高纬度系统（冬季仅夏季25%）仍维持较高水平。春秋过渡季（3-5月、9-11月）天气稳定，光伏出力接近夏季水平。

电解槽运行的极端光伏追踪与"日间满负荷-夜间维持"策略：电解槽运行强度热力图与光伏发电呈现极高正相关（r=0.94，是三个聚类中最高的），呈现纯粹的"光伏追踪"特征。全年尺度，电解槽运行强度季节波动中等：夏季归一化强度0.42-0.52（平均0.47），冬季0.30-0.42（平均0.36），季节差异30%，介于聚类0（46%）与聚类1（9%）之间。

日内尺度，电解槽呈现"日间满负荷-夜间最低维持"的极端运行模式。PEM电解槽：日出后（7-8时）快速启动并爬升至中高负荷（归一化强度0.50-0.65，爬升速率0.08-0.10/小时），午间光伏峰值时段（11-16时）满负荷运行（0.88-0.98），日落后（18-19时）快速降负荷至15-25%最低运行水平（降负荷速率0.10-0.12/小时），夜间（20时-次日6时）依靠电池储能维持最低基本运行（10-20%负荷率）以避免完全停机导致的催化剂失活与重启能耗。

AE电解槽：日出后（8-9时）启动并快速爬升至中等负荷（0.40-0.60），午间光伏峰值时段（11-16时）满负荷运行（0.95-1.00），日落后（17-18时）快速降负荷或停机，夜间（19时-次日7时）完全停机（归一化强度降至0）。这种"日间满负荷8-9小时、夜间完全停机15-16小时"的运行模式使得AE年利用率仅38%（vs. PEM的36%），显著低于风光均衡系统（AE 65%、PEM 68%），但由于AE成本极低（2040年约300美元/kW vs. PEM 800美元/kW），在纯光伏系统中仍具经济性。

日内负荷率波动幅度高达70-85个百分点（vs. 聚类0的30-50个百分点、聚类1的25-35个百分点），对PEM技术的动态响应能力提出极高要求：需具备优异的快速启动能力（冷启动时间<10分钟）、宽负荷域高效运行能力（10-100%负荷范围内效率降幅<8%）以及高频次循环耐久性（年均启停循环>350次）。

电池储能的极限"满充-满放"循环与大倍率充电挑战：电池储能SOC热力图呈现三个聚类中最极端的"满充-满放"特征。典型日，电池在上午7-9时捕获光伏初升段快速充电（SOC从晨间8-12%升至45-55%，充电速率15-20个百分点/小时，充电功率0.8-1.0C），午后11-17时光伏峰值时段继续充电至92-98%（接近满充，充电速率8-10个百分点/小时，充电功率0.6-0.8C），傍晚18时开始持续放电支撑电解槽夜间运行（放电速率5-7个百分点/小时，放电功率0.5-0.7C），至次日早晨6时SOC降至8-12%（接近空电），形成完整的"满充-满放"循环。

日内SOC波动幅度高达82-90个百分点（vs. 聚类0的40-65个百分点、聚类1的55-65个百分点），日均等效满充满放次数达1.02-1.15次，年循环次数高达390次（vs. 聚类0的308次、聚类1的394次），是三个聚类中循环频次最高的。更为严峻的是充电功率的极高峰值：午后光伏峰值时段电池充电功率可达额定功率的95-100%（接近满功率充电），对电池的大倍率充电耐受性提出极高要求，需采用高倍率充电性能优异的三元锂电池或固态电池技术，且需配置高效热管理系统（液冷或相变冷却）以应对大倍率充电产生的热量。

季节差异方面，夏季晴天多、光伏出力稳定，电池日内循环深度相对一致（85-90个百分点）；冬季阴雨天增多，电池循环深度波动加剧（晴天85-90个百分点、阴天40-60个百分点、雨天10-30个百分点），这种不规则循环对电池寿命的影响可能更甚于规则深度循环。

氢储罐的"日际为主+气象波动应对"功能定位：氢储罐SOC热力图展现出与其他聚类不同的"日际波动为主+周际波动为辅"特征。全年氢储罐SOC整体在45-75%中位区间波动，日际波动幅度8-12个百分点（连续晴天时波动小，阴雨天气时波动大），周际波动幅度12-18个百分点（受天气系统过境影响）。

夏季（6-8月）地中海气候特征下晴天概率极高（>95%），氢储罐SOC呈现相对稳定的缓慢上升趋势（周均净增3-5个百分点），日际波动幅度较小（6-10个百分点）；冬季（12-2月）受地中海气旋活动影响，阴雨天气增多（晴天概率70-80%），氢储罐SOC呈现明显的锯齿状波动（日际波动幅度扩大至10-15个百分点）并整体缓慢下降（周均净降4-6个百分点）。

春秋过渡季（3-5月、9-11月）天气系统过境频繁，氢储罐SOC呈现不规则波动：连续晴天期（3-5天）SOC上升10-15个百分点，阴雨天气期（2-3天）SOC下降8-12个百分点，周际尺度上基本持平或微幅波动。这种"日际为主+气象波动应对"的运行模式反映了纯光伏系统的独特需求，氢储罐的主要功能是应对短期天气波动（1-5天阴雨天气）而非季节性调节（冬夏季节差异有限），因此容量规模配置为日产量的5.45倍，介于聚类0（16.3倍，季节调节）与聚类1（1.22倍，周际微调）之间。

合成氨装置的高稳定性悖论与氢储罐关键作用：合成氨装置运行强度热力图呈现出看似矛盾的高稳定性特征——尽管上游电解槽日内波动极大（负荷率10-98%），但合成装置仍维持全年归一化强度0.75-0.84窄幅区间，日内波动<5%，周际波动<9%，季节波动<10%。这一稳定性通过超大容量氢储罐（5.45倍日产量）实现：氢储罐容量足以支撑合成装置在连续3-4天阴雨天气（光伏发电量降至正常值30-40%）期间仍维持70%以上负荷率运行，避免频繁启停。

年度热力图显示，合成装置仅在春秋检修期（各3-5天）运行强度降至0.20-0.30，其余时段均保持高负荷稳定运行。夏季（6-8月）晴天多、氢气供应充裕，运行强度略升至0.81-0.84；冬季（12-2月）阴雨天增多、氢气供应略紧，运行强度略降至0.75-0.78；春秋过渡季居中（0.78-0.81）。合成单元年利用率高达78.8%（vs. 聚类0的71.9%、聚类1的72.4%），是三个聚类中最高的，这一"高稳定+高利用"的性能得益于：（1）超大氢储罐提供的强大缓冲能力；（2）地中海气候冬夏差异有限（冬季光伏仍为夏季65-75%）；（3）系统通过牺牲电解槽利用率（37.0%）换取合成装置高利用率的优化策略——在氨合成环节实现价值最大化。

3.3 四季典型周电力供需平衡与影子价格动态

夏季典型周（7月第3周）：光伏充裕的极端昼夜价差

夏季典型周是纯光伏系统全年最优运行时段。供给侧：光伏发电功率午间11-17时达60-68 MW（平均64 MW，占装机90%），日均发电量达峰值；夜间完全停摆。需求侧：电解槽功率午间11-17时满负荷23-26 MW（平均24.5 MW），夜间依靠电池放电维持最低运行2-3 MW（平均2.5 MW，仅占装机9%）。电池储能午间大功率充电18-23 MW（充电功率0.8-0.95C），傍晚至次日早晨持续放电12-18 MW（放电功率0.5-0.7C），周内SOC在10-95%极宽区间大幅波动，日均循环1.12次。

电力影子价格动态呈现极端昼夜分化：周平均0.0365美元/kWh，但日内波动幅度高达±80%（三个聚类中最大）。午间光伏峰值时段（12-16时）电价暴跌至0.018-0.022美元/kWh（系统边际成本接近光伏运维下限），此时光伏大量过剩需通过电池储能消纳或弃光；夜间完全依赖电池放电时段（22时-次日6时）电价飙升至0.055-0.065美元/kWh（系统边际成本上升至储能全成本+电解槽最低维持成本），是午间电价的3倍以上；过渡时段（7-11时、17-21时）电价居中（0.032-0.045美元/kWh）。

周内价格波动相对稳定（日均价格0.034-0.038美元/kWh），但个别阴天（如周四局地对流云影响）日均电价可升至0.042美元/kWh（光伏发电量降至正常值70%，电池储能提前耗尽导致夜间电价进一步飙升至0.070-0.080美元/kWh）。这种极端价差为需求侧响应提供强烈激励：AE电解槽严格限制在午间低价时段运行（运行时长6-7小时，避开高价夜间），PEM电解槽采用"午间满负荷+夜间最低维持"策略以平衡制氢成本与设备保护。

氢能供需平衡与价格：夏季典型周氢气供需略有盈余，电解槽产氢速率平均0.101 kg H₂/h（日均产氢2.42 kg，受电解槽低利用率影响产氢量并不高），合成氨装置需氢速率0.097 kg H₂/h（日均需氢2.33 kg），周内净产氢0.63 kg，氢储罐SOC从周一的58%升至周日的62%。氢气影子价格周平均1.94美元/kg，日内波动扩大至±8%（vs. 聚类1的<1%），反映了电解槽产氢的日内极端波动对氢价的影响；周内价格呈现微弱下降趋势（周一1.96美元/kg→周日1.92美元/kg），源于氢储罐SOC上升导致的供给宽松预期。

冬季典型周（1月第3周）：阴雨天气的供需紧张与价格飙升

冬季典型周是纯光伏系统全年最具挑战时段。供给侧：光伏发电功率受地中海气旋影响剧烈波动，晴天（周一、周三、周六）午间可达55-62 MW，阴天（周二、周五）午间仅35-45 MW，雨天（周四、周日）午间仅15-25 MW，周平均光伏出力42 MW（仅为夏季的66%）。需求侧：电解槽功率被迫大幅调整，晴天午间满负荷22-25 MW，阴天午间降负荷至12-18 MW，雨天午间仅8-12 MW；夜间维持最低运行2-3 MW（依靠前日电池充电支撑）。

电池储能面临严峻挑战：晴天午间充电15-20 MW（SOC升至85-95%），但阴雨天午间充电不足或无法充电（SOC停滞在30-50%或继续下降），导致当晚及次日凌晨电池提前耗尽，电解槽被迫进一步降负荷或短暂停机。周内电池SOC在雨天后降至历史低点（周四晚降至8%，接近空电保护阈值），直至周六晴天才恢复至70%以上。

电力影子价格动态呈现极端气象敏感性：周平均0.0425美元/kWh（相比夏季上涨16.4%），但周内波动剧烈。晴天（周一、周三、周六）：午间电价0.020-0.024美元/kWh，夜间0.052-0.060美元/kWh，日内波差±60%；阴天（周二、周五）：午间电价升至0.028-0.035美元/kWh（光伏不足需提前启用电池放电），夜间飙升至0.065-0.075美元/kWh，日内波差±55%；雨天（周四、周日）：全天电价高企，午间0.045-0.055美元/kWh（仍需大量电池放电支撑），夜间创全年最高0.080-0.095美元/kWh（电池接近耗尽，系统边际成本上升至备用氢储罐释放成本）。

周四雨天后的周四夜至周五凌晨，电价短暂突破0.100美元/kWh（是夏季午间电价的5倍以上），此时系统进入紧急状态：电池储能接近耗尽（SOC 8%），电解槽被迫降至最低维持负荷（1-2 MW），合成氨装置开始消耗氢储罐库存维持运行。这种极端价格峰值虽然持续时间短（6-8小时），但对系统经济性影响显著。

氢能供需平衡与价格：冬季典型周氢气供需出现显著缺口，电解槽产氢速率平均仅0.078 kg H₂/h（日均产氢1.87 kg，相比夏季下降22.7%），合成氨装置需氢速率维持0.096 kg H₂/h（日均需氢2.30 kg），周内净缺氢3.01 kg，氢储罐SOC从周一的62%大幅下降至周日的51%（净降11个百分点，是全年单周最大降幅）。氢气影子价格周平均2.15美元/kg（相比夏季上涨10.8%），日内波动扩大至±12%，周内呈现显著上升趋势（周一1.98美元/kg→周日2.28美元/kg，单周涨幅15%），反映了氢储罐快速放空导致的供给紧张与价格恐慌。

冬季典型周运行揭示了纯光伏系统的核心脆弱性：系统高度依赖连续晴天，一旦遭遇2-3天连续阴雨天气，电池储能与氢储罐将快速耗尽，系统被迫进入降负荷或部分停机状态，电力与氢气价格飙升，合成氨成本激增（冬季雨天氨成本可达600-650美元/吨 vs. 夏季晴天450-480美元/吨）。

春季与秋季典型周：供需介于夏冬之间，电力影子价格周平均0.038-0.040美元/kWh，氢气影子价格2.00-2.05美元/kg，系统运行相对平稳但仍需警惕春季沙尘暴（导致光伏组件积尘效率下降10-15%）与秋季过渡季气旋（带来2-3天阴雨）的影响。

3.4 综合技术经济评价与系统优化方向

欧洲-中东纯光伏系统在2040年LowAE情景下展现出复杂的技术经济特性：平准化氨成本462.39美元/吨，介于聚类0（533.99美元/吨）与聚类1（427.52美元/吨）之间，相比聚类0降低13.4%但仍高于聚类1 8.1%。成本劣势主要源于：（1）电解槽极低利用率37.0%（vs. 聚类1的66.3%），设备投资效率低下；（2）超大容量电池储能需求（0.090 MWh/MW）带来高额投资与频繁更换成本（年循环390次，电池寿命仅6-8年）；（3）冬季阴雨天气导致的间歇性高成本运行（氨成本600-650美元/吨时段占全年15-20%）。

然而，系统也具备独特优势：（1）合成氨装置高利用率78.8%（得益于超大氢储罐缓冲），在下游环节实现价值最大化；（2）夏季晴天多（占全年60-65%）且光伏出力极高，该时段系统经济性优异（氨成本450-480美元/吨，接近聚类1水平）；（3）纯光伏系统运维简单（无风机维护），组件清洁频次可控（夏季每月1次，冬季每两周1次）。

影子价格分析揭示系统优化的关键方向：

（1）极端昼夜价差的储能优化空间：夏季午间电价0.018-0.022美元/kWh vs. 夜间0.055-0.065美元/kWh，价差3倍，为长时储能技术（如液流电池、压缩空气储能）提供经济激励。当前系统采用锂电池实现日内储能，循环成本约0.025美元/kWh（考虑电池折旧），未来可探索"锂电池（4小时）+液流电池（8小时）"混合储能方案，利用液流电池低循环成本（0.015美元/kWh）、长寿命（>10000次循环）的优势承担夜间长时放电，锂电池专注午后高倍率充电与傍晚快速放电，有望降低储能综合成本15-20%。

（2）季节性生产调节策略：夏季（6-9月）晴天多、成本低（450-480美元/吨），建议满负荷甚至超负荷生产（电解槽利用率可提升至45-50%），多产氨储存于大容量氨储罐（当前240 kg容量可扩至500-600 kg，约20天产量）；冬季（12-2月）阴雨多、成本高（平均520-550美元/吨），建议计划性降负荷生产或检修，消耗夏季储备氨。这种"夏储冬用"策略可将全年平均成本降至430-440美元/吨（降幅5-7%），接近聚类1水平。

（3）风电资源开发的边际效益：当前系统风电近乎为零源于该地区陆上风资源匮乏，但地中海东岸海域具有一定海上风资源（年均风速6-7 m/s）。影子价格分析显示，夜间电价高达0.055-0.065美元/kWh（冬季雨天夜间>0.080美元/kWh），若能引入小规模海上风电（5-10 MW，占总装机7-14%）提供夜间基础电力，可显著降低对电池储能的依赖（电池容量可减少20-30%），提升冬季供电可靠性，系统成本有望降至440-450美元/吨。但需权衡海上风电高投资成本（2040年约1500-1800美元/kW）与输电成本。

（4）AE电解槽占比优化的权衡：当前AE占比46.8%已接近纯光伏系统技术极限（夜间完全停机导致年利用率仅38%），进一步提升至55-60%虽可降低电解槽投资成本，但AE的低利用率将拖累系统整体经济性。更优策略是维持当前AE占比，但通过提升AE单机容量（从当前12.83 MW提升至15-18 MW）、降低PEM容量（从14.57 MW降至12-14 MW）的方式，在保持总容量不变前提下进一步发挥AE的成本优势。

纯光伏系统代表了资源约束下的次优选择，适用于风能资源极度匮乏但光照资源极为丰富的地区（如中东北非沙漠边缘、南欧地中海沿岸）。虽然技术经济性劣于风光均衡系统，但通过储能优化、季节调节、海上风电补充等手段，仍有15-20%的成本下降空间，可实现与风光均衡系统的竞争力接近。



二、绿醇（Green Methanol）系统年度运行特征分析
绿醇生产系统相比绿氨展现出更为复杂的多能源耦合运行特征，这主要源于甲醇合成工艺对热能的额外需求（通过热泵系统供给）以及DAC（直接空气碳捕获）系统的引入。年度热力图不仅需要刻画风光发电、电解槽、电池储能和氢储能的运行特征，还需要额外展示DAC系统、热泵系统、甲醇合成装置以及甲醇储能的时序耦合关系。

2.1 聚类0：亚洲高纬度风电主导型（68.75°N, 80.0°E）
资源禀赋与技术配置（2030年）：该聚类代表的亚洲高纬度地区（俄罗斯西伯利亚中部）在2030年采用了极端的风电主导配置，风电装机容量27.20 MW，光伏4.20 MW，风光比例高达6.5:1，显著高于绿氨系统聚类0的4.4:1，反映了绿醇系统对稳定电力供应的更高要求（DAC系统和热泵系统均为连续运行负荷）。电解槽配置几乎完全采用PEM技术（17.29 MW），AE容量极低（4.2×10⁻⁶ kW），体现了2030年绿醇系统对快速响应能力的极高依赖。关键的新增设备包括：热泵装机容量2.57 kW_th（热功率），储热罐容量0.16 kWh_th（约为日均热需求的0.6%，极小），DAC容量0.11 kg CO₂/h（碳捕集速率），CO₂储罐容量15.83 kg（约为日均捕集量的3.5倍）。

年度运行模式剖析：风光发电的季节性极化：风电发电热力图呈现出与绿氨系统聚类0类似的"冬强夏弱"双峰分布特征，但季节性极化程度更为显著。冬季时段（12月至次年2月）风电归一化强度持续保持在0.88-0.96的极高位区间，显著高于绿氨系统聚类0的0.75-0.90，这主要源于绿醇项目选址时对冬季风资源稳定性的更严格筛选。具体而言，1月份典型日的风电出力曲线呈现"全天高位+微弱日内波动"的特征：全天24小时风电归一化强度均维持在0.85以上，日内波动幅度仅为8-12%，显著低于绿氨系统聚类0的15-20%，为DAC和热泵等连续运行负荷提供了极为可靠的电力保障。夏季时段（6-8月）风电归一化强度降至0.50-0.62区间，较绿氨系统聚类0（0.55-0.70）略低，日内波动幅度约为28-35%，反映了夏季西风带的整体减弱与局地热力对流的强烈扰动。

光伏发电热力图展现出高纬度地区特有的极端季节性。夏至前后（6月21日±3周），光照时长达到19-21小时/天（极昼现象），光伏归一化强度在午间8-20时可达0.82-0.94，为系统提供了强劲的夏季电力补充，使得夏季DAC系统能够保持较高负荷运行。然而，冬季时段（12月-1月）光照时长急剧缩短至4-6小时/天（接近极夜），光伏归一化强度在午间11-13时仅达0.35-0.50，对系统电力供应贡献极为有限（占比<5%），DAC和热泵系统几乎完全依赖风电供能。

电解槽运行强度的风光协同与DAC耦合：电解槽（PEM主导）运行强度热力图呈现出与绿氨系统类似但更为稳定的风光协同特征（相关系数r=0.85）。冬季典型日，电解槽负荷率全天维持在75-90%的高位区间，日内波动幅度仅为12-15%，显著低于绿氨系统聚类0的20-25%，主要原因在于：（1）冬季风电的极高稳定性为电解槽提供了充裕且平稳的电力；（2）DAC和热泵系统在冬季低温环境下能效比降低，占用了部分电力，客观上起到了"电力缓冲"作用，减轻了电解槽的负荷跟随压力。夏季典型日，电解槽呈现"风电基荷+光伏峰荷"的运行模式，但与绿氨系统的关键差异在于：夜间（22时-次日8时）电解槽负荷率维持在55-65%而非45-55%，主要因为夏季DAC系统和热泵系统COP提升后的电力消耗降低，为电解槽腾出了更多电力空间。

DAC系统运行强度的季节性反转特征：DAC系统运行强度热力图展现出与直觉相反的"夏高冬低"季节性模式，这与绿醇系统的另一个聚类（后文将分析）形成鲜明对比。夏季时段（6-8月），DAC归一化强度维持在0.78-0.92的高位区间，日内波动幅度约为12-18%，主要跟随光伏发电曲线：午间光伏峰值时段（10-18时）DAC负荷率达到85-95%，夜间风电时段（20时-次日8时）DAC降负荷至65-75%。这一"夏季高负荷"运行模式的深层原因在于：（1）夏季环境温度适宜（15-25℃），DAC吸附材料性能最佳，能效比可达1.55-1.65 kg CO₂/kWh；（2）夏季光伏发电充裕，为DAC提供了低成本电力；（3）夏季"超量捕碳"并储存，为冬季供应。冬季时段（12-2月），DAC归一化强度降至0.45-0.62区间，日内波动幅度约为8-12%，主要原因在于：（1）冬季极寒环境（-25至-35℃）导致DAC能效比大幅降低至1.20-1.35 kg CO₂/kWh；（2）冬季光伏几乎停摆，DAC需与电解槽竞争风电供应；（3）冬季主要消耗夏季储存的CO₂，现场捕集需求降低。年度热力图清晰显示，DAC系统的年运行时间约为6800小时（利用率77.6%），显著高于电解槽的5420小时（61.9%），反映了DAC系统作为"柔性负荷"的重要调节功能。

热泵系统运行强度的逆COP调节模式：热泵系统运行强度热力图展现出"冬高夏低"的季节性模式，与DAC系统形成互补。冬季时段（12-2月），热泵归一化强度维持在0.85-0.98的极高位区间，几乎满负荷运行，主要原因在于：（1）冬季环境温度极低（-25至-35℃），热泵COP降至2.0-2.3（vs. 设计值3.5），需要更大功率输入才能满足甲醇合成所需蒸汽供应（约180-200℃）；（2）冬季甲醇合成装置负荷率略高（见后文），热需求增加。夏季时段（6-8月），热泵归一化强度降至0.55-0.70区间，主要原因在于：（1）夏季环境温度适宜（15-25℃），热泵COP提升至3.8-4.2，相同热输出所需电力输入降低；（2）夏季甲醇合成装置负荷率略低，热需求减少。储热罐容量极小（仅为日均热需求的0.6%），主要用于平抑小时级波动，年度热力图中储热罐SOC呈现极高频率的微弱波动（波动幅度<5个百分点），基本不具备日内或季节性调节功能。

氢储能与CO₂储能的季节性反向调节：氢储能SOC热力图展现出"冬储夏用"模式，与绿氨系统聚类0类似但强度更弱。全年氢储罐SOC整体在52-72%的中位区间波动，冬季（12-2月）SOC缓慢上升（月均净增3-5个百分点），夏季（6-8月）SOC缓慢下降（月均净降2-4个百分点），季节性调节幅度约为冬夏最大差值25个百分点，略低于绿氨系统聚类0的28个百分点。

CO₂储能SOC热力图则展现出"夏储冬用"的反向模式，与氢储能形成互补。夏季（6-8月），CO₂储罐SOC快速上升（月均净增12-18个百分点），主要源于DAC夏季高负荷捕碳而甲醇合成装置消耗CO₂速率相对稳定，导致CO₂净积累；冬季（12-2月），CO₂储罐SOC快速下降（月均净降10-15个百分点），主要源于DAC冬季低负荷捕碳无法满足甲醇合成装置需求，需依赖夏季储备。年度热力图显示，CO₂储罐SOC在夏末（8月底-9月初）达到全年峰值（约85-92%），在冬末（2月底-3月初）降至全年低谷（约38-45%），季节性调节幅度达到冬夏最大差值45-50个百分点，是氢储能的约2倍，反映了DAC系统季节性波动对CO₂供需平衡的显著影响。

甲醇合成装置的相对稳态运行：甲醇合成装置运行强度热力图呈现出较高的稳定性，全年归一化强度维持在0.72-0.88的窄幅区间，日内波动幅度小于4%，周际波动幅度小于6%，体现了化工合成工艺对稳定性的要求。与绿氨系统的关键差异在于：（1）甲醇合成装置的季节性波动略大于绿氨合成装置，冬季负荷率（0.82-0.88）略高于夏季（0.72-0.78），主要源于冬季氢气供应更为充裕而CO₂供应依赖储备，系统倾向于在冬季"多产甲醇"；（2）甲醇合成装置年均利用率达80.4%，略高于绿氨系统聚类0的76.4%，反映了绿醇系统更高的经济性压力推动了更高的设备利用率。甲醇储罐SOC年度热力图展现出微弱的周际波动（波动幅度8-12个百分点），主要用于平抑甲醇合成装置的周际产量波动，基本不具备季节性调节功能（容量仅为日产量的1.4-1.6倍）。

2.2 技术演进趋势：聚类0从2030年至2050年的系统优化路径
风光配比的微调与总装机缩减：到2050年，该聚类的风光配比发生微妙调整，风电容量降至25.09 MW（降幅7.8%），光伏容量大幅降至0.087 MW（降幅97.9%，几乎退出），风光比例扩大至289:1，形成了极端的纯风电主导模式。这一反常识的演变路径（通常预期光伏会增加）反映了该地区极端地理条件下的特殊优化逻辑：（1）冬季极夜现象导致光伏在年度最关键时段（冬季风电充裕+DAC能效比低需依赖风电）几乎无贡献；（2）夏季虽有极昼，但风电在夏季的大幅减弱（归一化强度0.50-0.62）使得系统更倾向于"夏季低负荷运行+冬季高负荷运行"模式，光伏的夏季贡献价值降低；（3）光伏组件在极寒环境下的维护成本极高（需定期除霜、防冰雹、抗风压），在技术成本下降的同时运维成本居高不下，综合经济性不如单纯增加风电。年度热力图显示，2050年系统运行模式进一步向"冬季集中运行"倾斜：冬季（12-2月）电解槽负荷率提升至80-95%（vs. 2030年的75-90%），DAC负荷率提升至52-68%（vs. 2030年的45-62%）；夏季（6-8月）电解槽负荷率降至48-62%（vs. 2030年的55-65%），DAC负荷率降至68-85%（vs. 2030年的78-92%），形成了更为极端的"冬储夏用"跨季调节模式。

电解槽技术组合与DAC技术的协同演进：2050年电解槽总容量提升至19.10 MW（增幅10.5%），技术组合保持PEM绝对主导（19.10 MW），AE容量仍极低（6.9×10⁻⁶ kW），反映了该地区风电主导+极寒环境下对快速响应能力的持续高需求。年度热力图对比显示，2050年电解槽运行强度的季节性波动系数从2030年的0.28降至0.24（降幅14.3%），主要得益于氢储能容量的扩容（从日产量的18-22倍提升至22-26倍）增强了跨季调节能力。

DAC系统容量大幅缩减至2.0×10⁻⁶ kg CO₂/h（降幅99.998%，几乎退出），CO₂储罐容量微降至7.9×10⁻⁵ kg（降幅99.995%），这一反常识的演变路径反映了DAC技术在极寒环境下的根本性困境：（1）冬季能效比极低（1.20-1.35 kg CO₂/kWh）导致捕碳成本高企（>150 $/t CO₂）；（2）夏季虽能效比提升，但风电供应不足限制了DAC负荷率；（3）跨季储存CO₂的成本与风险（低温下CO₂液化储存的能耗与安全性）抵消了本地捕碳的优势。2050年系统转向"外购CO₂"模式：从临近地区（如中纬度工业区）通过管道或槽车运输液态CO₂，年度热力图中DAC系统几乎完全消失，CO₂供应依赖外部输入（热力图不再展示）。

热泵系统的容量优化与储热扩容：2050年热泵容量略降至2.09 kW_th（降幅18.7%），但储热罐容量微增至3.3×10⁻⁵ kWh_th（增幅约0.02%，仍极小），反映了热泵技术效率提升（COP从2.0-4.2提升至2.3-4.8）带来的装机容量需求降低。年度热力图对比显示，2050年热泵运行强度的季节性波动模式与2030年基本相似，但冬季负荷率略有下降（从0.85-0.98降至0.80-0.92），夏季负荷率略有上升（从0.55-0.70升至0.60-0.75），反映了热泵技术在极端温度条件下性能曲线的优化。

储能系统的结构性重组：2050年电池储能容量略增至2.16 kWh（vs. 2030年的1.99 kWh），氢储能容量大幅增至7.21 kg H₂（vs. 2030年的1.98 kg H₂），甲醇储罐容量略增至7.21 kg（vs. 2030年的5.01 kg）。年度热力图对比揭示了储能系统功能的分化：（1）电池储能专注于日内小时级波动平抑，SOC日内波动幅度从2030年的平均45个百分点略降至2050年的42个百分点；（2）氢储能强化季节性调节，SOC季节性波动幅度从2030年的25个百分点扩大至2050年的35个百分点；（3）甲醇储罐则主要用于平抑周际波动，SOC周际波动幅度基本保持不变（10-12个百分点）。

2.3 聚类1：亚洲中纬度风光均衡型（47.5°N, 54.0°E）
资源禀赋与技术配置（2030年）：该聚类代表的亚洲中纬度地区（哈萨克斯坦中部）采用了风光相对均衡的配置模式，风电装机容量26.65 MW，光伏31.86 MW，风光比例为0.84:1，是绿醇系统三个聚类中风光配置最为均衡的。电解槽配置仍以PEM技术为主导（16.26 MW），但AE电解槽开始出现微量配置（0.04 kW），体现了2030年技术路径的初步分化。关键设备配置包括：热泵2.43 kW_th，储热罐0.79 kWh_th（约为日均热需求的3.0%，显著高于聚类0的0.6%），DAC容量0.39 kg CO₂/h（是聚类0的3.5倍），CO₂储罐容量35.70 kg（是聚类0的2.3倍）。

年度运行模式剖析：风光均衡互补的优势：风电发电热力图呈现出相对平缓的季节性波动，夏季（6-8月）风电归一化强度维持在0.70-0.82区间，冬季（12-2月）降至0.58-0.70区间，季节性波动系数约为0.18，显著低于聚类0的0.32，反映了中纬度西风带的全年相对稳定性。光伏发电热力图则展现出标准的季节性周期，夏季光伏归一化强度在午间10-17时可达0.88-0.96，冬季午间11-15时维持在0.72-0.85，全年日照时长稳定在10-15小时/天。最关键的是，风光发电的日内互补效应极为显著：夏季典型日，夜间（22时-次日8时）风电归一化强度0.72-0.78，白天（10-18时）光伏归一化强度0.88-0.96，两者的"无缝衔接"使得全天电力供应极为平稳，电解槽和DAC系统能够保持高负荷稳定运行。

电解槽与DAC的协同高效运行：电解槽运行强度热力图呈现出全年高位稳定的特征，冬季负荷率维持在70-85%，夏季维持在75-88%，年均利用率达78.0%，是三个聚类中最高的，显著高于聚类0的61.9%。这一优异表现源于风光资源的高度互补性使得电力供应波动最小化。DAC系统运行强度热力图则展现出与聚类0截然不同的"全年高位+夏季微峰"模式：全年DAC归一化强度维持在0.78-0.92的高位区间，夏季（6-8月）略微提升至0.85-0.92，冬季（12-2月）略微下降至0.78-0.88，季节性波动幅度仅为10-15个百分点，显著低于聚类0的35-40个百分点。这一模式反映了：（1）中纬度温和气候（冬季-5至-15℃，夏季20-30℃）使得DAC全年能效比保持在1.48-1.68 kg CO₂/kWh的较高水平；（2）风光均衡供电为DAC提供了全年充裕且低成本的电力；（3）DAC年运行时间约为7280小时（利用率83.1%），显著高于聚类0的6800小时（77.6%）。

热泵系统的全年高效运行：热泵系统运行强度热力图展现出"全年中高位+季节性微调"的模式，冬季热泵归一化强度维持在0.78-0.92，COP约为2.5-2.8（vs. 聚类0的2.0-2.3），夏季归一化强度降至0.62-0.75，COP提升至3.5-3.9（vs. 聚类0的3.8-4.2）。储热罐容量相对较大（日均热需求的3.0%），使得储热罐SOC呈现明显的日内波动（波动幅度15-25个百分点），主要用于平抑光伏发电的日内波动对热泵供热的影响：午间光伏峰值时段（11-16时）热泵满负荷运行并向储热罐充热，傍晚和夜间（18时-次日8时）热泵降负荷运行并从储热罐放热，实现了热能的跨时段调配。

氢储能与CO₂储能的弱季节性调节：氢储能SOC热力图展现出"周际波动为主+季节性微调"的模式，全年SOC整体在58-75%的中高位区间波动,周际波动幅度约为8-12个百分点，季节性波动幅度仅为冬夏最大差值18个百分点，显著低于聚类0的25个百分点。CO₂储能SOC热力图同样展现出弱季节性特征，夏季CO₂储罐SOC微增（月均净增5-8个百分点），冬季微降（月均净降4-6个百分点），季节性调节幅度约为冬夏最大差值22个百分点，显著低于聚类0的45-50个百分点。这些特征共同反映了风光均衡地区对长周期储能需求的大幅降低。

甲醇合成装置的高负荷稳定运行：甲醇合成装置运行强度热力图呈现出极高的稳定性和负荷率，全年归一化强度维持在0.76-0.88的窄幅区间，季节性波动幅度仅为12个百分点（冬季0.82-0.88，夏季0.76-0.82），日内波动幅度小于3%，周际波动幅度小于5%。年平均利用率达82.7%，是三个聚类中最高的，显著高于聚类0的80.4%，反映了风光均衡地区为甲醇合成装置提供了最为稳定的氢气和CO₂供应，实现了化工装置的高效运行。

2.4 技术演进趋势：聚类1从2030年至2050年的系统优化路径
风光配比的优化调整：到2050年，该聚类的风光配比发生显著调整，风电容量降至22.62 MW（降幅15.1%），光伏容量增加至38.22 MW（增幅20.0%），风光比例调整为0.59:1，更加偏向光伏主导。这一调整反映了光伏技术成本在该地区的快速下降与该地区良好光照资源的协同效应。年度热力图显示，2050年风光发电的日内互补效应进一步增强：夏季典型日，夜间风电归一化强度0.68-0.75，白天光伏归一化强度0.90-0.98，全天电力供应的平稳性进一步提升，电解槽负荷率标准差从2030年的0.12降至2050年的0.09（降幅25%）。

电解槽技术组合的渐进式优化：2050年电解槽总容量调整至17.63 MW（增幅8.4%），技术组合发生显著变化：AE电解槽容量提升至4.69 MW（增幅117倍），PEM容量降至12.94 MW（降幅20.4%），AE技术占比从2030年的0.2%提升至2050年的26.6%，是三个聚类中AE技术占比增长最快的。年度热力图对比显示，2050年电解槽运行强度的全年稳定性进一步提升：年均负荷率从2030年的78.0%提升至2050年的82.5%（提升5.8%），季节性波动系数从0.12降至0.09（降幅25%）。技术组合优化的关键在于：夏季光伏充裕时段（10-18时）AE和PEM电解槽均满负荷运行，夜间风电时段（20时-次日8时）PEM电解槽维持高负荷（75-85%）而AE电解槽降至中等负荷（50-65%），利用两类技术的差异化响应特性实现了系统经济性与灵活性的最优平衡。

DAC与热泵系统的技术升级：2050年DAC容量微降至1.87×10⁻⁷ kg CO₂/h（降幅99.995%），CO₂储罐容量微降至3.3×10⁻⁵ kg（降幅99.991%），热泵容量降至1.87 kW_th（降幅23.0%），储热罐容量微降至1.7×10⁻⁵ kWh_th（降幅99.998%）。这一反常识的演变与聚类0类似，反映了DAC技术在成本和能效方面仍面临的挑战，系统转向"外购CO₂"模式更具经济性。热泵容量降低主要得益于热泵COP从2.5-3.9提升至2.8-4.5，相同热输出所需装机容量降低。

储能系统的精细化配置：2050年电池储能容量增至2.64 kWh（vs. 2030年的1.69 kWh，增幅56.2%），氢储能容量增至4.38 kg H₂（vs. 2030年的1.87 kg H₂，增幅134.2%），甲醇储罐容量略降至4.38 kg（vs. 2030年的1.98 kg，增幅121.2%）。年度热力图对比揭示了储能系统配置优化的逻辑：（1）电池储能扩容主要用于应对光伏占比提升后日内波动的增强，SOC日内波动幅度从2030年的35个百分点略增至2050年的38个百分点；（2）氢储能扩容主要用于强化周际调节功能（而非季节性调节，因该地区季节性波动本就较小），SOC周际波动幅度从2030年的10个百分点扩大至2050年的14个百分点；（3）甲醇储罐则保持相对稳定的配置水平。
聚类簇2（南美洲-36.25°S/-70.25°W，纯光伏主导系统） 代表了南半球中纬度地区典型的太阳能资源富集区。该代表点位于智利中部，具有世界级的太阳辐照条件和相对温和的气候特征。2030年基准配置显示出极端的光伏主导特征，风电装机容量仅为0.00000142 MW（接近于零），而光伏装机达到68.81 MW，形成了风光比0.00002:1的纯光伏系统。这种配置反映了该地区"阿塔卡马沙漠效应"——全球最干燥地区之一带来的极高太阳辐照强度（年均DNI超过3000 kWh/m²）与极低的云量覆盖。电解系统初期仍以PEM技术为主（9.01 MW），AE电解槽装机微乎其微（0.022 MW，占比0.24%），电解槽年利用小时数达到6774小时（利用率77.3%），显示出光伏发电的高容量因子特性。储能配置呈现"电池主导、氢储补充"的典型光伏系统特征：电池储能容量42.82 MWh、功率12.22 MW，年利用循环次数251次；氢储罐容量3.77 kg H₂，主要用于日内至周内的负荷平滑。DAC系统装机0.20 kg CO₂/h，年捕集量1432 kg CO₂，利用率81.7%，显示出与中纬度地区类似的全年高稳定运行特性。热泵系统装机0.42 MW，配合光伏发电的日间高峰期运行，年运行小时数7156小时（81.7%），为甲醇合成提供稳定的蒸汽供给。甲醇合成单元装机0.31 kg MeOH/h，年产量2200 kg，合成单元利用率80.8%，略低于风电主导系统但显著高于纯光伏的氨系统（63.1%），反映了DAC-热泵-合成的协同优化效果。

年度运行特征热图揭示了纯光伏系统的极端昼夜循环规律。光伏发电呈现"日出而作、日落而息"的严格昼夜模式，每日6:00-18:00时段为高发电期（归一化强度0.8-1.0），夜间完全停机，形成矩形波动模式。这种极端的昼夜变化导致系统必须依赖大容量电池储能进行日内平衡：电池SOC曲线呈现标准的"日间充电、夜间放电"模式，每日充放电深度达50-70%，年循环次数高达251次，远超风电主导系统（北美61.5°N为119次）。电解槽运行策略采取"追光模式"，日间时段满负荷运行（归一化强度0.9-1.0），夜间通过电池放电维持40-60%的基础负荷，避免频繁启停对设备寿命的影响。氢储罐SOC呈现"日积月累"特征，主要在周末及光伏出力波动较大的阴天积累氢气库存，平均储氢量维持在50-70%水平。CO₂储罐则表现出与氢储相反的"季节性对冲"模式：夏季（12月-2月，南半球）光伏高发期DAC大量捕集CO₂导致储罐快速充满（SOC达90%以上），冬季（6月-8月）光伏出力下降时逐步消耗库存支撑甲醇合成，形成典型的"夏储冬用"季节调节功能。甲醇合成单元在全年维持相对稳定的运行强度（0.7-0.9），通过CO₂储罐的季节调节平滑了光伏发电的季节性波动，实现80.8%的高利用率。

2030年至2050年的技术演进路径显示出碱性电解槽在纯光伏系统中的快速渗透。2050年配置中，风光比调整为0.00003:1（风电0.00000205 MW，光伏68.81 MW），仍保持纯光伏特征，但电解系统结构发生根本性变化：AE装机大幅增至4.03 MW，占比达44.8%，而PEM装机降至4.96 MW（占比55.2%），AE渗透率提升幅度在所有系统中最为显著。这一变化反映了光伏发电"日间集中、夜间停机"的负荷特性与碱性电解槽"适应大功率波动"的技术特性高度匹配——日间光伏高峰时段AE满负荷运行，夜间完全停机进行维护，避免了PEM技术在低负荷运行时的效率损失。电池储能容量扩张至66.87 MWh（+56.2%），功率提升至19.06 MW（+56.0%），年循环次数增至306次，单次循环深度保持在60%左右，显示出储能系统在应对更高比例AE间歇运行中的关键支撑作用。氢储罐容量扩大至8.83 kg H₂（+134.2%），但年平均储氢量仍维持在55%水平，主要用于周内至月内的负荷平滑。CO₂储罐容量略降至2471 kg CO₂（-5.9%），反映了系统在技术成熟后对季节储能需求的优化——通过更高效的DAC运行策略（利用率从81.7%提升至82.5%）和热泵系统效率提升（COP从3.2提升至3.8），减少了对大容量CO₂储存的依赖。甲醇合成单元装机维持0.31 kg MeOH/h不变，但年产量增至2246 kg（+2.1%），合成单元利用率提升至82.4%，系统整体平准化成本从2030年的4.89美元/kg MeOH降至2050年的3.24美元/kg MeOH（-33.7%），成本下降幅度在所有甲醇系统中位居前列，充分体现了光伏发电成本快速下降与碱性电解槽技术进步的协同效应。

绿色甲醇系统的跨区域比较与技术演进规律 揭示了以下关键特征：

（1）资源禀赋主导系统架构：高纬度地区（68.75°N）呈现风电绝对主导（风光比6.5:1），中纬度地区（47.5°N）实现风光均衡（风光比0.84:1至0.59:1），低纬度纯光伏区（-36.25°S）形成极端光伏依赖（风光比<0.0001:1）。风光比的地理梯度直接决定了储能配置策略：风电主导系统依赖氢储罐进行周至季节调节（氢储容量4.48 kg占主导），风光均衡系统采用"电池+氢储"双轨制（电池66.96 MWh + 氢储8.34 kg），纯光伏系统强化电池日内平衡（电池66.87 MWh，年循环306次）。

（2）DAC-热泵-合成三联供系统的地理适应性：DAC运行模式呈现"纬度分异"特征——高纬度极寒地区（68.75°N）受冬季低温限制形成"夏高冬低"模式（冬季利用率<20%），中纬度温和地区（47.5°N）实现全年高稳定运行（利用率83.1%），纯光伏区（-36.25°S）维持全年高运行但季节性储存CO₂（利用率81.7%，夏季储罐充满）。热泵系统则与DAC形成"反向耦合"：高纬度地区冬季供暖需求与DAC低运行率矛盾，导致热泵利用率仅75.6%；中低纬度地区热泵全年稳定运行（利用率>81%），与DAC协同性更优。甲醇合成单元利用率在风光均衡系统中最高（82.7%），纯光伏系统次之（80.8%），风电主导系统最低（77.4%），反映了风光互补性对下游化工单元稳定运行的重要性。

（3）碱性电解槽的差异化渗透路径：2030至2050年间，AE渗透率呈现"光伏系统快、风电系统慢"的规律——纯光伏系统AE占比从0.24%跃升至44.8%（增长187倍），风光均衡系统从1.1%增至26.6%（增长24倍），风电主导系统从0.11%增至15.5%（增长141倍但绝对水平最低）。这一差异源于技术特性匹配度：光伏发电的"日间集中"特性与AE"适应大功率波动"能力高度契合，而风电的"连续波动"特性更适合PEM的快速响应。电解槽年利用小时数在风电主导系统中最高（7671小时，87.6%），风光均衡系统次之（6819小时，77.9%），纯光伏系统最低（6774小时，77.3%），但2050年随着AE渗透，纯光伏系统利用率提升最显著（+2.3个百分点）。

（4）储能系统的功能分化与协同优化：电池储能主导日内平衡，年循环次数随光伏占比提升而增加（风电主导119次 < 风光均衡241次 < 纯光伏306次）；氢储罐主导周至月调节，储氢容量随风电波动性增强而增大（纯光伏8.83 kg < 风光均衡8.34 kg < 风电主导4.48 kg，但风电系统单位产能氢储需求更高）；CO₂储罐主导季节对冲，容量配置与气候季节性正相关（中纬度四季分明地区2627 kg > 纯光伏温和地区2471 kg > 高纬度极端季节地区，但高纬度地区储罐利用率最低反映冬季DAC停机问题）。三级储能的协同优化实现了"日-周-季"三重时间尺度的能量管理，确保甲醇合成单元在全年维持高稳定运行（利用率77.4%-82.7%）。

（5）技术经济性的地理与时间演进：2030年平准化成本呈现"中纬度最优、高纬度次之、纯光伏最高"格局（中纬度4.32美元/kg < 高纬度4.58美元/kg < 纯光伏4.89美元/kg），反映了风光互补性与系统稳定性的经济价值。2050年成本下降幅度呈现"光伏系统快、风电系统慢"特征（纯光伏-33.7% > 中纬度-30.8% > 高纬度-29.3%），得益于光伏组件成本的陡降曲线（2030-2050年光伏LCOE下降60% vs. 风电下降40%）与碱性电解槽的规模化部署（AE CAPEX下降50% vs. PEM下降30%）。到2050年，中纬度风光均衡系统的成本优势进一步扩大（2.99美元/kg MeOH），成为全球绿色甲醇生产的最优技术路径。
聚类簇2（南美洲-36.25°S/-70.25°W，纯光伏主导系统） 代表了南半球中纬度地区典型的太阳能资源富集区。该代表点位于智利中部，具有世界级的太阳辐照条件和相对温和的气候特征。2030年基准配置显示出极端的光伏主导特征，风电装机容量仅为0.00000142 MW（接近于零），而光伏装机达到68.81 MW，形成了风光比0.00002:1的纯光伏系统。这种配置反映了该地区"阿塔卡马沙漠效应"——全球最干燥地区之一带来的极高太阳辐照强度（年均DNI超过3000 kWh/m²）与极低的云量覆盖。电解系统初期仍以PEM技术为主（9.01 MW），AE电解槽装机微乎其微（0.022 MW，占比0.24%），电解槽年利用小时数达到6774小时（利用率77.3%），显示出光伏发电的高容量因子特性。储能配置呈现"电池主导、氢储补充"的典型光伏系统特征：电池储能容量42.82 MWh、功率12.22 MW，年利用循环次数251次；氢储罐容量3.77 kg H₂，主要用于日内至周内的负荷平滑。DAC系统装机0.20 kg CO₂/h，年捕集量1432 kg CO₂，利用率81.7%，显示出与中纬度地区类似的全年高稳定运行特性。热泵系统装机0.42 MW，配合光伏发电的日间高峰期运行，年运行小时数7156小时（81.7%），为甲醇合成提供稳定的蒸汽供给。甲醇合成单元装机0.31 kg MeOH/h，年产量2200 kg，合成单元利用率80.8%，略低于风电主导系统但显著高于纯光伏的氨系统（63.1%），反映了DAC-热泵-合成的协同优化效果。

年度运行特征热图揭示了纯光伏系统的极端昼夜循环规律。光伏发电呈现"日出而作、日落而息"的严格昼夜模式，每日6:00-18:00时段为高发电期（归一化强度0.8-1.0），夜间完全停机，形成矩形波动模式。这种极端的昼夜变化导致系统必须依赖大容量电池储能进行日内平衡：电池SOC曲线呈现标准的"日间充电、夜间放电"模式，每日充放电深度达50-70%，年循环次数高达251次，远超风电主导系统（北美61.5°N为119次）。电解槽运行策略采取"追光模式"，日间时段满负荷运行（归一化强度0.9-1.0），夜间通过电池放电维持40-60%的基础负荷，避免频繁启停对设备寿命的影响。氢储罐SOC呈现"日积月累"特征，主要在周末及光伏出力波动较大的阴天积累氢气库存，平均储氢量维持在50-70%水平。CO₂储罐则表现出与氢储相反的"季节性对冲"模式：夏季（12月-2月，南半球）光伏高发期DAC大量捕集CO₂导致储罐快速充满（SOC达90%以上），冬季（6月-8月）光伏出力下降时逐步消耗库存支撑甲醇合成，形成典型的"夏储冬用"季节调节功能。甲醇合成单元在全年维持相对稳定的运行强度（0.7-0.9），通过CO₂储罐的季节调节平滑了光伏发电的季节性波动，实现80.8%的高利用率。

2030年至2050年的技术演进路径显示出碱性电解槽在纯光伏系统中的快速渗透。2050年配置中，风光比调整为0.00003:1（风电0.00000205 MW，光伏68.81 MW），仍保持纯光伏特征，但电解系统结构发生根本性变化：AE装机大幅增至4.03 MW，占比达44.8%，而PEM装机降至4.96 MW（占比55.2%），AE渗透率提升幅度在所有系统中最为显著。这一变化反映了光伏发电"日间集中、夜间停机"的负荷特性与碱性电解槽"适应大功率波动"的技术特性高度匹配——日间光伏高峰时段AE满负荷运行，夜间完全停机进行维护，避免了PEM技术在低负荷运行时的效率损失。电池储能容量扩张至66.87 MWh（+56.2%），功率提升至19.06 MW（+56.0%），年循环次数增至306次，单次循环深度保持在60%左右，显示出储能系统在应对更高比例AE间歇运行中的关键支撑作用。氢储罐容量扩大至8.83 kg H₂（+134.2%），但年平均储氢量仍维持在55%水平，主要用于周内至月内的负荷平滑。CO₂储罐容量略降至2471 kg CO₂（-5.9%），反映了系统在技术成熟后对季节储能需求的优化——通过更高效的DAC运行策略（利用率从81.7%提升至82.5%）和热泵系统效率提升（COP从3.2提升至3.8），减少了对大容量CO₂储存的依赖。甲醇合成单元装机维持0.31 kg MeOH/h不变，但年产量增至2246 kg（+2.1%），合成单元利用率提升至82.4%，系统整体平准化成本从2030年的4.89美元/kg MeOH降至2050年的3.24美元/kg MeOH（-33.7%），成本下降幅度在所有甲醇系统中位居前列，充分体现了光伏发电成本快速下降与碱性电解槽技术进步的协同效应。

绿色甲醇系统的跨区域比较与技术演进规律 揭示了以下关键特征：

（1）资源禀赋主导系统架构：高纬度地区（68.75°N）呈现风电绝对主导（风光比6.5:1），中纬度地区（47.5°N）实现风光均衡（风光比0.84:1至0.59:1），低纬度纯光伏区（-36.25°S）形成极端光伏依赖（风光比<0.0001:1）。风光比的地理梯度直接决定了储能配置策略：风电主导系统依赖氢储罐进行周至季节调节（氢储容量4.48 kg占主导），风光均衡系统采用"电池+氢储"双轨制（电池66.96 MWh + 氢储8.34 kg），纯光伏系统强化电池日内平衡（电池66.87 MWh，年循环306次）。

（2）DAC-热泵-合成三联供系统的地理适应性：DAC运行模式呈现"纬度分异"特征——高纬度极寒地区（68.75°N）受冬季低温限制形成"夏高冬低"模式（冬季利用率<20%），中纬度温和地区（47.5°N）实现全年高稳定运行（利用率83.1%），纯光伏区（-36.25°S）维持全年高运行但季节性储存CO₂（利用率81.7%，夏季储罐充满）。热泵系统则与DAC形成"反向耦合"：高纬度地区冬季供暖需求与DAC低运行率矛盾，导致热泵利用率仅75.6%；中低纬度地区热泵全年稳定运行（利用率>81%），与DAC协同性更优。甲醇合成单元利用率在风光均衡系统中最高（82.7%），纯光伏系统次之（80.8%），风电主导系统最低（77.4%），反映了风光互补性对下游化工单元稳定运行的重要性。

（3）碱性电解槽的差异化渗透路径：2030至2050年间，AE渗透率呈现"光伏系统快、风电系统慢"的规律——纯光伏系统AE占比从0.24%跃升至44.8%（增长187倍），风光均衡系统从1.1%增至26.6%（增长24倍），风电主导系统从0.11%增至15.5%（增长141倍但绝对水平最低）。这一差异源于技术特性匹配度：光伏发电的"日间集中"特性与AE"适应大功率波动"能力高度契合，而风电的"连续波动"特性更适合PEM的快速响应。电解槽年利用小时数在风电主导系统中最高（7671小时，87.6%），风光均衡系统次之（6819小时，77.9%），纯光伏系统最低（6774小时，77.3%），但2050年随着AE渗透，纯光伏系统利用率提升最显著（+2.3个百分点）。

（4）储能系统的功能分化与协同优化：电池储能主导日内平衡，年循环次数随光伏占比提升而增加（风电主导119次 < 风光均衡241次 < 纯光伏306次）；氢储罐主导周至月调节，储氢容量随风电波动性增强而增大（纯光伏8.83 kg < 风光均衡8.34 kg < 风电主导4.48 kg，但风电系统单位产能氢储需求更高）；CO₂储罐主导季节对冲，容量配置与气候季节性正相关（中纬度四季分明地区2627 kg > 纯光伏温和地区2471 kg > 高纬度极端季节地区，但高纬度地区储罐利用率最低反映冬季DAC停机问题）。三级储能的协同优化实现了"日-周-季"三重时间尺度的能量管理，确保甲醇合成单元在全年维持高稳定运行（利用率77.4%-82.7%）。

（5）技术经济性的地理与时间演进：2030年平准化成本呈现"中纬度最优、高纬度次之、纯光伏最高"格局（中纬度4.32美元/kg < 高纬度4.58美元/kg < 纯光伏4.89美元/kg），反映了风光互补性与系统稳定性的经济价值。2050年成本下降幅度呈现"光伏系统快、风电系统慢"特征（纯光伏-33.7% > 中纬度-30.8% > 高纬度-29.3%），得益于光伏组件成本的陡降曲线（2030-2050年光伏LCOE下降60% vs. 风电下降40%）与碱性电解槽的规模化部署（AE CAPEX下降50% vs. PEM下降30%）。到2050年，中纬度风光均衡系统的成本优势进一步扩大（2.99美元/kg MeOH），成为全球绿色甲醇生产的最优技术路径。

四、绿氨系统跨区域综合比较与技术演进规律

4.1 资源禀赋对系统架构的决定性影响

三个聚类的对比分析清晰揭示了资源禀赋对绿氨系统架构设计的决定性作用。风光配比的地理梯度呈现明确规律：高纬度地区（61.5°N）风光比1.22:1，仍维持风电主导但光伏显著增强（相比2030年的4.4:1大幅收敛）；低纬度地区（17.5°N）风光比0.83:1，实现接近完美的风光均衡；纯光伏区（37.5°N）风光比0.000001:1，风电资源完全缺失。这种地理梯度直接源于全球大气环流格局：高纬度西风带提供稳定风能但光照季节性极端，低纬度信风带与太阳直射带实现风光全年均衡，中纬度地中海气候区光照充沛但地形屏蔽风能。

储能配置策略的资源适应性分化同样显著。电池储能单位装机配比呈现"风光均衡最高（0.128 MWh/MW）> 纯光伏次之（0.090 MWh/MW）> 风电主导最低（0.028 MWh/MW）"的规律，反映了风光互补系统虽然总体稳定性最优，但日内多峰充放电模式（"三充两放"）对电池调节频次的更高需求；纯光伏系统虽然仅有昼夜单一周期，但极端的充放电深度（82-90个百分点）同样需要大容量电池支撑；风电主导系统连续发电特性使得电池仅需应对小时级波动。

氢储罐容量配置呈现截然相反的规律："风电主导最高（16.3倍日产量）> 纯光伏次之（5.45倍）> 风光均衡最低（1.22倍）"，反映了不同时间尺度储能需求的本质差异。高纬度风电系统需要氢储罐承担季节调节功能（"冬储夏用"），容量需覆盖数月尺度的供需失衡；纯光伏系统需要氢储罐应对多日天气波动（2-5天连续阴雨），容量需覆盖周尺度缺口；风光均衡系统风光互补效应极佳，氢储罐仅需应对单日至周际的微小波动，容量需求降至最低。两种储能的功能分化清晰："电池储能主导日内平衡，氢储能主导周至季节调节"，且两者需求呈现"此消彼长"关系——风光互补性越好，电池需求略增但氢储需求大幅下降，系统总储能成本最优。

4.2 电解槽技术渗透的地理与经济双重驱动

AE电解槽在2040年LowAE情景下的渗透呈现明显的地理分异：风光均衡系统渗透率最高（51.5%），纯光伏系统次之（46.8%），风电主导系统最低（12.7%）。这一规律背后是技术特性与资源特征的匹配度差异：

风光均衡系统（聚类1）：风光"夜间风电峰+午间光伏峰"的双峰供电模式与AE"适应大功率波动、不适宜频繁启停"的技术特性高度契合。系统采用"PEM全时运行（68%利用率）+AE峰时满负荷（65%利用率）"的技术分工，AE在夜间风电峰和午间光伏峰时段满负荷运行（日运行时长12-14小时），过渡时段停机或低负荷运行，既发挥了AE的成本优势（投资成本仅PEM的37.5%），又维持了较高利用率（65% vs. 理论最大值约58%，即14/24）。关键成功因素是全年气候稳定（季节波动小）与风光互补性好（双峰供电全年持续），使得AE的峰时运行模式在全年大部分时段均可实现。

纯光伏系统（聚类2）：光伏"午间集中发电、夜间完全停摆"的极端特性理论上非常适合AE技术（午间6-8小时满负荷、夜间完全停机），AE占比46.8%接近纯光伏系统的技术极限。但AE利用率仅38%（vs. PEM的36%），显著低于风光均衡系统的65%，主要受制于：（1）光伏发电时长短（夏季13-14小时、冬季10-11小时、年平均11.5小时），AE理论最大利用率即48%；（2）冬季阴雨天气导致光伏间歇性低发甚至停发，AE被迫频繁停机，实际运行时长进一步缩短。尽管如此，AE的极低成本（2040年约300美元/kW）仍使其在纯光伏系统中具备经济性，但进一步提升占比的空间有限（提升至55-60%将导致AE利用率降至30%以下，边际经济效益递减）。

风电主导系统（聚类0）：风电"连续波动、快速爬坡"的出力特性与AE"分钟级响应、窄负荷域（20-100%）"的技术局限性存在根本矛盾。系统虽然引入12.7%的AE电解槽，但其运行策略高度保守：仅在凌晨和傍晚风电高峰时段（负荷率>85%、持续时间>2小时）启动，其余时段停机，导致AE年利用率仅52%（vs. PEM的61%）。AE渗透率低的核心原因是高纬度地区极端季节性波动（冬夏风电差异50%）与日内三峰波动模式（峰谷差20-30%）要求电解系统具备极强的动态响应能力，PEM的秒级响应与宽负荷域（0-110%）优势在此场景下不可替代。未来即使AE成本进一步下降，其在风电主导系统中的占比也难以超过20-25%，除非配套氢储罐大幅扩容（提升至30-40倍日产量）以缓冲AE的间歇运行，但这将显著增加系统总成本。

技术经济权衡的量化结论：AE技术渗透的经济临界点取决于"AE成本优势"与"利用率劣势"的平衡。以2040年成本参数（AE 300美元/kW、PEM 800美元/kW）计算，当AE年利用率>45%时，其平准化制氢成本低于PEM（1.68 vs. 2.24美元/kg）；当AE利用率<35%时，成本优势消失甚至劣于PEM。风光均衡系统AE利用率65%远超临界点，可继续提升至60-65%占比；纯光伏系统AE利用率38%接近临界点，占比已达上限；风电主导系统AE利用率52%虽超临界点但受技术特性限制，占比难以大幅提升。

4.3 系统运行稳定性的风光互补效应量化

电解槽运行稳定性指标清晰量化了风光互补的价值。电解槽负荷率季节波动系数（标准差/均值）：风光均衡系统0.09（最优）< 纯光伏系统0.30 < 风电主导系统0.46（最差），反映了风光均衡系统全年运行最平稳、纯光伏系统受冬季阴雨影响出现中等波动、风电主导系统受极端季节性影响波动最剧烈。

日内负荷率波动幅度：风光均衡系统25-35个百分点（最小）< 风电主导系统30-50个百分点 < 纯光伏系统70-85个百分点（最大），反映了纯光伏系统昼夜极端不均对电解槽动态调节能力的极高要求。电解槽年利用率：风光均衡66.3%（最高）> 风电主导58.6% > 纯光伏37.0%（最低），直接体现在设备投资效率与制氢成本上。

合成氨装置运行稳定性的地理差异同样显著。合成单元负荷率标准差：风光均衡0.052（最优）< 纯光伏0.065 < 风电主导0.078（最差），反映了风光均衡系统通过小容量氢储罐（1.22倍日产量）即可实现下游高稳定运行，纯光伏系统需超大容量氢储罐（5.45倍）才能维持稳定，风电主导系统即使配置大容量氢储罐（16.3倍）仍难以完全平抑上游波动。

合成单元年利用率：纯光伏78.8%（最高，看似矛盾）> 风光均衡72.4% > 风电主导71.9%，这一"倒挂"现象源于系统优化策略差异：纯光伏系统通过"牺牲电解槽利用率（37.0%）+配置超大氢储罐（5.45倍）"的策略，将有限的氢气优先保障下游合成装置高负荷稳定运行，在合成环节实现价值最大化；风光均衡与风电主导系统则追求电解槽与合成装置的均衡利用。

4.4 系统经济性的地理梯度与优化潜力

平准化氨成本的地理梯度明确：风光均衡427.52美元/吨（最优）< 纯光伏462.39美元/吨（+8.1%）< 风电主导533.99美元/吨（+24.9%），清晰揭示了低纬度风光均衡系统的绝对经济优势。成本差异的来源分解：

电解环节成本差异（占总成本55-65%）：风光均衡系统电解槽高利用率（66.3%）与AE高占比（51.5%）使得制氢成本降至1.68美元/kg，相比风电主导（2.24美元/kg）降低25%、相比纯光伏（1.94美元/kg）降低13%。电解环节成本优势贡献了系统总成本差异的60-70%。

储能环节成本差异（占总成本10-20%）：风光均衡系统虽然电池储能配比最高（0.128 MWh/MW），但氢储罐配比极低（1.22倍日产量），两者综合后储能总成本占比仅12%；风电主导系统电池储能配比低（0.028 MWh/MW）但氢储罐配比极高（16.3倍），储能总成本占比达18%；纯光伏系统电池（0.090 MWh/MW）与氢储（5.45倍）双高，储能总成本占比达15%。关键洞察：氢储罐单位容量成本（约150-200美元/kg H₂）显著高于电池储能单位能量成本（约120-150美元/kWh），且氢储罐年利用率低（季节储能年循环仅2-4次 vs. 电池日内储能年循环300-400次），导致氢储罐平准化成本远高于电池储能。风光均衡系统通过最小化氢储罐需求实现储能成本最优。

合成环节成本差异（占总成本15-25%）：三个系统合成装置容量相近（12.70-13.91 kg NH₃/h），投资成本差异小，但运行稳定性差异导致能耗与催化剂消耗差异：风光均衡系统高稳定运行（负荷率>70%占比88%）使得吨氨能耗降至最低（约9.5 GJ/t NH₃），催化剂寿命最长（>4年）；风电主导与纯光伏系统受上游波动影响，吨氨能耗略高（10.0-10.5 GJ/t NH₃），催化剂寿命略短（3-3.5年）。合成环节成本差异虽然占比较小（5-8%），但反映了系统稳定性的隐性价值。

影子价格波动幅度的经济含义：风光均衡系统电力影子价格年波动范围0.0265-0.0285美元/kWh（差异7.5%）、日内波动±17-23%，价格信号微弱，系统已接近均衡优化状态；风电主导系统电力价格年波动0.029-0.038美元/kWh（差异31%）、日内波动±25-65%，价格信号强烈，存在较大优化空间（如季节性生产调节、需求侧响应）；纯光伏系统电力价格年波动0.0365-0.0425美元/kWh（差异16%）、日内波动±60-80%，极端价差为长时储能与混合储能提供经济激励。

氢气影子价格波动同样反映系统紧张度：风光均衡系统氢价年波动1.68-1.75美元/kg（差异4%）、周内波动<2%，供需高度平衡；风电主导系统氢价年波动2.15-2.38美元/kg（差异11%）、周内波动±2-5%，夏季供需偏紧；纯光伏系统氢价年波动1.94-2.28美元/kg（差异18%）、周内波动±8-15%（冬季阴雨周），供需波动较大。氢价波动幅度与氢储罐容量配置呈负相关：氢储罐容量越大，价格平抑效果越好，但投资成本越高。

4.5 未来优化路径与技术演进方向

基于三个聚类的比较分析，绿氨系统未来优化路径呈现明确的地理分异策略：

**风光均衡系统（聚类1）**已接近技术经济最优配置，优化空间有限（<5%）。主要方向：（1）AE占比进一步提升至60-65%，需权衡电池储能扩容成本（增幅10-15%）与制氢成本下降（降幅8-10%）；（2）电池储能循环深度优化，从当前85%降至60-70%，以延长电池寿命（从6-8年延长至10-12年），虽然需增加电池容量15-20%，但全生命周期成本可降低10-12%；（3）探索氢储罐容量精简可能性（从1.22倍降至0.8-1.0倍），需评估极端天气（连续7天阴雨）风险，潜在成本节约2-3%。

**风电主导系统（聚类0）**优化空间较大（10-15%）。主要方向：（1）继续提升光伏装机比例（从当前44.9%提升至55-60%），风光比从1.22:1收敛至0.8-0.9:1，充分利用光伏成本持续下降（2040-2050年预计再降30-40%）改善夏季供电，可降低氢储罐季节调节需求（容量从16.3倍降至12-14倍），系统成本可降低8-10%；（2）季节性生产调节策略，冬季高风电期满负荷生产并储氨（扩大氨储罐至150-200 kg，约15-20天产量），夏季低风电期计划性降负荷或检修，全年平均成本可降低5-7%；（3）AE占比适度提升至18-22%，配合氢储罐扩容（提升至20-25倍日产量）缓冲AE间歇运行，需评估储能投资增加与制氢成本下降的平衡点。

**纯光伏系统（聚类2）**优化空间最大（15-20%）。主要方向：（1）混合储能策略，"锂电池（4小时，应对午后峰值与傍晚过渡）+液流电池（8小时，承担夜间长时放电）"配合，利用液流电池低循环成本（0.015美元/kWh）与长寿命（>10000次）优势，可降低储能综合成本15-20%；（2）强制性季节调节，夏季（6-9月）满负荷生产+大容量氨储存（扩容至500-600 kg），冬季（12-2月）降负荷+消耗储备，全年平均成本可降低5-7%；（3）探索海上风电补充（5-10 MW，占总装机7-14%），提供夜间基础电力，可降低电池需求20-30%，但需权衡海上风电高投资（1500-1800美元/kW）与输电成本；（4）光伏跟踪系统（单轴或双轴）提升发电量15-25%，可提升电解槽利用率至42-45%，但需权衡跟踪系统投资（增加光伏成本20-30%）与发电量提升的经济性。

跨区域技术转移的可行性评估：风光均衡系统的技术配置（AE高占比51.5%、氢储罐小容量1.22倍、电池储能大容量0.128 MWh/MW）是否可复制至其他聚类？答案是高度受限于资源禀赋。若将该配置强制应用于风电主导系统，AE电解槽将在夏季长期停机（风电不足以支撑满负荷）、冬季频繁启停（风电三峰波动），年利用率将暴跌至25-30%（vs. 当前PEM配置的58.6%），系统成本不降反升15-20%；小容量氢储罐（1.22倍）完全无法应对季节性波动，夏季将出现严重缺氢，合成装置被迫长期停机。同样，若将纯光伏系统配置应用于风光均衡系统，超大容量氢储罐（5.45倍）将长期闲置（年利用率<30%），电池储能容量不足（0.090 MWh/MW）无法应对风光双峰充放电，系统成本增加10-12%。

核心结论：绿氨系统优化必须遵循"资源禀赋→系统架构→运行策略→技术组合"的适配逻辑，不存在普适性的"最优配置"，只有因地制宜的"局部最优解"。全球绿氨产业布局应优先选择低纬度风光均衡地区（北纬15-25°、南纬15-25°），在此基础上通过技术进步（AE成本下降、先进储能技术）与运营优化（季节调节、需求响应）持续提升经济性；高纬度与纯光伏地区可作为补充，但需接受较高成本（10-25%）并通过政策支持（如碳价、绿氢补贴）弥补经济性劣势。

5.1 聚类0：亚洲高纬度风电主导型（68.75°N, 80.0°E）

技术配置与碳氢耦合系统架构

聚类0代表点位于亚洲高纬度地区（西伯利亚中部，68.75°N, 80.0°E），处于北极圈以北的极端气候带，纬度比绿氨系统聚类0（61.5°N）更高，面临更为严酷的极地气候挑战。技术配比图显示系统采用极端风电主导配置：风电装机24.46 MW，光伏装机7.26 MW，风光比3.37:1，相比绿氨系统聚类0（风光比1.22:1）风电优势更为突出。这一配置调整反映了极高纬度地区（>65°N）光伏资源的急剧恶化——冬至前后极夜期长达2-3个月（vs. 61.5°N的1-1.5个月），光伏年发电量降低40-50%，光伏装机的边际经济效益显著下降，系统被迫更加依赖风电资源。

电解系统配置总容量17.74 MW，几乎完全采用PEM技术（AE装机仅1.67×10⁻⁵ kW，实质为零），这一"纯PEM"配置与绿氨系统聚类0（AE占比12.7%）形成对比，反映了绿醇系统对氢气供应连续性与响应速度的更严格要求。绿醇生产需要H₂和CO₂的精确同步供应（化学计量比H₂:CO₂=3:1），任何一方短缺或波动都会导致甲醇合成效率下降甚至中断，因此系统完全依赖PEM的"秒级响应+宽负荷域（0-110%）"技术优势，牺牲AE的成本优势换取供应稳定性。电解槽年利用率59.7%，略高于绿氨系统聚类0（58.6%），反映了甲醇系统对电解槽保守运行策略的更高要求——为确保氢气供应不中断，电解槽避免过度降负荷（最低负荷率维持在40%以上 vs. 绿氨系统的30%）。

储能系统配置呈现"氢储主导+电池辅助"的风电系统典型特征：氢储罐4.98 kg H₂，约为日均产氢量的9.34倍（按年产氢1947 kg计算，日均5.33 kg），远高于绿氨系统聚类0（2.98 kg，16.3倍日产量但日产量基数小）；电池储能1.85 MWh、功率16.60 MW，单位装机储能比0.058 MWh/MW，略高于绿氨系统聚类0（0.028 MWh/MW），反映了光伏装机比例提升（从18.6%增至22.9%）带来的日内波动加剧。

碳氢耦合系统的独特配置与超大容量CO₂储罐：DAC（直接空气捕集）装机2.12 kg CO₂/h，年捕集14000 kg CO₂（日均38.36 kg），利用率75.3%；热泵装机2.12 MW_th（热功率），为甲醇合成反应提供所需蒸汽（反应温度250-300°C，压力50-100 bar），热储能仅0.27 kWh_th（容量极小，主要用于数分钟至数小时的热量缓冲）；甲醇合成装机13.29 tpy（吨/年产能，折合约0.52 kg/h峰值产能），甲醇储罐210.13 kg（约16天产量），年产甲醇10000 kg，合成单元利用率75.3%（与DAC完全一致，反映两者的紧密耦合）。

最显著的特征是CO₂储罐的超大容量配置：CO₂储罐容量10037 kg，约为日均捕集量的261倍，而氢储罐仅为日均产氢量的9.34倍，两者容量倍数差异高达28倍。这一极端不对称配置反映了碳氢耦合系统在极高纬度地区面临的核心挑战：DAC捕集受气候影响极大，冬季极寒导致捕集效率暴跌甚至完全停摆，必须依赖超大容量CO₂储罐进行跨季调节；氢气生产由风电驱动，风电虽有季节性但全年持续供应，氢储罐主要应对周际波动即可。CO₂储罐投资约150万美元（按150美元/kg储存成本计算），占系统总投资（约850-950万美元）的16-18%，是储能系统中投资占比最高的单元（电池储能约占10-12%，氢储罐约占5-7%）。

年度运行特征热力图的极端季节性与碳氢异步调节

风电发电的"冬强夏弱"极端模式与日内三峰结构：风电发电热力图呈现高纬度地区典型的季节性分布，但极端程度进一步加剧。冬季时段（11月至次年3月，长达5个月）为风电高发期，归一化强度稳定在0.80-0.92区间（平均0.86），受极地涡旋南侵和西风急流增强的双重影响，该时段风速可达14-18 m/s，日内波动幅度约12-18%。具体而言，1月份典型日的风电出力曲线呈现"凌晨高峰-午后回落-傍晚次峰"的三段式结构：凌晨2-6时为第一高峰（强度0.88-0.92），此时夜间地表逆温层破坏前风速最强；午后12-16时为相对低谷（强度0.75-0.82），受地表加热大气混合层发展影响风速略有减弱；傍晚18-22时为第二高峰（强度0.82-0.88），受地表快速冷却形成的局地环流影响出现次高峰。

夏季时段（6-8月）风电发电量大幅下降，归一化强度降至0.50-0.65区间（平均0.58），仅为冬季的65-70%水平，主要源于极地高压增强导致西风带北抬与局地热力对流的扰动加剧。日内波动幅度扩大至25-32%（冬季仅12-18%），反映了夏季气象不稳定性增强。春秋过渡季（4-5月、9-10月）风电强度居于冬夏之间（0.65-0.78），但过渡期较短（各约1.5-2个月），系统很快从高风期切换至低风期或反之。全年风电季节差异达48%（冬夏强度比1.48:1），略高于绿氨系统聚类0（46%），反映了更高纬度带来的季节性恶化。

光伏发电的极端"极昼-极夜"效应与年均贡献率暴跌：光伏发电热力图展现出极高纬度地区（68.75°N）特有的极端季节性，远超绿氨系统聚类0（61.5°N）。夏至前后（6月21日±3周，约持续6周）极昼期光照时长达20-24小时/天，太阳几乎不落，但受极低太阳高度角限制（正午太阳高度角仅43-45° vs. 61.5°N的52-55°），光伏归一化强度午间仅达0.60-0.75（vs. 61.5°N的0.80-0.92），峰值持续时长虽长（14-16小时 vs. 7小时），但单位时间出力强度显著降低。更严重的问题是极昼期太阳高度角的日内变化极小（午夜30-35° vs. 正午43-45°，差异仅10-15°），导致光伏出力曲线呈现"平顶波"形态（归一化强度全天维持在0.55-0.75窄幅区间），缺乏明显的午间峰值，对系统日内负荷调节的贡献有限。

冬至前后（12月21日±6周，约持续12周，长达3个月）极夜期光照时长降至0-4小时/天，受极低太阳高度角（正午太阳高度角仅0-3°，太阳几乎贴地平线）与漫长极夜影响，光伏几乎完全停摆（归一化强度<0.05），对系统电力供应贡献趋近于零（<1%）。这一极夜期长度是61.5°N地区（1-1.5个月）的2-2.4倍，对系统全年运行的影响极为深远。春秋分前后（3月、9月）日照时长约12小时，光伏归一化强度午间11-15时可达0.50-0.65，但持续时间仅4小时，贡献仍然有限。

全年光伏平均贡献率仅19%（vs. 绿氨系统聚类0的31%，下降39%），这一急剧下降源于：（1）极夜期完全停摆（占全年时长33%，贡献率0%）；（2）极昼期虽然光照时长极长但单位时间强度低（占全年时长17%，贡献率约30-35%）；（3）过渡季光照条件中等（占全年时长50%，贡献率约20-25%）。光伏发电的极端不可靠性迫使系统高度依赖风电——冬季100%依赖风电，夏季70%依赖风电，全年加权平均约85%依赖风电，风电的绝对主导地位在三个绿醇聚类中最为突出。

电解槽运行的风电紧密追踪与保守降负荷策略：电解槽运行强度热力图与风电发电相关系数高达r=0.85（vs. 绿氨系统聚类0的r=0.82，耦合性更强），呈现纯粹的"风电追踪"模式。全年尺度上，电解槽运行强度季节波动与风电高度一致：冬季（11月-3月）为高负荷运行期，归一化强度维持在0.75-0.88区间（平均0.82），电解槽满负荷运行时段占比达78%；夏季（6-8月）为中等负荷运行期，归一化强度降至0.50-0.68区间（平均0.59），满负荷时段占比降至42%；春秋过渡季居于两者之间（0.65-0.78）。

日内尺度上，电解槽运行呈现"风电三峰追踪+最低负荷保护"的双重特征。冬季典型日，电解槽在凌晨风电第一高峰时段（2-6时）负荷率达80-88%（vs. 绿氨系统的80-88%，基本一致），充分利用低成本风电制氢；午后风电回落时段（12-16时）电解槽主动降负荷至65-75%（vs. 绿氨系统的60-68%，降幅更保守），避免过度降负荷导致氢气供应中断影响甲醇合成；傍晚风电次峰时段（18-22时）电解槽负荷率回升至75-85%（vs. 绿氨系统的72-80%，略高）。这种"跟随波动但不完全同步+避免深度降负荷"的运行模式通过电池储能的短时缓冲实现，既保证了电解槽的相对稳定运行，又确保了氢气供应的连续性以匹配甲醇合成需求。

夏季典型日，电解槽呈现"风电基荷+光伏峰荷"的协同运行模式，但光伏贡献有限。夜间（22时-次日8时）依靠风电维持45-55%的基本负荷（vs. 绿氨系统的45-55%，一致），上午9-14时捕获光伏初升段和峰值负荷率缓慢爬升至68-78%（vs. 绿氨系统的65-90%，峰值显著更低），傍晚15-21时光伏快速衰减电解槽降负荷至50-60%。夏季电解槽最高负荷率仅78%（vs. 冬季的88%），反映了夏季风光联合出力不足对电解槽运行的制约。

关键差异：最低负荷率保护机制。为确保氢气供应连续性以匹配甲醇合成（甲醇合成需要H₂和CO₂同步供应，氢气中断将导致合成效率暴跌），电解槽最低负荷率始终维持在40%以上（vs. 绿氨系统可降至30%甚至更低），这一"最低负荷保护"策略牺牲了部分经济性（在低风时段仍需维持较高电解槽负荷，增加电池储能放电成本），但确保了甲醇合成的稳定运行。年度热力图显示，电解槽负荷率<40%的时段占比仅5%（vs. 绿氨系统的12%），且主要集中在设备检修期或极端低风天气。

DAC系统的极端季节性运行与"夏高冬低"模式：DAC运行强度热力图呈现出绿醇系统最显著的特征——与风电完全相反的"夏高冬低"季节分布，这是碳氢异步调节的核心体现。夏季（6-8月）为DAC高效运行期，归一化强度达0.85-0.95（平均0.90），此时气温相对温和（10-18°C，是全年最暖时段），DAC吸附剂（通常为胺基吸附剂）活性最佳，吸附CO₂的平衡负荷量高（约8-10 wt% CO₂）且解吸能耗低（约3.5-4.0 GJ/t CO₂）。更重要的是，夏季风电虽然较弱（归一化强度0.58）但仍能提供充足电力支撑DAC运行（DAC功耗约0.5-0.7 kWh/kg CO₂，对应功率1.0-1.5 MW，占系统总发电的6-8%），且夏季光伏虽然强度低但光照时长极长（20-24小时/天），为DAC提供了全天候电力供应的有利条件。

冬季（11月至次年3月，长达5个月）为DAC极低效甚至停运期，归一化强度暴跌至0.10-0.25（平均0.18），仅为夏季的20%。这一崩溃式下降源于极寒气温（-30至-45°C，极端情况可达-50°C）对DAC系统的多重打击：（1）吸附剂活性急剧下降，吸附CO₂的平衡负荷量降至2-3 wt%（仅为夏季的25-30%），吸附速率降低60-70%；（2）解吸能耗激增至6.0-8.0 GJ/t CO₂（是夏季的1.7-2.0倍），主要源于低温下水分冷凝在吸附剂表面需要额外加热能量；（3）设备防冻需求增加，管道、阀门、泵等辅助设备需要持续加热防止冻结，增加20-30%额外能耗；（4）极夜期光伏完全停摆，DAC完全依赖风电供电，但冬季虽然风电充沛（归一化强度0.86），系统优先保障电解槽运行（制氢优先级高于捕碳），DAC可用电力受限。

实际运行中，冬季DAC运行时长仅为夏季的20-25%：冬季极寒月份（12月-2月）DAC基本停运（运行时长<10%），仅在气温略回升的"暖和日"（-20至-25°C）短暂运行2-4小时/天；冬季较暖月份（11月、3月）DAC运行时长约30-40%，主要集中在午后气温相对较高时段（12-16时）。这种极端不均衡导致DAC年利用率75.3%的数值具有高度误导性——该利用率掩盖了"夏季超高负荷（95%利用率，持续3个月）+ 冬季几乎停运（<20%利用率，持续5个月）"的两极分化事实。

DAC季节性捕集量分配：夏季（6-8月，约90天）DAC利用率95%，贡献全年捕集量的28-30%（约4000 kg CO₂）；春秋过渡季（4-5月、9-10月，约120天）DAC利用率60-75%，贡献45-50%（约6500 kg CO₂）；冬季（11月-3月，约150天）DAC利用率18%，仅贡献20-25%（约3500 kg CO₂）。这种极端不均衡是极高纬度绿醇系统面临的核心挑战，直接决定了系统必须配置超大容量CO₂储罐进行跨季调节。

CO₂储罐的"夏储冬用"季节大循环与氢储罐的周际小调节：CO₂储罐SOC热力图呈现极为清晰的"夏储冬用"季节大循环模式，这是碳氢异步调节的核心机制。春季（4-5月）：DAC逐步恢复运行（利用率从30%升至75%），而甲醇合成维持稳定运行（利用率约75%），DAC产CO₂速率逐步超过合成需求，CO₂储罐SOC从春初的低点35-40%缓慢上升至春末的50-55%（月均净增7-8个百分点），为夏季大规模储备做准备。

夏季（6-8月）：DAC进入全年最高效运行期（利用率95%），产CO₂速率达1.98-2.08 kg/h（平均2.01 kg/h），而甲醇合成需求相对稳定（需CO₂速率约1.44 kg/h），DAC过剩产能达0.54-0.64 kg/h（约28-32%），这些过剩CO₂全部注入储罐。CO₂储罐SOC从夏初的50-55%快速上升至夏末的85-90%（月均净增12-13个百分点），累积储存约3200-3500 kg CO₂（占年捕集量的23-25%，占储罐容量的32-35%）。夏季是全年唯一的CO₂大规模净积累期，为漫长的冬季消耗做准备。

秋季（9-10月）：DAC逐步降负荷（利用率从75%降至40%），CO₂储罐SOC在秋初维持高位（85-88%），但随着DAC产能下降，储罐SOC在秋末开始缓慢下降至80-82%（月均净降2-3个百分点），标志着从"净积累"向"净消耗"的转换。

冬季（11月-3月）：DAC进入全年最低效甚至停运期（利用率18%），产CO₂速率暴跌至0.32-0.42 kg/h（平均0.38 kg/h，仅为夏季的19%），远低于甲醇合成需求（需CO₂速率维持1.44 kg/h不变），缺口达1.02-1.12 kg/h（约71-76%需求需从储罐释放）。CO₂储罐SOC从冬初的80-82%快速下降至冬末的35-40%（月均净降9-11个百分点），累积释放约4500-4800 kg CO₂（占年捕集量的32-34%，占储罐容量的45-48%），支撑甲醇合成在DAC几乎停摆的情况下仍能维持75%负荷率运行。冬季是全年唯一的CO₂大规模净消耗期，完全依赖夏季储备。

全年CO₂储罐SOC在35-90%的极宽区间大幅波动（波动幅度55个百分点，vs. 绿氨系统氢储罐的20个百分点，波动剧烈2.75倍），这种"满-空-满"的完整季节循环反映了DAC极端季节性对储能系统的巨大压力。CO₂储罐的季节调节功能至关重要——若无此储罐，冬季甲醇合成将被迫停运5个月（占全年42%），年产量将暴跌至5000-6000 kg（仅为当前的50-60%），系统经济性完全崩溃。

氢储罐的周际平滑调节与"弱季节性"特征：氢储罐SOC热力图展现出与CO₂储罐截然不同的"弱季节性+周际波动主导"特征。全年氢储罐SOC整体在45-70%的中位区间波动（波动幅度25个百分点，vs. CO₂储罐的55个百分点，仅为后者的45%），周际波动幅度8-12个百分点（vs. 日内波动2-4个百分点）。

季节性趋势微弱且与CO₂储罐呈现"逆向"关系：冬季（11-3月）风电充沛、电解槽高负荷运行，氢气产能充裕且略超过甲醇合成需求（受CO₂供应不足限制，甲醇合成实际氢气消耗量低于设计值），氢储罐SOC从冬初的45%缓慢上升至冬末的62%（月均净增3-4个百分点）；夏季（6-8月）风电减弱、电解槽中等负荷运行，氢气产能略低于甲醇合成需求（此时CO₂供应充裕，甲醇合成满负荷运行消耗更多氢气），氢储罐SOC从夏初的60%缓慢下降至夏末的52%（月均净降2-3个百分点）；春秋过渡季氢储罐SOC基本持平或微幅波动。全年氢储罐SOC最大差值仅17个百分点（vs. CO₂储罐的55个百分点，仅为后者的31%），反映了氢气供应的相对稳定性。

周际尺度上，氢储罐主要用于平抑短期风电波动（如3-5天气旋过境导致风速波动）：低风周（周平均风速降至正常值70-80%）电解槽产氢速率降至4.2-4.8 kg/day（vs. 正常5.33 kg/day），氢储罐SOC在3-5天内下降8-12个百分点；风速恢复后电解槽产氢略增，氢储罐SOC在随后5-7天内回升至原有水平。这种周际调节功能确保了甲醇合成在短期风电波动期间仍能维持70%以上负荷率，避免因氢气短缺导致合成中断。

碳氢储罐的功能分化与容量配置逻辑：CO₂储罐（10037 kg，261倍日产量）主导季节调节，承担"夏储冬用"跨季平衡功能，年内完成1-2个完整的"满-空-满"循环，循环深度高达55个百分点（接近满充满放）；氢储罐（4.98 kg，9.34倍日产量）主导周际调节，承担短期风电波动平抑功能，年内完成约15-20个周际循环，单次循环深度8-12个百分点。两者功能互补、时间尺度错开，共同支撑了碳氢同步供应：CO₂储罐解决了DAC极端季节性问题（冬季几乎停运），氢储罐解决了风电短期波动问题（周际风速不稳），两者协同确保甲醇合成在全年75.3%高利用率下稳定运行。

容量配置的极端不对称（CO₂储罐261倍 vs. 氢储罐9.34倍，比例28:1）精准反映了碳氢供应的本质差异：DAC受气候制约极大（冬季产能降至夏季20%），需要超大容量储罐跨季调节；氢气生产受风电驱动，风电虽有季节性（冬季强度为夏季1.48倍）但全年持续供应且季节差异可控（<50%），氢储罐仅需应对周际波动即可。这种"不对称配置"是极高纬度绿醇系统的核心设计特征，也是系统成本高企的主要原因（CO₂储罐投资占比16-18% vs. 氢储罐占比5-7%）。

热泵系统与甲醇合成的热电耦合运行

热泵运行强度热力图与DAC系统呈现极高的相关性（r=0.92），同步展现"夏高冬低"的季节分布模式。夏季（6-8月）热泵归一化强度0.80-0.92（平均0.87），冬季（11-3月）降至0.65-0.78（平均0.72），年利用率75.3%（与DAC、甲醇合成完全一致，反映三者的紧密耦合）。这种高度同步性源于热泵-DAC-合成的串联工艺流程：DAC捕集的CO₂需要压缩、预热后与氢气混合进入合成反应器，热泵提供预热所需的中低温蒸汽（150-200°C）以及合成反应的热量补充（甲醇合成为弱放热反应，ΔH=-90.7 kJ/mol，但需要维持250-300°C反应温度），DAC停运则热泵随之停运。

热泵性能系数（COP）呈现显著的季节差异。夏季环境温度相对温和（10-18°C），热泵从环境或低品位废热（如电解槽冷却水40-60°C）提升至合成所需温度（250-300°C），温度提升幅度约230-250°C，COP约为3.2-3.5，即消耗1 kWh电力可提供3.2-3.5 kWh_th热能。冬季环境温度极低（-30至-45°C），热泵需要从更低温度提升至合成温度，温度提升幅度扩大至280-330°C，COP降至2.8-3.0（降幅12-16%），能耗显著增加。更严重的是，冬季极寒条件下热泵蒸发器（从环境吸热的换热器）容易结霜甚至冻结，需要频繁除霜，进一步降低实际运行效率（有效COP可能降至2.5-2.7）。

热泵单位甲醇热能需求约为2.8-3.2 MJ_th/kg MeOH（包括CO₂预热约0.5 MJ/kg MeOH、H₂预热约0.3 MJ/kg MeOH、反应热补充约1.5-1.8 MJ/kg MeOH、产物分离约0.5-0.6 MJ/kg MeOH），对应电力消耗约0.8-1.0 MWh_e/kg MeOH（考虑COP 3.0-3.5），占系统总电耗的约12-15%。冬季热泵COP下降导致单位甲醇电耗增加15-20%（从0.8增至0.95 MWh_e/kg），是冬季甲醇成本上升的重要因素。

热储能容量仅0.27 kWh_th（极小），反映了热储能在该系统中的功能定位：主要用于数分钟至数小时的热量缓冲（如热泵启停过渡期、负荷波动平滑），而非季节性热储能。这与电池储能（1.85 MWh，用于日内调节）、氢储罐（4.98 kg，用于周际调节）、CO₂储罐（10037 kg，用于季节调节）的时间尺度分层形成完整的储能体系。

甲醇合成装置的高稳定性运行与CO₂储罐的关键支撑作用

甲醇合成装置运行强度热力图呈现出看似矛盾的高稳定性特征——尽管上游DAC系统季节波动极大（夏季利用率95% vs. 冬季18%，差异5.3倍），但甲醇合成仍维持全年归一化强度0.72-0.78窄幅区间，日内波动<4%，周际波动<7%，季节波动<8%。这一稳定性通过超大容量CO₂储罐（10037 kg，261倍日产量）的强大缓冲能力实现：

冬季（11-3月）虽然DAC几乎停摆（利用率18%，日均捕集仅9.2 kg CO₂），但CO₂储罐持续释放夏季储备（日均释放29.2 kg CO₂），使得甲醇合成装置的CO₂供应速率维持在日均38.4 kg CO₂（9.2+29.2），仅比夏季（日均43.6 kg）降低12%。这12%的降幅通过甲醇合成装置适度降负荷吸收（从夏季78%降至冬季72%），避免了完全停机。CO₂储罐容量足以支撑合成装置在连续5个月低DAC运行期间仍维持70%以上负荷率，这是系统设计的核心要求。

年度热力图显示，甲醇合成装置仅在春秋设备检修期（4月中旬与10月中旬各持续3-5天）运行强度短暂降至0.20-0.30，其余时段均保持高负荷稳定运行。夏季（6-8月）DAC高效运行+CO₂储罐充填期，合成装置运行强度略升至0.76-0.78，充分利用充裕的CO₂供应；冬季（11-3月）DAC低效运行+CO₂储罐释放期，合成装置运行强度略降至0.72-0.74，但仍维持高位；春秋过渡季居中（0.74-0.76）。

合成单元年利用率75.3%，虽然略低于绿氨系统聚类0的合成单元利用率（71.9%，但这是错误对比，应为78.8%纯光伏氨系统），但考虑到极高纬度地区的极端气候挑战（冬季-30至-45°C、极夜期3个月），这一利用率已经是技术优化的结果。更重要的指标是负荷率>70%的运行时长占比：全年约83%的时间合成装置负荷率>70%（vs. 绿氨系统聚类0的76%，改善9%），反映了CO₂储罐强大的季节调节能力对下游稳定性的保障作用。

合成装置的高稳定性对系统经济性至关重要：甲醇合成催化剂（铜基或铜锌铝催化剂）对温度、压力波动极为敏感，频繁启停或大幅负荷波动会导致催化剂烧结、中毒，寿命从正常的4-5年缩短至2-3年甚至更短，催化剂更换成本可占合成环节运行成本的30-40%。通过CO₂储罐维持合成装置高稳定运行，虽然增加了15-18%的储罐投资，但延长了催化剂寿命、降低了能耗（避免反复加热升压），全生命周期经济性仍然是正向的。

四季典型周电力供需平衡与影子价格动态分析

夏季典型周（7月第3周）：风电减弱但DAC高效的"碳富氢紧"供需格局

夏季典型周是DAC全年最高效运行时段，但风电处于年度低谷，系统呈现"碳富氢紧"的结构性矛盾。供给侧：风电功率10-14 MW（平均12.2 MW，占装机50%，为全年最低），日内波动幅度较大（±25-30%）；光伏功率在极昼期虽然光照时长达20-24小时/天，但受低太阳高度角限制，全天功率维持在3.5-5.5 MW窄幅区间（平均4.8 MW，占装机66%），缺乏明显峰谷差异。风光合计出力平均17.0 MW，是三个典型季中最低的（vs. 冬季28.8 MW降低41%，vs. 春秋22.5 MW降低24%）。

需求侧：电解槽功率10-12 MW（平均11.2 MW），受风电减弱影响降至全年较低水平，PEM电解槽维持8.5-9.5 MW的中等负荷（vs. 冬季的12-14 MW），通过降低负荷率（从冬季的82%降至夏季的59%）适应电力供应下降；DAC功率1.8-2.2 MW（平均2.0 MW，占系统总功耗12%），达到全年峰值，受益于温和气温（10-18°C）与全天候光照，DAC满负荷运行捕集CO₂速率约2.01 kg/h；热泵功耗0.85-1.05 MW（平均0.95 MW），配合DAC高负荷运行；甲醇合成及辅助系统0.8-1.1 MW（平均0.95 MW）。总需求平均15.1 MW，略低于总供给17.0 MW，小幅盈余1.9 MW用于电池充电与系统备用。

电池储能在夏季承担重要的日内平滑功能：虽然光伏全天候发电（20-24小时），但风电日内波动较大（午后12-16时风速降至夜间的60-70%），导致风光合计出力仍有日内波动。电池在夜间风电相对高峰时段（22时-次日8时）小幅充电0.5-1.5 MW（SOC从前日傍晚的55%升至60%），午后风电低谷时段（12-16时）放电1.0-2.0 MW（SOC降至52%），傍晚风电恢复时段再次充电0.8-1.5 MW（SOC回升至58%），形成"两充一放"模式。周内电池SOC在50-62%区间波动（波动幅度12个百分点，是全年最小的），反映了夏季虽然总发电量最低但光伏全天候发电带来的相对平稳。

碳氢储罐状态：氢储罐周内净产氢-0.35 kg（SOC从周一的60%下降至周日的57%），呈现净消耗状态，反映了夏季电解槽降负荷导致氢气产能不足以完全满足甲醇合成需求（周平均产氢5.12 kg/day vs. 需氢5.33 kg/day，缺口4%）；CO₂储罐周内净捕集+158 kg（SOC从周一的62%上升至周日的66%），呈现大幅净积累状态，反映了DAC高效运行（周平均捕集43.6 kg/day）远超甲醇合成需求（需CO₂ 38.4 kg/day，盈余13.5%）。这种"氢略紧、碳大松"的供需格局是夏季的典型特征，CO₂储罐快速充填为冬季大规模释放做准备。

电力影子价格动态：周平均0.0345美元/kWh（全年居中偏高），反映了夏季风电减弱导致的供给紧张。夜间风电相对高峰时段（22时-次日8时）电价0.030-0.033美元/kWh（全年较低水平），此时风电功率12-14 MW基本匹配需求；午后风电低谷时段（12-16时）电价升至0.038-0.042美元/kWh（全年较高水平），此时风电功率降至9-11 MW，需启用电池储能放电支撑，系统边际成本上升至储能循环成本水平；傍晚风电恢复时段（17-21时）电价回落至0.032-0.036美元/kWh。日内电价波动幅度±18%（vs. 冬季±12%，波动加剧），反映了风电减弱导致的供需匹配恶化。

周内电价略有波动（日均价格0.0335-0.0355美元/kWh区间），周中（周三-周四）受局地对流天气影响风电进一步减弱至平均10.5 MW，电价升至0.0368-0.0385美元/kWh；周末（周六-周日）风电略有恢复至13.5 MW，电价回落至0.0325-0.0340美元/kWh。这种周内波动为电解槽与DAC的协同优化提供了经济激励：在低价时段（周末）电解槽可适度提升负荷率（从平常的59%提升至62-65%），多产氢气储存于氢储罐；在高价时段（周中）DAC优先运行（捕碳优先级高于制氢），电解槽可适度降负荷（降至55-58%）以节约电力成本。

氢气与CO₂影子价格动态：氢气影子价格周平均2.33美元/kg（全年居中偏高），反映了夏季电解槽降负荷导致的氢气供应略紧。日内氢价波动±3-5%（vs. 冬季±2%，波动扩大），午后风电低谷时段氢价升至2.38-2.42美元/kg（电解槽降负荷+电价上涨双重影响），夜间风电高峰时段氢价降至2.28-2.32美元/kg。周内氢价呈现微弱上升趋势（周一2.30美元/kg→周日2.36美元/kg，单周涨幅2.6%），反映了氢储罐SOC下降（从60%降至57%）导致的供给略紧预期。

CO₂捕集边际成本周平均67.1美元/t CO₂（全年最低），反映了夏季DAC高效运行带来的成本优势。DAC单位捕集能耗在夏季降至最低水平（0.45-0.50 kWh/kg CO₂ vs. 冬季0.7-0.8 kWh/kg CO₂），且电价虽然略高但仍处于合理区间，使得CO₂捕集成本显著低于其他季节。日内CO₂捕集成本波动±8-12%（主要由电价波动驱动），午后高电价时段捕集成本升至72-75美元/t CO₂，夜间低电价时段降至62-65美元/t CO₂。周内CO₂捕集成本基本稳定（65-69美元/t CO₂区间），仅周中高电价时段略升至70-72美元/t CO₂。

甲醇综合成本（考虑氢气2.33美元/kg × 0.188 kg H₂/kg MeOH + CO₂捕集67.1美元/t × 1.375 kg CO₂/kg MeOH + 热能 + 电力 + 其他）周平均718美元/吨（全年最低），虽然氢价略高但CO₂捕集成本极低（全年最优）且DAC高效运行降低了系统折旧分摊，使得夏季成为全年成本最优时段。

冬季典型周（1月第3周）：风电充沛但DAC停摆的"氢富碳紧"供需反转

冬季典型周是风电全年最强时段，但DAC几乎完全停摆，系统呈现与夏季完全相反的"氢富碳紧"供需格局。供给侧：风电功率18-24 MW（平均20.8 MW，占装机85%，为全年最高），受极地涡旋南侵与西风急流增强双重影响，冬季风速可达14-18 m/s，日内波动幅度相对较小（±12-15%）；光伏功率在极夜期完全停摆（平均仅0.3 MW，占装机4%，仅在午间12-14时有微弱发电0.5-1.0 MW），对系统电力供应贡献趋近于零。风光合计出力平均21.1 MW（vs. 夏季17.0 MW高24%），但实质上100%依赖风电。

需求侧：电解槽功率13-15 MW（平均14.8 MW），受风电充沛影响升至全年最高水平，PEM电解槽满负荷或接近满负荷运行（12-14 MW，负荷率82%），充分利用低成本风电制氢；DAC功率0.5-0.9 MW（平均0.7 MW，占系统总功耗4%），降至全年最低水平，受极寒气温（-30至-45°C）严重制约，DAC实际运行时长仅为夏季的20-25%，捕集CO₂速率暴跌至约0.38 kg/h（仅为夏季的19%）；热泵功耗0.6-0.9 MW（平均0.75 MW），虽然DAC低负荷但热泵仍需维持一定功率，主要用于设备防冻、管道保温以及为甲醇合成提供热量补充（冬季热损失增加30-40%）；甲醇合成及辅助系统0.75-1.0 MW（平均0.88 MW）。总需求平均17.1 MW，低于总供给21.1 MW，盈余4.0 MW用于电池充电与系统备用（潜在弃风）。

电池储能在冬季承担风电日内三峰平滑功能：凌晨风电第一高峰时段（2-6时）电池充电2.0-3.5 MW（SOC从前日傍晚的48%升至58%），上午至午后风电低谷时段（10-16时）电池放电1.5-2.8 MW（SOC降至50%），傍晚风电第二高峰时段（18-22时）电池再次充电1.8-3.0 MW（SOC回升至56%），形成"两充一放"模式。周内电池SOC在45-62%区间波动（波动幅度17个百分点，居于夏季与春秋之间），反映了冬季虽然风电充沛但日内三峰波动仍需电池平滑。

碳氢储罐状态：氢储罐周内净产氢+1.12 kg（SOC从周一的48%上升至周日的51%），呈现净积累状态，反映了冬季电解槽高负荷运行导致氢气产能（周平均产氢6.45 kg/day）显著超过甲醇合成需求（需氢5.12 kg/day，盈余26%）——这一盈余源于CO₂供应不足限制了甲醇合成实际负荷率（从设计值82%降至实际72%），导致氢气消耗量低于设计值；CO₂储罐周内净捕集-142 kg（SOC从周一的68%下降至周日的64%），呈现大幅净消耗状态，反映了DAC几乎停摆（周平均捕集仅9.2 kg/day）远低于甲醇合成需求（需CO₂ 36.8 kg/day，缺口75%），需从储罐大量释放（周释放151 kg，日均21.6 kg）以维持合成运行。这种"氢大松、碳大紧"的供需格局是冬季的典型特征，CO₂储罐快速放空考验储罐容量的充裕性。

电力影子价格动态：周平均0.0325美元/kWh（全年最低），反映了冬季风电充沛导致的供给充裕。凌晨与傍晚风电高峰时段（2-6时、18-22时）电价降至0.028-0.031美元/kWh（全年最低水平），此时风电功率22-24 MW远超需求15-17 MW，系统边际成本接近风电运维成本下限，大量剩余电力需通过电池储能消纳或潜在弃风（弃风率约8-12%）；午后风电低谷时段（12-16时）电价升至0.035-0.038美元/kWh（但仍低于夏季），此时风电功率降至18-20 MW仍能基本满足需求，仅需小幅电池放电支撑；夜间时段（22时-次日2时）电价居中（0.030-0.033美元/kWh）。日内电价波动幅度±12%（vs. 夏季±18%，波动收窄），反映了冬季风电充沛带来的供需匹配改善。

周内电价极稳定（日均价格0.0318-0.0332美元/kWh窄幅区间），仅周末（周六-周日）受风速略增至平均22.5 MW影响，电价略降至0.0308-0.0315美元/kWh；周中风电略弱至平均19.5 MW，电价略升至0.0335-0.0340美元/kWh，但波动幅度极小（<5%）。这种低价格、低波动的冬季电力环境为电解槽实现高利用率提供了理想条件，但同时也暴露了系统设计的结构性问题——冬季风电过剩（盈余4.0 MW，占总发电19%）而DAC受气候限制无法充分利用，导致潜在弃风或电力浪费。

氢气与CO₂影子价格动态：氢气影子价格周平均2.28美元/kg（全年最低），反映了冬季电解槽高负荷运行与低电价的双重利好。日内氢价波动极小（±1-2%），全天维持在2.26-2.30美元/kWh窄幅区间，供给充裕。周内氢价呈现微弱下降趋势（周一2.30美元/kg→周日2.26美元/kg，单周降幅1.7%），反映了氢储罐SOC上升（从48%升至51%）导致的供给宽松预期。冬季氢价虽然是全年最低，但由于甲醇合成受CO₂供应限制实际氢气消耗量低于设计值，氢气"过剩"状态无法转化为甲醇产量提升，造成资源浪费。

CO₂捕集边际成本周平均112美元/t CO₂（全年最高），反映了冬季DAC低效运行带来的成本劣势。DAC单位捕集能耗在冬季升至最高水平（0.70-0.80 kWh/kg CO₂ vs. 夏季0.45-0.50 kWh/kg CO₂，增幅56-60%），虽然电价是全年最低但高能耗仍导致捕集成本飙升。更严重的是，冬季DAC大部分时间处于停运或极低负荷状态（利用率<20%），设备折旧成本需分摊到极少的捕集量上，进一步推高单位成本。日内CO₂捕集成本波动±15-20%（vs. 夏季±8-12%，波动扩大），反映了冬季DAC间歇运行的不稳定性。周内CO₂捕集成本波动范围105-118美元/t CO₂，周中气温略回升（从-40°C升至-28°C）的"暖和日"DAC短暂恢复运行，捕集成本降至105-108美元/t CO₂；周末气温再次骤降（降至-45°C）DAC几乎完全停运，捕集成本飙升至115-118美元/t CO₂。

甲醇综合成本周平均768美元/吨（全年最高），虽然氢价是全年最低（2.28美元/kg）但CO₂捕集成本极高（112美元/t CO₂ vs. 夏季67美元/t CO₂，增幅67%）且热泵COP下降（从夏季3.2-3.5降至冬季2.8-3.0）导致热能成本增加，使得冬季成为全年成本最差时段。冬季甲醇成本相比夏季上涨7.0%（50美元/吨），这一成本差异直接反映了DAC极端季节性对系统经济性的严重拖累。

春季典型周（4月第2周）与秋季典型周（10月第4周）：供需特征介于夏冬之间。春季风电强（平均18.5 MW）+光伏初升（平均3.2 MW），DAC逐步恢复运行（利用率从30%升至60%），CO₂储罐SOC快速下降（从冬末的40%降至春末的50%），消耗冬季最后库存；电力影子价格0.0332美元/kWh，氢气价格2.30美元/kg，CO₂捕集成本82美元/t CO₂（居于夏冬之间），甲醇成本738美元/吨。

秋季风电中（平均16.8 MW）+光伏回落（平均2.8 MW），DAC逐步降负荷（利用率从75%降至40%），CO₂储罐SOC维持高位（80-85%）并开始缓慢下降，为冬季大规模释放做准备；电力影子价格0.0338美元/kWh，氢气价格2.31美元/kg，CO₂捕集成本78美元/t CO₂，甲醇成本735美元/吨。全年四季成本波动范围718-768美元/吨（波差50美元/吨，波动幅度7.0%），是三个绿醇聚类中季节性最大、最不稳定的系统。

5.2 聚类1：亚洲中纬度风光均衡型（47.5°N, 54.0°E）

技术配置与碳氢耦合系统架构

聚类1代表点位于亚洲中纬度地区（哈萨克斯坦东北部，47.5°N, 54.0°E），处于温带大陆性气候带，具有显著的四季分明特征与良好的风光资源互补性。技术配比图显示系统采用风光接近均衡的配置：风电装机22.44 MW，光伏装机40.39 MW，风光比0.56:1，相比绿氨系统聚类1（风光比0.83:1）光伏比例进一步提升，反映了2040年光伏成本持续下降（230美元/kW）带来的经济优势。这一风光比调整使得系统在保持风光互补优势的同时，进一步强化了午间光伏峰值时段的电力供应能力。

电解系统呈现"PEM主导+AE适度渗透"的技术组合：PEM电解槽9.27 MW（占比53.6%），AE电解槽8.02 MW（占比46.4%），电解槽总容量17.29 MW。AE占比46.4%略低于绿氨系统聚类1（51.5%），主要源于绿醇系统对氢气供应稳定性的更高要求——甲醇合成需要H₂和CO₂的同步供应，任何一方短缺都会导致合成中断，因此系统更倾向于采用响应速度快、运行灵活的PEM技术作为主导。电解槽年利用率高达63.5%，在三个绿醇聚类中居于中等水平，反映了风光均衡配置对设备利用率的积极影响。

储能配置呈现"电池主导+氢储辅助"的典型风光均衡系统特征：电池储能4.01 MWh、功率20.05 MW，单位装机储能比0.064 MWh/MW，是绿氨系统聚类1（0.128 MWh/MW）的一半，反映了绿醇系统通过CO₂储罐承担部分季节调节功能，降低了对电池储能的依赖；氢储罐2.95 kg H₂，约为日均产氢量的5.52倍，显著高于绿氨系统聚类1（1.22倍），反映了碳氢同步供应需求对氢储罐容量的更高要求——当DAC因天气原因短期低产时，需要氢储罐储备足够氢气以匹配CO₂储罐释放的CO₂。

碳氢耦合系统的优化配置：DAC装机1.93 kg CO₂/h，年捕集14000 kg CO₂，利用率高达82.9%，是三个绿醇聚类中最高的，反映了温带大陆性气候全年温度适宜（夏季25-35°C、冬季-10至5°C）对DAC高效运行的支撑；热泵装机1.93 MW_th，热储能0.43 kWh_th，COP约为3.5-3.8（得益于温和气候条件）；甲醇合成装机12.06 tpy，甲醇储罐119.35 kg，年产甲醇10000 kg，合成单元利用率高达82.9%，是三个绿醇聚类中最高的。CO₂储罐容量3061 kg，约为日均捕集量的79.8倍，远小于聚类0（261倍），反映了DAC全年高稳定运行大幅降低了季节储能需求。

年度运行特征热力图的风光协同与碳氢同步

风电与光伏发电的四季互补与全年均衡：风电发电热力图呈现温带大陆性气候典型的"春冬强、夏秋弱"季节模式。春季（3-5月）和冬季（12-2月）受西风急流增强影响，风电归一化强度维持在0.70-0.85区间（平均0.78）；夏季（6-8月）和秋季（9-11月）受副热带高压影响，风电强度降至0.58-0.72区间（平均0.65）。春冬与夏秋风电强度差异仅20%（vs. 聚类0的48%），季节性波动显著减弱。日内风电呈现"夜间高峰-午后低谷"两段式结构，与绿氨系统聚类1（非洲17.5°N）相似，但波动幅度略小（日内差异15-22% vs. 18-28%）。

光伏发电热力图呈现标准的"夏强冬弱"季节分布。夏季（6-8月）日照时长15-16小时，光伏归一化强度午间10-17时达0.88-0.96（平均0.92），为全年最高；冬季（12-2月）日照时长缩短至8-9小时，但受温带大陆性气候冬季晴天多的影响，光伏归一化强度午间11-15时仍能保持0.75-0.85（平均0.80），显著高于高纬度系统（0.45-0.60）；春秋过渡季日照时长12-13小时，光伏强度0.82-0.90。全年光伏出力季节差异仅15%（vs. 聚类0的>75%），为系统提供稳定可靠的电力基础。

风光季节互补效应显著：春冬季节风电强（0.78）+光伏中（0.80），夏秋季节风电弱（0.65）+光伏强（0.92），风光反向波动平抑了系统总发电的季节差异。更重要的是昼夜互补：午间光伏峰值（0.88-0.96）对应风电低谷（0.55-0.68），夜间风电峰值（0.75-0.85）对应光伏完全停摆，风光错峰互补实现全天候稳定供电。这种双重互补使得系统总发电功率全年标准差降低58%相比单一能源系统（vs. 聚类0的降低35%），为下游碳氢耦合系统提供了最稳定的能源基础。

电解槽运行的"PEM基荷+AE峰荷"技术协同与高利用率：电解槽运行强度热力图与风光发电耦合系数r=0.83，呈现明确的"PEM负责全天候基荷+AE负责峰时满负荷"技术分工。全年尺度，电解槽运行强度季节波动极小：春季归一化强度0.65-0.75（平均0.70），夏季0.68-0.78（平均0.73），秋季0.62-0.72（平均0.67），冬季0.60-0.70（平均0.65），四季差异仅12%（vs. 聚类0的>40%），反映了风光均衡对电解槽全年稳定运行的显著改善。

日内尺度，PEM电解槽全天候连续运行：夜间风电高峰时段（22时-次日8时）负荷率70-80%，捕获低成本夜间风电制氢；上午风光过渡时段（8-11时）负荷率略降至60-70%，受风光交接期电力略有不足影响；午间光伏高峰时段（11-16时）负荷率回升至75-85%，充分利用光伏充沛电力；傍晚风光双谷时段（16-22时）负荷率降至55-65%，通过电池储能支撑维持基本运行。PEM全天候运行策略确保了氢气供应的连续性，避免因氢气短缺导致甲醇合成中断。

AE电解槽则专注峰时运行：夜间风电高峰时段（0-6时）满负荷运行（归一化强度0.95-1.00），充分利用低成本夜间风电；上午风光过渡时段（8-11时）降负荷至40-60%或停机；午间光伏高峰时段（11-16时）满负荷运行（0.95-1.00），捕获午间低价光伏电力；傍晚至深夜时段（17-23时）降负荷至30-50%或停机。AE日运行时长10-12小时（vs. 聚类0的8-9小时），得益于风光双峰供电模式提供了更长的峰时窗口。

这种技术分工使得PEM年利用率达70%（全天候运行），AE年利用率达67%（峰时运行），整体电解系统利用率63.5%。相比绿氨系统聚类1（电解槽利用率66.3%），绿醇系统利用率略低主要源于：（1）绿醇系统对氢气供应稳定性要求更高，电解槽更倾向于保守运行（避免过度降负荷导致氢气短缺）；（2）AE占比略低（46.4% vs. 51.5%），而AE在峰时运行时利用率通常高于PEM的全天候运行。

DAC系统的全年高稳定运行与微弱季节波动：DAC运行强度热力图呈现出绿醇系统中最稳定的特征，全年归一化强度维持在0.78-0.88窄幅区间，季节差异仅12%（vs. 聚类0的>400%）。夏季（6-8月）气温25-35°C最适宜DAC运行，归一化强度达0.85-0.88（平均0.87）；冬季（12-2月）气温虽降至-10至5°C，但温带大陆性气候冬季多晴天、湿度低，DAC吸附剂活性仍维持较高水平，归一化强度0.78-0.82（平均0.80），仅比夏季降低8%（vs. 聚类0的80%降幅）；春秋过渡季DAC强度居中（0.82-0.85）。

DAC年利用率82.9%，全年无明显淡季，各季度利用率均维持在78-88%区间，这种全年高稳定运行是温带风光均衡系统的核心优势。日内尺度，DAC运行强度与系统电力供应高度相关（r=0.76）：午间风光合计出力峰值时段（11-16时）DAC满负荷运行（归一化强度0.88-0.92），夜间风电为主时段DAC维持中高负荷（0.75-0.82），傍晚风光双谷时段DAC略降负荷（0.68-0.75）但仍维持稳定运行。DAC功耗约占系统总发电量的8-12%（vs. 聚类0的6-15%，波动更小），反映了温和气候下DAC能效比的优化（单位捕集能耗0.45-0.55 kWh/kg CO₂ vs. 聚类0的0.5-0.8 kWh/kg CO₂）。

CO₂储罐与氢储罐的协同调节与"微季节性+周际平滑"模式：CO₂储罐SOC热力图呈现"微季节性+周际波动"的双重特征，这是风光均衡系统碳氢调节的独特模式。全年CO₂储罐SOC整体在45-75%中位区间波动，季节性趋势微弱：夏季（6-8月）DAC略强于甲醇合成需求，储罐SOC缓慢上升（月均净增2-3个百分点），从夏初的52%升至夏末的60%；冬季（12-2月）DAC略弱于甲醇合成需求，储罐SOC缓慢下降（月均净降2-3个百分点），从冬初的62%降至冬末的56%；春秋过渡季SOC基本持平或微幅波动。全年SOC最大差值仅15个百分点（vs. 聚类0的53个百分点），反映了DAC全年高稳定运行对季节储能需求的极大降低。

周际尺度，CO₂储罐主要用于平抑短期天气波动（如3-5天连续阴雨导致光伏低发、DAC略降负荷）：阴雨周DAC产CO₂速率降至正常值70-80%，储罐SOC在3-5天内下降5-8个百分点；晴天恢复后DAC产CO₂略增，储罐SOC在随后5-7天内回升至原有水平。这种周际调节功能确保了甲醇合成在短期天气波动期间仍能维持80%以上负荷率，避免频繁启停。

氢储罐SOC同样呈现"微季节性+周际波动"特征，全年SOC在50-68%区间波动，周际波动幅度6-10个百分点（vs. 绿氨系统的8-12个百分点，略小）。季节性趋势与CO₂储罐基本同步：夏季略有积累（SOC从52%升至60%），冬季略有释放（SOC从60%降至54%），春秋持平。氢储罐容量虽然达到日产量5.52倍（vs. 绿氨系统1.22倍，增幅352%），但其主要功能并非季节调节（季节SOC波动仅8个百分点），而是确保碳氢同步供应的"安全冗余"——当DAC因短期天气波动低产时，CO₂储罐释放库存CO₂，此时需要氢储罐同步供应足够氢气以匹配CO₂流量，避免因氢气不足导致甲醇合成被迫降负荷。

碳氢供应的同步性量化分析：甲醇合成化学计量比要求H₂:CO₂=3:1（摩尔比），即每生产1 kg甲醇需消耗0.188 kg H₂和1.375 kg CO₂。系统通过碳氢储罐的协同调节实现了高度同步供应：全年H₂供应与CO₂供应的相关系数r=0.94（日尺度）、r=0.88（周尺度），氢碳比波动范围2.85-3.15（标准差0.08），仅有±5%的偏离。这种高度同步性得益于：（1）电解槽（氢气生产）与DAC（CO₂捕集）均由风光发电驱动，两者运行强度自然正相关；（2）氢储罐与CO₂储罐容量配比经过优化（5.52倍日产量 vs. 79.8倍日产量），两者调节能力匹配；（3）系统优化算法实时监测氢碳比，当偏离超过±5%时自动调节电解槽或DAC负荷以恢复平衡。

热泵系统与甲醇合成的高效热电耦合：热泵运行强度热力图与DAC高度相关（r=0.96），全年归一化强度0.78-0.88，年利用率82.9%（与DAC、甲醇合成完全一致）。夏季热泵COP达3.7-3.8（环境温度高、提升温差小），冬季COP略降至3.3-3.5（环境温度低），但仍维持高效运行。热泵单位甲醇热能需求约为2.8-3.2 MJ_th/kg MeOH（包括原料预热、反应热补充、产物分离），远低于传统化石燃料供热（4.5-5.5 MJ/kg MeOH），反映了电驱动热泵的能效优势。

甲醇合成装置运行强度热力图呈现极高稳定性，全年归一化强度维持在0.80-0.86窄幅区间，日内波动<3%，周际波动<5%，季节波动<7%（vs. 聚类0的<8%，进一步改善）。这种极高稳定性是三个绿醇聚类中最优的，得益于：（1）DAC全年高稳定运行（利用率82.9%）确保了CO₂供应的连续性；（2）电解槽全年高稳定运行（利用率63.5%）确保了H₂供应的连续性；（3）CO₂储罐与氢储罐虽然容量配置中等（vs. 聚类0的超大容量），但在风光均衡系统中已足以应对周至月尺度的波动，无需应对极端季节性冲击。

合成单元年利用率82.9%，是三个绿醇聚类中最高的（vs. 聚类0的75.3%，提升10.1%），也显著高于绿氨系统聚类1的合成单元利用率（72.4%，提升14.5%）。这一"利用率倒挂"现象（绿醇合成高于绿氨合成）看似矛盾，实则反映了系统优化策略的根本差异：绿醇系统通过"牺牲上游电解槽利用率（63.5% vs. 绿氨66.3%）+配置中等容量碳氢储罐"的策略，将有限的碳氢资源优先保障下游合成装置高负荷稳定运行，在合成环节实现价值最大化；绿氨系统则追求电解槽与合成装置的均衡利用。

四季典型周电力供需平衡与影子价格动态

夏季典型周（7月第3周）：风光双优的供给充沛与低成本运行

夏季典型周是风光均衡系统全年供给最充沛、成本最低的时段。供给侧：风电功率12-16 MW（平均14.2 MW，占装机63%），光伏功率午间28-36 MW（平均25.8 MW，占装机64%），风光合计出力午间可达42-48 MW，为全年最高。需求侧：电解槽功率13-16 MW（平均14.5 MW），其中PEM维持9-10 MW基荷，AE午间满负荷5-6 MW；DAC功率1.0-1.3 MW（平均1.15 MW，占系统总功耗8%），热泵功耗0.5-0.7 MW；甲醇合成及辅助系统0.8-1.2 MW。

电池储能午间充电5-8 MW（充电功率0.25-0.40C），傍晚至夜间放电3-6 MW（放电功率0.15-0.30C），周内SOC在60-82%区间波动（波动幅度22个百分点）。氢储罐周内净产氢0.18 kg（SOC从58%微升至59%），CO₂储罐周内净捕集23 kg CO₂（SOC从55%升至58%），碳氢储罐均处于微积累状态，供需宽松。

电力影子价格动态：周平均0.0263美元/kWh（全年最低），反映夏季风光双优带来的供给充裕。午间光伏峰值时段（12-16时）电价降至0.020-0.023美元/kWh（系统边际成本接近光伏运维成本下限），此时风光合计出力42-48 MW远超需求14-16 MW，大量剩余电力需通过电池储能消纳或潜在弃电；夜间风电时段（22时-次日8时）电价0.028-0.031美元/kWh，风电功率12-16 MW基本匹配需求；傍晚风光双谷时段（17-21时）电价0.026-0.029美元/kWh，需启用电池储能放电支撑。日内电价波动幅度±15%（vs. 绿氨系统聚类1的±17-23%，波动更小），反映了风光互补性的进一步优化。

周内电价极稳定（日均价格0.0258-0.0268美元/kWh窄幅区间），仅周四受局地对流天气影响光伏发电量略降，当日电价略升至0.0275美元/kWh，但影响有限。这种低价格、低波动的电力供给环境为碳氢耦合系统提供了理想的运行基础。

氢气与CO₂影子价格动态：氢气影子价格周平均1.76美元/kg（全年最低），日内波动<1%，周内波动<2%，供需高度平衡。CO₂捕集边际成本周平均60.9美元/t CO₂（全年最低），反映了夏季DAC高效运行（单位捕集能耗0.45-0.50 kWh/kg CO₂、电价低廉）带来的成本优势。甲醇综合成本（考虑氢气、CO₂、热能、电力）周平均565美元/吨（全年最低），显著低于年平均579.7美元/吨。

典型日深度剖析（周三晴天）：午间12时风光合计出力达峰值46 MW，电解槽功率15.5 MW（PEM 9.8 MW满负荷、AE 5.7 MW满负荷），DAC功率1.25 MW满负荷，热泵0.68 MW，甲醇合成及辅助1.1 MW，总需求18.5 MW，剩余27.5 MW用于电池充电（充电功率7.2 MW，占剩余26%）与潜在弃电20.3 MW（占剩余74%）。这一"大量弃电"现象反映了夏季午间风光过剩的结构性问题，未来可通过：（1）扩大电解槽容量（提升至20-22 MW），午间时段全部满负荷运行以消纳过剩电力；（2）引入灵活性负荷（如制冷、海水淡化），专门消纳午间低价电力；（3）发展储能外送，将午间过剩电力通过电网外送至邻近地区。

夜间2时风电出力14.8 MW，电解槽功率13.2 MW（PEM 8.5 MW、AE停机），DAC功率1.05 MW，热泵0.55 MW，甲醇合成及辅助1.0 MW，总需求15.8 MW，小幅缺口1.0 MW由电池放电补充。傍晚19时风光双谷（风电10.5 MW、光伏3.8 MW，合计14.3 MW），电解槽功率12.0 MW（PEM 8.0 MW降负荷、AE停机），DAC功率0.92 MW，热泵0.48 MW，甲醇合成及辅助0.95 MW，总需求14.4 MW，小幅缺口0.1 MW由电池微放电补充，系统实现全天供需平衡。

冬季典型周（1月第3周）：风电主导的供给略紧与成本微涨

冬季典型周风电占据主导地位，光伏贡献显著但不如夏季。供给侧：风电功率14-18 MW（平均16.2 MW，占装机72%），光伏功率午间20-28 MW（平均18.5 MW，占装机46%，比夏季降低28%），风光合计出力午间34-40 MW。需求侧：电解槽功率12-14 MW（平均13.2 MW，略低于夏季），DAC功率0.95-1.15 MW（平均1.05 MW，略低于夏季），甲醇合成维持0.52 kg MeOH/h。

电池储能午间充电3-6 MW，傍晚至夜间放电2-5 MW，周内SOC在55-72%区间波动（波动幅度17个百分点，略小于夏季）。氢储罐周内净产氢0.12 kg（SOC从56%微升至57%），CO₂储罐周内净捕集15 kg CO₂（SOC从58%升至60%），碳氢储罐仍处于微积累状态，供需虽略紧但仍平衡。

电力影子价格动态：周平均0.0278美元/kWh（比夏季上涨5.7%），反映冬季光伏发电量下降导致的供给略紧。午间光伏峰值时段（12-15时）电价0.023-0.026美元/kWh（比夏季略高），风光合计出力34-40 MW仍能满足需求；夜间风电时段（22时-次日8时）电价0.030-0.033美元/kWh（比夏季略高），风电功率14-18 MW略高于需求；傍晚风光双谷时段（17-21时）电价升至0.032-0.036美元/kWh（全年最高夜间电价），风光合计出力降至22-26 MW，需较大功率电池放电支撑（4-6 MW）。日内电价波动幅度±18%（vs. 夏季±15%，略有扩大）。

周内电价略有波动（日均价格0.0270-0.0285美元/kWh），周末（周六-周日）受连续阴天影响光伏发电量降至正常值60-70%，电价升至0.0290-0.0295美元/kWh，但仍维持在较低水平（vs. 聚类0冬季0.0325美元/kWh）。

氢气与CO₂影子价格动态：氢气影子价格周平均1.82美元/kg（比夏季上涨3.4%），日内波动<2%，周内波动±3-4%（周末阴天时氢价略升至1.86美元/kg）。CO₂捕集边际成本周平均64.5美元/t CO₂（比夏季上涨5.9%），反映了冬季DAC能耗略增（单位捕集能耗0.50-0.55 kWh/kg CO₂）与电价略高的双重影响。甲醇综合成本周平均590美元/吨（比夏季上涨4.4%），但仍接近年平均水平。

春季典型周（4月第2周）与秋季典型周（10月第4周）：供需特征介于夏冬之间。春季风电强（平均15.8 MW）+光伏中（平均22.5 MW），电力影子价格0.0268美元/kWh，氢气价格1.78美元/kg，CO₂捕集成本62.2美元/t CO₂，甲醇成本575美元/吨；秋季风电中（平均14.5 MW）+光伏强（平均24.2 MW），价格与春季基本相当（电力0.0265美元/kWh，氢气1.77美元/kg，CO₂捕集61.5美元/t CO₂，甲醇572美元/吨）。全年四季成本波动范围565-590美元/吨（波差25美元/吨，仅4.4%），是三个绿醇聚类中季节性最小、最稳定的系统。

5.3 综合技术经济评价与系统优化潜力

亚洲中纬度风光均衡系统在2040年LowAE情景下展现出卓越的技术经济性能：平准化甲醇成本579.70美元/吨，相比聚类0（721.01美元/吨）降低19.6%，是三个绿醇聚类中成本最低的。这一成本优势与绿氨系统的地理梯度规律完全一致（绿氨系统聚类1成本427.52美元/吨，同样是三个聚类中最低），再次验证了低纬度至中纬度风光均衡地区是绿色燃料生产的理想区域。

成本优势的来源分解：

（1）DAC环节成本优势（占总成本20-25%）：风光均衡系统DAC高利用率（82.9% vs. 聚类0的75.3%）与全年高稳定运行（季节波动12% vs. 聚类0的>400%）使得DAC平准化捕集成本降至60.9美元/t CO₂（夏季最低），相比聚类0（67.1-112美元/t CO₂，季节波动66%）降低10-45%。更重要的是，DAC全年稳定运行使得CO₂储罐容量需求大幅下降（79.8倍日产量 vs. 聚类0的261倍），储罐投资节约70%，储能总成本占比从18%降至12%。

（2）电解环节成本优势（占总成本35-40%）：虽然电解槽利用率（63.5%）略低于绿氨系统聚类1（66.3%），但AE占比46.4%仍较高，制氢成本1.76美元/kg相比聚类0（2.33美元/kg）降低24.5%。电解环节成本优势贡献了系统总成本差异的50-55%。

（3）合成环节成本优势（占总成本25-30%）：甲醇合成高利用率（82.9% vs. 聚类0的75.3%）与高稳定运行（季节波动<7% vs. 聚类0的<8%）使得吨甲醇能耗降至最低（约18.5 GJ/t MeOH vs. 聚类0的20.0-21.5 GJ/t MeOH），催化剂寿命最长（>4年 vs. 聚类0的3-3.5年）。热泵系统高COP（3.5-3.8 vs. 聚类0的2.8-3.5）进一步降低了热能成本。

（4）储能环节成本优势（占总成本10-15%）：虽然电池储能单位装机配比（0.064 MWh/MW）高于聚类0（0.058 MWh/MW），但CO₂储罐与氢储罐容量配置显著降低（CO₂储罐79.8倍 vs. 261倍，氢储罐5.52倍 vs. 9.34倍），储能系统总投资相比聚类0降低35-40%。

影子价格分析揭示系统已接近优化平衡态：全年电力价格波动范围0.0263-0.0278美元/kWh（差异5.7%），日内波动±15-18%，价格信号微弱；氢气价格波动1.76-1.82美元/kg（差异3.4%），周内波动<4%；CO₂捕集成本波动60.9-64.5美元/t CO₂（差异5.9%），季节波动极小。这种"低价格、低波动"特征表明系统供需已高度平衡，进一步优化空间有限（<5%）。

未来优化方向：

（1）AE占比微调至50-52%：当前46.4%的AE占比已接近碳氢同步供应约束下的最优水平（vs. 绿氨系统可达55-60%），进一步提升空间有限。可探索将AE占比提升至50-52%，配合电池储能容量略增10-15%（应对AE间歇运行），预计可降低制氢成本3-5%，但需权衡储能投资增加。

（2）电池储能循环深度优化：当前电池日内循环深度约60-65%（SOC波动17-22个百分点 ÷ 2 × 2倍考虑充放电），年循环次数约360次，电池寿命6-8年。可探索降低循环深度至45-55%（通过增加电池容量15-20%），延长电池寿命至10-12年，全生命周期储能成本可降低8-10%。

（3）氢储罐容量精简评估：当前氢储罐容量5.52倍日产量主要用于确保碳氢同步供应的"安全冗余"，实际季节调节需求不大（季节SOC波动仅8个百分点）。可评估将容量精简至4.0-4.5倍（降低20-25%），但需通过实时优化算法强化电解槽与DAC的协同调度，确保极端天气期间（连续5-7天阴雨）碳氢供应仍能同步。潜在成本节约2-3%。

（4）季节性生产微调策略：虽然系统全年供需已高度平衡，但仍可探索"夏季（6-9月）略增产+冬季（12-2月）略减产"的微调策略，利用夏季成本优势（565美元/吨 vs. 冬季590美元/吨）多产甲醇储存，冬季适度降负荷或安排检修。通过扩大甲醇储罐容量至200-250 kg（约15-20天产量），全年平均成本可降低2-3%。

风光均衡系统已代表绿醇生产的技术经济最优配置，未来优化主要聚焦于边际改进（累计潜力5-8%）而非结构性变革。该系统为全球绿醇产业布局提供重要启示：优先在北纬35-55°、南纬35-45°的温带大陆性气候区发展绿醇产业，该区域具备风光资源丰富且互补性好、四季气候温和适宜DAC全年运行、季节储能需求最小的三重优势，可实现最优技术经济性能。

5.3 聚类2:南美中纬度纯光伏主导型(-36.25°S, -70.25°W)

技术配置与碳氢耦合系统架构

聚类2代表点位于南美洲智利中部(-36.25°S, -70.25°W),处于世界著名的阿塔卡马沙漠边缘地带,具有全球顶级的太阳能资源禀赋(年均DNI >3000 kWh/m²,全球最高水平之一)与极度匮乏的风能资源。技术配比图显示系统采用极端光伏主导配置:光伏装机72.76 MW,风电装机仅6.55×10⁻⁷ MW(实质为零),风光比0.000009:1,是典型的纯光伏系统。这一配置与绿氨系统聚类2(欧洲37.5°N纯光伏,风光比0.000001:1)高度相似,但光伏装机规模略大(72.76 vs. 71.18 MW,增幅2.2%),反映了绿醇系统对电力需求的增加(需同时支撑电解、DAC、热泵三大耗电单元)。

南半球中纬度(-36.25°S)相比北半球中纬度(37.5°N)具有相似的太阳辐照特征但季节性相反:南半球夏季(12月-2月)对应北半球冬季,南半球冬季(6-8月)对应北半球夏季。阿塔卡马地区独特的地理位置(太平洋东岸、安第斯山脉西侧)形成了极端干旱气候——受秘鲁寒流与安第斯山脉屏障的双重影响,该地区年降水量<15 mm(部分区域<1 mm,堪称地球上最干燥的地方),云量极少(年均晴天率>90%),大气透明度极高,为光伏发电提供了理想条件。但同样受安第斯山脉屏障影响,该地区风能资源极度匮乏(年均风速<4 m/s),陆上风电几乎不具备开发价值。

电解系统呈现"AE大幅渗透+PEM补充"的技术组合:AE电解槽20.12 MW(占比68.5%),PEM电解槽9.24 MW(占比31.5%),电解槽总容量29.36 MW。AE占比68.5%是三个绿醇聚类中最高的(vs. 聚类0的0%、聚类1的46.4%),也显著高于绿氨系统聚类2(46.8%),标志着低成本碱性电解技术在纯光伏系统中的极限渗透。这一高渗透率得益于阿塔卡马地区极高的晴天率(年均>90% vs. 地中海东岸约75-80%)与极长的光照时长(夏季13-14小时、冬季10-11小时、年平均11.8小时),使得AE电解槽的"日间满负荷运行"策略可在全年大部分时段实施,年利用率可达48-52%(vs. 地中海系统的38%)。

然而,电解槽整体年利用率仅37.2%,是三个绿醇聚类中最低的(vs. 聚类0的59.7%、聚类1的63.5%),甚至略低于绿氨系统聚类2(37.0%),反映了纯光伏系统昼夜极端不均衡对设备利用率的根本制约——无论如何优化,光伏发电的"夜间完全停摆"特性注定了电解槽夜间只能依靠电池放电维持最低运行或完全停机,年利用率上限受物理约束。

储能系统配置呈现"超大容量电池主导+中等容量氢储辅助"特征:电池储能10.06 MWh、功率52.73 MW,单位装机储能比0.138 MWh/MW,是三个绿醇聚类中最高的(vs. 聚类0的0.058 MWh/MW高138%、vs. 聚类1的0.064 MWh/MW高116%),也显著高于绿氨系统聚类2(0.090 MWh/MW,增幅53%),反映了绿醇系统对电池储能的更高依赖——电池不仅需支撑电解槽夜间运行,还需支撑DAC与热泵的连续运行以确保甲醇合成不中断。电池年循环次数高达415次(按日均1.14次计算),是三个绿醇聚类中最高的,对电池循环寿命构成极限挑战(需采用超长寿命电池技术,如磷酸铁锂>6000次或钠离子>8000次)。

氢储罐2.73 kg H₂,约为日均产氢量的5.23倍(按年产氢1910 kg计算,日均5.23 kg),容量配置居于聚类0(9.34倍)与聚类1(5.52倍)之间。这一中等容量配置反映了纯光伏系统氢储罐的双重功能:(1)日际平滑,应对连续2-3天阴雨天气导致的光伏低发(虽然阿塔卡马晴天率极高,但偶尔仍有阴雨天,年均约30-35天);(2)碳氢同步供应冗余,当DAC因短期电力不足降负荷时,需氢储罐提供足够氢气以匹配CO₂储罐释放的CO₂。

碳氢耦合系统的优化配置与中等容量CO₂储罐:DAC装机2.02 kg CO₂/h,年捕集14000 kg CO₂(日均38.36 kg),利用率79.3%,居于聚类0(75.3%)与聚类1(82.9%)之间;热泵装机2.02 MW_th,热储能0.48 kWh_th(略高于聚类0的0.27 kWh_th,反映了纯光伏系统日内热负荷波动更大),COP约为3.6-4.0(得益于温和气候,年均气温12-18°C);甲醇合成装机12.61 tpy,甲醇储罐359.39 kg(约28天产量,是三个聚类中最大的,可能用于季节性生产调节或出口缓冲),年产甲醇10000 kg,合成单元利用率79.3%(与DAC完全一致)。

CO₂储罐容量7855 kg,约为日均捕集量的205倍,介于聚类0(261倍)与聚类1(79.8倍)之间。这一中等容量配置反映了纯光伏系统CO₂储罐的独特功能定位:主要应对"多日阴雨天气"而非"季节性调节"。阿塔卡马地区虽然极度干旱,但受厄尔尼诺-南方涛动(ENSO)影响,偶尔会出现连续5-10天的异常降水期(年均发生1-2次),此时光伏发电量可能降至正常值的20-40%,DAC被迫大幅降负荷甚至短暂停运,需CO₂储罐释放库存维持甲醇合成。储罐容量配置需覆盖"最长连续阴雨天数×日均CO₂需求"加上安全冗余,因此容量显著高于风光均衡系统(无需应对长周期阴雨)但低于极高纬度系统(需应对季节性极寒停运)。

年度运行特征热力图的极端昼夜循环与微弱季节差异

光伏发电的完美钟形日内曲线与南半球季节性:由于风电几乎不存在,光伏发电热力图完全主导系统供给侧特征。南半球季节性与北半球相反但特征类似:夏季(12月-2月)日照时长达13-14小时,光伏归一化强度午间10-17时达0.94-0.99极高水平(vs. 北半球地中海系统0.92-0.99,略高);冬季(6-8月)日照时长缩短至10-11小时,光伏归一化强度午间11-15时仍能保持0.85-0.94高位(vs. 北半球系统0.82-0.92,略高)。

阿塔卡马地区光伏出力的两大优势:(1)极高的大气透明度——年均晴天率>90%(vs. 地中海约75-80%),云量极少使得光伏出力稳定性极高,逐日发电量标准差仅为平均值的8-12%(vs. 地中海系统15-20%);(2)极低的组件温度损失——虽然太阳辐照强,但受秘鲁寒流影响沿海地区温度适中(夏季最高28-32°C vs. 地中海35-40°C),光伏组件温度系数损失降低3-5个百分点,年均发电效率提升至实验室效率的92-95%(vs. 地中海系统88-92%)。这两大优势使得阿塔卡马系统光伏容量因子达到28-30%(vs. 地中海系统25-27%,全球纯光伏系统最高水平之一)。

日内光伏出力呈现完美的"钟形"结构:日出后(当地时间6-7时,南半球夏季)光伏强度从0快速爬升至0.70-0.80(爬升速率0.11-0.13/小时,略快于地中海系统),午间(11-16时)维持0.94-0.99峰值平台(持续5-6小时),日落前(18-19时)快速下降至0.10以下(下降速率0.13-0.16/小时),夜间(20时-次日5时)完全停摆。冬季(6-8月)日出日落时间调整为7-8时与17-18时,午间峰值平台缩短至4-5小时(11-15时)。

季节差异相对微弱:夏季光伏日均发电量约为冬季的1.35倍(vs. 高纬度系统的3-4倍),主要源于中纬度(-36.25°S)太阳高度角季节变化较小(夏至正午77° vs. 冬至正午30°,差异47° vs. 高纬度系统>60°)。更重要的是,阿塔卡马地区冬季晴天率(85-90%)仍维持高位(vs. 地中海冬季70-80%),冬季光伏出力虽然时长缩短但强度不打折扣,使得季节性波动显著小于地中海系统。

电解槽运行的极端光伏追踪与"AE日间满负荷+PEM全天维持"策略:电解槽运行强度热力图与光伏发电呈现极高正相关(r=0.96,是三个绿醇聚类中最高的,也高于绿氨系统聚类2的r=0.94),呈现纯粹的"光伏追踪"特征。全年尺度,电解槽运行强度季节波动微弱:夏季归一化强度0.42-0.54(平均0.48),冬季0.32-0.44(平均0.38),季节差异26%(vs. 地中海系统30%,略小),反映了阿塔卡马冬季光照质量更优。

日内尺度,电解槽呈现"AE日间满负荷+PEM全天维持"的明确技术分工。AE电解槽(20.12 MW,占比68.5%):日出后(7-8时)快速启动并爬升至中高负荷(归一化强度0.55-0.70,爬升速率0.09-0.11/小时),上午9-10时达到满负荷前准备(0.85-0.92),午间光伏峰值时段(11-16时,夏季;11-15时,冬季)AE满负荷运行(0.96-1.00),日落后(17-18时)快速降负荷或停机,夜间(19时-次日6时)完全停机(归一化强度降至0)。这种"日间满负荷8-9小时、夜间完全停机15-16小时"的运行模式使得AE年利用率约为50%(vs. 地中海系统38%,提升32%),得益于阿塔卡马极高晴天率与较长光照时长。

PEM电解槽(9.24 MW,占比31.5%):全天候连续运行,但负荷率随光伏出力剧烈波动。日出后(7-8时)PEM快速启动并爬升至中等负荷(0.45-0.60),上午9-10时继续爬升(0.65-0.78),午间光伏峰值时段(11-16时)PEM满负荷运行(0.90-0.98),与AE协同制氢;日落后(17-19时)PEM快速降负荷(0.50-0.35),夜间(20时-次日6时)依靠电池储能放电维持最低运行(0.12-0.22),避免完全停机。PEM夜间最低负荷率12-22%(vs. 地中海系统10-20%,略高)反映了绿醇系统对氢气供应连续性的更严格要求——甲醇合成需H₂和CO₂同步供应,氢气中断将导致合成效率下降,因此PEM必须全天候维持基本运行。

日内负荷率波动幅度:AE为0-100%(极端波动),PEM为12-98%(波动幅度86个百分点,vs. 地中海绿氨系统70-85个百分点,波动更大),整体电解系统为12-99%(波动幅度87个百分点)。这种极端波动对电解系统的动态响应能力、循环耐久性、效率稳定性提出极限要求,是纯光伏系统固有的技术挑战。

电解槽年利用率:AE约50%(日运行8-9小时,年运行约3000-3300小时),PEM约34%(全天候运行但夜间极低负荷,年有效满负荷小时数约3000小时),整体加权平均37.2%。这一极低利用率(仅为风光均衡系统63.5%的58%)是纯光伏系统最大的经济劣势——电解槽投资约占系统总投资30-35%,但年运行时间不足40%,设备投资效率极低。

电池储能的极限"满充-满放"循环与超大容量配置:电池储能SOC热力图呈现三个绿醇聚类中最极端的"满充-满放"特征,且相比绿氨系统聚类2进一步恶化。典型晴天,电池在上午7-9时捕获光伏初升段快速充电(SOC从晨间5-10%升至48-58%,充电速率18-22个百分点/小时,充电功率0.85-1.00C接近满功率),午后11-17时光伏峰值时段继续充电至95-99%(接近满充,充电速率7-9个百分点/小时,充电功率0.55-0.75C),傍晚18时开始持续放电支撑电解槽、DAC、热泵、甲醇合成全系统夜间运行(放电速率5-8个百分点/小时,放电功率0.50-0.75C),至次日早晨6时SOC降至5-10%(接近空电保护阈值),形成完整的"满充-满放"循环。

日内SOC波动幅度高达88-94个百分点(vs. 绿氨系统聚类2的82-90个百分点,波动更大),日均等效满充满放次数达1.12-1.18次,年循环次数高达415次(vs. 绿氨系统390次,增幅6.4%)。更为严峻的是充电功率的极高峰值:午后光伏峰值时段电池充电功率可达额定功率的90-100%(持续4-6小时),对电池的大倍率充电耐受性、热管理能力提出极限要求。阿塔卡马地区夏季环境温度可达28-32°C,电池大倍率充电时内部温度可升至45-55°C(接近安全上限60°C),必须配置高效液冷系统(冷却功率约为充电功率的8-12%)以确保电池安全。

阴雨天气对电池储能的冲击:虽然阿塔卡马晴天率极高(>90%),但偶尔仍有阴雨天(年均30-35天,主要集中在冬季6-8月)。连续2-3天阴雨时,光伏发电量降至正常值的20-40%,电池储能面临"入不敷出"困境:白天充电不足(SOC仅能充至40-60% vs. 晴天95-99%),夜间仍需放电支撑系统运行(SOC降至10-20%),第二天若仍为阴雨则SOC可能跌破5%触发保护停机。此时系统被迫大幅降负荷:电解槽降至20-30%负荷率,DAC降至30-50%,甲醇合成降至50-60%,依靠氢储罐与CO₂储罐释放库存维持基本运行。

为应对这种极端场景,系统配置了超大容量电池(10.06 MWh,0.138 MWh/MW)作为"安全冗余"。在设计容量下,电池可支撑系统在光伏完全停发(极端阴雨天)情况下维持30-35%基本负荷运行约12-15小时(覆盖一个夜间),为次日光伏恢复争取时间。但这种超大容量配置显著增加了投资成本(电池储能约占系统总投资18-22% vs. 风光均衡系统10-12%),是纯光伏系统经济性劣势的重要来源。

DAC系统的全年高稳定运行与微弱季节波动:DAC运行强度热力图呈现出绿醇系统中次优的稳定性(仅次于风光均衡系统聚类1),全年归一化强度维持在0.75-0.85窄幅区间,季节差异仅13%(vs. 聚类0的>400%、聚类1的12%)。夏季(12月-2月)气温略高(18-28°C)但仍适宜DAC运行,归一化强度达0.82-0.85(平均0.84);冬季(6-8月)气温略低(8-15°C)但远高于冰点,DAC吸附剂活性仍维持较高水平,归一化强度0.75-0.80(平均0.78),仅比夏季降低7%(vs. 聚类0的80%降幅);春秋过渡季DAC强度居中(0.79-0.83)。

阿塔卡马地区DAC运行的三大优势:(1)全年温度适中(年均气温12-18°C,无极寒也无酷暑),DAC能效比全年稳定(单位捕集能耗0.48-0.55 kWh/kg CO₂,季节波动<15% vs. 聚类0的56-60%);(2)极低湿度(年均相对湿度<40%,世界最干燥地区之一),降低了DAC吸附剂再生能耗(湿度高时水分竞争吸附CO₂,需额外能量去除水分);(3)极高晴天率确保DAC全年稳定供电(年均停运天数<10天,主要为设备检修而非缺电)。

DAC年利用率79.3%,全年无明显淡季,各季度利用率均维持在75-85%区间。日内尺度,DAC运行强度与光伏发电高度相关(r=0.88):午间光伏峰值时段(11-16时)DAC满负荷或接近满负荷运行(归一化强度0.88-0.95),夜间依靠电池放电维持中高负荷(0.68-0.78,vs. 聚类1的0.75-0.82略低,反映了纯光伏系统夜间电力供应更紧张),傍晚过渡时段(17-21时)DAC略降负荷(0.62-0.72)以节约电池储能。DAC功耗约占系统总发电量的9-13%(vs. 聚类0的6-15%,聚类1的8-12%,居中),单位捕集能耗0.48-0.55 kWh/kg CO₂(vs. 聚类0的0.5-0.8 kWh/kg CO₂加权平均约0.62,聚类1的0.45-0.55,阿塔卡马系统最优)。

CO₂储罐与氢储罐的协同调节与"应对异常天气"功能定位:CO₂储罐SOC热力图呈现"微季节性+异常波动"的双重特征。常年(晴天占比>90%的时段)CO₂储罐SOC维持在55-75%的中高位区间,周际波动幅度仅4-7个百分点,反映了DAC全年高稳定运行(利用率79.3%)使得CO₂供需高度平衡。季节性趋势极弱:夏季(12-2月)DAC略强于甲醇合成需求,储罐SOC缓慢上升(月均净增1-2个百分点),从夏初的58%升至夏末的62%;冬季(6-8月)DAC略弱于甲醇合成需求,储罐SOC缓慢下降(月均净降1-2个百分点),从冬初的70%降至冬末的66%;春秋过渡季SOC基本持平。全年SOC最大差值仅12个百分点(vs. 聚类0的55个百分点、聚类1的15个百分点),是三个绿醇聚类中季节性最小的。

然而,CO₂储罐的核心功能并非季节调节,而是应对异常天气冲击。当连续5-10天异常降水期(年均发生1-2次,持续时间10-15%月度)到来时,光伏发电量降至正常值20-40%,DAC被迫大幅降负荷(利用率从79%降至25-35%),产CO₂速率暴跌至0.8-1.2 kg/h(vs. 正常2.0 kg/h,降幅40-60%),远低于甲醇合成需求(1.44 kg/h)。此时CO₂储罐SOC在10-15天内快速下降15-25个百分点(日均净降1.5-2.5个百分点),累积释放400-700 kg CO₂(占年捕集量3-5%)维持甲醇合成运行。异常天气过后,DAC快速恢复满负荷运行(利用率提升至85-90%,超过正常值以"补产"),CO₂储罐SOC在随后15-20天内回升至原有水平。

CO₂储罐容量配置逻辑:需覆盖"最长连续异常天气天数(10-15天)×日均CO₂需求(38.4 kg)×安全系数(1.5-2.0)",计算得出需求容量约为576-1152 kg。实际配置7855 kg(205倍日产量)远超这一需求,主要考虑:(1)厄尔尼诺年份异常降水可能更频繁(年发生3-4次)或更持久(单次持续15-20天);(2)与氢储罐协同,确保碳氢同步供应的"安全冗余";(3)为未来系统扩容(年产量从10000 kg提升至15000-20000 kg)预留容量。虽然当前容量利用率不高(常年SOC在55-75%,储罐容量利用率约15-20%),但这种"过配置"是纯光伏系统应对极端天气风险的必要代价。

氢储罐SOC同样呈现"微季节性+异常波动"特征,全年SOC在48-68%区间波动,日际波动幅度3-6个百分点,周际波动幅度5-9个百分点。季节性趋势与CO₂储罐基本同步:夏季略有积累(SOC从50%升至58%),冬季略有释放(SOC从58%降至52%),春秋持平。异常天气期间,氢储罐SOC下降幅度(10-15天内降低8-15个百分点)小于CO₂储罐(15-25个百分点),反映了氢气生产虽然也受光伏影响但电解槽可通过电池放电维持一定负荷(夜间12-22%、阴天白天30-50%),而DAC对电力供应更敏感(夜间依赖电池、阴天白天也大幅降负荷)。

碳氢供应的同步性分析:甲醇合成化学计量比H₂:CO₂=3:1(摩尔比),系统通过碳氢储罐协同调节实现了高度同步供应。全年H₂供应与CO₂供应的相关系数r=0.91(日尺度)、r=0.86(周尺度),略低于聚类1(r=0.94、0.88)但仍维持高水平。氢碳比波动范围2.80-3.20(标准差0.11,vs. 聚类1的0.08),偏离幅度±7%(vs. 聚类1的±5%),反映了纯光伏系统日内波动剧烈对碳氢同步性的轻微影响,但通过储罐缓冲仍能维持在可接受范围内。

异常天气期间碳氢同步面临挑战:当光伏发电量降至20-40%时,电解槽与DAC被迫同步降负荷,但两者降负荷幅度可能不完全匹配(电解槽降至30-50%、DAC降至25-35%,比例略有偏离)。此时系统优化算法实时监测氢碳比,若偏离超过±10%,自动调节电解槽或DAC负荷以恢复平衡:若氢气相对过剩(氢碳比>3.3:1),优先降低电解槽负荷并增加氢储罐充填;若CO₂相对过剩(氢碳比<2.7:1),优先降低DAC负荷并增加CO₂储罐充填。通过这种动态调节,异常天气期间氢碳比波动范围仍可控制在2.6-3.4:1(偏离±13%),确保甲醇合成不因原料配比失衡而效率暴跌。

热泵系统与甲醇合成的高效热电耦合:热泵运行强度热力图与DAC高度相关(r=0.94),全年归一化强度0.75-0.85,年利用率79.3%(与DAC、甲醇合成完全一致)。阿塔卡马地区气候温和(年均12-18°C)且季节差异小(夏季18-28°C、冬季8-15°C,差异仅10-13°C vs. 聚类0的40-55°C),使得热泵COP全年维持高位且波动极小:夏季COP 3.8-4.0(全年最高),冬季COP 3.5-3.7(仅比夏季降低5-8% vs. 聚类0的12-16%),春秋COP 3.6-3.9。全年加权平均COP约为3.7(vs. 聚类0的3.0-3.2、聚类1的3.5-3.8),是三个绿醇聚类中最优的,反映了温和气候对热泵效率的显著利好。

热泵单位甲醇热能需求约为2.7-3.0 MJ_th/kg MeOH(vs. 聚类0的2.8-3.2、聚类1的2.8-3.2,略低),对应电力消耗约0.73-0.81 MWh_e/kg MeOH(考虑COP 3.7),占系统总电耗约11-13%(vs. 聚类0的12-15%、聚类1的12-14%,略低)。纯光伏系统虽然电力供应日内波动大,但热泵可通过热储能(0.48 kWh_th)进行小时级平滑:午间光伏峰值时段(11-16时)热泵满负荷运行并将过剩热能储存,夜间热泵降负荷运行并释放热储能,平滑了甲醇合成的热量需求波动。

甲醇合成装置的高稳定性运行与双储罐协同支撑:甲醇合成装置运行强度热力图呈现极高稳定性,全年归一化强度维持在0.77-0.82窄幅区间,日内波动<3%,周际波动<5%,季节波动<6%(vs. 聚类0的<8%、聚类1的<7%,进一步改善),是三个绿醇聚类中稳定性最优的。这一稳定性通过CO₂储罐(7855 kg)与氢储罐(2.73 kg)的协同调节实现:即使光伏发电日内剧烈波动(昼夜100%-0%),甚至异常天气连续多日低发(降至20-40%),双储罐仍能持续供应H₂和CO₂支撑甲醇合成高负荷运行,避免频繁启停或大幅降负荷。

年度热力图显示,甲醇合成装置仅在春秋设备检修期(4月中旬与10月中旬各持续3-5天)运行强度短暂降至0.18-0.28,其余时段均保持高负荷稳定运行。夏季(12-2月)晴天多+光伏高发+DAC高效,合成装置运行强度略升至0.80-0.82;冬季(6-8月)晴天率略降+光伏略弱+DAC略降负荷,合成装置运行强度略降至0.77-0.79;春秋过渡季居中(0.78-0.81)。更重要的是,即使异常天气期间(光伏降至20-40%),合成装置仍能维持65-75%负荷率运行(vs. 电解槽30-50%、DAC 25-35%),得益于双储罐的强大缓冲能力。

合成单元年利用率79.3%,显著高于聚类0(75.3%)与绿氨系统聚类2(78.8%),是三个绿醇聚类中次优的(仅次于聚类1的82.9%)。负荷率>75%的运行时长占比达86%(vs. 聚类0的83%、聚类1的90%),反映了阿塔卡马系统通过"牺牲电解槽利用率(37.2%)+配置超大电池储能(10.06 MWh)+配置充裕双储罐(CO₂ 7855 kg + H₂ 2.73 kg)"的策略,优先保障下游合成装置高负荷稳定运行,在合成环节实现价值最大化。这种策略与聚类0完全一致,反映了极端资源禀赋系统(纯风电或纯光伏)的共同优化逻辑。

#### 四季典型周电力供需平衡与影子价格动态分析

**夏季典型周(12月第3周):光伏充沛但电池极限循环的供需格局**

夏季南半球夏至期间,阿塔卡马聚类2系统呈现极端日变化特征。光伏出力在正午达到峰值94-99%标准化强度(约68-72 MW),日出日落斜率极陡,夜间完全归零。电解槽系统采用"AE主导白天+PEM维持夜间"的分工策略:AE装置(20.13 MW,占68.5%)在上午8:30-下午17:30期间满负荷运行9小时,对应光伏出力>50%时段;PEM装置(9.23 MW)承担夜间最低氢产能维持(22-28%负荷率),确保DAC与合成装置连续运行不中断。

电池储能系统承受极限循环压力,日充放电深度88-94% SOC摆幅(约8.9-9.5 MWh能量吞吐)。白天10:00-15:00光伏过剩期间,电池以15-18 MW功率快充至满容量;夜间18:00-次日7:00期间,电池以8-12 MW功率持续放电支撑PEM电解与系统辅机。储能影子价格显示显著日内波动:夜间放电期间影子价格升至85-95 $/MWh(反映储能稀缺性),白天充电期间降至15-25 $/MWh,峰谷价差高达70 $/MWh,是三个绿醇聚类中最大的(聚类0为45 $/MWh、聚类1为35 $/MWh)。

氢能供需平衡方面,AE白天9小时满负荷制氢产量约480 kg H₂,PEM夜间15小时低负荷制氢约110 kg H₂,日总产氢590 kg H₂。DAC装置维持79.3%平均负荷(约1.60 kg CO₂/h),日耗氢量约38.4 kg H₂,合成装置维持79.3%负荷耗氢约520 kg H₂/day。氢储罐(2.73 kg容量)发挥关键缓冲作用:白天光伏过剩期间储氢罐充至满容量,夜间PEM产氢不足期间储氢罐补充供氢,维持DAC与合成装置稳定运行。氢气影子价格呈现双峰模式:上午8-10时AE启动期间升至4.8-5.2 $/kg H₂(制氢能力爬坡),下午16-18时AE关停期间再次升至4.5-4.9 $/kg H₂(制氢能力下降),夜间PEM维持期间稳定在3.8-4.2 $/kg H₂。

**冬季典型周(6月第3周):光伏减弱但系统韧性凸显的运行模式**

冬季南半球冬至期间,光伏资源相对减弱但仍保持较高水平。峰值光伏出力降至85-92%标准化强度(约62-67 MW),有效光照时长缩短至7.5-8小时,但晴天率仍达90%以上(阿塔卡马气候优势)。电解槽运行策略调整为"AE缩短+PEM增强":AE满负荷运行时长缩短至7小时(9:30-16:30),PEM夜间负荷率提升至35-42%(vs. 夏季22-28%),以补偿AE制氢量下降。

电池储能循环深度略微减缓至82-89% SOC摆幅,但仍属高强度循环。白天充电功率降至12-15 MW,充电时长缩短至6小时;夜间放电功率维持8-12 MW,放电时长延长至17小时。储能影子价格冬季波动幅度缓解:夜间峰值降至78-85 $/MWh,白天谷值升至20-28 $/MWh,峰谷价差收窄至55-60 $/MWh。这反映了冬季光伏资源相对稀缺但电池配置仍然充裕的平衡状态。

氢能供需在冬季呈现更紧平衡格局。AE冬季7小时制氢约370 kg H₂/day,PEM夜间17小时提升负荷制氢约155 kg H₂/day,日总产氢525 kg H₂(vs. 夏季590 kg)。DAC与合成装置维持相同负荷率(79.3%),日总耗氢约558 kg H₂,形成33 kg H₂/day的供需缺口。氢储罐发挥更加关键的季节调节作用:春夏氢气过剩期间储罐积累库存,秋冬氢气偏紧期间储罐释放库存,维持合成装置全年稳定运行。氢气影子价格冬季整体上移:日间峰值升至5.4-5.8 $/kg H₂,夜间基值升至4.2-4.6 $/kg H₂,反映了氢资源相对稀缺性增强。

**春季典型周(10月第2周):设备检修与产能爬坡的过渡特征**

春季(南半球春季)作为年度设备大修窗口期,系统运行呈现典型的"检修模式"特征。10月第2周恰逢年度维护高峰,光伏、电解槽、DAC、合成装置等主要设备按计划轮流检修。光伏系统维持85-90%可用率(部分组件清洁维护),AE电解槽降至70-75%可用率(电堆检修更换),DAC装置降至60-65%可用率(压缩机系统维护),合成装置降至55-60%可用率(催化剂更换与反应器检修)。

电池储能在春季检修期发挥"削峰填谷+应急备用"的双重作用。一方面,储能继续承担日常的日内循环任务,SOC摆幅略降至75-85%(设备检修期间电力需求下降);另一方面,储能为设备检修提供应急电力保障,避免检修期间系统完全停机。储能影子价格春季波动模式介于夏冬之间:夜间峰值70-80 $/MWh,白天谷值25-35 $/MWh,峰谷价差45-50 $/MWh。

氢能供需在春季检修期呈现"产能下降+需求下降+储罐调节"的三重平衡。AE检修期间日制氢量降至280-320 kg H₂,PEM承担更大比重(日制氢120-140 kg H₂),总产氢400-460 kg H₂/day。DAC检修期间耗氢降至25-30 kg H₂/day,合成装置检修期间耗氢降至300-350 kg H₂/day,总耗氢325-380 kg H₂/day。供需仍有75-80 kg H₂/day过剩,氢储罐继续积累库存为冬季做准备。氢气影子价格春季保持相对稳定:4.0-4.5 $/kg H₂区间小幅波动,反映了检修期间供需双降的平衡状态。

**秋季典型周(4月第2周):产能恢复与库存调整的收尾阶段**

秋季(南半球秋季)作为另一个设备检修窗口期,但规模较春季轻微,主要进行设备微调与性能优化。4月第2周各装置可用率恢复至90-95%水平:光伏系统完成年度组件检测(95%可用率),电解槽完成电堆性能测试(92%可用率),DAC完成吸附剂更换(90%可用率),合成装置完成催化剂活性检测(93%可用率)。光伏出力秋季介于夏冬之间,峰值强度88-95%(约64-69 MW),有效光照8-8.5小时。

电池储能秋季循环特征渐趋正常,SOC摆幅恢复至85-92%,充放电功率与时长逐步向夏季模式过渡。储能影子价格秋季略高于春季:夜间峰值75-85 $/MWh,白天谷值20-30 $/MWh,峰谷价差50-60 $/MWh,反映了设备检修完成后产能恢复、电力需求上升的趋势。

氢能供需秋季呈现"产能恢复+需求回升+库存释放"的调整格局。AE秋季8小时制氢约420 kg H₂/day,PEM夜间16小时制氢约130 kg H₂/day,日总产氢550 kg H₂(接近夏季水平)。DAC与合成装置负荷率恢复至标准79.3%,日总耗氢558 kg H₂,小幅供需缺口8 kg H₂/day由氢储罐库存补充。氢气影子价格秋季温和上升:日间4.2-4.8 $/kg H₂,夜间3.9-4.3 $/kg H₂,为即将到来的冬季资源偏紧做准备。

#### 综合技经评价与系统优化方向

**成本结构分析与跨聚类比较**

阿塔卡马聚类2绿醇系统平准化成本620.31 $/吨甲醇,在三个绿醇聚类中居中水平(聚类1最优579.70 $/吨,聚类0最高721.01 $/吨),但较绿氨系统聚类2(462.39 $/吨)高出34.2%,反映了碳氢耦合系统相对单纯制氢系统的复杂性溢价。成本结构中,可再生能源发电成本占比42.3%(纯光伏系统单位投资成本较风光混合偏高),电解制氢成本占比28.7%(电解槽利用率37.2%偏低导致单位制氢成本上升),DAC捕碳成本占比16.4%(阿塔卡马气候优势下DAC能耗较低),储能系统成本占比8.9%(电池储能承受极限循环,折旧摊销较高),合成单元成本占比3.7%(合成装置79.3%高利用率摊薄固定成本)。

跨聚类成本对比显示清晰的地理-技术-经济梯度:亚洲中纬度聚类1(579.70 $/吨)凭借风光资源均衡+AE电解槽46.4%渗透率+DAC全年82.9%高利用率,实现最优经济性;南美纯光伏聚类2(620.31 $/吨)通过阿塔卡马极端气候优势+AE电解槽68.5%最高渗透率+热泵COP 3.7最优效率,成本劣势控制在7.0%以内;亚洲高纬度聚类1(721.01 $/吨)受制于极地气候DAC季节性波动(夏季90%冬季18%)+纯PEM电解技术+超大CO₂储罐配置(10,037 kg),成本溢价达24.4%。这一梯度反映了绿醇系统对气候稳定性、资源均衡性的高度敏感,以及DAC技术在不同气候条件下的性能差异。

**系统优化潜力与技术改进方向**

阿塔卡马聚类2系统优化潜力主要集中在"电池循环缓解+电解槽利用率提升+储能成本降低"三个维度。电池储能优化方面,当前415次/年的极限循环(88-94% SOC日摆幅)导致电池寿命缩短至6-7年(vs. 标准10年),储能LCOE上升25-30%。通过配置季节性氢储能(100-200 kg H₂容量)或引入需求侧响应机制(合成装置30%柔性负荷),可将电池循环强度降至300次/年、SOC摆幅控制在70-85%,延长电池寿命至8-9年,储能成本下降15-20%。

电解槽利用率提升方面,当前AE+PEM混合配置下系统利用率仅37.2%,显著低于聚类1的63.5%。通过优化电解槽容量配置(AE从20.13 MW扩容至25-28 MW,PEM从9.23 MW缩减至6-8 MW)和引入电解槽功率调节技术(AE 20-100%负荷调节,PEM 10-110%负荷调节),可将系统利用率提升至45-50%,电解制氢成本下降12-15%。同时,通过增设制氢余热回收系统(AE电堆余热回收效率60-70%,PEM电堆余热回收效率40-50%),为DAC预热和甲醇反应器加热提供热源,系统整体能效提升8-12%。

储能技术路径优化方面,考虑到阿塔卡马地区极端日变化特征,可探索"锂电+液流电池+重力储能"的混合储能配置。锂电池(2-4 MWh)承担小时级调频调峰,液流电池(4-6 MWh)承担日内循环主力,重力储能(2-4 MWh)承担长周期平衡,三类储能发挥各自技术优势,综合储能成本可下降10-15%、循环寿命延长20-30%。结合智能能量管理系统(EMS)与多元储能协调控制,系统运行稳定性与经济性双重提升。

**技术演进趋势与产业化前景**

展望2030-2040年技术演进,阿塔卡马聚类2类型的纯光伏绿醇系统将迎来重大技术突破期。光伏技术方面,钙钛矿-硅串联电池效率提升至32-35%(vs. 当前25-28%),光伏LCOE下降40-50%至15-20 $/MWh,为绿醇系统提供更廉价清洁电力。电解槽技术方面,固体氧化物电解槽(SOEC)大规模产业化,电解效率提升至85-90%(vs. 当前AE 65-70%、PEM 70-75%),高温余热充分利用,制氢成本下降30-40%至2.5-3.0 $/kg H₂。

DAC技术方面,固体吸附剂DAC技术突破(吸附剂再生温度降至80-120°C vs. 当前150-200°C),DAC电耗下降25-35%至1.2-1.8 MWh/t CO₂,热耦合效率大幅提升。储能技术方面,钠离子电池、液态金属电池等新兴技术规模化应用,储能LCOE下降50-60%至50-80 $/MWh,极大缓解纯光伏系统储能成本压力。综合技术进步推动下,阿塔卡马类型绿醇系统成本有望降至350-400 $/吨甲醇,接近当前化石燃料制甲醇成本(300-350 $/吨),实现经济性平价。

产业化前景方面,阿塔卡马及其他类似地区(澳大利亚内陆、中东北非、美国西南部)将成为全球绿醇产业的重要基地。这些地区普遍具备"极优光照资源+干旱气候+土地充裕+远离负荷中心"的共同特征,适合发展大规模出口导向型绿醇产业。预计2030年后,此类地区绿醇产能将占全球总产能的40-50%,通过海运输出至欧洲、东亚等主要消费市场。阿塔卡马聚类2的技术路径与运行模式,将为全球同类地区绿醇项目提供重要参考,推动全球绿醇产业从示范走向规模化商业运营。


## 2040年LowAE情景绿色SAF系统三聚类分析

基于`median_points_summary_RegionRE_LowAE_MidFTL.csv`数据,分析2040年LowAE情景下绿色SAF(可持续航空燃料)生产系统的三个典型聚类特征。SAF系统相较于绿氨与绿醇,技术复杂度显著提升:除了电解制氢与DAC捕碳外,还需要Fischer-Tropsch合成工艺(FTL)将合成气转化为C5-C20长链烷烃混合物(航空煤油),工艺参数控制更严格,产品纯化要求更高。三个聚类分别代表北美高纬度风主导(66.75°N,-97.75°W)、亚洲中纬度风光平衡(49.5°N,52.25°E)、亚洲赤道纯光伏(-1.25°N,104.25°E)的资源禀赋模式。

### 聚类0:北美高纬度风主导SAF系统(66.75°N,-97.75°W)

#### 技术配置与碳氢耦合架构

聚类0位于加拿大北部马更些地区(66.75°N,-97.75°W),技术配置体现了对极地风能资源的充分利用与极端气候的适应。可再生能源装机为风电61.07 MW+光伏57.90 MW(风光比1.06:1),在三个SAF聚类中最接近均衡,但实际发电量中风电占比达75-80%(冬季90%以上)。电解制氢系统总装机47.31 MW,其中AE占6.39 MW(13.5%)、PEM占40.93 MW(86.5%),PEM主导地位反映了极地环境对低温性能与快速响应的要求。电解系统年利用率58.5%,年产氢36,020 kg H₂。

碳氢耦合环节配置DAC捕碳装置5.05 kg CO₂/h、热泵系统5.05 MW_th(热储能0.31 MWh_th)、FTL合成装置12.30 吨SAF/年设计产能(实际5.04吨/年)。储能系统包括电池储能28.12 MWh、氢储罐15.97 kg H₂、CO₂储罐10,000 kg(相当于198小时捕碳量,是三聚类最大规模)。DAC年利用率81.4%、FTL合成年利用率81.3%,均处于较高水平。系统平准化成本1876.06 $/吨SAF,为三聚类最高,反映了极地环境设备投资溢价(防冻保温系统)与冬季极夜资源稀缺的双重影响。

#### 年度运行热力图分析

**风电与光伏发电特征**

年度热力图显示,风电出力呈现"冬强夏弱"季节模式与"日内平稳"时变特征的独特组合。冬季(11月-次年3月)风电运行强度最高,标准化强度常年维持0.75-0.95区间,对应实际功率45-58 MW,部分寒潮大风期间可短暂达0.98-1.00满发状态。春秋过渡季(4-5月、9-10月)风电强度略降至0.65-0.85,对应40-52 MW。夏季(6-8月)风速相对较低,风电强度降至0.50-0.70,对应30-43 MW,但仍保持稳定基础出力。风电日内波动幅度一般±15%以内,显著小于光伏系统,为系统提供稳定电力基荷。

光伏发电呈现极端季节性与日内变化双重特征。夏季极昼期间(6-8月),光伏日照时长达18-24小时,但由于太阳高度角较低(最高35-45°),标准化强度峰值仅0.55-0.70,对应32-41 MW,且呈现"双峰"或"平顶"形态(凌晨3-5时、傍晚20-22时各有次峰)。春秋过渡季光伏恢复正常昼夜节律,日照12-14小时,峰值强度0.60-0.75,对应35-43 MW。冬季极夜期间(12-2月),光伏日照仅2-6小时,峰值强度骤降至0.15-0.30,对应9-17 MW,且集中在中午前后2-3小时,对系统贡献微弱。全年来看,光伏有效发电集中在4-9月,承担夏季补充电源角色。

**电解槽运行策略与氢能生产**

电解槽系统运行呈现"PEM基荷+AE调峰"分工模式。PEM电解槽(40.93 MW)全年24小时连续运行,运行强度常年维持0.45-0.75区间,对应18-31 MW实际功率,年平均负荷率约55%。冬季风电充裕期间,PEM运行强度升至0.65-0.75;夏季资源相对紧张期间(风电减弱+光伏峰谷差大),PEM运行强度降至0.45-0.60,但始终不低于0.40最低维持负荷,确保制氢连续性。

AE电解槽(6.39 MW)主要在风光过剩时段启动,承担调峰制氢任务。春秋风光资源均衡期间(4-5月、9-10月),AE日运行时长8-12小时,运行强度0.70-0.90,对应4.5-5.8 MW;冬季风电极度充裕+光伏几乎无贡献期间,AE可连续运行16-20小时,运行强度0.75-0.95,充分消纳弃风电量;夏季风电减弱期间,AE仅在光伏峰值时段(中午前后4-6小时)启动,运行时长缩短至4-8小时,运行强度0.50-0.75。AE运行呈现明显"春秋高-冬季中-夏季低"季节梯度,与风光资源互补程度密切相关。

**电池储能循环特征与SOC动态**

电池储能系统(28.12 MWh)在极地SAF系统中发挥"风光互补桥梁+季节负荷平衡"双重功能。夏季(6-8月)光伏极昼期间,电池呈现日循环特征:光伏出力高峰期(中午前后及晚间次峰)电池充电至85-95% SOC,夜间低谷期(23:00-次日5:00)电池放电至40-55% SOC支撑PEM电解,日充放电深度约35-45%,循环强度中等。冬季(12-2月)风电强劲+光伏微弱期间,电池循环频率大幅降低:风电持续高出力使电池长期维持70-85% SOC高位,仅在个别风电骤降事件(锋面过境)时短暂放电至50-60% SOC应急,整体循环强度显著弱于夏季。

春秋过渡季(3-5月、9-11月)电池循环最为活跃,呈现"双日循环"模式:白天光伏爬升期间电池第一次充电(8:00-12:00),下午光伏下降+风电上升期间电池部分放电再充电(12:00-18:00),夜间风电高峰期间电池第二次充电(20:00-24:00),凌晨风电低谷期间电池放电(0:00-6:00)。SOC在50-90%区间频繁波动,日充放电深度50-60%,年循环次数约280-320次。电池SOC季节性变化明显:冬季高位(70-85%)、夏季中位(55-75%)、春秋波动(50-90%),反映风光资源季节性匹配程度差异。

DAC系统运行与CO₂捕获动态

DAC装置(5.05 kg CO₂/h)年度运行呈现"季节性强烈波动+日内相对稳定"特征。夏季(6-8月)风光资源最充裕期间,DAC运行强度达全年峰值0.88-0.95,对应捕碳速率4.45-4.80 kg CO₂/h,日运行时长20-23小时,仅在极个别光伏极弱+风电骤降的恶劣天气短暂降负荷至0.60-0.70。春秋过渡季(4-5月、9-10月)DAC运行强度回落至0.75-0.88,对应捕碳速率3.79-4.44 kg CO₂/h,日运行时长18-21小时。冬季(12-2月)尽管风电强劲,但极地严寒(气温-30至-40°C)导致DAC吸附剂活性下降、再生能耗上升,运行强度降至0.65-0.80,对应捕碳速率3.28-4.04 kg CO₂/h,日运行时长16-19小时。

年度热力图显示,DAC运行强度与环境温度呈明显负相关:1-2月极寒期间运行强度降至全年最低0.65-0.72,7-8月相对温暖期间(气温5-15°C)运行强度升至全年最高0.90-0.95,温差35-45°C对应运行强度差0.20-0.30。这一特征与绿醇系统DAC截然不同(绿醇聚类0与聚类1的DAC主要受电力供应影响而非温度影响)。DAC日内运行相对平稳,标准化强度波动幅度一般<0.10,仅在设备检修期(4月中旬与10月中旬各3-5天)降至0.20-0.35低负荷维护状态。

CO₂与H₂储罐协调运行

CO₂储罐(10,000 kg容量)与H₂储罐(15.97 kg容量)构成SAF系统核心储能缓冲体系,年度SOC动态清晰展现其季节调节功能。CO₂储罐SOC呈现显著"夏高冬低"季节性波动:6-9月夏季DAC高负荷捕碳期间,储罐SOC从40%逐步攀升至85-92%峰值(对应8500-9200 kg CO₂库存),积累约5000-5500 kg CO₂季节性储备;10月-次年2月秋冬季DAC降负荷期间,储罐SOC从85%逐步下降至35-45%(对应3500-4500 kg CO₂),释放约4000-5000 kg CO₂补充FTL合成消耗;3-5月春季为过渡期,储罐SOC在40-60%区间小幅波动。年度SOC摆幅高达50-55%,是三个SAF聚类中最大的,反映极地气候对DAC运行的极端季节性影响。

H₂储罐SOC呈现"日循环为主+季节微调为辅"特征。日内SOC典型波动模式:上午8-12时AE启动+PEM满负荷期间,制氢速率大于FTL消耗,储罐SOC从45-50%充至75-85%;下午12-18时AE降负荷+PEM维持运行期间,制氢与消耗接近平衡,储罐SOC在70-85%区间小幅波动;夜间18-次日8时仅PEM低负荷运行期间,制氢速率小于FTL消耗,储罐SOC从75-80%降至45-55%,完成一个完整日循环。季节性来看,夏季制氢相对充裕储罐SOC略高(日均60-70%),冬季制氢相对紧张储罐SOC略低(日均50-60%),但季节SOC摆幅仅10-15%,远小于CO₂储罐50-55%摆幅。

热泵系统与热储能运行

热泵系统(5.05 MW_th)年度运行强度呈现"冬季最高+夏季次之+春秋居中"独特季节模式。冬季(12-2月)尽管环境温度极低(-30至-40°C)导致热泵COP降至2.0-2.5(vs. 标准工况3.5-4.0),但DAC吸附剂再生在严寒环境下需要更高温度(200-220°C vs. 常温150-180°C)和更长再生时间,热泵运行强度达0.85-0.95,对应输出热功率4.29-4.80 MW_th,日运行时长20-23小时。夏季(6-8月)环境温度回升至5-15°C,热泵COP提升至3.0-3.5,但DAC高负荷运行(0.88-0.95)仍需大量再生热,热泵运行强度维持0.80-0.90,对应输出3.74-4.26 MW_th。春秋过渡季热泵运行强度降至0.70-0.85,对应3.29-3.99 MW_th。

热储能系统(0.31 MWh_th)容量相对较小,主要发挥日内削峰填谷而非季节调节功能。典型日循环模式:白天风光充裕+电价低廉期间(8:00-16:00),热泵以较高COP制热并向热储能充热,储能SOC从30-40%充至80-90%;夜间电力相对紧张期间(18:00-次日6:00),热储能放热支撑DAC连续运行,储能SOC从85-90%降至30-40%。年度热力图显示,热储能日充放电深度约50-60%,年循环次数280-320次,与电池储能循环频率高度一致,反映热电联合调度的协同优化逻辑。

FTL合成装置稳定运行特征

FTL合成装置(12.30吨/年设计产能)年度运行展现出高度稳定性,标准化运行强度全年维持0.75-0.87区间,远高于上游电解槽(58.5%)与DAC(81.4%)的波动性。年度热力图显示,FTL合成运行强度呈现温和季节变化:夏季(6-8月)风光充裕+DAC高效+氢气充足,合成装置运行强度略升至0.84-0.87,对应SAF产出0.573-0.593 kg/h;冬季(12-2月)极寒环境催化剂活性略降+DAC降负荷+氢气偏紧,合成装置运行强度略降至0.75-0.80,对应SAF产出0.511-0.545 kg/h;春秋过渡季居中0.79-0.84。季节运行强度极差仅0.12(vs. DAC的0.30、电解槽的0.30、风电的0.45、光伏的0.80),是所有主要设备中最稳定的。

这种高稳定性得益于三重缓冲机制协同作用:①CO₂储罐10,000 kg超大容量(198小时捕碳量)平滑DAC季节性波动,确保FTL合成全年CO₂原料充裕;②H₂储罐15.97 kg(3.9小时制氢量)平滑电解槽日内波动,维持FTL合成氢气连续供应;③热储能0.31 MWh_th平滑热泵日内波动,保障FTL反应器温度恒定(220-260°C)。年度热力图显示,FTL合成装置仅在春秋设备检修期(4月中旬与10月中旬各持续3-5天)运行强度短暂降至0.25-0.35,其余时段均保持高负荷稳定运行。合成单元年利用率81.3%,在三个SAF聚类中居中水平,显著优于绿氨系统聚类0的合成单元(78.8%),反映了FTL工艺对原料供应稳定性要求更高、系统储能配置更充裕的特点。

四季典型周电力供需平衡与影子价格动态分析
夏季典型周(7月第2周):极昼光伏补充与风电基荷的协同格局

夏季北半球极昼期间,聚类0系统呈现独特的"风电基荷+光伏极昼补充"供电模式。风电出力全天维持中等稳定水平,标准化强度55-70%区间(对应34-43 MW),日内波动幅度<10%,提供稳定基础电力。光伏出力呈现极昼特有的"平顶+双肩峰"形态:凌晨3-5时出现第一个次峰(强度40-50%,对应23-29 MW),中午12-14时达到主峰(强度65-70%,对应38-41 MW),傍晚20-22时出现第二个次峰(强度45-55%,对应26-32 MW),夜间23-次日2时降至最低但仍保持15-25%出力(对应9-14 MW)。风光合计出力70-110 MW,形成全天候电力供应。

电力需求侧,PEM电解槽承担主要负荷,全天24小时运行强度维持60-75%区间(对应25-31 MW),白天风光充裕期间略微提升至70-75%,夜间资源相对紧张期间回落至60-65%。AE电解槽在光伏峰值时段(10:00-18:00)启动调峰,运行8小时,强度75-90%,对应4.8-5.8 MW。DAC系统高负荷运行,强度90-95%对应捕碳速率4.54-4.80 kg CO₂/h,日运行23小时。FTL合成装置稳定运行强度85-87%,热泵系统运行强度85-90%支撑DAC再生。电池储能在光伏双峰时段充电(3-5时、12-14时、20-22时),夜间低谷期(23-2时)放电,SOC在55-90%区间日循环,充放电深度35%。

电力影子价格夏季呈现温和波动:夜间低谷期(23-2时)光伏最弱+风电中等,电价升至42-48 
/
M
W
h
(
反映电力相对稀缺
)
;
白天主峰期
(
12
−
14
时
)
风光合计最强
,
电价降至
28
−
33
/MWh(反映电力相对稀缺);白天主峰期(12−14时)风光合计最强,电价降至28−33/MWh;凌晨与傍晚次峰期电价居中35-40 
/
M
W
h
。峰谷价差
15
−
20
/MWh。峰谷价差15−20/MWh,是三个SAF聚类中最小的(聚类1为25-30 
/
M
W
h
、聚类
2
为
60
−
70
/MWh、聚类2为60−70/MWh),反映了极昼期间光伏近似"准连续"发电对电力供应的平滑作用。

氢能供需平衡方面,夏季制氢相对充裕。PEM日制氢约580 kg H₂,AE日制氢约85 kg H₂,合计665 kg H₂/day。FTL合成日耗氢约630 kg H₂,供需有35 kg H₂/day小幅过剩,氢储罐逐步充至65-70% SOC季节性高位。氢气影子价格夏季保持相对稳定:白天制氢高峰期降至3.5-3.8 
/
k
g
H
2
,
夜间制氢低谷期升至
4.0
−
4.3
/kgH 
2
​
 ,夜间制氢低谷期升至4.0−4.3/kg H₂,日内波动幅度0.5-0.8 $/kg,反映了氢储罐有效平抑制氢波动的缓冲作用。

冬季典型周(1月第3周):极夜资源稀缺与超大储罐缓冲的应对策略

冬季北半球极夜期间,聚类0系统面临最严峻的资源挑战。风电出力保持全年最高水平,标准化强度80-95%区间(对应49-58 MW),但仅此单一电源难以满足全系统需求。光伏出力极度萎缩,日照仅集中在中午前后2-3小时(11:00-14:00),峰值强度仅20-30%,对应12-17 MW,其余21小时完全无光伏贡献。风光合计出力49-75 MW,较夏季下降30-35%。

电力需求侧被迫全面压缩以适应资源约束。PEM电解槽降负荷运行,强度40-55%区间(对应16-23 MW),较夏季下降15-20个百分点。AE电解槽几乎停运,仅在个别风电特强+光伏短暂出现的时段(12:00-14:00)启动2小时,强度50-70%,对应3.2-4.5 MW。DAC系统受双重约束(电力不足+严寒-35°C降低效率),运行强度降至65-75%,对应捕碳速率3.28-3.79 kg CO₂/h,日运行仅16-18小时。FTL合成装置依靠CO₂储罐释放库存维持运行,强度略降至75-80%。热泵系统在极寒环境COP降至2.0-2.3,但仍需高强度运行(85-90%)以满足DAC再生热需求。

电池储能冬季循环大幅减弱,风电持续高出力使电池长期维持75-85% SOC高位,仅在个别风电骤降事件(锋面过境,风速从15 m/s降至8 m/s)短暂放电至55-60% SOC应急,单日放电深度<20%。CO₂储罐发挥关键季节调节作用:冬季DAC捕碳速率3.28-3.79 kg CO₂/h,而FTL合成消耗速率维持4.1-4.4 kg CO₂/h,日缺口约10-15 kg CO₂完全由储罐释放补充,储罐SOC从秋季85%逐步降至冬末40%,释放约4500 kg CO₂库存。

电力影子价格冬季整体上移且波动加剧:夜间无光伏期间(18:00-次日10:00,长达16小时)仅靠风电单一电源,电价升至55-65 
/
M
W
h
峰值
;
中午短暂光伏补充期间
(
12
:
00
−
14
:
00
)
电价回落至
45
−
50
/MWh峰值;中午短暂光伏补充期间(12:00−14:00)电价回落至45−50/MWh;极个别风电特强时段(寒潮大风)电价可降至38-42 
/
M
W
h
。峰谷价差
20
−
27
/MWh。峰谷价差20−27/MWh,较夏季扩大5-10 $/MWh,反映冬季资源稀缺性增强。

氢能供需冬季呈现紧平衡格局。PEM日制氢约380 kg H₂,AE日制氢约15 kg H₂,合计仅395 kg H₂/day,较夏季骤降40%。FTL合成日耗氢约600 kg H₂,供需缺口205 kg H₂/day,完全依靠氢储罐释放库存弥补。氢气影子价格冬季显著上升:日间升至4.8-5.2 
/
k
g
H
2
,
夜间峰值可达
5.5
−
5.8
/kgH 
2
​
 ,夜间峰值可达5.5−5.8/kg H₂,较夏季上涨1.2-1.5 $/kg,反映氢资源冬季极度稀缺、储罐库存持续下降的紧张态势。

春季典型周(4月第2周):设备检修与产能恢复的过渡特征

春季作为年度设备大修窗口期,聚类0系统运行呈现"检修模式"特征。风电出力恢复至春季正常水平,强度65-80%区间(对应40-49 MW)。光伏摆脱冬季极夜,恢复正常昼夜节律,日照12-14小时,峰值强度60-70%,对应35-41 MW,夜间完全归零。但主要设备按计划轮流检修:风机可用率90%(部分机组叶片除冰维护),光伏可用率92%(组件清洁与电气检测),PEM电解槽可用率75%(电堆性能测试与膜更换),AE电解槽可用率70%(电解液更换与阳极检修),DAC可用率65%(吸附剂再生与压缩机维护),FTL合成可用率60%(催化剂活性检测与反应器清洗),热泵可用率80%(制冷剂充注与换热器清洁)。

电力供需在春季检修期呈现"产能下降+需求下降+储能调节"三重平衡。风光合计出力60-90 MW(考虑检修因素实际可用54-81 MW)。PEM电解槽检修期间负荷率降至35-50%,对应14-20 MW,AE日运行缩短至6小时强度60-75%。DAC检修期间运行强度降至55-70%,对应捕碳速率2.78-3.54 kg CO₂/h。FTL合成检修期间运行强度降至50-65%,主要进行催化剂更换,部分时段完全停机。电池储能继续承担日内调节,SOC在50-80%区间波动,同时为检修设备提供启停应急电力保障。

CO₂与H₂储罐在春季发挥"检修缓冲"功能。DAC检修期间捕碳量大幅下降(日捕碳约50-65 kg CO₂ vs. 正常90-120 kg),但FTL合成检修期间消耗同步下降(日耗CO₂ 75-95 kg vs. 正常140-160 kg),供需仍有20-30 kg CO₂/day缺口由储罐补充,储罐SOC从60%略降至50-55%。氢能类似,PEM检修期间制氢量降至280-340 kg H₂/day,FTL检修期间耗氢降至420-480 kg H₂/day,缺口140 kg H₂/day由储罐释放,储罐SOC从55%降至45%。

电力影子价格春季波动模式介于冬夏之间:夜间无光伏期间(20:00-次日6:00)电价升至48-55 
/
M
W
h
,
白天光伏补充期间
(
10
:
00
−
16
:
00
)
电价降至
32
−
38
/MWh,白天光伏补充期间(10:00−16:00)电价降至32−38/MWh,峰谷价差15-20 
/
M
W
h
。氢气影子价格春季温和上升
:
日间
4.2
−
4.6
/MWh。氢气影子价格春季温和上升:日间4.2−4.6/kg H₂,夜间4.5-4.9 $/kg H₂,为即将到来的夏季制氢过剩做准备,指导系统在春末开始积累氢储罐库存。

秋季典型周(10月第2周):产能恢复与库存调整的收尾阶段

秋季作为另一个设备检修窗口期,但规模较春季轻微,主要进行设备微调与性能优化。风电出力秋季恢复至70-85%强度(对应43-52 MW),光伏峰值强度60-75%,对应35-43 MW,日照13-14小时。设备可用率逐步恢复:风机95%,光伏94%,PEM 90%,AE 85%,DAC 88%,FTL 90%,热泵92%。

电力供需秋季渐趋正常,风光合计出力70-95 MW(考虑检修因素实际可用66-88 MW)。PEM电解槽负荷率恢复至50-70%,对应20-29 MW。AE日运行10小时强度70-85%,对应4.5-5.4 MW。DAC运行强度恢复至80-90%,对应捕碳速率4.04-4.55 kg CO₂/h。FTL合成运行强度恢复至80-85%,完成催化剂活性检测确认性能达标。电池储能秋季循环特征渐趋活跃,SOC在55-88%区间波动,日充放电深度30-35%。

CO₂与H₂储罐秋季呈现"产能恢复+需求回升+库存释放"调整格局。DAC秋季捕碳速率4.04-4.55 kg CO₂/h,日捕碳约95-105 kg CO₂,FTL合成日耗CO₂ 135-150 kg,供需仍有40-45 kg CO₂/day缺口由储罐补充,储罐SOC从70%降至55%,为冬季深度释放预留空间。氢能供需秋季接近平衡:PEM+AE日制氢约550 kg H₂,FTL日耗氢约570 kg H₂,小幅缺口20 kg H₂/day由储罐补充,储罐SOC从60%略降至55%。

电力影子价格秋季略高于春季:夜间50-58 
/
M
W
h
,
白天
34
−
40
/MWh,白天34−40/MWh,峰谷价差15-20 
/
M
W
h
,
反映设备检修完成后产能恢复、电力需求上升的趋势。氢气影子价格秋季温和上升
:
日间
4.4
−
4.8
/MWh,反映设备检修完成后产能恢复、电力需求上升的趋势。氢气影子价格秋季温和上升:日间4.4−4.8/kg H₂,夜间4.7-5.1 $/kg H₂,为即将到来的冬季氢资源紧张发出预警信号,指导系统在秋末尽可能多制氢积累库存。

综合技经评价与系统优化方向
成本结构分析与跨聚类比较

聚类0 SAF系统平准化成本1876.06 
/
吨
,
在三个
S
A
F
聚类中最高
(
聚类
1
为
1528.53
/吨,在三个SAF聚类中最高(聚类1为1528.53/吨、聚类2为1621.84 
/
吨
)
,
较绿醇系统成本溢价显著
(
绿醇聚类
0
为
721.01
/吨),较绿醇系统成本溢价显著(绿醇聚类0为721.01/吨,SAF比绿醇高160%)。成本结构中,可再生能源发电成本占比38.2%(极地环境风机防冻设备投资高+光伏冬季极夜利用率低),电解制氢成本占比26.5%(电解槽利用率58.5%偏低+PEM占比86.5%投资成本高),DAC捕碳成本占比18.7%(严寒环境再生能耗高+防冻保温设备投资高),储能系统成本占比10.3%(CO₂超大储罐10,000 kg配置+极地环境储罐保温防冻成本高),FTL合成成本占比6.3%(反应器防冻保温+产品分离精馏塔额外能耗)。

跨聚类成本对比显示清晰的地理-气候-技术梯度:亚洲中纬度聚类1(1528.53 
/
吨
)
凭借风光资源均衡
+
温和气候
(
无极端温度
)
+
电解槽利用率
64.1
/吨)凭借风光资源均衡+温和气候(无极端温度)+电解槽利用率64.1/吨)通过全年高光照+AE电解槽78.1%最高渗透率+FTL利用率92.4%最高,成本劣势控制在6.1%以内;北美高纬度聚类0(1876.06 $/吨)受制于极地严寒环境设备溢价+冬季极夜资源稀缺+CO₂超大储罐配置,成本溢价达22.7%。这一梯度反映了SAF系统对气候极端性的高度敏感,极地环境的防冻保温成本与季节性资源波动是成本控制的核心挑战。

系统优化潜力与技术改进方向

聚类0系统优化潜力主要集中在"CO₂储罐配置优化+冬季制氢增强+热泵低温性能提升"三个维度。CO₂储罐优化方面,当前10,000 kg超大容量(198小时捕碳量)主要应对冬季DAC降负荷+严寒降效的双重挑战,但配置过于保守导致储罐投资成本占比过高。通过优化DAC吸附剂(采用耐低温新型MOF材料,−40°C活性保持率从65%提升至80%),冬季捕碳速率可从3.28-4.04 kg CO₂/h提升至4.0-4.9 kg CO₂/h,季节性捕碳缺口收窄,CO₂储罐容量可缩减至7000-8000 kg,储罐投资成本下降20-25%,系统总成本下降2.0-2.5%。

冬季制氢增强方面,当前冬季极夜期间光伏几乎无贡献、电解槽利用率骤降至40-55%,导致氢供应紧张、储罐库存持续下降。通过配置小型核能热电联供系统(SMR,10-15 MWe+20-30 MWth)作为冬季补充电源,在极夜期间(12-2月,约60天)提供稳定基荷电力,PEM电解槽利用率可从45%提升至60-65%,冬季日制氢量从395 kg H₂增至520-550 kg H₂,氢供需缺口从205 kg H₂/day收窄至50-80 kg H₂/day,氢储罐配置可从15.97 kg缩减至10-12 kg。虽然SMR增加初始投资,但通过提升冬季系统整体利用率+缩减储氢罐投资,综合系统成本可下降8-12%。

热泵低温性能提升方面,当前冬季环境温度−40°C时热泵COP降至2.0-2.5,制热效率显著下降。通过采用CO₂跨临界循环热泵技术(工作温度范围−50至+40°C,低温COP可达2.8-3.2)替代常规蒸汽压缩热泵,冬季制热效率提升25-30%,热泵电耗从当前4.3-4.8 MW降至3.4-3.8 MW,节省电力0.9-1.0 MW可用于增强电解制氢,系统整体能效提升5-8%,成本下降3-5%。综合三项优化措施,聚类0系统成本有望从1876.06 
/
吨降至
1650
−
1720
/吨降至1650−1720/吨,降幅8-12%,接近聚类2水平。

技术演进趋势与极地SAF产业化前景

展望2030-2040年技术演进,极地地区SAF生产系统将迎来关键技术突破期。电解技术方面,固体氧化物电解槽(SOEC)在极地环境展现独特优势:高温运行(700-850°C)不受环境低温影响,电解效率85-90%(vs. 当前PEM 70-75%),高温余热(600-700°C)可直接用于DAC再生无需热泵,综合能效提升至75-80%(vs. 当前65-70%)。DAC技术方面,低温高效MOF吸附剂(−50°C活性保持90%+)与机械蒸汽再压缩(MVR)技术结合,DAC能耗可降至1.2-1.5 MWh/t CO₂(vs. 当前1.8-2.2 MWh/t),严寒环境能耗劣势基本消除。

储能技术方面,液态空气储能(LAES)在极地展现天然优势:环境低温降低液化能耗,储能密度200-250 kWh/m³,往返效率60-70%,非常适合极地长周期季节性储能。通过配置LAES替代部分电池储能,储能系统成本可下降30-40%。FTL合成技术方面,微通道反应器技术(单通道直径<1 mm)实现精确温控+快速传热,在极地环境保温性能优异,催化剂用量降低50%,反应器体积缩减60%,投资成本下降40-50%。

综合技术进步推动下,极地SAF系统成本有望在2035-2040年降至1100-1300 
/
吨
,
接近当前化石航煤成本
(
900
−
1100
/吨,接近当前化石航煤成本(900−1100/吨含碳税后成本)。产业化前景方面,加拿大北部、阿拉斯加、俄罗斯西伯利亚、北欧等极地地区将成为全球SAF生产的重要基地,利用丰富风能资源+廉价土地+远离人口中心的地理优势,发展大规模出口导向型SAF产业。预计2035年后,极地地区SAF产能将占全球总产能的15-20%,通过管道运输至南部机场或海运出口至全球市场,聚类0的技术路径与运行经验将为全球极地SAF项目提供重要参考。


聚类1:亚洲中纬度风光平衡SAF系统(49.5°N,52.25°E)
技术配置与碳氢耦合架构
聚类1位于哈萨克斯坦中部草原地区(49.5°N,52.25°E),技术配置体现了温带大陆性气候下风光资源均衡互补的典型特征。可再生能源装机为风电54.22 MW+光伏98.26 MW(风光比0.55:1),光伏装机占比64.5%显著高于聚类0,但实际发电量中风光各占50%左右,实现了真正的资源互补。电解制氢系统总装机44.07 MW,其中AE占21.45 MW(48.7%)、PEM占22.62 MW(51.3%),AE渗透率是三个SAF聚类中最高的,反映了温和气候对AE电解槽运行的友好性。电解系统年利用率64.1%,为三聚类最高,年产氢36,047 kg H₂。

碳氢耦合环节配置DAC捕碳装置4.91 kg CO₂/h、热泵系统4.91 MW_th(热储能0.64 kWh_th,几乎可忽略)、FTL合成装置11.95 吨SAF/年设计产能(实际5.02吨/年)。储能系统包括电池储能45.36 MWh(是三聚类中最大)、氢储罐8.48 kg H₂(最小)、CO₂储罐10,000 kg(与聚类0相同)。DAC年利用率83.8%、FTL合成年利用率83.7%,均为三聚类最高水平。系统平准化成本1528.53 $/吨SAF,为三聚类最优,较聚类0降低18.5%,反映了温和气候+风光均衡+高利用率的综合优势。

年度运行热力图分析
风电与光伏发电特征

年度热力图显示,风电与光伏出力呈现高度互补的季节性模式,形成"风强光弱-风弱光强"的完美对冲格局。风电出力冬春季最强(11月-次年4月),标准化强度常年维持0.70-0.90区间,对应实际功率38-49 MW,部分冷锋过境大风期间可短暂达0.95-1.00满发。夏秋过渡季(5-10月)风电强度回落至0.50-0.70,对应27-38 MW,但仍保持稳定中等出力。风电日内波动幅度一般±10-15%,呈现温带大陆性气候典型的"夜间风速略高+白天风速略低"特征,与光伏形成完美昼夜互补。

光伏发电呈现与风电反向的季节性特征。夏季(6-8月)光伏资源最充裕,日照时长达14-16小时,标准化强度峰值0.85-0.95,对应实际功率83-93 MW,接近装机容量满发。春秋过渡季(3-5月、9-11月)光伏峰值强度0.70-0.85,对应69-83 MW,日照12-14小时。冬季(12-2月)光伏峰值强度降至0.45-0.60,对应44-59 MW,日照仅8-10小时,但相较聚类0极地地区(冬季仅0.15-0.30)仍保持相当贡献。全年来看,光伏有效发电覆盖全年,冬季虽弱但仍能提供30-40%装机容量出力,与风电形成稳定互补。

风光合计出力全年维持70-140 MW区间,季节性波动幅度仅70 MW(vs. 聚类0的75-110 MW波动45 MW但冬季资源极度稀缺),且任何季节风光合计出力均不低于70 MW基础水平,为系统提供了极为稳定的电力供应基础。这种风光高度互补的资源禀赋是聚类1实现最优经济性的核心优势。

电解槽运行策略与氢能生产

电解槽系统运行呈现"AE-PEM近似均衡分工+全年高负荷稳定运行"的独特模式。AE电解槽(21.45 MW,占48.7%)与PEM电解槽(22.62 MW,占51.3%)装机容量几乎相当,运行策略上形成"双基荷"模式:两者均全年24小时连续运行,AE运行强度常年维持0.55-0.75区间,对应11.8-16.1 MW实际功率;PEM运行强度常年维持0.50-0.70区间,对应11.3-15.8 MW实际功率。与聚类0"PEM基荷+AE调峰"截然不同,聚类1的AE与PEM共同承担基础负荷,充分发挥AE成本优势与PEM性能优势。

季节性来看,冬春季风电强+光伏弱期间,AE运行强度略升至0.65-0.75(利用风电稳定出力),PEM运行强度维持0.55-0.65;夏秋季风电弱+光伏强期间,AE运行强度略降至0.55-0.65,PEM运行强度略升至0.60-0.70(利用光伏峰值快速响应)。这种"AE偏重风电基荷+PEM偏重光伏调峰"的协同策略,使得电解系统整体利用率达到64.1%,较聚类0的58.5%提升5.6个百分点,是三个SAF聚类中最高的。

年度热力图显示,电解槽运行强度全年维持在0.50-0.75区间,季节性波动幅度仅0.25(vs. 聚类0的0.40-0.75波动0.35),日内波动幅度<0.10,展现出极高的运行稳定性。这种稳定性直接转化为制氢成本优势:高利用率摊薄固定成本+平稳运行降低启停损耗,使得聚类1制氢成本较聚类0降低约15-18%。

电池储能循环特征与SOC动态

电池储能系统(45.36 MWh,是三聚类最大)在风光平衡系统中承担"日内削峰填谷+风光互补桥梁+季节微调"三重功能。夏季(6-8月)光伏主导期间,电池呈现典型日循环特征:白天光伏峰值期(10:00-16:00)电池充电至80-92% SOC,夜间风电基荷期(20:00-次日6:00)电池放电至45-60% SOC支撑电解与DAC,日充放电深度约30-40%,循环强度中等。冬季(12-2月)风电主导期间,电池循环模式反转:夜间风电强劲期(18:00-次日8:00)电池充电至75-88% SOC,白天光伏补充期(10:00-16:00)电池放电至50-65% SOC,日充放电深度约25-35%,循环强度略低于夏季。

春秋过渡季(3-5月、9-11月)风光资源最为均衡,电池循环呈现"多峰多谷"复杂模式:早晨光伏爬升期电池第一次充电(7:00-10:00),中午光伏峰值期电池深度充电(11:00-14:00至85-95% SOC),下午光伏下降+风电上升过渡期电池部分放电(15:00-18:00),夜间风电高峰期电池第二次充电(19:00-23:00),凌晨风电低谷期电池深度放电(0:00-6:00至40-55% SOC)。SOC在40-95%区间频繁波动,日充放电深度可达50-55%,年循环次数约320-360次,是三聚类中最活跃的。

年度热力图显示,电池SOC季节性变化相对平缓:夏季日均SOC 65-70%,冬季日均SOC 60-68%,春秋日均SOC 60-75%,季节摆幅仅10-15%,远小于聚类0的25-30%摆幅。这种平缓的季节性变化反映了风光资源全年均衡互补,电池主要承担日内而非季节调节功能。尽管循环次数较高(320-360次/年 vs. 聚类0的280-320次/年),但由于单次循环深度适中(日均35-45% vs. 聚类0春秋季50-60%),电池寿命仍可维持8-10年标准水平。

DAC系统运行与CO₂捕获动态

DAC装置(4.91 kg CO₂/h)年度运行呈现"全年高负荷稳定+温和季节波动"的理想特征。年度热力图显示,DAC运行强度全年维持0.75-0.92区间,是三个聚类中波动幅度最小的(季节极差仅0.17 vs. 聚类0的0.30)。夏季(6-8月)光伏充裕+温度适宜(20-30°C),DAC运行强度达全年峰值0.88-0.92,对应捕碳速率4.32-4.52 kg CO₂/h,日运行时长22-23小时。春秋过渡季(3-5月、9-11月)风光均衡+温度温和(10-20°C),DAC运行强度维持0.82-0.88,对应捕碳速率4.03-4.32 kg CO₂/h,日运行时长20-22小时。冬季(12-2月)风电主导+温度偏低(-10至0°C),DAC运行强度略降至0.75-0.82,对应捕碳速率3.68-4.03 kg CO₂/h,日运行时长18-20小时,但相较聚类0极地严寒(-30至-40°C导致0.65-0.72)性能保持显著更优。

温带大陆性气候对DAC运行的友好性体现在两方面:①冬季温度虽低但不极端(-10至0°C vs. 聚类0的-30至-40°C),吸附剂活性保持率80-85%(vs. 聚类0的65-75%),再生能耗增加幅度仅10-15%(vs. 聚类0的15-20%);②夏季温度适宜(20-30°C)且相对湿度较低(40-55%),是DAC运行的最佳温度区间,捕获效率达到峰值水平。年度热力图显示,DAC运行强度与环境温度呈现温和正相关,但相关性远弱于聚类0(聚类0温差35-45°C对应强度差0.20-0.30,聚类1温差30-40°C仅对应强度差0.13-0.17)。

DAC日内运行极为平稳,标准化强度波动幅度一般<0.08,显著小于聚类0的<0.10。仅在设备检修期(4月中旬与10月中旬各3-5天)降至0.25-0.40低负荷维护状态。全年来看,DAC年利用率83.8%,较聚类0的81.4%提升2.4个百分点,是三个SAF聚类中最高的,充分体现了温和气候+风光均衡双重优势对DAC稳定运行的支撑作用。

CO₂与H₂储罐协调运行

CO₂储罐(10,000 kg容量)与H₂储罐(8.48 kg容量)构成聚类1储能体系,但其运行特征与聚类0显著不同。CO₂储罐SOC呈现"温和季节波动+日内微调为主"特征:6-9月夏秋季DAC高负荷捕碳期间,储罐SOC从55%逐步攀升至75-82%峰值(对应7500-8200 kg CO₂库存),积累约2000-2500 kg CO₂季节性储备,远小于聚类0的5000-5500 kg;10月-次年3月冬春季DAC略降负荷期间,储罐SOC从75%逐步下降至50-58%(对应5000-5800 kg CO₂),释放约1700-2200 kg CO₂补充FTL合成消耗;4-5月春季为过渡期,储罐SOC在55-65%区间小幅波动。年度SOC摆幅仅20-27%,不到聚类0的一半(50-55%),反映了风光均衡带来的DAC全年稳定运行,储罐主要承担日内而非季节调节功能。

H₂储罐(8.48 kg容量,仅为聚类0的53%)配置显著偏小,但运行特征更加活跃。日内SOC典型波动模式:上午7-12时风光爬升+双电解槽满负荷期间,制氢速率大于FTL消耗,储罐SOC从40-50%快速充至70-82%;下午12-18时风光维持高位期间,制氢与消耗接近平衡,储罐SOC在65-80%区间小幅波动;夜间18-次日7时风电基荷+光伏归零期间,制氢速率小于FTL消耗,储罐SOC从70-75%降至40-50%,完成一个完整日循环。日充放电深度约30-40%,高于聚类0的20-30%,反映了储罐容量偏小但制氢稳定性更高,可以承受更频繁的充放循环。

季节性来看,夏季制氢相对充裕储罐SOC略高(日均55-65%),冬季制氢相对紧张储罐SOC略低(日均45-55%),但季节SOC摆幅仅10-15%,与聚类0相同。年度热力图显示,氢储罐全年维持"日循环主导+季节微调辅助"运行模式,与聚类0类似但日循环更加规律。储罐配置虽小但得益于电解槽64.1%高利用率+风光均衡带来的制氢稳定性,氢供应充裕度实际优于聚类0(聚类0夏季供需过剩35 kg/day、冬季供需缺口205 kg/day,聚类1全年供需基本平衡±20 kg/day以内波动)。

热泵系统与热储能运行

热泵系统(4.91 MW_th)年度运行强度呈现"全年稳定+温和季节波动"特征,与聚类0"冬季最高+夏季次之"截然不同。温带气候环境温度全年维持-10至+30°C区间,热泵COP全年保持3.0-3.8高水平(vs. 聚类0冬季仅2.0-2.5),制热效率显著更优。夏季(6-8月)环境温度20-30°C,热泵COP达全年峰值3.5-3.8,DAC吸附剂再生温度需求标准(150-180°C),热泵运行强度0.80-0.88,对应输出热功率3.93-4.32 MW_th,日运行时长22-23小时。春秋过渡季(3-5月、9-11月)环境温度10-20°C,热泵COP维持3.2-3.6,热泵运行强度0.75-0.85,对应输出3.68-4.17 MW_th,日运行时长20-22小时。冬季(12-2月)环境温度-10至0°C,热泵COP降至3.0-3.3(仍显著高于聚类0的2.0-2.5),DAC再生温度需求略升至160-190°C,热泵运行强度0.78-0.86,对应输出3.83-4.22 MW_th,日运行时长19-21小时。

年度热力图显示,热泵运行强度全年维持0.75-0.88区间,季节波动幅度仅0.13(vs. 聚类0的0.70-0.95波动0.25),日内波动幅度<0.08,展现出极高的稳定性。这种稳定性直接转化为能效优势:热泵全年高COP(3.0-3.8)意味着单位热输出耗电量仅为聚类0的70-85%,制热成本显著降低。更重要的是,温和的运行环境大幅延长热泵设备寿命(15-18年 vs. 聚类0极寒环境10-12年),进一步摊薄固定成本。

热储能系统(0.64 kWh_th,仅为聚类0的2%)配置极小,几乎可忽略不计。这一配置差异反映了两个系统的根本不同:聚类0极地环境冬季电力极度稀缺、热储能需要跨季节调节,因此配置0.31 MWh_th较大容量;聚类1风光均衡电力全年充裕、热泵可全天候稳定运行,热储能仅需承担小时级微调即可。年度热力图显示,热储能SOC全年维持40-80%区间小幅波动,日充放电深度<20%,年循环次数<100次,主要在电力短时波动(风光同时低谷)期间发挥缓冲作用,其余时段热泵直供DAC再生无需储能中转。这种"准直供"模式使得热力系统整体效率达到88-92%(vs. 聚类0储能中转损耗后82-86%),进一步强化了温和气候的能效优势。

FTL合成装置稳定运行特征

FTL合成装置(11.95吨/年设计产能)年度运行展现出三个SAF聚类中最高的稳定性,标准化运行强度全年维持0.78-0.88区间,季节波动幅度仅0.10(vs. 聚类0的0.75-0.87波动0.12)。年度热力图显示,FTL合成运行强度呈现极温和的季节变化:夏季(6-8月)风光充裕+DAC高效+氢气充足+温度适宜(20-30°C催化剂活性最优),合成装置运行强度略升至0.85-0.88,对应SAF产出0.580-0.600 kg/h;冬季(12-2月)风电主导+DAC略降负荷+氢气基本平衡+温度偏低(-10至0°C催化剂活性略降5-8%),合成装置运行强度略降至0.78-0.82,对应SAF产出0.532-0.559 kg/h;春秋过渡季居中0.81-0.86。

这种极高稳定性得益于四重保障机制协同作用:①风光全年均衡互补提供稳定电力基础,电解槽64.1%高利用率保障氢气全年充裕供应;②DAC全年83.8%高利用率保障CO₂连续充足供应,CO₂储罐10,000 kg超大容量平滑季节微小波动(仅20-27%摆幅);③温带气候对FTL催化剂友好(-10至+30°C全年保持85-95%活性,vs. 聚类0极寒冬季活性降至75-80%);④热泵系统全年高COP(3.0-3.8)稳定提供反应器加热,保障反应温度恒定在220-260°C最佳区间。

年度热力图显示,FTL合成装置仅在春秋设备检修期(4月中旬与10月中旬各持续3-5天)运行强度短暂降至0.30-0.40,其余时段均保持高负荷稳定运行,日内波动幅度<0.05,是所有主要设备中最稳定的。合成单元年利用率83.7%,在三个SAF聚类中最高(聚类0为81.3%,聚类2为92.4%但后者为纯光伏极端配置),较聚类0提升2.4个百分点。这种高稳定性+高利用率直接转化为成本优势:单位SAF产出的固定成本摊销较聚类0降低约12-15%,是聚类1实现最优经济性的关键因素之一。

四季典型周电力供需平衡与影子价格动态分析
夏季典型周(7月第2周):光伏主导与风电基荷的黄金搭档

夏季北半球日照最充沛期间,聚类1系统呈现"光伏主导+风电稳定基荷"的最优供电模式。光伏出力白天呈现完美钟形曲线:早晨6:00日出后快速爬升,上午10:00达到80%强度(对应79 MW),中午12-14时达到峰值90-95%强度(对应88-93 MW),下午16:00降至75%强度,傍晚20:00日落前降至10%,夜间完全归零。风电出力全天维持中低稳定水平,标准化强度50-65%区间(对应27-35 MW),日内波动幅度<10%,夜间(20:00-次日6:00)略强至60-65%,白天(8:00-18:00)略弱至50-60%,与光伏形成完美昼夜互补。风光合计出力白天峰值115-128 MW,夜间基础27-35 MW,全天平均85-95 MW。

电力需求侧呈现"双电解槽高负荷+DAC高效+FTL稳定"格局。AE电解槽全天运行强度65-75%,对应13.9-16.1 MW,白天光伏充裕期略升至70-75%,夜间风电基荷期略降至65-70%。PEM电解槽全天运行强度60-70%,对应13.6-15.8 MW,运行模式与AE类似但略偏重光伏峰值响应。双电解槽合计耗电27-32 MW,占总供电的30-35%。DAC系统高负荷运行强度88-92%,对应捕碳速率4.32-4.52 kg CO₂/h耗电约7-8 MW,日运行23小时。FTL合成装置稳定运行强度85-88%耗电约5-6 MW(含电加热与辅机),热泵系统运行强度82-88%耗电约4-5 MW(COP 3.6-3.8换算)。系统总需求50-58 MW,供需缺口60-70 MW用于电池充电+弃电(夏季光伏过剩期间约5-8%弃光率)。

电池储能在光伏峰值时段深度充电:上午10-14时以25-35 MW功率快充,SOC从55-60%充至85-92%;下午14-18时以15-25 MW功率继续充电至90-95%;夜间20-次日6时以10-15 MW功率持续放电支撑风电不足,SOC降至45-55%。日充放电深度约40-45%,充放电比例约1.8:1(充电多于放电,夏季储能季节性积累)。

电力影子价格夏季呈现明显昼夜波动:夜间低谷期(0:00-6:00)仅风电单一电源,电价升至38-45 
/
M
W
h
;
清晨过渡期
(
6
:
00
−
9
:
00
)
光伏爬升
+
风电维持
,
电价降至
32
−
36
/MWh;清晨过渡期(6:00−9:00)光伏爬升+风电维持,电价降至32−36/MWh;白天峰值期(10:00-16:00)风光合计最强,电价降至全天最低25-30 
/
M
W
h
;
傍晚过渡期
(
17
:
00
−
20
:
00
)
光伏下降
+
风电略升
,
电价回升至
32
−
38
/MWh;傍晚过渡期(17:00−20:00)光伏下降+风电略升,电价回升至32−38/MWh。峰谷价差15-20 
/
M
W
h
,
略低于聚类
0
(
15
−
20
/MWh,略低于聚类0(15−20/MWh),反映了风光互补对电力供应的平滑作用。

氢能供需平衡方面,夏季制氢充裕。AE日制氢约310 kg H₂,PEM日制氢约295 kg H₂,合计605 kg H₂/day。FTL合成日耗氢约590 kg H₂,供需小幅过剩15 kg H₂/day,氢储罐逐步充至60-65% SOC。氢气影子价格夏季保持稳定低位:白天制氢高峰期降至3.2-3.5 
/
k
g
H
2
,
夜间制氢略降期升至
3.6
−
3.9
/kgH 
2
​
 ,夜间制氢略降期升至3.6−3.9/kg H₂,日内波动幅度0.4-0.7 $/kg,反映了双电解槽全天候稳定制氢带来的氢供应充裕性。

冬季典型周(1月第3周):风电主导与光伏补充的稳健模式

冬季北半球风能资源最充沛期间,聚类1系统切换为"风电主导+光伏显著补充"模式,与夏季形成镜像互补。风电出力全天维持高位,标准化强度75-90%区间(对应41-49 MW),夜间(18:00-次日8:00)最强可达85-90%,白天(9:00-17:00)略降至75-85%,日内波动幅度<15%。光伏出力冬季显著减弱但仍有相当贡献:日照时长缩短至8-10小时(8:00-17:00),峰值强度仅50-60%,对应49-59 MW(中午12-14时),早晚(8-9时、16-17时)降至20-30%强度,夜间完全归零。风光合计出力白天峰值90-108 MW,夜间基础41-49 MW(纯风电),全天平均70-85 MW,较夏季下降15-20%但仍保持充裕。

电力需求侧适度压缩以适应资源略降。AE电解槽运行强度60-70%,对应12.9-15.0 MW,冬季风电稳定使其保持较高负荷。PEM电解槽运行强度55-65%,对应12.4-14.7 MW,略低于夏季以节省电力。双电解槽合计耗电25-30 MW,占总供电的35-40%。DAC系统受低温略影响,运行强度75-82%,对应捕碳速率3.68-4.03 kg CO₂/h耗电约6-7 MW,日运行19-20小时。FTL合成装置依靠CO₂储罐释放库存维持运行强度78-82%,耗电约4.5-5.5 MW。热泵系统在冬季低温环境COP降至3.0-3.3,但仍需运行强度80-86%耗电约4.5-5.0 MW。系统总需求48-55 MW,供需基本平衡,电池充放电切换为"夜充日放"模式。

电池储能冬季循环模式反转:夜间风电强劲期(19:00-次日7:00)以12-18 MW功率充电,SOC从50-55%充至75-85%;白天光伏补充期(10:00-16:00)以8-12 MW功率放电支撑电解与DAC,SOC降至55-65%。日充放电深度约25-30%,低于夏季(40-45%),反映了冬季风电持续高出力对储能需求降低。

CO₂储罐冬季发挥季节调节作用:DAC捕碳速率3.68-4.03 kg CO₂/h日捕碳约70-80 kg,而FTL合成消耗速率维持4.2-4.5 kg CO₂/h日耗CO₂ 80-90 kg,日缺口10 kg由储罐释放补充,储罐SOC从秋季75%逐步降至冬末55%,释放约2000 kg CO₂库存,远小于聚类0的4500 kg释放量,反映了温和气候对DAC运行的友好性。

电力影子价格冬季整体略升但波动缓和:夜间风电强劲期(20:00-次日7:00)电价降至全天最低32-38 
/
M
W
h
;
白天光伏补充期
(
10
:
00
−
15
:
00
)
电价升至
42
−
48
/MWh;白天光伏补充期(10:00−15:00)电价升至42−48/MWh;早晚过渡期(7-9时、16-19时)电价居中38-42 
/
M
W
h
。峰谷价差
10
−
16
/MWh。峰谷价差10−16/MWh,较夏季收窄,反映了冬季风电持续充裕对电力供应的稳定作用。

氢能供需冬季呈现平衡偏紧格局。AE日制氢约285 kg H₂,PEM日制氢约265 kg H₂,合计550 kg H₂/day,较夏季下降9%。FTL合成日耗氢约570 kg H₂,供需小幅缺口20 kg H₂/day由储罐释放弥补。氢气影子价格冬季温和上升:白天升至3.8-4.2 
/
k
g
H
2
,
夜间峰值
4.3
−
4.6
/kgH 
2
​
 ,夜间峰值4.3−4.6/kg H₂,较夏季上涨0.5-0.8 
/
k
g
,
但远低于聚类
0
冬季的
4.8
−
5.8
/kg,但远低于聚类0冬季的4.8−5.8/kg,反映了温和气候+风光均衡带来的氢供应相对充裕性。

春季典型周(4月第2周):设备检修与资源过渡的平稳切换

春季作为年度设备大修窗口期,聚类1系统运行呈现"检修降负荷+风光过渡"特征。风电出力从冬季高位回落,强度65-80%区间(对应35-43 MW)。光伏摆脱冬季弱势,恢复正常水平,日照12-13小时,峰值强度65-75%,对应64-74 MW。风光合计出力白天峰值99-117 MW,夜间基础35-43 MW。但主要设备按计划轮流检修:风机可用率92%,光伏可用率94%,AE电解槽可用率75%(电解液更换与阳极检修),PEM电解槽可用率80%(电堆测试与膜检测),DAC可用率68%(吸附剂再生测试),FTL合成可用率65%(催化剂活性检测),热泵可用率82%(制冷剂检查)。

电力供需在春季检修期呈现"产能下降+需求下降+储能调节"平衡。风光合计出力91-110 MW(考虑检修因素实际可用84-103 MW)。AE电解槽检修期间负荷率降至45-60%,对应9.7-12.9 MW。PEM电解槽检修期间负荷率降至50-65%,对应11.3-14.7 MW。DAC检修期间运行强度降至55-72%,对应捕碳速率2.70-3.54 kg CO₂/h耗电约5-6 MW。FTL合成检修期间运行强度降至52-68%耗电约3.5-4.5 MW。系统总需求35-45 MW,供需仍有充裕盈余,电池储能继续承担日内调节。

CO₂与H₂储罐在春季发挥"检修缓冲"功能。DAC检修期间日捕碳约50-65 kg CO₂,FTL检修期间日耗CO₂ 75-95 kg,缺口20-30 kg由储罐补充(vs. 聚类0相同20-30 kg),储罐SOC从65%略降至58%。氢能类似,双电解槽检修期间日制氢约380-440 kg H₂,FTL检修期间日耗氢约430-500 kg H₂,缺口50-60 kg由储罐释放,储罐SOC从55%降至48%。

电力影子价格春季波动模式介于冬夏之间:夜间风电基荷期35-42 
/
M
W
h
,
白天光伏补充期
28
−
35
/MWh,白天光伏补充期28−35/MWh,峰谷价差7-14 
/
M
W
h
,
是全年最小的
,
反映了春季风光过渡期资源最为均衡
+
检修期需求下降的双重作用。氢气影子价格春季保持中位
:
日间
3.5
−
3.9
/MWh,是全年最小的,反映了春季风光过渡期资源最为均衡+检修期需求下降的双重作用。氢气影子价格春季保持中位:日间3.5−3.9/kg H₂,夜间3.8-4.2 $/kg H₂,为即将到来的夏季制氢过剩做准备。

秋季典型周(10月第2周):产能恢复与库存调整的黄金时段

秋季作为另一个检修窗口期但规模更轻,主要进行设备性能验证。风电出力秋季恢复上升,强度70-85%区间(对应38-46 MW),逐步向冬季高位过渡。光伏仍保持较高水平,峰值强度70-80%,对应69-79 MW,日照11-12小时,尚未进入冬季弱势期。风光合计出力白天峰值107-125 MW,夜间基础38-46 MW。设备可用率逐步恢复:风机95%,光伏96%,AE 88%,PEM 90%,DAC 85%,FTL 88%,热泵90%。

电力供需秋季渐趋正常。AE电解槽负荷率恢复至58-72%,对应12.4-15.4 MW。PEM电解槽负荷率恢复至56-68%,对应12.7-15.4 MW。DAC运行强度恢复至78-88%,对应捕碳速率3.83-4.32 kg CO₂/h耗电约6.5-7.5 MW。FTL合成运行强度恢复至80-86%耗电约5.0-5.8 MW。系统总需求46-55 MW,供需盈余52-70 MW用于电池充电与季节性库存积累。

CO₂与H₂储罐秋季呈现"产能恢复+库存释放为主"调整格局。DAC秋季捕碳日均90-100 kg CO₂,FTL合成日耗CO₂ 135-150 kg,供需缺口40-50 kg由储罐补充(vs. 春季缺口20-30 kg,秋季缺口更大是为了冬季深度释放预留空间),储罐SOC从78%降至62%。氢能供需秋季接近平衡:双电解槽日制氢约560 kg H₂,FTL日耗氢约575 kg H₂,小幅缺口15 kg由储罐补充,储罐SOC从58%略降至52%。

电力影子价格秋季略高于春季:夜间38-45 
/
M
W
h
,
白天
30
−
36
/MWh,白天30−36/MWh,峰谷价差8-15 
/
M
W
h
,
反映设备检修完成后产能恢复、电力需求上升的趋势。氢气影子价格秋季温和上升
:
日间
3.7
−
4.1
/MWh,反映设备检修完成后产能恢复、电力需求上升的趋势。氢气影子价格秋季温和上升:日间3.7−4.1/kg H₂,夜间4.0-4.4 $/kg H₂,为即将到来的冬季氢资源偏紧发出预警信号,指导系统在秋末尽可能多制氢积累库存。

综合技经评价与系统优化方向
成本结构分析与跨聚类比较

聚类1 SAF系统平准化成本1528.53 
/
吨
,
在三个
S
A
F
聚类中最优
(
聚类
0
为
1876.06
/吨,在三个SAF聚类中最优(聚类0为1876.06/吨、聚类2为1621.84 $/吨),较聚类0降低18.5%,成本优势显著。成本结构中,可再生能源发电成本占比35.8%(温和气候无极端设备投资+风光均衡高利用率),电解制氢成本占比24.2%(电解槽利用率64.1%最高+AE占比48.7%降低投资成本),DAC捕碳成本占比16.5%(温和气候能耗低+高利用率83.8%),储能系统成本占比15.3%(电池储能45.36 MWh最大但高循环效率+CO₂储罐季节调节需求小),FTL合成成本占比8.2%(高利用率83.7%+温和环境催化剂活性最优)。

跨聚类成本对比展现清晰的"温和气候+资源均衡"最优范式:聚类1(1528.53 $/吨)凭借四重优势实现最优经济性:①温带大陆性气候(-10至+30°C)对所有设备友好,无极地防冻(聚类0)或热带散热(聚类2)额外投资;②风光全年均衡互补(风电冬强夏弱+光伏夏强冬弱),电解槽利用率64.1%最高摊薄固定成本;③DAC全年高效运行(83.8%利用率+COP 3.0-3.8),温和气候能耗最优;④储能系统尽管电池容量最大(45.36 MWh),但得益于风光互补日内调节为主、季节调节需求小(CO₂储罐SOC摆幅仅20-27% vs. 聚类0的50-55%),综合储能成本合理。

聚类0(1876.06 
/
吨
)
成本劣势主要来自极地严寒环境
:
①防冻保温设备投资高
15
−
25
/吨)成本劣势主要来自极地严寒环境:①防冻保温设备投资高15−25/吨)成本略高于聚类1(+6.1%),主要因为:①纯光伏系统极端日变化需要超大电池储能(127.61 MWh,是聚类1的2.8倍);②AE电解槽78.1%最高渗透率虽降低投资成本,但电解槽整体利用率37.0%显著低于聚类1的64.1%,固定成本摊销不利;③赤道气候高温高湿对DAC不利,能耗略高于温带。

系统优化潜力与技术改进方向

聚类1系统已经接近最优配置,优化潜力主要集中在"电池储能循环优化+AE渗透率微调+热储能配置增强"三个边际改进维度。电池储能优化方面,当前45.36 MWh配置是三聚类最大,年循环次数320-360次处于高位,春秋季日充放电深度可达50-55%,导致电池寿命略短于理想值(8-9年 vs. 标准10年)。通过引入需求侧响应机制(电解槽±10%柔性负荷、DAC ±15%柔性负荷、FTL合成±5%柔性负荷),可将电池日充放电深度控制在40-48%以内,年循环次数降至280-320次,延长电池寿命至9-10年,储能成本下降8-12%,系统总成本下降1.2-1.8%。

AE渗透率微调方面,当前AE占比48.7%、PEM占比51.3%接近均衡,是三聚类中AE渗透率最高的。考虑到2030-2040年AE技术持续进步(响应速度提升至30-50%负荷调节能力 vs. 当前20-100%,启动时间缩短至5-10分钟 vs. 当前15-30分钟),可将AE渗透率进一步提升至55-60%,在保持系统灵活性前提下降低电解投资成本5-8%,系统总成本下降1.2-1.5%。但需要注意AE渗透率不宜超过60%,否则在风光短时同时低谷期间系统响应速度不足,需要配置更大电池储能补偿,得不偿失。

热储能配置增强方面,当前0.64 kWh_th极小容量(仅为聚类0的2%)主要因为风光均衡电力充裕、热泵可全天候运行无需储能中转。但在电力短时波动(风光同时低谷)期间,热泵被迫降负荷导致DAC再生热供应不足,DAC运行强度短暂降至60-70%低位。通过将热储能容量从0.64 kWh_th扩容至15-20 MWh_th(约25-30倍,投资增加约80-100万美元),可在风光低谷期(每月2-3次、每次持续3-5小时)释放储热维持DAC稳定运行,DAC年利用率从83.8%提升至85.5-86.2%,DAC捕碳成本下降2-3%,系统总成本下降0.3-0.5%。综合三项优化措施,聚类1系统成本有望从1528.53 
/
吨降至
1480
−
1500
/吨降至1480−1500/吨,降幅约2-3%,进一步巩固最优经济性地位。

技术演进趋势与全球SAF产业布局

展望2030-2040年技术演进,温带风光均衡地区SAF生产系统将成为全球主流模式。电解技术方面,新一代AE电解槽突破响应速度瓶颈(30-100%负荷调节能力+5-10分钟冷启动),在保持成本优势(投资成本较PEM低30-40%)同时获得接近PEM的灵活性,使得AE渗透率可提升至60-70%(vs. 当前最高48.7%)而不损失系统灵活性。DAC技术方面,新型固体吸附剂(MOF/COF材料)在温带气候展现最佳性能:吸附容量提升50%(2.0-2.5 mol CO₂/kg vs. 当前1.2-1.5 mol),再生温度降低至100-120°C(vs. 当前150-180°C),能耗降至0.9-1.2 MWh/t CO₂(vs. 当前1.5-1.8 MWh/t),温带气候能效优势进一步扩大。

储能技术方面,液流电池(铁铬/锌溴体系)规模化应用于日内循环储能,循环寿命20,000-25,000次(vs. 锂电池5,000-8,000次),特别适合聚类1类型的高频日循环场景(年循环320-360次,电池全生命周期25-30年)。通过配置液流电池替代50-60%锂电池储能(保留快响应锂电+大容量液流的混合配置),储能系统成本可下降25-35%。FTL合成技术方面,等离子体辅助催化技术(PAC)在温带环境展现独特优势:反应温度降至150-180°C(vs. 当前220-260°C),CO转化率提升至85-90%(vs. 当前70-75%),催化剂寿命延长至5-8年(vs. 当前2-3年),大幅降低合成成本。

综合技术进步推动下,温带风光均衡SAF系统成本有望在2035-2040年降至900-1050 
/
吨
,
实现与化石航煤平价
(
含碳税后成本
900
−
1100
/吨,实现与化石航煤平价(含碳税后成本900−1100/吨)。产业布局前景方面,全球温带风光资源均衡地区将成为SAF产业核心区:中国西北(新疆、甘肃、宁夏)、中亚五国(哈萨克斯坦、乌兹别克斯坦)、美国中部(德克萨斯、俄克拉荷马)、南美潘帕斯草原(阿根廷、乌拉圭)、澳大利亚内陆等地区,普遍具备"风光均衡+土地充裕+气候温和+远离人口中心"特征,适合发展大规模SAF产业。预计2035年后,温带风光均衡地区SAF产能将占全球总产能的50-60%,成为绝对主力,聚类1的技术路径与运行经验将为全球同类地区提供最佳实践范本,引领全球SAF产业从示范走向大规模商业化部署。

5.3 聚类2:亚洲赤道纯光伏SAF系统(-1.25°N,104.25°E)
技术配置与碳氢耦合架构
聚类2位于印度尼西亚苏门答腊岛中部赤道地区(-1.25°N,104.25°E),技术配置体现了热带雨林气候下纯光伏资源的极端特征。可再生能源装机为风电0.0000002 MW(几乎为零)+光伏209.55 MW,是典型的纯光伏系统,光伏装机占比>99.9999%。赤道地区全年太阳高度角高(正午80-90°)且昼夜等长(日照约12小时/天),光伏发电呈现"高强度+高稳定性"特征,但完全缺乏风电补充导致极端的日变化特性。电解制氢系统总装机76.23 MW,其中AE占59.42 MW(78.0%)、PEM占16.80 MW(22.0%),AE渗透率是三个SAF聚类中最高的,反映了纯光伏系统对低成本大容量电解的需求。电解系统年利用率37.0%,为三聚类最低,年产氢36,000 kg H₂。

碳氢耦合环节配置DAC捕碳装置4.45 kg CO₂/h、热泵系统4.45 MW_th(热储能0.046 MWh_th)、FTL合成装置10.82 吨SAF/年设计产能(实际4.91吨/年)。储能系统包括电池储能127.61 MWh(是三聚类中最大,为聚类1的2.8倍)、氢储罐7.34 kg H₂、CO₂储罐10,000 kg(与聚类0/1相同)。DAC年利用率92.4%、FTL合成年利用率92.4%,均为三聚类最高水平,显著优于聚类0/1。系统平准化成本1621.84 
/
吨
S
A
F
,
在三聚类中居中
(
聚类
1
最优
1528.53
/吨SAF,在三聚类中居中(聚类1最优1528.53/吨、聚类0最高1876.06 $/吨),较聚类1高出6.1%,反映了纯光伏极端日变化需要超大储能配置的成本压力,但通过极高利用率部分抵消了劣势。

年度运行热力图分析
光伏发电特征与极端日变化

年度热力图显示,光伏出力呈现赤道地区典型的"全年高度一致+完美日钟形曲线"特征。赤道地区太阳辐射全年稳定(年际变化<5%),光伏发电季节性极弱:全年任何月份,光伏出力曲线几乎完全重合,标准化强度峰值常年维持0.90-0.98区间(对应188-205 MW),日照时长稳定在11.5-12.5小时。典型日光伏出力模式:清晨6:00日出后快速爬升,上午9:00达到70%强度(对应147 MW),中午11:30-12:30达到峰值94-98%强度(对应197-205 MW),下午15:00降至65%强度,傍晚18:00日落前降至5%,夜间18:30-次日5:30完全归零(持续11小时黑暗期)。

年度热力图最显著特征是光伏出力的"完美重复性":将全年8760小时热力图按日(24小时)折叠,365天的曲线几乎完全叠加,标准差<3%。这种极高的预测性是赤道纯光伏系统的核心优势,但同时带来极端日变化挑战:白天峰值205 MW vs. 夜间0 MW,日内波动幅度达到装机容量的98%,是三个SAF聚类中最大的(聚类0风电日内波动<15%、聚类1风光互补日内合计波动<30%)。这种极端日变化决定了系统必须配置超大电池储能(127.61 MWh)来平抑供需,储能投资占比显著高于其他聚类。

电解槽运行策略与氢能生产

电解槽系统运行呈现纯光伏系统典型的"白天满负荷+夜间最低维持"极端分工模式。AE电解槽(59.42 MW,占78.0%)主要在光伏充裕时段(8:00-17:00)运行,白天运行强度70-95%区间,对应41.6-56.4 MW实际功率,峰值时段(11:00-14:00)可达90-95%满负荷;夜间(18:00-次日7:00)完全停机以节省电力。AE年运行时长约3200-3400小时(占全年37-39%),集中在白天9-10小时,充分利用光伏峰值电力大规模制氢。PEM电解槽(16.80 MW,占22.0%)承担24小时基础负荷制氢任务,夜间(18:00-次日7:00)运行强度15-25%,对应2.5-4.2 MW最低维持负荷,确保FTL合成装置连续供氢不中断;白天(8:00-17:00)运行强度提升至40-60%,对应6.7-10.1 MW,与AE协同制氢。

这种"AE白天冲刺+PEM夜间守夜"的极端分工策略,使得电解系统整体利用率仅37.0%,远低于聚类1的64.1%和聚类0的58.5%,是三聚类中最低的。低利用率直接导致单位制氢成本上升(固定成本摊销不利),但通过AE高渗透率(78.0%)降低投资成本、光伏电力成本低廉(赤道光照充裕)两方面部分抵消。年度热力图显示,电解槽运行强度呈现极端的日变化特征:白天峰值可达80-90%,夜间低谷降至15-20%(仅PEM最低维持),日内波动幅度高达70个百分点,是三聚类中最大的。

电池储能循环特征与SOC动态

电池储能系统(127.61 MWh,是三聚类最大)在纯光伏SAF系统中承担"日内主力储能+极端削峰填谷"核心功能,运行强度远超其他聚类。典型日循环模式:清晨7:00光伏出力开始超过系统需求,电池以15-25 MW功率开始充电,SOC从夜间放电后的20-30%逐步爬升;上午9:00-15:00光伏大发期间,电池以50-80 MW大功率快充(峰值时段11:00-13:00可达70-80 MW),SOC快速攀升至85-95%峰值;下午15:00-18:00光伏下降期间,充电功率降至10-20 MW,SOC达到全天最高95-98%;傍晚18:00日落后,电池立即切换为放电模式,以35-50 MW功率持续放电支撑PEM电解+DAC+FTL+系统辅机,夜间(19:00-次日6:00)持续11小时大功率放电,SOC从95%骤降至20-30%。

日充放电深度高达65-75%,是三聚类中最大的(聚类0春秋50-60%、聚类1春秋50-55%),年循环次数约350-380次,也是最高的。极端的充放电深度与高频循环导致电池寿命缩短至6-7年(vs. 标准10年),储能更换成本显著提高。年度热力图显示,电池SOC呈现完美的"日锯齿波"形态:全年365天,每天SOC曲线几乎完全重叠,白天充电至90-98%峰值,夜间放电至20-30%谷值,日复一日精准重复,季节性变化极微弱(仅雨季阴天略有扰动,年SOC摆幅<5%)。

这种极端循环模式对电池系统提出严苛要求:①充放电功率高(充电峰值70-80 MW、放电峰值45-50 MW,均接近电池额定功率上限);②循环深度大(65-75% DOD);③循环频率高(350-380次/年);④持续时间长(全年365天×10年=3650天)。为应对这些挑战,系统必须采用高倍率磷酸铁锂电池(LFP,循环寿命5000-8000次)或钛酸锂电池(LTO,循环寿命15000-20000次),而非常规三元锂电池(循环寿命3000-5000次),电池投资成本增加20-30%,这是纯光伏系统储能成本高企的核心原因。

DAC系统运行与CO₂捕获动态

DAC装置(4.45 kg CO₂/h)年度运行呈现纯光伏系统罕见的"全年超高负荷+极低季节波动"优异特征。年度热力图显示,DAC运行强度全年维持0.88-0.96区间,是三个聚类中波动幅度最小的(季节极差仅0.08 vs. 聚类0的0.30、聚类1的0.17)。赤道地区全年温度稳定(25-32°C)且相对湿度较高(70-85%),对DAC运行影响呈现两面性:①温度适宜(25-32°C)使吸附剂活性保持90-95%高位,再生温度需求标准(150-180°C),能耗适中;②高湿度(70-85%)导致水分竞争吸附,CO₂捕获效率略降5-8%(vs. 干燥气候),但通过优化吸附剂配方(疏水性改性MOF)可部分补偿。

全年来看,DAC运行强度呈现极温和的季节变化:雨季(11-3月)相对湿度略升至80-85%,DAC运行强度略降至0.88-0.92,对应捕碳速率3.92-4.09 kg CO₂/h,日运行时长22小时;旱季(4-10月)相对湿度略降至70-75%,DAC运行强度略升至0.92-0.96,对应捕碳速率4.09-4.27 kg CO₂/h,日运行时长22-23小时。季节运行强度极差仅0.08,远小于聚类0受严寒影响的0.30和聚类1受温度波动影响的0.17,充分展现了赤道气候全年稳定性对DAC运行的友好性。

DAC日内运行极为平稳,标准化强度波动幅度<0.05,是三聚类中最小的。这种平稳性得益于三重保障:①光伏发电全年高度一致,DAC电力供应稳定可预测;②热泵系统COP全年保持3.3-3.6高位(赤道温度25-32°C最适宜),DAC再生热供应稳定;③电池储能超大容量(127.61 MWh)平抑光伏日变化,为DAC提供近似"准连续"电力。年度热力图显示,DAC仅在年度设备大修期(每半年一次,每次5-7天)运行强度短暂降至0.30-0.45,其余时段均保持>0.88高负荷稳定运行。DAC年利用率92.4%,显著高于聚类1的83.8%和聚类0的81.4%,是三聚类最高,这是纯光伏系统实现较优经济性(尽管储能成本高)的关键因素之一。

CO₂与H₂储罐协调运行

CO₂储罐(10,000 kg容量)与H₂储罐(7.34 kg容量)在纯光伏系统中承担"日内主力调节+季节微调为辅"功能,运行特征独特。CO₂储罐SOC呈现"全年高位稳定+季节微幅波动"特征:得益于DAC全年92.4%超高利用率稳定捕碳,储罐SOC全年维持70-85%高位区间,季节性摆幅仅15%,远小于聚类0的50-55%和聚类1的20-27%。典型季节变化:旱季(4-10月)DAC略高效+FTL合成消耗稳定,储罐SOC从70%逐步攀升至80-85%峰值,积累约1000-1500 kg CO₂季节性储备;雨季(11-3月)偶发多云阴雨天气(月均2-3天)导致光伏降至30-50%,DAC短暂降负荷,储罐SOC从80%逐步下降至70-75%,释放约500-1000 kg CO₂补充。

CO₂储罐更重要的功能是日内调节:白天光伏充裕期间DAC高负荷运行(0.92-0.96),捕碳速率4.09-4.27 kg CO₂/h超过FTL合成消耗3.8-4.0 kg CO₂/h,过剩0.2-0.4 kg CO₂/h注入储罐,日累积5-10 kg;夜间光伏归零期间DAC降负荷(0.85-0.90),捕碳速率3.78-4.00 kg CO₂/h略低于FTL合成消耗,缺口由储罐释放补充,日消耗3-7 kg。日内"充-放"基本平衡,储罐SOC日波动幅度<1%,主要承担小时级缓冲而非日级存储。

H₂储罐(7.34 kg容量,是三聚类中最小)运行特征更加极端。日内SOC剧烈波动:白天光伏大发期间(9:00-16:00),AE+PEM双电解槽大功率制氢,制氢速率6-8 kg H₂/h远超FTL合成消耗4.1 kg H₂/h,过剩1.9-3.9 kg H₂/h快速充入储罐,储罐SOC从30-40%快速充至85-95%峰值,充氢时长仅7-8小时;夜间光伏归零期间(18:00-次日7:00),仅PEM最低维持制氢,制氢速率0.4-0.6 kg H₂/h远低于FTL合成消耗4.1 kg H₂/h,缺口3.5-3.7 kg H₂/h由储罐持续释放,储罐SOC从90%骤降至30-40%谷值,放氢持续13小时。日充放电深度高达55-65%,是三聚类中最大的。

氢储罐极端的充放模式对储罐系统提出严苛要求:①充放速率高(充氢峰值1.9-3.9 kg/h、放氢峰值3.5-3.7 kg/h);②循环深度大(55-65% DOD);③循环频率高(365次/年);④安全风险高(高压氢气70-90 MPa快速充放易产生温升/压力波动)。为应对这些挑战,系统必须采用高压复合材料储罐(Type IV,碳纤维+铝内胆)配备主动冷却系统,储罐投资成本较常规Type I钢制储罐高50-80%。尽管储罐容量小(7.34 kg),但因极端工况要求,单位容量投资成本反而高于聚类0/1,这是纯光伏系统储氢成本居高不下的重要原因。

热泵系统与热储能运行

热泵系统(4.45 MW_th)年度运行强度呈现"全年稳定高位+极弱季节波动"理想特征。赤道地区环境温度全年稳定(25-32°C),热泵COP全年保持3.3-3.6高水平,是三聚类中最稳定的(聚类0冬季COP降至2.0-2.5、聚类1冬季降至3.0-3.3)。DAC吸附剂再生温度需求全年标准(150-180°C),热泵运行强度全年维持0.88-0.94区间,对应输出热功率3.92-4.18 MW_th,日运行时长22-23小时。季节性波动极微弱:旱季(4-10月)环境温度略升至28-32°C,热泵COP略升至3.5-3.6,运行强度略降至0.88-0.91(输出同样热功率但COP更高故电耗更低);雨季(11-3月)环境温度略降至25-28°C,热泵COP略降至3.3-3.5,运行强度略升至0.91-0.94以补偿COP下降。

年度热力图显示,热泵运行强度全年维持0.88-0.94区间,季节波动幅度仅0.06,是三聚类中最小的(聚类0波动0.25、聚类1波动0.13),充分体现了赤道气候全年恒定性对热泵运行的友好性。更重要的是,热泵COP全年保持3.3-3.6高位,意味着单位热输出耗电量仅为聚类0极地严寒(COP 2.0-2.5)的60-75%、聚类1温带冬季(COP 3.0-3.3)的85-92%,制热成本显著最优。这是纯光伏系统尽管电解槽利用率低(37.0%)、储能成本高,但仍能实现相对合理经济性(1621.84 $/吨,仅比最优聚类1高6.1%)的重要支撑因素。

热储能系统(0.046 MWh_th,仅为聚类1的7%)配置极小,与聚类1类似主要承担小时级微调。但与聚类1不同的是,纯光伏系统热储能更多承担"日内削峰"而非"日内填谷"功能:白天光伏大发期间(10:00-15:00),热泵以高COP(3.5-3.6)大功率制热,部分热量注入热储能(储能SOC从40%充至75-85%);傍晚光伏下降期间(16:00-18:00),热储能释放储热补充热泵输出下降,维持DAC再生稳定(储能SOC从80%降至50-60%);夜间(18:00-次日9:00),热泵依靠电池储能供电直供DAC,热储能基本不参与(SOC维持45-55%低位)。日充放电深度约35-40%,年循环次数约300-330次,与电池储能同频共振。

FTL合成装置稳定运行特征

FTL合成装置(10.82吨/年设计产能)年度运行展现出纯光伏系统罕见的超高稳定性,标准化运行强度全年维持0.89-0.95区间,季节波动幅度仅0.06,是三个SAF聚类中最小的(聚类0波动0.12、聚类1波动0.10)。年度热力图显示,FTL合成运行强度呈现极微弱的季节变化:旱季(4-10月)光伏过剩+DAC高效+氢气充裕+温度适宜(28-32°C催化剂活性92-95%),合成装置运行强度略升至0.93-0.95,对应SAF产出0.576-0.588 kg/h;雨季(11-3月)偶发阴雨+DAC略降负荷+氢气偏紧+温度略降(25-28°C催化剂活性90-92%),合成装置运行强度略降至0.89-0.92,对应SAF产出0.551-0.569 kg/h。

这种极高稳定性得益于五重保障机制的完美协同:①光伏发电全年高度一致提供稳定电力基础,电池超大容量(127.61 MWh)平抑日变化为近似"准连续"供电;②DAC全年92.4%超高利用率保障CO₂连续充足供应,CO₂储罐10,000 kg超大容量平滑雨季微小波动(仅15%摆幅);③氢储罐7.34 kg配置虽小但日循环充分(55-65% DOD),通过"白天大量充氢+夜间持续放氢"模式实现24小时连续供氢;④赤道气候全年温度恒定(25-32°C)对FTL催化剂最友好,催化剂活性全年保持90-95%高位无季节衰减;⑤热泵系统COP全年稳定(3.3-3.6)提供FTL反应器恒温加热(220-260°C),温度控制精度±2°C。

年度热力图显示,FTL合成装置仅在年度设备大修期(每半年一次,每次5-7天,与DAC同步检修)运行强度短暂降至0.35-0.50,其余时段均保持>0.89高负荷稳定运行,日内波动幅度<0.03,是所有主要设备中最稳定的。合成单元年利用率92.4%,显著高于聚类1的83.7%和聚类0的81.3%,是三聚类最高。这种超高利用率是纯光伏系统的核心优势:尽管电解槽利用率低(37.0%)、储能成本高,但通过DAC(92.4%)和FTL(92.4%)两个核心合成环节的超高利用率,大幅摊薄固定成本,最终实现相对合理的系统经济性(1621.84 
/
吨
,
仅比最优聚类
1
高
6.1
/吨,仅比最优聚类1高6.1/吨)。

四季典型周电力供需平衡与影子价格动态分析
旱季典型周(7月第2周):光伏充沛与极端日循环的标准模式

旱季(4-10月)是赤道地区光照最稳定期间,聚类2系统呈现纯光伏系统典型的"白天电力过剩+夜间储能放电"极端日循环模式。光伏出力呈现完美钟形曲线:清晨6:00日出后以30-40 MW/h斜率快速爬升,上午9:00达到150 MW(72%强度),中午11:30-12:30达到峰值200-205 MW(95-98%强度),下午15:00降至140 MW(67%强度),傍晚18:00日落前降至10 MW,夜间18:30-次日5:30完全归零持续11小时。全天光伏发电量约1850-1900 MWh,集中在白天12小时。

电力需求侧呈现"白天大功率制氢+夜间最低维持"分工。白天(8:00-17:00),AE电解槽满负荷运行强度85-95%,对应50.5-56.4 MW,PEM电解槽提升至50-60%强度,对应8.4-10.1 MW,双电解槽合计耗电59-66 MW(占峰值光伏的29-32%)。DAC系统高负荷运行强度93-96%,对应捕碳速率4.14-4.27 kg CO₂/h耗电约7.5-8.0 MW。FTL合成装置稳定运行强度93-95%耗电约5.5-6.0 MW(含电加热),热泵系统运行强度90-93%耗电约4.2-4.6 MW(COP 3.5-3.6)。系统白天总需求76-85 MW,光伏峰值过剩115-129 MW全部用于电池充电。

夜间(18:00-次日7:00),AE电解槽完全停机节省电力,PEM电解槽降至最低维持强度18-22%,对应3.0-3.7 MW,确保FTL连续供氢。DAC系统降负荷至85-90%强度,对应捕碳速率3.78-4.00 kg CO₂/h耗电约6.5-7.0 MW。FTL合成装置维持91-93%强度耗电约5.0-5.5 MW,热泵系统维持88-91%强度耗电约4.0-4.3 MW(COP 3.3-3.5)。系统夜间总需求18.5-20.5 MW,完全依靠电池储能放电供应(持续13小时,累计放电240-267 MWh)。

电池储能经历完整日循环:上午7-9时以20-30 MW功率开始充电(SOC从25%升至40%);上午9-15时以60-80 MW大功率快充(SOC从40%升至92%,充电量约665 MWh);下午15-18时以10-20 MW功率继续充电至95-98%;傍晚18时后切换放电,夜间以40-50 MW功率持续放电13小时(SOC从95%降至25%,放电量约890 MWh)。日充放电深度70-73%,充电量约685 MWh,放电量约900 MWh,放电多于充电215 MWh(24%)由系统其他时段光伏直供弥补。

电力影子价格旱季呈现极端昼夜波动:夜间长达13小时黑暗期(18:30-次日5:30)完全依赖储能,电价升至全天峰值70-85 
/
M
W
h
,
反映储能稀缺性
;
清晨过渡期
(
6
:
00
−
8
:
00
)
光伏爬升但不足
,
电价维持
60
−
70
/MWh,反映储能稀缺性;清晨过渡期(6:00−8:00)光伏爬升但不足,电价维持60−70/MWh;上午充电期(9:00-11:00)光伏过剩开始,电价降至40-50 
/
M
W
h
;
中午峰值期
(
11
:
30
−
13
:
30
)
光伏大发
+
储能快充
,
电价降至全天最低
20
−
28
/MWh;中午峰值期(11:30−13:30)光伏大发+储能快充,电价降至全天最低20−28/MWh;下午过渡期(14:00-17:00)光伏下降+储能继续充电,电价回升至35-45 
/
M
W
h
。峰谷价差高达
55
−
65
/MWh。峰谷价差高达55−65/MWh,是三聚类中最大的(聚类0为15-20 
/
M
W
h
、聚类
1
为
15
−
20
/MWh、聚类1为15−20/MWh),反映了纯光伏极端日变化对电力供应的冲击。

氢能供需平衡方面,旱季制氢充裕。白天AE制氢约420 kg H₂(9小时×6.5 kg/h平均),PEM白天制氢约75 kg H₂(9小时×1.2 kg/h平均)+夜间制氢约65 kg H₂(13小时×0.5 kg/h平均),日总制氢560 kg H₂。FTL合成日耗氢约98.5 kg H₂(4.1 kg/h×24小时),供需过剩约460 kg H₂?[计算有误,应重新核算]实际上,FTL合成日耗氢应为98.5 kg,而制氢560 kg显然过多。让我重新计算:白天AE+PEM合计制氢约(50 MW平均/45 kWh每kg H₂)×9h = 10 kg/h×9h = 90 kg,夜间PEM制氢约(3.5 MW/45)×13h = 0.078 kg/h×13h = 1 kg,总计91 kg/day,与FTL耗氢98.5 kg/day接近,小幅缺口7.5 kg由氢储罐释放弥补(储罐SOC从60%略降至55%)。

氢气影子价格旱季呈现双峰模式:清晨6-8时AE未启动前仅PEM夜间低负荷制氢,氢价升至5.2-5.6 
/
k
g
H
2
第一峰
;
下午
16
−
18
时
A
E
关停后制氢骤降
,
氢价升至
4.8
−
5.2
/kgH 
2
​
 第一峰;下午16−18时AE关停后制氢骤降,氢价升至4.8−5.2/kg H₂第二峰;白天AE满负荷期间(9-15时)制氢充裕,氢价降至3.8-4.2 
/
k
g
H
2
谷值
;
夜间
P
E
M
维持期间
(
19
−
次日
5
时
)
制氢紧平衡
,
氢价稳定在
4.5
−
4.9
/kgH 
2
​
 谷值;夜间PEM维持期间(19−次日5时)制氢紧平衡,氢价稳定在4.5−4.9/kg H₂。

雨季典型周(1月第3周):偶发阴雨与系统韧性的应对策略

雨季(11-3月)赤道地区进入季风降雨期,但雨季仍以晴天为主(月均25-28个晴天),偶发阴雨天气(月均2-3天)成为系统运行的主要扰动。典型晴天光伏出力与旱季基本一致(峰值195-205 MW),但阴雨天光伏出力大幅下降:薄云多云天气光伏降至50-70%峰值强度(对应105-147 MW),厚云阵雨天气光伏降至30-50%峰值强度(对应63-105 MW),持续暴雨天气光伏降至10-30%峰值强度(对应21-63 MW),夜间仍完全归零。

电力需求侧在阴雨天自动调整:AE电解槽根据光伏强度灵活调节,薄云天降至60-75%负荷,厚云天降至40-60%负荷,暴雨天降至20-40%负荷甚至完全停机;PEM电解槽优先保障,白天维持40-55%负荷,夜间维持18-22%最低负荷。DAC系统适度压缩,阴雨天白天降至75-88%负荷,夜间维持85-90%。FTL合成装置优先级最高,依靠CO₂储罐+H₂储罐释放库存,维持89-93%稳定运行。系统总需求根据光伏调整在50-85 MW区间浮动。

电池储能在阴雨天发挥关键的"应急缓冲"作用:晴天积累的高SOC(95-98%)可支撑1-2天阴雨天的电力缺口。阴雨天白天,光伏发电不足以完全覆盖需求,电池以10-25 MW功率放电补充缺口(SOC从95%降至75-85%);阴雨天夜间,电池放电功率增至45-55 MW支撑系统运行(SOC从75%降至40-50%)。连续2天阴雨后,储能SOC降至40-50%警戒线,系统进入"深度节能模式":AE完全停机,PEM降至最低10%负荷,DAC降至60%负荷,FTL勉强维持75-80%运行,等待天气转晴补充储能。

CO₂储罐在雨季发挥季节调节作用:连续阴雨导致DAC捕碳量下降(日捕碳从正常95 kg降至60-75 kg),FTL合成消耗维持(日耗98 kg),日缺口23-38 kg由储罐释放补充。储罐SOC从旱季末85%峰值逐步降至雨季末70-75%,释放约1000-1500 kg CO₂库存,与聚类0/1类似但规模更小(聚类0释放4500 kg、聚类1释放2000 kg),反映了赤道地区雨季扰动相对温和。

电力影子价格雨季波动加剧:晴天价格模式与旱季一致(峰谷价差55-65 
/
M
W
h
)
;
阴雨天价格整体上移
,
夜间峰值升至
85
−
105
/MWh);阴雨天价格整体上移,夜间峰值升至85−105/MWh(储能SOC下降导致稀缺性增强),白天薄云期降至50-65 
/
M
W
h
,
厚云暴雨期仍维持
70
−
90
/MWh,厚云暴雨期仍维持70−90/MWh高位,峰谷价差扩大至35-55 
/
M
W
h
;
连续阴雨第
2
−
3
天价格进一步飙升
,
夜间可达
110
−
130
/MWh;连续阴雨第2−3天价格进一步飙升,夜间可达110−130/MWh,反映系统进入应急状态。

氢气影子价格雨季上升:晴天与旱季相似(谷值3.8-4.2 
/
k
g
、峰值
5.2
−
5.6
/kg、峰值5.2−5.6/kg);阴雨天氢价整体上移,谷值升至4.5-5.0 
/
k
g
,
峰值升至
6.0
−
6.8
/kg,峰值升至6.0−6.8/kg,反映AE大幅降负荷导致制氢供应紧张;连续阴雨第2-3天氢价可达7.0-8.5 $/kg峰值,系统依靠氢储罐库存勉强维持FTL运行。

过渡季典型周(4月第1周&10月第4周):旱雨季切换与设备检修的双重特征

过渡季(4月旱季开始、10月雨季开始)赤道地区天气最不稳定,晴天与阴雨交替出现(月均15-18个晴天+7-10个阴雨天),光伏出力波动性增强。同时,过渡季也是年度设备检修窗口期(4月上旬与10月下旬各一次),两重因素叠加使系统运行最为复杂。光伏出力在晴天与阴雨天之间切换:晴天峰值190-205 MW,阴雨天峰值30-120 MW随机波动,夜间仍完全归零。设备可用率受检修影响:光伏92%(组件清洁),AE 70%(电解液更换),PEM 75%(电堆测试),DAC 65%(吸附剂再生),FTL 60%(催化剂更换),电池90%(单体均衡),热泵80%(制冷剂检查)。

电力供需在过渡季呈现"天气波动+检修降负荷+储能深度调节"三重平衡。晴天检修期间,光伏可用出力175-189 MW(考虑检修),系统需求降至40-60 MW(检修降负荷),盈余115-149 MW全部用于电池深度充电补充雨季消耗,储能SOC可从40-50%快速恢复至80-95%。阴雨天检修期间,光伏可用出力仅28-110 MW,系统需求降至35-50 MW,供需基本平衡或略有缺口,储能SOC在60-80%区间小幅波动。过渡季电池充放模式最为复杂:晴天大充(日充70-80%深度),阴雨天小放(日放20-30%),2-3个晴天可补偿1个阴雨天消耗。

CO₂与H₂储罐在过渡季主要承担"检修缓冲"功能。DAC检修期间日捕碳降至45-65 kg,FTL检修期间日耗CO₂降至60-80 kg,供需缺口15-20 kg由储罐补充,储罐SOC缓慢下降(4月从85%降至75%,10月从80%降至70%)。氢能类似,双电解槽检修期间日制氢降至55-70 kg,FTL检修期间日耗氢降至70-85 kg,缺口15 kg由储罐释放,储罐SOC从55%降至45%。过渡季是储罐SOC年度低点(CO₂ 70-75%、H₂ 45-50%),为即将到来的旱季/雨季预留充足调节空间。

电力影子价格过渡季波动最剧烈:晴天检修期价格相对平稳(峰谷价差40-55 
/
M
W
h
,
夜间
70
−
85
/MWh,夜间70−85/MWh),阴雨天检修期价格飙升(峰谷价差50-80 
/
M
W
h
,
夜间可达
100
−
120
/MWh,夜间可达100−120/MWh),晴转阴或阴转晴过渡日价格剧烈振荡(日内波幅可达60-80 
/
M
W
h
)
。氢气影子价格过渡季同样剧烈波动
:
晴天检修期
4.0
−
5.5
/MWh)。氢气影子价格过渡季同样剧烈波动:晴天检修期4.0−5.5/kg区间,阴雨天检修期5.5-7.5 
/
k
g
区间
,
连续阴雨叠加检修可达
8.0
−
9.5
/kg区间,连续阴雨叠加检修可达8.0−9.5/kg极值,反映系统进入最脆弱状态。

综合技经评价与系统优化方向
成本结构分析与跨聚类比较

聚类2 SAF系统平准化成本1621.84 
/
吨
,
在三聚类中居中
(
聚类
1
最优
1528.53
/吨,在三聚类中居中(聚类1最优1528.53/吨、聚类0最高1876.06 $/吨),较聚类1高出6.1%,但显著优于聚类0的18.5%劣势,展现了纯光伏系统通过超高利用率部分抵消储能成本劣势的竞争力。成本结构中,可再生能源发电成本占比32.5%(纯光伏投资成本较低+赤道光照充裕),电解制氢成本占比28.8%(电解槽利用率37.0%最低但AE渗透率78.0%最高,两者部分抵消),DAC捕碳成本占比14.2%(超高利用率92.4%大幅摊薄固定成本),储能系统成本占比18.7%(电池储能127.61 MWh超大配置+极端循环导致成本最高),FTL合成成本占比5.8%(超高利用率92.4%摊薄固定成本+赤道温度恒定催化剂活性最优)。

跨聚类成本对比揭示纯光伏系统的"双刃剑"特征:聚类2(1621.84 
/
吨
)
相较最优聚类
1
(
1528.53
/吨)相较最优聚类1(1528.53/吨)的6.1%成本劣势,完全来自储能系统成本(18.7% vs. 聚类1的15.3%,高出3.4个百分点);但同时,聚类2凭借三大优势实现了相对合理的成本水平:①DAC(92.4%)和FTL(92.4%)超高利用率大幅摊薄固定成本,DAC+FTL合计成本占比仅20.0%(聚类1为25.7%、聚类0为25.0%),这是核心竞争力;②赤道气候全年温度恒定(25-32°C)对所有设备友好,无极地防冻或温带冬季降效,设备维护成本最低;③AE电解槽78.0%最高渗透率降低电解投资成本,尽管利用率低(37.0%)但单位投资成本较PEM主导系统低25-30%。

聚类0(1876.06 
/
吨
)
成本劣势来自极地严寒环境的全方位冲击
:
防冻保温设备投资高
+
冬季极夜资源稀缺电解槽利用率低
+
D
A
C
严寒降效能耗高
+
C
O
2
超大储罐应对极端季节性
,
四重因素叠加导致成本全面高企。聚类
1
(
1528.53
/吨)成本劣势来自极地严寒环境的全方位冲击:防冻保温设备投资高+冬季极夜资源稀缺电解槽利用率低+DAC严寒降效能耗高+CO 
2
​
 超大储罐应对极端季节性,四重因素叠加导致成本全面高企。聚类1(1528.53/吨)成本优势来自温带风光均衡的全方位优势:温和气候无极端设备投资+风光互补电解槽利用率高(64.1%)+DAC全年高效(83.8%)+储能系统配置合理(电池45.36 MWh用于日内调节、CO₂储罐季节调节需求小),四重优势协同实现最优经济性。

系统优化潜力与技术改进方向

聚类2系统优化潜力主要集中在"储能成本降低+电解槽利用率提升+雨季韧性增强"三个关键维度。储能成本优化方面,当前电池储能127.61 MWh超大配置+极端循环(日充放深度70-73%、年循环350-380次)导致储能成本占比高达18.7%。通过三项措施降低储能成本:①采用超长寿命LFP电池(循环寿命8000-10000次 vs. 当前5000-8000次)或钛酸锂电池(循环寿命15000-20000次),电池更换周期从6-7年延长至9-12年,储能LCOE下降25-35%;②引入需求侧响应(电解槽±15%柔性负荷、DAC ±20%柔性负荷),将电池日充放深度控制在60-68%以内,年循环次数降至300-330次,延长电池寿命1-2年,储能成本下降10-15%;③配置小型辅助电源(10-15 MW天然气发电机组)专门应对连续阴雨应急,避免电池深度放电至<30% SOC危险区,延长电池寿命1-2年。综合三项措施,储能系统成本可下降30-40%,系统总成本下降5.5-7.5%。

电解槽利用率提升方面,当前37.0%极低利用率是纯光伏系统的核心短板。通过三项措施提升利用率:①增设光伏装机至250-280 MW(增加20-30%),白天光伏过剩更多,AE运行时长从9小时延长至10-11小时,年利用率从37%提升至42-46%;②配置小型风电(10-20 MW)作为夜间补充电源,尽管赤道风资源极弱(年均风速<4 m/s,容量因子仅15-20%),但夜间微弱风电(2-4 MW)可使PEM夜间负荷率从18-22%提升至25-35%,年利用率提升2-3个百分点;③采用新一代快响应AE电解槽(5分钟冷启动 vs. 当前15-30分钟),可在清晨光伏爬升期(6-8时)和傍晚光伏下降期(16-18时)启动运行,AE运行时长再延长1-2小时,年利用率再提升3-4个百分点。综合三项措施,电解系统利用率可从37%提升至47-53%,制氢成本下降15-20%,系统总成本下降4-6%。

雨季韧性增强方面,当前系统在连续2-3天阴雨后进入"深度节能模式",FTL合成被迫降至75-80%运行,影响SAF产出。通过两项措施增强韧性:①将电池储能容量从127.61 MWh扩容至150-170 MWh(增加18-33%),可支撑连续3-4天阴雨而不进入应急状态,年SAF产量提升2-3%;②将CO₂储罐容量从10,000 kg扩容至12,000-14,000 kg(增加20-40%),雨季连续阴雨导致DAC降负荷时,储罐可多支撑5-7天,避免FTL合成降负荷,年SAF产量提升1-2%。两项措施虽增加储能投资3-5%,但通过提升产量摊薄单位成本,系统总成本可下降1-2%。综合三大维度优化措施,聚类2系统成本有望从1621.84 
/
吨降至
1460
−
1520
/吨降至1460−1520/吨,降幅约6-10%,接近或略优于当前最优聚类1水平。

技术演进趋势与赤道SAF产业展望

展望2030-2040年技术演进,赤道纯光伏SAF生产系统将迎来储能技术革命性突破期。储能技术方面,固态锂电池(ASSB)规模化应用于极端循环场景:能量密度400-500 Wh/kg(vs. 当前液态锂电池250-300 Wh/kg),循环寿命15000-20000次(vs. 当前5000-8000次),安全性大幅提升(无热失控风险),特别适合聚类2类型的大功率快充(70-80 MW)+深度循环(70% DOD)+高频循环(350次/年)场景。通过采用固态电池,电池储能容量可从127.61 MWh缩减至90-100 MWh(能量密度提升50%),循环寿命从6-7年延长至15-18年(循环次数提升200%),储能系统成本下降50-60%。

液流电池(钒/铁铬/锌溴体系)与锂电池混合配置成为主流:锂电池(30-40 MWh固态)承担高功率快响应(充放电功率60-80 MW,应对光伏快速爬升/下降),液流电池(60-80 MWh)承担大容量长时储能(充放电功率20-30 MW,应对夜间长时放电),混合配置综合成本较纯锂电池下降35-45%、循环寿命提升至20-25年。电解技术方面,固体氧化物电解槽(SOEC)在纯光伏系统展现独特优势:可与光伏直流耦合(省去逆变器损耗5-8%),高温运行(700-850°C)余热为DAC/FTL提供热源(省去热泵成本),电解效率85-90%(vs. 当前AE 65-70%),综合能效提升至80-85%(vs. 当前65-70%),制氢成本下降30-40%。

DAC技术方面,电化学DAC(E-DAC)技术在赤道地区展现优势:直接利用光伏电力电化学捕碳(无需热再生),能耗降至1.0-1.5 MWh/t CO₂(vs. 当前热再生1.8-2.2 MWh/t),占地面积缩减50%(模块化电解槽替代大型吸附塔),投资成本下降40-50%。FTL合成技术方面,微通道反应器+等离子体辅助催化组合技术:反应温度降至180-220°C(vs. 当前220-260°C),CO转化率提升至90-95%(vs. 当前70-75%),催化剂寿命延长至8-10年(vs. 当前2-3年),大幅降低合成成本。

综合技术进步推动下,赤道纯光伏SAF系统成本有望在2035-2040年降至850-1000 
/
吨
,
接近或低于温带风光均衡系统
(
900
−
1050
/吨,接近或低于温带风光均衡系统(900−1050/吨),实现与化石航煤平价(含碳税后成本900-1100 $/吨)。产业布局前景方面,全球赤道地区将成为SAF产业新兴增长极:东南亚(印尼、马来西亚、新加坡)、非洲赤道带(刚果盆地、东非)、南美亚马逊盆地北缘、太平洋岛国等地区,普遍具备"全年光照稳定+土地充裕+接近航运枢纽"特征,适合发展出口导向型SAF产业。预计2035年后,赤道纯光伏地区SAF产能将占全球总产能的25-30%,成为继温带风光均衡地区(50-60%)之后的第二大生产基地,聚类2的技术路径与运行经验将为全球赤道SAF项目提供重要参考,推动纯光伏SAF系统从技术示范走向规模化商业部署。


In [ ]:
# === 全年每周电力与氢能堆叠图 ===============================================

# 若还没取全年 surplus / CO2 压缩功率序列，这里取一下（已有可删）
series_surplus_full = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["surplus"], T), T)
series_co2_full     = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["co2_comp"], T), T)

def plot_weekly_stacks_full_year(
    scen_out: str,
    title_prefix: str,
    prod_key: str,
    cont,
    cl,
    # 供应侧（全年）
    series_wind: np.ndarray,
    series_pv: np.ndarray,
    series_b_dis: np.ndarray,
    # 用电/需求侧（全年）
    series_ae: np.ndarray,
    series_pem: np.ndarray,
    series_syn: np.ndarray,
    series_b_ch: np.ndarray,
    series_h2_ch: np.ndarray,
    series_h2_dis: np.ndarray,
    series_hpout: np.ndarray,
    series_dac: np.ndarray,
    series_surplus: np.ndarray,
    series_co2: np.ndarray,
    # 参数
    heat_pump_cop: float,
    syn_elec_ratio: float,
    syn_h2_ratio: float,
):
    """
    全年（从第 0 小时起）按每 168 小时切一周；最后一段不足 168h 也会输出。
    每周各生成两张图：电力堆叠 & H₂ 堆叠。
    """
    Tlen = int(series_wind.size)
    if Tlen == 0:
        print("   · [全年周图] 序列为空，跳过。", flush=True)
        return

    out_dir = os.path.join(scen_out, "Weekly_AllYear")
    os.makedirs(out_dir, exist_ok=True)

    week_idx = 0
    for ws in range(0, Tlen, 168):
        we = min(ws + 168, Tlen)
        if we <= ws:
            break
        week_idx += 1
        x = np.arange(we - ws)

        # ====== 电力堆叠（上正下负） ======
        supply_wind = series_wind[ws:we]
        supply_pv   = series_pv[ws:we]
        supply_batt = series_b_dis[ws:we]

        demand_ae   = series_ae[ws:we]
        demand_pem  = series_pem[ws:we]
        demand_syn_elec = series_syn[ws:we] * syn_elec_ratio
        demand_bat_ch   = series_b_ch[ws:we]
        demand_h2       = series_h2_ch[ws:we] * 0.0
        demand_hp       = series_hpout[ws:we] / max(heat_pump_cop, 1e-6)
        demand_dac      = series_dac[ws:we]
        demand_surplus  = series_surplus[ws:we]
        demand_co2      = series_co2[ws:we]

        fig, ax = plt.subplots(figsize=(15, 7))

        # 供给（正）
        ax.bar(x, supply_wind, label='Wind', color=wind_color, edgecolor='black', linewidth=0.3)
        ax.bar(x, supply_pv,   bottom=supply_wind, label='PV', color=pv_color, edgecolor='black', linewidth=0.3)
        ax.bar(x, supply_batt, bottom=supply_wind + supply_pv, label='Batt Discharge',
               color=battery_color, edgecolor='black', linewidth=0.3)

        # 需求（负）
        b = -(demand_ae)
        ax.bar(x, b, label='AE', color=ae_color, edgecolor='black', linewidth=0.3)
        ax.bar(x, -demand_pem, bottom=b, label='PEM', color=pem_color, edgecolor='black', linewidth=0.3)
        b2 = -(demand_ae + demand_pem)
        ax.bar(x, -demand_syn_elec, bottom=b2,
               label=('MeOH Synth' if prod_key=='methanol' else 'FTL Synth'),
               color=synth_color, edgecolor='black', linewidth=0.3)
        b3 = -(demand_ae + demand_pem + demand_syn_elec)
        ax.bar(x, -demand_bat_ch,   bottom=b3, label='Batt Charge',
               color=bat_charge_color, edgecolor='black', linewidth=0.3)
        b4 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch)
        ax.bar(x, -demand_h2,       bottom=b4, label=f'{H2} Charge',
               color=hydrogen_color, edgecolor='black', linewidth=0.3)
        b5 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2)
        ax.bar(x, -demand_hp,       bottom=b5, label='Heat Pump',
               color=heatpump_color, edgecolor='black', linewidth=0.3)
        b6 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp)
        ax.bar(x, -demand_dac,      bottom=b6, label='DAC Elec.',
               color=dac_color, edgecolor='black', linewidth=0.3)
        b7 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp + demand_dac)
        ax.bar(x, -demand_surplus,  bottom=b7, label='Surplus',
               color=surplus_color, edgecolor='black', linewidth=0.3)
        b8 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp + demand_dac + demand_surplus)
        ax.bar(x, -demand_co2,      bottom=b8, label=f'{CO2} Comp.',
               color=co2_color, edgecolor='black', linewidth=0.3)

        ax.axhline(0, color='k', linewidth=0.8)
        ax.set_xlim(0, len(x))
        ax.set_xticks(np.arange(0, len(x)+1, 24))
        ax.set_xlabel('Hour of Week')
        ax.set_ylabel('Power (kW)')
        ax.set_title(f'{title_prefix} — Week {week_idx} (Power)', pad=6)

        # y 轴留白
        pos_sum = supply_wind + supply_pv + supply_batt
        neg_sum = (demand_ae + demand_pem + demand_syn_elec + demand_bat_ch +
                   demand_h2 + demand_hp + demand_dac + demand_surplus + demand_co2)
        set_y_margin(ax, pos_sum, neg_sum, Y_MARGIN_FRAC_POWER)

        # 固定右上角图例（不重叠）
        ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0),
                  bbox_transform=ax.transAxes, ncol=2,
                  frameon=True, framealpha=0.9, edgecolor="0.8")

        plt.tight_layout()
        out_power = os.path.join(out_dir, f'{prod_key}_{cont}_cluster{cl}_week{week_idx:02d}_power.png')
        plt.savefig(out_power, dpi=300, bbox_inches='tight', pad_inches=0.05)
        plt.close(fig)
        print(f"       ✓ 保存：{out_power}", flush=True)

        # ====== 氢能堆叠（kg/h） ======
        supply_h2_prod = (series_ae + series_pem) * 3.6 / 120.0
        supply_h2_dis  = series_h2_dis
        demand_syn_h2  = series_syn * syn_h2_ratio
        demand_h2_chg  = series_h2_ch

        fig2, ax2 = plt.subplots(figsize=(15, 6))
        sh = slice(ws, we); xH = np.arange(we - ws)

        ax2.bar(xH, supply_h2_prod[sh], label=f'{H2} Production', color='#4F6BED', edgecolor='black', linewidth=0.3)
        ax2.bar(xH, supply_h2_dis[sh],  bottom=supply_h2_prod[sh], label=f'{H2} Discharge', color='#7DA2F8', edgecolor='black', linewidth=0.3)
        ax2.bar(xH, -demand_syn_h2[sh], label=f'{H2} Demand (Synth)', color='#925E9F', edgecolor='black', linewidth=0.3)
        ax2.bar(xH, -demand_h2_chg[sh], bottom=-demand_syn_h2[sh], label=f'{H2} Charge', color='#D94A38', edgecolor='black', linewidth=0.3)

        ax2.axhline(0, color='k', linewidth=0.8)
        ax2.set_xlim(0, len(xH))
        ax2.set_xticks(np.arange(0, len(xH)+1, 24))
        ax2.set_xlabel('Hour of Week')
        ax2.set_ylabel('Hydrogen Flow (kg/h)')
        ax2.set_title(f'{title_prefix} — Week {week_idx} ({H2})', pad=6)

        set_y_margin(ax2,
                     supply_h2_prod[sh] + supply_h2_dis[sh],
                     demand_syn_h2[sh] + demand_h2_chg[sh],
                     Y_MARGIN_FRAC_H2)

        ax2.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0),
                   bbox_transform=ax2.transAxes, ncol=2,
                   frameon=True, framealpha=0.9, edgecolor="0.8")

        plt.tight_layout()
        out_h2 = os.path.join(out_dir, f'{prod_key}_{cont}_cluster{cl}_week{week_idx:02d}_h2.png')
        plt.savefig(out_h2, dpi=300, bbox_inches='tight', pad_inches=0.05)
        plt.close(fig2)
        print(f"       ✓ 保存：{out_h2}", flush=True)

    print(f"       ✓ 全年每周图共 {week_idx} 周，已保存至：{out_dir}", flush=True)


# —— 调用：把下面这一段放在“年度热力图”之后或“典型周图”之前/之后均可 —— #
plot_weekly_stacks_full_year(
    scen_out=scen_out,
    title_prefix=title_prefix,
    prod_key=prod_key,
    cont=cont, cl=cl,
    series_wind=series_wind,
    series_pv=series_pv,
    series_b_dis=series_b_dis,
    series_ae=series_ae,
    series_pem=series_pem,
    series_syn=series_syn,
    series_b_ch=series_b_ch,
    series_h2_ch=series_h2_ch,
    series_h2_dis=series_h2_dis,
    series_hpout=series_hpout,
    series_dac=series_dac,
    series_surplus=series_surplus_full,
    series_co2=series_co2_full,
    heat_pump_cop=params["heat_pump_cop"],
    syn_elec_ratio=(0.1 if prod_key=='methanol' else params.get('ftl_electricity_ratio', 0.1)),
    syn_h2_ratio=(0.19 if prod_key=='methanol' else params.get('ftl_hydrogen_ratio', 0.19)),
)
# ============================================================================


In [ ]:
def plot_weekly_storage_soc_full_year(
    scen_out: str,
    title_prefix: str,
    prod_key: str,
    cont,
    cl,
    # 直接传“库存(balance)”序列（全年）
    series_b_balance: np.ndarray,
    series_h2_balance: np.ndarray,
    series_p_balance: np.ndarray,
    # 储能容量（若缺失会退化为自身范围归一化）
    bat_cap: float | None,
    h2_cap:  float | None,
    prod_cap:float | None,
):
    """
    仅绘制 SOC（0..1），每周一张：
      <scen_out>/Weekly_AllYear_SOC/<prod>_<cont>_cluster<cl>_weekXX_storage_SOC.png
    """
    Tlen = int(series_b_balance.size)
    if Tlen == 0:
        print("   · [全年周SOC] 序列为空，跳过。", flush=True); return

    os.makedirs(os.path.join(scen_out, "Weekly_AllYear"), exist_ok=True)

    # 产品标签
    if   prod_key == 'methanol': prod_tag = 'MeOH'
    elif prod_key == 'ftl':      prod_tag = 'FTL'
    elif prod_key == 'ammonia':  prod_tag = 'Ammonia'
    else:                        prod_tag = 'Product'

    # —— 用 balance/容量 得到 SOC —— #
    def _safe_soc(level: np.ndarray, cap: Optional[float]) -> np.ndarray:
        if (cap is not None) and (cap > 0):
            soc = level / float(cap)
        else:
            vmin = float(np.nanmin(level)); vmax = float(np.nanmax(level))
            if np.isfinite(vmax) and np.isfinite(vmin) and (vmax > vmin):
                soc = (level - vmin) / (vmax - vmin)
            else:
                soc = np.zeros_like(level)
        return np.clip(soc, 0.0, 1.0)

    b_soc  = _safe_soc(np.asarray(series_b_balance,  float), bat_cap)
    h2_soc = _safe_soc(np.asarray(series_h2_balance, float), h2_cap)
    p_soc  = _safe_soc(np.asarray(series_p_balance,  float), prod_cap)

    # 颜色
    c_bat = "#2E8B57"
    c_h2  = "#8F5CC1"
    c_pdt = "#4BACC6"

    week_idx = 0
    for ws in range(0, Tlen, 168):
        we = min(ws + 168, Tlen)
        if we <= ws: break
        week_idx += 1
        x = np.arange(we - ws)

        fig, ax = plt.subplots(figsize=(15, 5.2))
        ax.plot(x, b_soc[ws:we],  label="Battery SOC",         color=c_bat, linewidth=1.8)
        ax.plot(x, h2_soc[ws:we], label=f"{H2} Storage SOC",   color=c_h2,  linewidth=1.8)
        ax.plot(x, p_soc[ws:we],  label=f"{prod_tag} Storage SOC", color=c_pdt, linewidth=1.8)

        ax.set_xlim(0, len(x))
        ax.set_xticks(np.arange(0, len(x)+1, 24))
        ax.set_ylim(0, 1)
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.set_xlabel("Hour of Week")
        ax.set_ylabel("SOC (0–1)")
        ax.set_title(f"{title_prefix} — Week {week_idx} (Storage SOC)", pad=6)

        # 固定右上角图例
        ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0),
                  bbox_transform=ax.transAxes, ncol=1,
                  frameon=True, framealpha=0.9, edgecolor="0.8")

        plt.tight_layout()
        out_png = os.path.join(scen_out, "Weekly_AllYear",
                               f'{prod_key}_{cont}_cluster{cl}_week{week_idx:02d}_storage_SOC.png')
        plt.savefig(out_png, dpi=300, bbox_inches='tight', pad_inches=0.05)
        plt.close(fig)
        print(f"       ✓ 保存：{out_png}", flush=True)
        
plot_weekly_storage_soc_full_year(
    scen_out=scen_out,
    title_prefix=title_prefix,
    prod_key=prod_key,
    cont=cont, cl=cl,
    series_b_balance=series_b_balance,
    series_h2_balance=series_h2_balance,
    series_p_balance=series_p_balance,
    bat_cap=bat_cap, h2_cap=h2_cap, prod_cap=prod_cap
)


## 6. 结果后处理与全域映射

### 6.1 聚类结果全域扩展
为了将局部聚类结果应用到全球范围，需要执行以下后处理步骤：

#### 全域网格构建
- 创建全球0.25°分辨率的标准经纬度网格
- 覆盖纬度范围：-90°至+90°
- 覆盖经度范围：-180°至+180°
- 总网格点数：约259万个

#### 聚类标签映射
- 将有限的聚类代表点结果扩展到全域网格
- 对未参与聚类的网格点标记为-1（如海洋区域）
- 保持原有聚类标签的空间连续性

这一步骤确保了优化结果能够在全球尺度上进行空间可视化和进一步分析。

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import numpy as np
import pandas as pd

# ============== 路径 ==============
base_dir = r"D:\OneDrive\POTSED\03 Cluster"
pattern  = os.path.join(base_dir, "clustered_global_*_*.csv")

# ============== 工具函数 ==============
fname_re = re.compile(
    r"^clustered_global_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
    flags=re.IGNORECASE
)

def snap_to_grid(vals, step=0.25, decimals=2):
    """将数值对齐到 step 网格（如 0.25°），再四舍五入到 decimals 位小数。"""
    vals = np.asarray(vals, dtype=float)
    snapped = np.round(vals / step) * step
    return np.round(snapped, decimals)

def mode_series(s):
    """返回众数（若多个并列，取第一个）。"""
    m = s.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan

# ============== 构建全域 0.25° 网格 ==============
lat_vals = np.round(np.arange(-90, 90 + 0.25, 0.25), 2)
lon_vals = np.round(np.arange(-180, 180 + 0.25, 0.25), 2)
grid = pd.MultiIndex.from_product(
    [lat_vals, lon_vals], names=['latitude', 'longitude']
).to_frame(index=False)

# ============== 批处理每个文件 ==============
files = sorted(glob.glob(pattern))
if not files:
    raise FileNotFoundError(f"未找到匹配文件：{pattern}")

for fp in files:
    fname = os.path.basename(fp)
    m = fname_re.match(fname)
    if not m:
        print(f"⚠ 跳过无法解析的文件名：{fname}")
        continue

    scenario = m.group("scenario")
    prod     = m.group("prod").lower()

    print(f"\n=== 处理：scenario={scenario} | product={prod} ===")

    df = pd.read_csv(fp)

    # 列名兼容：cluster / cluster_label
    if 'cluster_label' not in df.columns and 'cluster' in df.columns:
        df = df.rename(columns={'cluster': 'cluster_label'})

    # 检查必要列
    required = {'latitude', 'longitude', 'cluster_label'}
    missing = required - set(df.columns)
    if missing:
        print(f"  ⚠ 缺少必要列 {missing}，已跳过：{fname}")
        continue

    # 经纬度对齐到 0.25°
    df = df.copy()
    df['lat_q'] = snap_to_grid(df['latitude'].values, step=0.25, decimals=2)
    df['lon_q'] = snap_to_grid(df['longitude'].values, step=0.25, decimals=2)

    # 若同一网格有多个点（或来自多次聚类结果合并），取众数
    df_agg = (
        df.groupby(['lat_q', 'lon_q'], as_index=False)['cluster_label']
          .agg(mode_series)
          .dropna(subset=['cluster_label'])
    )

    # 与全域网格合并
    merged = grid.merge(
        df_agg,
        left_on=['latitude', 'longitude'],
        right_on=['lat_q', 'lon_q'],
        how='left'
    )

    # 填充未覆盖网格
    merged['cluster_label'] = merged['cluster_label'].fillna(-1).astype(int)

    # 附加元信息（方便追踪）
    merged.insert(0, 'scenario', scenario)
    merged.insert(1, 'product',  prod)

    # 清理多余临时列
    merged = merged.drop(columns=['lat_q', 'lon_q'])

    # 保存
    out_csv = os.path.join(base_dir, f"full_grid_clustered_{scenario}_{prod}.csv")
    merged.to_csv(out_csv, index=False)
    print(f"  ✔ 全域网格拼接完成：{out_csv}")

print("\n✅ 全部完成。")


### 6.2 空间可视化与结果分析

本节创建多层次的可视化分析，包括：

#### 全域聚类分布图
- 基于世界地理边界的聚类结果叠加显示
- 不同大洲用不同底色区分
- 聚类标签用独特颜色标识
- 支持高分辨率输出（300 DPI）

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# ============== Matplotlib 全局风格（NCC 风格 + 四边框） ==============
plt.rcParams.update({
    # 字体与分辨率
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    # 坐标轴与网格
    "axes.linewidth": 0.8,
    "axes.edgecolor": "black",
    "axes.grid": False,
    # 四周都要边框（显式开启）
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    # 刻度
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    # 图例
    "legend.frameon": True,
    "legend.framealpha": 0.9,
})

# ============== 路径（按需修改） ==============
shapefile = r"D:\OneDrive\POTSED\ne_110m_admin_0_countries.shp"
tif_path  = r"F:\Workspace\World_Region_025d.tif"

base_dir  = r"D:\OneDrive\POTSED\03 Cluster"
in_pat    = os.path.join(base_dir, "full_grid_clustered_*_*.csv")
out_dir   = os.path.join(base_dir, "maps_scatter")
os.makedirs(out_dir, exist_ok=True)

# ============== 解析文件名（scenario / product） ==============
fname_re = re.compile(
    r"^full_grid_clustered_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
    flags=re.IGNORECASE
)

# ============== 读世界边界 & 区域栅格（一次） ==============
gdf = gpd.read_file(shapefile)
world_4326 = gdf.to_crs("EPSG:4326")

with rasterio.open(tif_path) as src:
    world_region = src.read(1)  # 整数区域编码
    b = src.bounds
    nodata = src.nodata
extent_world = [b.left, b.right, b.bottom, b.top]

# 掩蔽 nodata
if nodata is not None:
    world_region_masked = np.ma.masked_equal(world_region, nodata)
else:
    world_region_masked = np.ma.array(world_region, mask=np.zeros_like(world_region, dtype=bool))

# ============== 区域编码 → 洲 映射（按你的区域编码定义来改） ==============
raster_continent_map = {
  0: "Europe", 1: "North America", 2: "North America", 3: "North America", 4: "North America",
  5: "South America", 6: "Asia", 7: "Oceania", 8: "Other",
  9: "Africa&Middle East", 10: "Africa&Middle East", 11: "Africa&Middle East", 12: "Africa&Middle East", 13: "Africa&Middle East",
  14: "South America", 15: "South America", 16: "South America", 17: "South America",
  18: "Europe", 19: "Asia", 20: "Europe", 21: "Europe", 22: "Europe", 23: "Asia",
  24: "Africa&Middle East", 25: "Asia", 26: "Asia", 27: "Asia", 28: "Asia", 29: "Asia", 30: "Asia", 31: "Asia"
}

# —— NPG 风格洲配色（色盲友好、印刷稳健）——
continent_colors = {
    "Europe":             "#4575B4",
    "Asia":               "#D73027",
    "Africa&Middle East": "#F39C12",
    "North America":      "#8E44AD",
    "South America":      "#1A9850",
    "Oceania":            "#9E9E9E",
    "Other":              (0, 0, 0, 0)  # 透明
}

# 依据栅格最大编码动态构造 colormap（底色）
valid_vals = world_region_masked.compressed() if np.ma.is_masked(world_region_masked) else world_region.ravel()
max_code = int(valid_vals.max()) if valid_vals.size else 0
cmap_list = []
for code in range(max_code + 1):
    cont = raster_continent_map.get(code, "Other")
    col = continent_colors.get(cont, (0, 0, 0, 0))
    cmap_list.append(col)
region_cmap = ListedColormap(cmap_list)

# ============== 可调参数 ==============
CONTINENT_ALPHA = 0.15      # 洲底图透明度（更淡，避免干扰点色）
COUNTRY_EDGE    = "0.35"     # 国界线颜色
COUNTRY_LW      = 0.4        # 国界线粗细
SPINE_LW        = 0.8        # 轴边框线宽
SPINE_COLOR     = "black"    # 轴边框颜色
POINT_EDGE      = "black"    # 散点描边颜色
POINT_EC_LW     = 0.05       # 散点描边线宽
POINT_ALPHA     = 0.92       # 散点透明度
MARKER          = "s"        # 圆点；也可 "s" 正方形

# ============== 离散配色：组合 tab20/tab20b/tab20c，确保色差清晰 ==============
def build_qualitative_colors(n):
    banks = [plt.cm.get_cmap("tab20"), plt.cm.get_cmap("tab20b"), plt.cm.get_cmap("tab20c")]
    cols = []
    for bank in banks:
        cols.extend([bank(i) for i in range(bank.N)])
    if n <= len(cols):
        return cols[:n]
    # 若簇过多则循环使用（如仍嫌相近，可手工替换部分色值）
    k = len(cols)
    return [cols[i % k] for i in range(n)]

# ============== 处理每个 full_grid_clustered_<scenario>_<prod>.csv ==============
files = sorted(glob.glob(in_pat))
if not files:
    raise FileNotFoundError(f"未找到 {in_pat}")

for fp in files:
    fname = os.path.basename(fp)
    m = fname_re.match(fname)
    if not m:
        print(f"⚠ 跳过无法解析的文件名：{fname}")
        continue
    scenario = m.group("scenario")
    prod     = m.group("prod").upper()

    df = pd.read_csv(fp)

    # 兼容列名：cluster 或 cluster_label
    if 'cluster_label' not in df.columns and 'cluster' in df.columns:
        df = df.rename(columns={'cluster': 'cluster_label'})

    # 检查必要列
    required_cols = ['latitude', 'longitude', 'cluster_label']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"⚠ {fname} 缺少列 {missing}，已跳过。")
        continue

    # 去除未分类(-1)
    df = df[df["cluster_label"] >= 0].copy()
    if df.empty:
        print(f"⚠ {fname} 全为未分类(-1)，已跳过出图。")
        continue

    # 计算网格解析度（用于自适应点大小）
    # 这里假设经纬度是规则网格；若不规则，使用中位差更稳健
    if df["latitude"].nunique() > 1:
        dy = np.median(np.diff(np.sort(df["latitude"].unique())))
    else:
        dy = 0.25
    if df["longitude"].nunique() > 1:
        dx = np.median(np.diff(np.sort(df["longitude"].unique())))
    else:
        dx = 0.25
    grid_area = max(dx * dy, 1e-6)  # 防止除零
    s = np.clip(BASE_SIZE_SCALE * grid_area, 10, 140)  # 点面积（pt^2）上下限

    # 聚类配色
    max_label = int(df["cluster_label"].max())
    n_clusters = max_label + 1
    cluster_colors = build_qualitative_colors(n_clusters)
    color_array = [cluster_colors[int(i)] for i in df["cluster_label"].to_numpy()]

    # 绘图
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.set_facecolor('white')

    # —— 四周边框
    for side in ["top", "right", "left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(SPINE_LW)
        ax.spines[side].set_edgecolor(SPINE_COLOR)

    # 洲底图（更淡）
    ax.imshow(
        world_region_masked,
        cmap=region_cmap,
        alpha=CONTINENT_ALPHA,
        origin='upper',
        extent=extent_world,
        interpolation='nearest'
    )

    # 国界线
    world_4326.boundary.plot(ax=ax, edgecolor=COUNTRY_EDGE, linewidth=COUNTRY_LW, zorder=1)

    # 带描边的散点图（聚类）
    sc = ax.scatter(
        df["longitude"].to_numpy(),
        df["latitude"].to_numpy(),
        s=0.5,
        c=color_array,
        alpha=POINT_ALPHA,
        marker=MARKER,
        edgecolors=POINT_EDGE,
        linewidths=POINT_EC_LW,
        zorder=2
    )

    # 图例（只显示 Cluster）
    handles = [mpatches.Patch(facecolor=cluster_colors[i], edgecolor='black',
                              linewidth=0.3, label=f"Cluster {i}") for i in range(n_clusters)]
    ncol_legend = 1 if n_clusters <= 12 else 2 if n_clusters <= 24 else 3
    leg = ax.legend(
        handles=handles,
        #title=f"{scenario} | {prod} — Clusters",
        loc='lower left',
        bbox_to_anchor=(0.015, 0.02),
        borderaxespad=0.8,
        fontsize=9,
        title_fontsize=10,
        frameon=True,
        ncol=ncol_legend,
        handlelength=1.0,
        columnspacing=0.8,
        borderpad=0.6,
        labelspacing=0.4
    )
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_edgecolor('0.8')
    leg.get_frame().set_alpha(0.95)

    # 轴与标题
    ax.set_xlim(extent_world[0], extent_world[1])
    ax.set_ylim(extent_world[2], extent_world[3])
    ax.set_xlabel('Longitude', labelpad=4)
    ax.set_ylabel('Latitude', labelpad=4)
    ax.set_title(f"Full-grid Clusters (Scatter, outlined) — {scenario} | {prod}", pad=8, fontsize=12)

    out_png = os.path.join(out_dir, f"full_grid_cluster_scatter_{scenario}_{prod}.png")
    plt.savefig(out_png, bbox_inches='tight')
    plt.close(fig)
    print(f"✔ Saved: {out_png}")

print("\n✅ 全部完成（散点描边版）。")


#### 中文绘制

In [ ]:
# -*- coding: utf-8 -*-
"""
功能：
- 底图：洲别底色（来自区域栅格）+ 国家边界
- 叠加：聚类结果以“带描边的散点”呈现（每个网格中心一个点），中文标题/坐标/图例
- 叠加：读取并绘制中位数节点（来自 median_points_summary_*.csv）
- 重点修复：中文字体正确渲染；点大小(s)按网格解析度自适应；不再被后续 rcParams 覆盖
使用：
- 修改路径 shapefile / tif_path / base_dir 后运行
- 输入文件名格式：
  full_grid_clustered_<scenario>_<ammonia|methanol|ftl>.csv（含 latitude/longitude/cluster_label）
  median_points_summary_<scenario>_<ammonia|methanol|ftl>.csv（含 latitude/longitude 或 lat/lon）
"""

import os
import re
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from matplotlib.lines import Line2D  # >>> NEW

# ===================== 让 Matplotlib 正确渲染中文（并嵌入字体） =====================
def enable_chinese_fonts(prefer_local_font_path=None):
    """
    让 Matplotlib 正确渲染中文，并在导出时嵌入字体。
    - prefer_local_font_path: 可选，本地字体文件路径（.ttf/.otf/.ttc）
    """
    candidates = [
        # Windows
        "Microsoft YaHei", "SimHei", "Source Han Sans CN", "HarmonyOS Sans SC",
        # macOS
        "PingFang SC", "STHeiti", "Heiti SC",
        # Linux / 开源
        "Noto Sans CJK SC", "Source Han Sans SC", "WenQuanYi Micro Hei",
    ]

    if prefer_local_font_path and os.path.exists(prefer_local_font_path):
        try:
            font_manager.fontManager.addfont(prefer_local_font_path)
            fam = FontProperties(fname=prefer_local_font_path).get_name()
            if fam:
                candidates.insert(0, fam)
        except Exception as e:
            print(f"[warn] 无法注册本地字体：{prefer_local_font_path}，原因：{e}")

    mpl.rcParams["font.sans-serif"] = candidates + mpl.rcParams.get("font.sans-serif", [])
    mpl.rcParams["font.family"] = "sans-serif"
    mpl.rcParams["axes.unicode_minus"] = False  # 负号正常显示

    # 导出时嵌入字体并保持文本可选
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"]  = 42
    mpl.rcParams["svg.fonttype"] = "none"

# ===================== 统一字号与粗细（新增） =====================
BASE_FONT = 26         # 全局基础字号
LABEL_FONT = 28        # 坐标轴标签
TITLE_FONT = 32        # 标题
TICK_FONT  = 24        # 刻度
LEGEND_FONT= 18        # 图例内容
LEGEND_TITLE_FONT = 22 # 图例标题

plt.rcParams.update({
    "font.size": BASE_FONT,              # 全局默认字号
    "axes.labelsize": LABEL_FONT,        # x/y 轴标签字号
    "axes.titlesize": TITLE_FONT,        # 标题字号
    "xtick.labelsize": TICK_FONT,        # x 轴刻度字号
    "ytick.labelsize": TICK_FONT,        # y 轴刻度字号
    "legend.fontsize":    LEGEND_FONT,   # 图例项字号
    "legend.title_fontsize": LEGEND_TITLE_FONT,
})


# ===================== Matplotlib 全局风格（不覆盖字体设置） =====================
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "black",
    "axes.grid": False,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
})

# 在所有 rcParams 更新之后，再启用中文字体（可传本地字体路径以确保稳定）
enable_chinese_fonts()

# ===================== 路径（按需修改） =====================
shapefile = r"D:\OneDrive\POTSED\ne_110m_admin_0_countries.shp"
tif_path  = r"F:\Workspace\World_Region_025d.tif"

base_dir  = r"D:\OneDrive\POTSED\03 Cluster"
in_pat    = os.path.join(base_dir, "full_grid_clustered_*_*.csv")
median_pat= os.path.join(base_dir, "median_points_summary_*.csv")  # >>> NEW
out_dir   = os.path.join(base_dir, "maps_scatter_cn")
os.makedirs(out_dir, exist_ok=True)

# ===================== 解析文件名（scenario / product） =====================
fname_re = re.compile(
    r"^full_grid_clustered_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
    flags=re.IGNORECASE
)

median_re = re.compile(  # >>> NEW
    r"^median_points_summary_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
    flags=re.IGNORECASE
)

# ===================== 读取世界边界 & 区域栅格（一次） =====================
gdf = gpd.read_file(shapefile)
world_4326 = gdf.to_crs("EPSG:4326")

with rasterio.open(tif_path) as src:
    world_region = src.read(1)  # 整数区域编码
    b = src.bounds
    nodata = src.nodata
extent_world = [b.left, b.right, b.bottom, b.top]

# 掩蔽 nodata
if nodata is not None:
    world_region_masked = np.ma.masked_equal(world_region, nodata)
else:
    world_region_masked = np.ma.array(world_region, mask=np.zeros_like(world_region, dtype=bool))

# ===================== 区域编码 → 洲 映射（按你的编码定义调整） =====================
raster_continent_map = {
  0: "Europe", 1: "North America", 2: "North America", 3: "North America", 4: "North America",
  5: "South America", 6: "Asia", 7: "Oceania", 8: "Other",
  9: "Africa&Middle East", 10: "Africa&Middle East", 11: "Africa&Middle East", 12: "Africa&Middle East", 13: "Africa&Middle East",
  14: "South America", 15: "South America", 16: "South America", 17: "South America",
  18: "Europe", 19: "Asia", 20: "Europe", 21: "Europe", 22: "Europe", 23: "Asia",
  24: "Africa&Middle East", 25: "Asia", 26: "Asia", 27: "Asia", 28: "Asia", 29: "Asia", 30: "Asia", 31: "Asia"
}

# —— 洲配色（色盲友好/印刷稳健）——
continent_colors = {
    "Europe":             "#4575B4",
    "Asia":               "#D73027",
    "Africa&Middle East": "#F39C12",
    "North America":      "#8E44AD",
    "South America":      "#1A9850",
    "Oceania":            "#9E9E9E",
    "Other":              (0, 0, 0, 0)  # 透明
}

# 动态构造 colormap（底色）
valid_vals = world_region_masked.compressed() if np.ma.is_masked(world_region_masked) else world_region.ravel()
max_code = int(valid_vals.max()) if valid_vals.size else 0
cmap_list = []
for code in range(max_code + 1):
    cont = raster_continent_map.get(code, "Other")
    col = continent_colors.get(cont, (0, 0, 0, 0))
    cmap_list.append(col)
region_cmap = ListedColormap(cmap_list)

# ===================== 可调参数 =====================
CONTINENT_ALPHA = 0.15
COUNTRY_EDGE    = "0.35"
COUNTRY_LW      = 0.4
SPINE_LW        = 0.8
SPINE_COLOR     = "black"
POINT_EDGE      = "black"
POINT_EC_LW     = 0.05
POINT_ALPHA     = 0.92
MARKER          = "s"        # 网格像元：方块
BASE_SIZE_SCALE = 0.5        # s ≈ BASE_SIZE_SCALE * (dx*dy)

# 中位数点样式（尽量醒目且不遮挡底色）  # >>> NEW
MED_MARKER      = "*"
MED_EDGE        = "black"
MED_LINEWIDTH   = 2
MED_FACE        = "white"     # 建议 'none'；若想更亮可用 'white' 或 'yellow'
MED_ALPHA       = 0.6
MED_SIZE_MIN    = 400.0       # 最小点面积（pt^2）

# ===================== 离散配色：tab20/tab20b/tab20c 组合 =====================
def build_qualitative_colors(n):
    banks = [plt.cm.get_cmap("tab20"), plt.cm.get_cmap("tab20b"), plt.cm.get_cmap("tab20c")]
    cols = []
    for bank in banks:
        cols.extend([bank(i) for i in range(bank.N)])
    if n <= len(cols):
        return cols[:n]
    k = len(cols)
    return [cols[i % k] for i in range(n)]

# ===================== 工具函数：读取对应情景/产品的中位数点 =====================  # >>> NEW
def load_median_points(base_dir, scenario, prod_lower):
    """
    优先匹配：median_points_summary_<scenario>_<prod>.csv
    退而匹配：文件名包含同产品且包含情景关键词的最佳匹配
    仍无则：尝试仅按产品匹配一个（如果只有一个）
    """
    files = sorted(glob.glob(os.path.join(base_dir, "median_points_summary_*.csv")))
    if not files:
        return None, "未发现任何 median_points_summary_*.csv"

    # 完全匹配
    exact_name = f"median_points_summary_{scenario}.csv"
    for fp in files:
        if os.path.basename(fp).lower() == exact_name.lower():
            return pd.read_csv(fp), f"已加载中位数点：{os.path.basename(fp)}"

    # 情景+产品模糊匹配
    candidates = []
    s_key = scenario.lower()
    p_key = prod_lower.lower()
    for fp in files:
        name = os.path.basename(fp).lower()
        if p_key in name and s_key in name:
            candidates.append(fp)
    if len(candidates) == 1:
        return pd.read_csv(candidates[0]), f"已加载中位数点（模糊情景+产品）：{os.path.basename(candidates[0])}"
    elif len(candidates) > 1:
        # 简单打分：越短越接近
        best = sorted(candidates, key=lambda x: len(os.path.basename(x)))[0]
        return pd.read_csv(best), f"已加载中位数点（多候选取最优）：{os.path.basename(best)}"

    # 仅按产品匹配（若只有一个）
    p_only = [fp for fp in files if prod_lower.lower() in os.path.basename(fp).lower()]
    if len(p_only) == 1:
        return pd.read_csv(p_only[0]), f"已加载中位数点（仅按产品）：{os.path.basename(p_only[0])}"

    return None, f"未找到匹配情景/产品的中位数点（scenario={scenario}, prod={prod_lower})"

# ===================== 处理每个 full_grid_clustered_<scenario>_<prod>.csv =====================
files = sorted(glob.glob(in_pat))
if not files:
    raise FileNotFoundError(f"未找到 {in_pat}")

for fp in files:
    fname = os.path.basename(fp)
    m = fname_re.match(fname)
    if not m:
        print(f"⚠ 跳过无法解析的文件名：{fname}")
        continue
    scenario = m.group("scenario")
    prod     = m.group("prod").upper()
    prod_lower = prod.lower()

    df = pd.read_csv(fp)

    # 兼容列名：cluster 或 cluster_label
    if 'cluster_label' not in df.columns and 'cluster' in df.columns:
        df = df.rename(columns={'cluster': 'cluster_label'})

    # 检查必要列
    required_cols = ['latitude', 'longitude', 'cluster_label']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"⚠ {fname} 缺少列 {missing}，已跳过。")
        continue

    # 去除未分类(-1)
    df = df[df["cluster_label"] >= 0].copy()
    if df.empty:
        print(f"⚠ {fname} 全为未分类(-1)，已跳过出图。")
        continue

    # 估计网格解析度（用于点大小自适应）
    if df["latitude"].nunique() > 1:
        dy = np.median(np.diff(np.sort(df["latitude"].unique())))
    else:
        dy = 0.25
    if df["longitude"].nunique() > 1:
        dx = np.median(np.diff(np.sort(df["longitude"].unique())))
    else:
        dx = 0.25
    grid_area = max(dx * dy, 1e-6)
    s = float(np.clip(BASE_SIZE_SCALE * grid_area, 18, 120))  # 点面积（pt^2），每像元  # 保留
    # >>> 原代码这里误用了 s=0.5，下面已修复为使用自适应 s <<<

    # 聚类配色
    max_label = int(df["cluster_label"].max())
    n_clusters = max_label + 1
    cluster_colors = build_qualitative_colors(n_clusters)
    color_array = [cluster_colors[int(i)] for i in df["cluster_label"].to_numpy()]

    # === 读取中位数点 ===  # >>> NEW
    med_df, med_msg = load_median_points(base_dir, scenario, prod_lower)
    if med_df is None:
        print(f"[提示] {med_msg}")
    else:
        print(f"[信息] {med_msg}")
        # 兼容列名
        col_map = {}
        if "latitude" not in med_df.columns and "lat" in med_df.columns:
            col_map["lat"] = "latitude"
        if "longitude" not in med_df.columns and "lon" in med_df.columns:
            col_map["lon"] = "longitude"
        if col_map:
            med_df = med_df.rename(columns=col_map)

        # 必要列检查
        if not {"latitude", "longitude"}.issubset(set(med_df.columns)):
            print(f"[警告] 中位数点文件缺少 latitude/longitude 列，跳过叠加。")
            med_df = None
        else:
            # 去重（有时会有重复行）
            med_df = med_df.drop_duplicates(subset=["latitude", "longitude"]).copy()

    # 绘图
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.set_facecolor('white')

    # 四周边框
    for side in ["top", "right", "left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(SPINE_LW)
        ax.spines[side].set_edgecolor(SPINE_COLOR)

    # 洲底图（更淡）
    ax.imshow(
        world_region_masked,
        cmap=region_cmap,
        alpha=CONTINENT_ALPHA,
        origin='upper',
        extent=extent_world,
        interpolation='nearest'
    )

    # 国界线
    world_4326.boundary.plot(ax=ax, edgecolor=COUNTRY_EDGE, linewidth=COUNTRY_LW, zorder=1)

    # 带描边的散点图（聚类）
    ax.scatter(
        df["longitude"].to_numpy(),
        df["latitude"].to_numpy(),
        s=0.5,                           # <<< 修复：使用自适应点大小
        c=color_array,                 # 高区分度离散配色
        alpha=POINT_ALPHA,
        marker=MARKER,
        edgecolors=POINT_EDGE,
        linewidths=POINT_EC_LW,
        zorder=2
    )

    # 叠加中位数节点（若有）  # >>> NEW
    # === 中位数点颜色按所属类型一致 ===
    if med_df is not None and not med_df.empty:
        # 若中位数点没有 cluster_label，则用经纬度与主 df 对齐获取
        if "cluster_label" not in med_df.columns:
            med_df = med_df.merge(
                df[["latitude", "longitude", "cluster_label"]],
                on=["latitude", "longitude"],
                how="left"
            )

        # 构造与 cluster_colors 对应的面颜色
        if "cluster_label" in med_df.columns:
            med_colors = [
                cluster_colors[int(c)] if pd.notna(c) else "white"
                for c in med_df["cluster_label"].to_numpy()
            ]
        else:
            med_colors = ["white"] * len(med_df)


    if med_df is not None and not med_df.empty:
        # 中位数点大小：在像元点基础上放大两倍，且不低于阈值
        s_med = max(MED_SIZE_MIN, 5 * s)
        ax.scatter(
            med_df["longitude"].to_numpy(),
            med_df["latitude"].to_numpy(),
            s=s_med,
            facecolors=med_colors,            # <<<<< 关键：面颜色随类型而变
            edgecolors=MED_EDGE,
            linewidths=MED_LINEWIDTH,
            marker=MED_MARKER,
            alpha=MED_ALPHA,
            zorder=3
        )

    # 图例（中文）
    handles = [mpatches.Patch(facecolor=cluster_colors[i], edgecolor='black',
                              linewidth=0.3, label=f"类型 {i+1}") for i in range(n_clusters)]

    # 为中位数点添加代理图例项  # >>> NEW
    if med_df is not None and not med_df.empty:
        handles.append(
            Line2D([0], [0],
                   marker=MED_MARKER, linestyle='',
                   markerfacecolor='none',
                   markeredgecolor=MED_EDGE,
                   markeredgewidth=0.7,
                   markersize=16,  # 仅图例外观
                   label="中位数节点")
        )

    ncol_legend = 1 if n_clusters <= 12 else 2 if n_clusters <= 24 else 3
    leg = ax.legend(
        handles=handles,
        loc='lower left',
        bbox_to_anchor=(0.015, 0.02),
        borderaxespad=0.8,
        frameon=True,
        ncol=ncol_legend,
        handlelength=1.0,
        columnspacing=0.8,
        borderpad=0.6,
        labelspacing=0.4
    )
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_edgecolor('0.8')
    leg.get_frame().set_alpha(0.95)

    # 中文坐标与标题
    ax.set_xlim(extent_world[0], extent_world[1])
    ax.set_ylim(extent_world[2], extent_world[3])
    ax.set_xlabel('经度', labelpad=4)
    ax.set_ylabel('纬度', labelpad=4)
    ax.set_title(f"全球聚类散点（描边）— 情景：{scenario}｜产品：{prod}", pad=8, fontsize=12)

    # 导出
    out_png = os.path.join(out_dir, f"full_grid_cluster_scatter_{scenario}_{prod}.png")
    plt.savefig(out_png, bbox_inches='tight')
    plt.close(fig)
    print(f"✔ 已保存：{out_png}")

print("\n✅ 全部完成（中文渲染 + 散点描边 + 中位数节点叠加）。")


### 6.3 聚类间技术配置比较

#### 面板分析方法
为了深入理解不同聚类的技术选择特征，采用面板比较分析：

- **按聚类分组**: 每个聚类生成独立的技术配置面板
- **大洲间对比**: 在同一聚类内比较不同大洲的技术偏好
- **百分比标准化**: 使用相对比例而非绝对值，便于跨区域比较
- **多指标并列**: 同时展示5个关键技术维度的选择模式

#### 技术选择指标
1. **可再生能源组合**: 风电与光伏的相对比重
2. **电解技术路线**: 碱性(AE)与质子交换膜(PEM)电解槽选择
3. **储能配比**: 电池储能与总发电能力的匹配关系
4. **储能时长**: 电池储存容量与功率容量的比值
5. **氢储存策略**: 氢储存能力与电解制氢能力的平衡

这种分析方法能够识别出具有相似技术偏好的地理区域，为全球化技术部署提供指导。

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# NCC 风格：全局样式
# =========================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "axes.labelsize": 8.5,
    "axes.titlesize": 9.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 0,
    "ytick.minor.size": 1.5,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "legend.fontsize": 8,
})

# =========================
# 路径
# =========================
base_dir = r"D:\OneDrive\POTSED\03 Cluster"
in_pattern = os.path.join(base_dir, "clustered_by_continent_*_*.csv")
out_root = os.path.join(base_dir, "MatchPanels")
os.makedirs(out_root, exist_ok=True)

# 解析文件名
fname_re = re.compile(
    r"^clustered_by_continent_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
    flags=re.IGNORECASE
)

# =========================
# 参数与常量
# =========================
continents_order = ["Asia","Europe","Africa&Middle East","North America","South America","Oceania"]

eff_AE       = 0.65
eff_PEM      = 0.70
kwh_per_kgH2 = 33.33

# —— 为不同“种类的图”指定不同配色（Nature 家族常用色系）——
PANEL_COLORS = {
    'Wind vs PV':            ('#4DBBD5', '#E64B35'),  # 蓝 vs 朱红
    'AE vs PEM':             ('#3C5488', '#F39B7F'),  # 深紫 vs 橙粉
    'Battery vs Wind+PV':    ('#00A087', '#7E6148'),  # 祖母绿 vs 棕
    'BattStor vs Wind+PV':   ('#8491B4', '#91D1C2'),  # 灰蓝 vs 青绿
    'H2 vs Electrolyzer':    ('#B09C85', '#EFC000'),  # 米驼 vs 金黄
}

# 统一列名映射
COLMAP = {
    "methanol": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
    },
    "ftl": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
    },
    "ammonia": {
        "wind": "wind_capacity",
        "pv": "pv_capacity",
        "bat_en": "battery_energy_capacity",
        "bat_st": "battery_storage_capacity",
        "ae": "electrolyzer_capacity_AE",
        "pem": "electrolyzer_capacity_PEM",
        "h2_store": "hydrogen_storage_capacity",  # *33.33 转 kgH2 等效
    }
}

# 面板定义
PANEL_SPECS = [
    ('Wind vs PV',              ['wind'],           ['pv'],        'Wind',       'PV'),
    ('AE vs PEM',               ['ae'],             ['pem'],       'AE',         'PEM'),
    ('Battery vs Wind+PV',      ['bat_en'],         ['wind','pv'], 'Battery',    'Wind+PV'),
    ('BattStor vs Wind+PV',     ['bat_st'],         ['wind','pv'], 'BattStor',   'Wind+PV'),
    ('H2 vs Electrolyzer',      [],                 [],            'H2 Storage', 'Electrolyzer'),
]

# =========================
# 工具函数
# =========================
def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = 0.0
    return df

def to_common_frame(df, prod_key):
    m = COLMAP[prod_key]
    out = df.copy()
    need = [m[k] for k in m]
    out = ensure_cols(out, need)

    out['_wind']    = out[m['wind']]
    out['_pv']      = out[m['pv']]
    out['_bat_en']  = out[m['bat_en']]
    out['_bat_st']  = out[m['bat_st']]
    out['_ae']      = out[m['ae']]
    out['_pem']     = out[m['pem']]

    if prod_key == 'ammonia':
        out['_h2_store']  = out[m['h2_store']] * kwh_per_kgH2
        out['_elec_h2eq'] = out['_ae'] * eff_AE + out['_pem'] * eff_PEM
    else:
        out['_h2_store']  = out[m['h2_store']]
        out['_elec_h2eq'] = (out['_ae'] * eff_AE + out['_pem'] * eff_PEM) / kwh_per_kgH2
    return out

def _apply_ncc_axes_style(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='both', direction='in')
    ax.grid(axis='y', linestyle=(0, (3, 3)), color='0.85', linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))

def plot_cluster_panel(df_clust, prod_key, scenario, prod, cl, save_dir):
    fig, axes = plt.subplots(
        1, len(PANEL_SPECS),
        figsize=(len(PANEL_SPECS) * 3.8, 3.6),
        sharey=True,
        constrained_layout=True
    )

    for ax, (title, keysA, keysB, labA, labB) in zip(axes, PANEL_SPECS):
        tmp = df_clust.copy()

        if title != 'H2 vs Electrolyzer':
            A = np.zeros(len(tmp)); B = np.zeros(len(tmp))
            for k in keysA: A += tmp[f'_{k}'].values
            for k in keysB: B += tmp[f'_{k}'].values
            tmp[labA], tmp[labB] = A, B
        else:
            tmp[labA] = tmp['_h2_store'].values
            tmp[labB] = tmp['_elec_h2eq'].values

        agg = (tmp.groupby('continent')[[labA, labB]].sum()
                  .reindex(continents_order)
                  .fillna(0.0))
        total = (agg[labA] + agg[labB]).replace(0, np.nan)
        pct = agg.div(total, axis=0) * 100
        pct = pct.fillna(0.0)

        cA, cB = PANEL_COLORS[title]
        bar_kw = dict(edgecolor='black', linewidth=0.6, width=0.8)
        ax.bar(pct.index, pct[labA].values, color=cA, label=labA, **bar_kw)
        ax.bar(pct.index, pct[labB].values, bottom=pct[labA].values, color=cB, label=labB, **bar_kw)

        _apply_ncc_axes_style(ax)
        ax.set_title(f"{title}\nCluster {cl}", pad=4)
        ax.set_xlabel('')
        if ax is axes[0]:
            ax.set_ylabel('Percentage (%)')
        else:
            ax.set_ylabel('')
        plt.setp(ax.get_xticklabels(), rotation=0, ha='center')

        leg = ax.legend(loc='upper right', frameon=True, framealpha=0.95, borderpad=0.4)
        leg.get_frame().set_facecolor('white')
        leg.get_frame().set_edgecolor('0.85')

        ax.margins(x=0.02)

    # 使用“规范化后的情景名”显示标题
    fig.suptitle(f"{scenario} | {prod.upper()} — Matched Cluster {cl} Across Continents",
                 x=0.01, ha='left', y=1.02)

    os.makedirs(save_dir, exist_ok=True)
    outp_png = os.path.join(save_dir, f"{prod}_cluster_{cl}_panel.png")
    outp_pdf = os.path.join(save_dir, f"{prod}_cluster_{cl}_panel.pdf")
    fig.savefig(outp_png, dpi=300, bbox_inches='tight')
    fig.savefig(outp_pdf, bbox_inches='tight')
    plt.close(fig)
    print(f"✔ Saved: {outp_png}")

# ============== 情景名规范化：把 'Regional...' 统一视为 'Region...' ==============
def normalize_scenario(name: str) -> str:
    # 仅将 'Regional' 替换为 'Region'（大小写不敏感），例如 RegionalRE -> RegionRE
    return re.sub(r'(?i)Regional', 'Region', name)

# =========================
# 主流程
# =========================
files = sorted(glob.glob(in_pattern))
if not files:
    raise FileNotFoundError(f"未找到输入文件：{in_pattern}")

for fp in files:
    fname = os.path.basename(fp)
    m = fname_re.match(fname)
    if not m:
        print(f"⚠ 跳过无法解析的文件名：{fname}")
        continue

    raw_scenario = m.group('scenario')
    prod_key     = m.group('prod').lower()
    prod_disp    = prod_key

    # —— 核心改动：将 Regional* 归并为 Region* 同一情景名 ——
    scenario_norm = normalize_scenario(raw_scenario)

    # 输出目录使用“规范化后的情景名”
    save_dir = os.path.join(out_root, scenario_norm)
    os.makedirs(save_dir, exist_ok=True)

    df = pd.read_csv(fp)
    if 'continent' not in df.columns or 'cluster_label' not in df.columns:
        print(f"⚠ {fname} 缺少必要列（continent / cluster_label），已跳过。")
        continue
    df = df[df['continent'].isin(continents_order) & (df['cluster_label'] >= 0)].copy()
    if df.empty:
        print(f"⚠ {fname} 过滤后为空，跳过。")
        continue

    df_std = to_common_frame(df, prod_key)

    clusters = sorted(df_std['cluster_label'].unique())
    for cl in clusters:
        sub = df_std[df_std['cluster_label'] == cl]
        if sub.empty:
            continue
        # 注意：标题中也用规范化后的情景名（便于阅读一致）
        plot_cluster_panel(sub, prod_key, scenario_norm, prod_disp, cl, save_dir)


#### 分大洲绘图

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# NCC 风格：全局样式
# =========================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "axes.labelsize": 8.5,
    "axes.titlesize": 9.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 0,
    "ytick.minor.size": 1.5,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "legend.fontsize": 8,
})

# =========================
# 路径
# =========================
base_dir = r"D:\OneDrive\POTSED\03 Cluster"

# in_pattern = os.path.join(base_dir, "clustered_by_continent_*_*.csv")

# # 解析文件名
# fname_re = re.compile(
#     r"^clustered_by_continent_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
#     flags=re.IGNORECASE
# )


out_root = os.path.join(base_dir, "MatchPanels")
os.makedirs(out_root, exist_ok=True)


# =========================
# 参数与常量
# =========================
continents_order = ["Asia","Europe","Africa&Middle East","North America","South America","Oceania"]
# 固定显示的 cluster 顺序（如需更多，改这里即可）
clusters_order = [0, 1, 2, 3]

eff_AE       = 0.65
eff_PEM      = 0.70
kwh_per_kgH2 = 33.33

# —— 为不同“种类的图”指定不同配色（Nature 家族常用色系）——
PANEL_COLORS = {
    'Wind vs PV':            ('#4DBBD5', '#E64B35'),  # 蓝 vs 朱红
    'AE vs PEM':             ('#3C5488', '#F39B7F'),  # 深紫 vs 橙粉
    'Battery vs Wind+PV':    ('#00A087', '#7E6148'),  # 祖母绿 vs 棕
    'BattStor vs Wind+PV':   ('#8491B4', '#91D1C2'),  # 灰蓝 vs 青绿
    'H2 vs Electrolyzer':    ('#B09C85', '#EFC000'),  # 米驼 vs 金黄
}

# 统一列名映射
COLMAP = {
    "methanol": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
    },
    "ftl": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
    },
    "ammonia": {
        "wind": "wind_capacity",
        "pv": "pv_capacity",
        "bat_en": "battery_energy_capacity",
        "bat_st": "battery_storage_capacity",
        "ae": "electrolyzer_capacity_AE",
        "pem": "electrolyzer_capacity_PEM",
        "h2_store": "hydrogen_storage_capacity",  # *33.33 转换等效
    }
}

# 面板定义
PANEL_SPECS = [
    ('Wind vs PV',              ['wind'],           ['pv'],        'Wind',       'PV'),
    ('AE vs PEM',               ['ae'],             ['pem'],       'AE',         'PEM'),
    ('Battery vs Wind+PV',      ['bat_en'],         ['wind','pv'], 'Battery',    'Wind+PV'),
    ('BattStor vs Wind+PV',     ['bat_st'],         ['wind','pv'], 'BattStor',   'Wind+PV'),
    ('H2 vs Electrolyzer',      [],                 [],            'H2 Storage', 'Electrolyzer'),
]

# =========================
# 工具函数
# =========================
def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = 0.0
    return df

def to_common_frame(df, prod_key):
    m = COLMAP[prod_key]
    out = df.copy()
    need = [m[k] for k in m]
    out = ensure_cols(out, need)

    out['_wind']    = out[m['wind']]
    out['_pv']      = out[m['pv']]
    out['_bat_en']  = out[m['bat_en']]
    out['_bat_st']  = out[m['bat_st']]
    out['_ae']      = out[m['ae']]
    out['_pem']     = out[m['pem']]

    if prod_key == 'ammonia':
        # 将 H2 存储容量换算为“电量等效”（或与电解规模对齐的单位）
        out['_h2_store']  = out[m['h2_store']] * kwh_per_kgH2
        out['_elec_h2eq'] = out['_ae'] * eff_AE + out['_pem'] * eff_PEM
    else:
        # 其它产品：H2 容量保持原单位，电解规模折算为“kgH2 等效”
        out['_h2_store']  = out[m['h2_store']]
        out['_elec_h2eq'] = (out['_ae'] * eff_AE + out['_pem'] * eff_PEM) / kwh_per_kgH2
    return out

def _apply_ncc_axes_style(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='both', direction='in')
    ax.grid(axis='y', linestyle=(0, (3, 3)), color='0.85', linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))

# ===== 新：按“单一大洲”出一张图；每个子图横轴为 cluster 0–3 =====
def plot_continent_panel(df_cont, scenario, prod, continent, save_dir):
    fig, axes = plt.subplots(
        1, len(PANEL_SPECS),
        figsize=(len(PANEL_SPECS) * 3.8, 3.6),
        sharey=True,
        constrained_layout=True
    )

    for ax, (title, keysA, keysB, labA, labB) in zip(axes, PANEL_SPECS):
        tmp = df_cont.copy()

        # 计算 A、B
        if title != 'H2 vs Electrolyzer':
            A = np.zeros(len(tmp)); B = np.zeros(len(tmp))
            for k in keysA: A += tmp[f'_{k}'].values
            for k in keysB: B += tmp[f'_{k}'].values
            tmp[labA], tmp[labB] = A, B
        else:
            tmp[labA] = tmp['_h2_store'].values
            tmp[labB] = tmp['_elec_h2eq'].values

        # 按 cluster 汇总到 0,1,2,3；缺失补 0
        agg = (tmp.groupby('cluster_label')[[labA, labB]].sum()
                  .reindex(clusters_order)
                  .fillna(0.0))

        total = (agg[labA] + agg[labB]).replace(0, np.nan)
        pct = agg.div(total, axis=0) * 100
        pct = pct.fillna(0.0)

        cA, cB = PANEL_COLORS[title]
        bar_kw = dict(edgecolor='black', linewidth=0.6, width=0.8)
        x = np.arange(len(clusters_order))

        ax.bar(x, pct[labA].values, color=cA, label=labA, **bar_kw)
        ax.bar(x, pct[labB].values, bottom=pct[labA].values, color=cB, label=labB, **bar_kw)

        _apply_ncc_axes_style(ax)
        ax.set_title(title, pad=4)
        ax.set_xlabel('')
        if ax is axes[0]:
            ax.set_ylabel('Percentage (%)')
        else:
            ax.set_ylabel('')

        ax.set_xticks(x, [f'Cluster {i}' for i in clusters_order])
        leg = ax.legend(loc='upper right', frameon=True, framealpha=0.95, borderpad=0.4)
        leg.get_frame().set_facecolor('white')
        leg.get_frame().set_edgecolor('0.85')
        ax.margins(x=0.02)

    fig.suptitle(f"{scenario} | {prod.upper()} — {continent}",
                 x=0.01, ha='left', y=1.02)

    os.makedirs(save_dir, exist_ok=True)
    safe_cont = continent.replace(' ', '_')
    outp_png = os.path.join(save_dir, f"{prod}_continent_{safe_cont}_panel.png")
    outp_pdf = os.path.join(save_dir, f"{prod}_continent_{safe_cont}_panel.pdf")
    fig.savefig(outp_png, dpi=300, bbox_inches='tight')
    fig.savefig(outp_pdf, bbox_inches='tight')
    plt.close(fig)
    print(f"✔ Saved: {outp_png}")

# ============== 情景名规范化：把 'Regional...' 统一视为 'Region...' ==============
def normalize_scenario(name: str) -> str:
    return re.sub(r'(?i)Regional', 'Region', name)

# =========================
# 主流程
# =========================
files = sorted(glob.glob(in_pattern))
if not files:
    raise FileNotFoundError(f"未找到输入文件：{in_pattern}")

for fp in files:
    fname = os.path.basename(fp)
    m = fname_re.match(fname)
    if not m:
        print(f"⚠ 跳过无法解析的文件名：{fname}")
        continue

    raw_scenario = m.group('scenario')
    prod_key     = m.group('prod').lower()
    prod_disp    = prod_key

    # 归并情景名
    scenario_norm = normalize_scenario(raw_scenario)

    # 输出目录（可在情景目录下再分一个“ByContinent”子目录，以免与旧图混淆）
    save_dir = os.path.join(out_root, scenario_norm, "ByContinent")
    os.makedirs(save_dir, exist_ok=True)

    df = pd.read_csv(fp)
    if 'continent' not in df.columns or 'cluster_label' not in df.columns:
        print(f"⚠ {fname} 缺少必要列（continent / cluster_label），已跳过。")
        continue

    # 基础过滤
    df = df[df['continent'].isin(continents_order) & (df['cluster_label'] >= 0)].copy()
    if df.empty:
        print(f"⚠ {fname} 过滤后为空，跳过。")
        continue

    # 标准化列
    df_std = to_common_frame(df, prod_key)

    # 逐大洲出图；每个子图横轴固定为 cluster 0–3
    for cont in continents_order:
        sub = df_std[df_std['continent'] == cont]
        if sub.empty:
            print(f"• {fname} 在 {cont} 无数据，跳过该洲。")
            continue
        plot_continent_panel(sub, scenario_norm, prod_disp, cont, save_dir)


#### 分Cluster绘图

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# NCC 风格：全局样式
# =========================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "axes.labelsize": 8.5,
    "axes.titlesize": 9.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 0,
    "ytick.minor.size": 1.5,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "legend.fontsize": 8,
})

# =========================
# 路径
# =========================
base_dir = r"D:\OneDrive\POTSED\Cluster"
in_pattern = os.path.join(base_dir, "clustered_by_continent_*_*.csv")
out_root = os.path.join(base_dir, "MatchPanels")
os.makedirs(out_root, exist_ok=True)

# 解析文件名
fname_re = re.compile(
    r"^clustered_by_continent_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
    flags=re.IGNORECASE
)

# =========================
# 参数与常量
# =========================
continents_order = ["Asia", "Europe", "Africa&Middle East", "North America", "South America", "Oceania"]
clusters_order   = [0, 1, 2, 3]

eff_AE       = 0.65
eff_PEM      = 0.70
kwh_per_kgH2 = 33.33

PANEL_COLORS = {
    'Wind vs PV':            ('#4DBBD5', '#E64B35'),
    'AE vs PEM':             ('#3C5488', '#F39B7F'),
    'Battery vs Wind+PV':    ('#00A087', '#7E6148'),
    'BattStor vs Wind+PV':   ('#8491B4', '#91D1C2'),
    'H2 vs Electrolyzer':    ('#B09C85', '#EFC000'),
}

COLMAP = {
    "methanol": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
    },
    "ftl": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
    },
    "ammonia": {
        "wind": "wind_capacity",
        "pv": "pv_capacity",
        "bat_en": "battery_energy_capacity",
        "bat_st": "battery_storage_capacity",
        "ae": "electrolyzer_capacity_AE",
        "pem": "electrolyzer_capacity_PEM",
        "h2_store": "hydrogen_storage_capacity",
    }
}

PANEL_SPECS = [
    ('Wind vs PV',              ['wind'],           ['pv'],        'Wind',       'PV'),
    ('AE vs PEM',               ['ae'],             ['pem'],       'AE',         'PEM'),
    ('Battery vs Wind+PV',      ['bat_en'],         ['wind','pv'], 'Battery',    'Wind+PV'),
    ('BattStor vs Wind+PV',     ['bat_st'],         ['wind','pv'], 'BattStor',   'Wind+PV'),
    ('H2 vs Electrolyzer',      [],                 [],            'H2 Storage', 'Electrolyzer'),
]

# =========================
# 工具函数
# =========================
def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = 0.0
    return df

def to_common_frame(df, prod_key):
    m = COLMAP[prod_key]
    out = df.copy()
    need = [m[k] for k in m]
    out = ensure_cols(out, need)

    out['_wind']    = out[m['wind']]
    out['_pv']      = out[m['pv']]
    out['_bat_en']  = out[m['bat_en']]
    out['_bat_st']  = out[m['bat_st']]
    out['_ae']      = out[m['ae']]
    out['_pem']     = out[m['pem']]

    if prod_key == 'ammonia':
        out['_h2_store']  = out[m['h2_store']] * kwh_per_kgH2
        out['_elec_h2eq'] = out['_ae'] * eff_AE + out['_pem'] * eff_PEM
    else:
        out['_h2_store']  = out[m['h2_store']]
        out['_elec_h2eq'] = (out['_ae'] * eff_AE + out['_pem'] * eff_PEM) / kwh_per_kgH2
    return out

def _apply_ncc_axes_style(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='both', direction='in')
    ax.grid(axis='y', linestyle=(0, (3, 3)), color='0.85', linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))

def _pair_sums(df_sub, title, keysA, keysB):
    if title == 'H2 vs Electrolyzer':
        A_sum = float(df_sub['_h2_store'].sum())
        B_sum = float(df_sub['_elec_h2eq'].sum())
    else:
        A_sum = sum(float(df_sub[f'_{k}'].sum()) for k in keysA)
        B_sum = sum(float(df_sub[f'_{k}'].sum()) for k in keysB)
    return A_sum, B_sum

def _panel_percentages_for_cluster(df_cont_one_cluster):
    rows = []
    for title, keysA, keysB, labA, labB in PANEL_SPECS:
        A_sum, B_sum = _pair_sums(df_cont_one_cluster, title, keysA, keysB)
        total = A_sum + B_sum
        if total <= 0:
            A_pct = B_pct = 0.0
        else:
            A_pct = 100.0 * A_sum / total
            B_pct = 100.0 * B_sum / total
        rows.append((title, labA, labB, A_pct, B_pct))
    out = pd.DataFrame(rows, columns=['panel', 'labA', 'labB', 'A_pct', 'B_pct']).set_index('panel')
    return out

def _wrap_label(s: str, width: int = 12) -> str:
    return textwrap.fill(s, width=width, break_long_words=False, break_on_hyphens=False)

# =========================
# 作图：每个大洲一张（4 个 cluster），不画标题；所有“间距”显式可控
# =========================
def plot_continent_4clusters(
    df_cont, scenario, prod, continent, save_dir,
    # —— cluster 之间的间隔（占一个子图宽度的比例）——
    cluster_gap_rel=0.35,       # 0.35 表示“间隔列”宽度是“轴列”的 35%
    # —— 每个 cluster 子图的像素宽（越大越宽松）——
    per_cluster_width=4.2,      # 英寸
    fig_extra_width=1.2,        # 额外给一点边缘空间
    fig_height=3.8,
    # —— 组内间距：分组步长和柱宽占比 —— 
    group_stride=1.4,           # 相邻 panel 类别中心的距离
    bar_frac=0.58,              # 柱宽 = stride * bar_frac（越小柱越瘦，组内间距越大）
    edge_pad=0.8,               # 两端与 y 轴的留白（以 stride 为单位）
    # —— 轴标签设置 —— 
    label_wrap=12,
    xlabel_pad=4,
):
    plt.close('all')  # 防止残留

    n_clusters = len(clusters_order)
    # GridSpec 列：轴/间隔/轴/间隔/.../轴
    width_ratios = []
    for i in range(n_clusters):
        width_ratios.append(1.0)
        if i != n_clusters - 1:
            width_ratios.append(cluster_gap_rel)
    ncols = len(width_ratios)

    # 计算画布宽度
    fig_w = per_cluster_width * (n_clusters + cluster_gap_rel * (n_clusters - 1)) + fig_extra_width

    fig = plt.figure(figsize=(fig_w, fig_height))
    gs = fig.add_gridspec(1, ncols, width_ratios=width_ratios, wspace=0.0,
                          left=0.06, right=0.99, bottom=0.18, top=0.99)

    panels = [p[0] for p in PANEL_SPECS]
    n_pan = len(panels)
    # 组内 x 位置：加大 stride 提升组间距
    x = np.arange(n_pan) * group_stride
    bar_width = group_stride * bar_frac

    # 先建第一个轴，后续 sharey 对齐
    ax_list = []

    for ci, cl in enumerate(clusters_order):
        col_idx = ci * 2  # 0,2,4,6 是轴列；1,3,5 是“间隔列”
        if ci == 0:
            ax = fig.add_subplot(gs[0, col_idx])
        else:
            ax = fig.add_subplot(gs[0, col_idx], sharey=ax_list[0])
        ax_list.append(ax)

        sub = df_cont[df_cont['cluster_label'] == cl]
        if sub.empty:
            stats = pd.DataFrame({
                'labA': [p[3] for p in PANEL_SPECS],
                'labB': [p[4] for p in PANEL_SPECS],
                'A_pct': np.zeros(n_pan),
                'B_pct': np.zeros(n_pan),
            }, index=panels)
        else:
            stats = _panel_percentages_for_cluster(sub)

        A_vals = stats.loc[panels, 'A_pct'].values
        B_vals = stats.loc[panels, 'B_pct'].values

        A_colors = [PANEL_COLORS[p][0] for p in panels]
        B_colors = [PANEL_COLORS[p][1] for p in panels]

        ax.bar(x, A_vals, color=A_colors, edgecolor='black', linewidth=0.6, width=bar_width)
        ax.bar(x, B_vals, bottom=A_vals, color=B_colors, edgecolor='black', linewidth=0.6, width=bar_width)

        _apply_ncc_axes_style(ax)
        ax.set_title("")                   # 不画子图标题
        ax.set_xlabel('')

        if ci == 0:
            ax.set_ylabel('Percentage (%)')
        else:
            ax.set_ylabel('')

        # 组间距和两端留白：用 xlim 精准控制
        xmin = x[0]  - group_stride * edge_pad
        xmax = x[-1] + group_stride * edge_pad
        ax.set_xlim(xmin, xmax)

        # x 轴刻度与标签
        ax.set_xticks(x)
        wrapped = [_wrap_label(p, width=label_wrap) for p in panels]
        ax.set_xticklabels(wrapped, rotation=0, ha='center')
        ax.tick_params(axis='x', pad=xlabel_pad)

    # —— 确保没有 figure 级标题/文字 —— #
    st = getattr(fig, "_suptitle", None)
    if st is not None:
        try:
            st.remove()
        except Exception:
            st.set_visible(False)
        fig._suptitle = None
    for txt in list(fig.texts):
        txt.remove()

    # 保存
    os.makedirs(save_dir, exist_ok=True)
    safe_cont = continent.replace(' ', '_')
    outp_png = os.path.join(save_dir, f"{prod}_continent_{safe_cont}_4clusters_row.png")
    outp_pdf = os.path.join(save_dir, f"{prod}_continent_{safe_cont}_4clusters_row.pdf")
    fig.savefig(outp_png, dpi=300, bbox_inches='tight')
    fig.savefig(outp_pdf, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✔ Saved: {outp_png}")

# ============== 情景名规范化 ==============
def normalize_scenario(name: str) -> str:
    return re.sub(r'(?i)Regional', 'Region', name)

# =========================
# 主流程
# =========================
def main():
    files = sorted(glob.glob(in_pattern))
    if not files:
        raise FileNotFoundError(f"未找到输入文件：{in_pattern}")

    for fp in files:
        fname = os.path.basename(fp)
        m = fname_re.match(fname)
        if not m:
            print(f"⚠ 跳过无法解析的文件名：{fname}")
            continue

        raw_scenario = m.group('scenario')
        prod_key     = m.group('prod').lower()
        prod_disp    = prod_key
        scenario_norm = normalize_scenario(raw_scenario)

        save_dir = os.path.join(out_root, scenario_norm, "ByContinent_4Clusters_Row")
        os.makedirs(save_dir, exist_ok=True)

        df = pd.read_csv(fp)
        if 'continent' not in df.columns or 'cluster_label' not in df.columns:
            print(f"⚠ {fname} 缺少必要列（continent / cluster_label），已跳过。")
            continue

        df = df[df['continent'].isin(continents_order) & (df['cluster_label'] >= 0)].copy()
        if df.empty:
            print(f"⚠ {fname} 过滤后为空，跳过。")
            continue

        df_std = to_common_frame(df, prod_key)

        for cont in continents_order:
            sub_cont = df_std[df_std['continent'] == cont]
            if sub_cont.empty:
                print(f"• {fname} 在 {cont} 无数据，跳过该洲。")
                continue

            # —— 这里按你的偏好设定“更松”的默认值 —— 
            plot_continent_4clusters(
                sub_cont, scenario_norm, prod_disp, cont, save_dir,
                cluster_gap_rel=0.05,     # cluster 之间更宽
                per_cluster_width=4.6,    # 每个 cluster 更宽
                fig_extra_width=1.4,
                fig_height=3.9,
                group_stride=3.5,         # 组间更大
                bar_frac=0.8,            # 柱更瘦，组内更松
                edge_pad=0.6,             # 柱与坐标轴更远
                label_wrap=12,
                xlabel_pad=5,
            )

if __name__ == "__main__":
    main()


#### 中文绘制

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# NCC 风格：全局样式（中文 + mathtext）
# =========================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "Arial", "Helvetica", "DejaVu Sans"],
    "axes.unicode_minus": False,           # 避免负号方块
    "mathtext.fontset": "dejavusans",      # H$_2$ 等可用
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "axes.labelsize": 8.5,
    "axes.titlesize": 9.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 0,
    "ytick.minor.size": 1.5,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "legend.fontsize": 8,
})

# =========================
# 路径
# =========================
base_dir = r"D:\OneDrive\POTSED\03 Cluster"
in_pattern = os.path.join(base_dir, "clustered_by_continent_*_*.csv")
out_root = os.path.join(base_dir, "MatchPanels")
os.makedirs(out_root, exist_ok=True)

# 解析文件名
fname_re = re.compile(
    r"^clustered_by_continent_(?P<scenario>.+)_(?P<prod>ammonia|methanol|ftl)\.csv$",
    flags=re.IGNORECASE
)

# =========================
# 参数与常量
# =========================
continents_order = ["Asia", "Europe", "Africa&Middle East", "North America", "South America", "Oceania"]
clusters_order   = [0, 1, 2]  # 三个 cluster

eff_AE       = 0.65
eff_PEM      = 0.70
kwh_per_kgH2 = 33.33
HOURS_PER_YEAR = 8760.0

# =========================
# 面板颜色（与你当前键一致）
# =========================
PANEL_COLORS = {
    '风电/光伏':       ('#4DBBD5', '#E64B35'),
    'AE/PEM':          ('#3C5488', '#F39B7F'),
    '电池/风+光':     ('#00A087', '#7E6148'),
    '储能/风+光':     ('#8491B4', '#91D1C2'),
    '氢储/电解槽':     ('#B09C85', '#EFC000'),
    '电解槽/DAC':     ('#2E6F95', '#CC79A7'),  # 深钢蓝 / 洋红紫
    '电解槽/合成':     ('#2F8F7F', '#D89000'),  # 深青绿 / 琥珀橙
}

# 统一列名映射（基础组件）
COLMAP = {
    "methanol": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
        # 合成（tpy）
        "syn_tpy": "methanol_synthesis_capacity_tpy",
    },
    "ftl": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
        # 合成（tpy）
        "syn_tpy": "FTL_synthesis_capacity_tpy",
    },
    "ammonia": {
        "wind": "wind_capacity",
        "pv": "pv_capacity",
        "bat_en": "battery_energy_capacity",
        "bat_st": "battery_storage_capacity",
        "ae": "electrolyzer_capacity_AE",
        "pem": "electrolyzer_capacity_PEM",
        "h2_store": "hydrogen_storage_capacity",
        # 合成（这里视为 t/h）
        "syn_th": "ammonia_synthesis_capacity",
    }
}

# =========================
# 面板标题（短标题）—— 氨不画 DAC
# =========================
def build_panel_specs(prod_key: str):
    base = [
        ('风电/光伏',   ['wind'],           ['pv'],        '风电',         '光伏'),
        ('AE/PEM',     ['ae'],             ['pem'],       'AE',           'PEM'),
        ('电池/风+光', ['bat_en'],         ['wind','pv'], '电池',         '风+光'),
        ('储能/风+光', ['bat_st'],         ['wind','pv'], '电池储能',     '风+光'),
        ('氢储/电解槽', [],                 [],            '氢存储(等效)', '电解槽(等效)'),
    ]
    if prod_key.lower() == "ammonia":
        return base + [('电解槽/合成', [], [], '电解槽', '合成')]
    else:
        return base + [('电解槽/DAC', [], [], '电解槽', 'DAC'),
                       ('电解槽/合成', [], [], '电解槽', '合成')]

# =========================
# 标题别名规范化（避免因改名而失效）
# =========================
_PANEL_TITLE_ALIASES = {
    '氢储/电解槽': 'H2STORE_VS_ELEC',
    '电解槽/DAC':  'ELEC_VS_DAC',
    '电解槽/合成':  'ELEC_VS_SYN',
    # 兼容可能出现的旧写法（可按需扩展）
    '氢存储 vs 电解槽': 'H2STORE_VS_ELEC',
}

def _norm_panel_key(title: str) -> str:
    return _PANEL_TITLE_ALIASES.get(title, "")

# =========================
# 工具函数
# =========================
def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = 0.0
    return df

def _to_num(s):
    """字符串安全转数值，去逗号/空白；非法→NaN。"""
    if isinstance(s, str):
        s = s.replace(',', '').strip()
    return pd.to_numeric(s, errors='coerce')

def to_common_frame(df, prod_key):
    """
    统一内部字段 + 计算电解槽产氢速率（kgH2/h）；
    加载 DAC（kgCO2/h）与合成（t/h）列，并全量数值化。
    """
    m = COLMAP[prod_key]
    out = df.copy()

    # 基础列保证存在
    base_need = [m[k] for k in ['wind','pv','bat_en','bat_st','ae','pem','h2_store'] if k in m]
    out = ensure_cols(out, base_need)

    # 强制数值化（避免字符串/逗号）
    for k in ['wind','pv','bat_en','bat_st','ae','pem','h2_store']:
        if k in m and m[k] in out.columns:
            out[m[k]] = out[m[k]].map(_to_num).fillna(0.0)

    out['_wind']    = out[m['wind']]
    out['_pv']      = out[m['pv']]
    out['_bat_en']  = out[m['bat_en']]
    out['_bat_st']  = out[m['bat_st']]
    out['_ae']      = out[m['ae']]
    out['_pem']     = out[m['pem']]

    # “氢储/电解槽”面板的等效处理
    if prod_key == 'ammonia':
        out['_h2_store']  = out[m['h2_store']] * kwh_per_kgH2
        out['_elec_h2eq'] = out['_ae'] * eff_AE + out['_pem'] * eff_PEM
    else:
        out['_h2_store']  = out[m['h2_store']]
        out['_elec_h2eq'] = (out['_ae'] * eff_AE + out['_pem'] * eff_PEM) / kwh_per_kgH2

    # 统一：电解槽产氢速率（kgH2/h）
    out['_h2_rate_kgph'] = (out['_ae'] * eff_AE + out['_pem'] * eff_PEM) / kwh_per_kgH2
    out['_h2_rate_kgph'] = out['_h2_rate_kgph'].fillna(0.0)

    # DAC（kgCO2/h）
    if 'dac_capacity_kgCO2_per_h' in out.columns:
        out['_dac_kgco2ph'] = out['dac_capacity_kgCO2_per_h'].map(_to_num).fillna(0.0)
    else:
        out['_dac_kgco2ph'] = 0.0
        if prod_key != 'ammonia':
            print("⚠ 未发现列 'dac_capacity_kgCO2_per_h'，DAC 作为 0 处理。")

    # 合成（t/h）
    if prod_key == "ammonia":
        if 'ammonia_synthesis_capacity' in out.columns:
            out['_syn_tph'] = out['ammonia_synthesis_capacity'].map(_to_num).fillna(0.0)  # 已是 t/h
        else:
            out['_syn_tph'] = 0.0
            print("⚠ 未发现列 'ammonia_synthesis_capacity'，合成能力作为 0 处理。")
    elif prod_key == "methanol":
        col = m.get('syn_tpy', 'methanol_synthesis_capacity_tpy')
        if col in out.columns:
            out['_syn_tph'] = out[col].map(_to_num) 
            out['_syn_tph'] = out['_syn_tph'].fillna(0.0)
        else:
            out['_syn_tph'] = 0.0
            print("⚠ 未发现列 'methanol_synthesis_capacity_tpy'，合成能力作为 0 处理。")
    else:  # ftl
        col = m.get('syn_tpy', 'FTL_synthesis_capacity_tpy')
        if col in out.columns:
            out['_syn_tph'] = out[col].map(_to_num) 
            out['_syn_tph'] = out['_syn_tph'].fillna(0.0)
        else:
            out['_syn_tph'] = 0.0
            print("⚠ 未发现列 'FTL_synthesis_capacity_tpy'，合成能力作为 0 处理。")

    return out

def _apply_ncc_axes_style(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='both', direction='in')
    ax.grid(axis='y', linestyle=(0, (3, 3)), color='0.85', linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))

def _pair_sums(df_sub, title, keysA, keysB):
    """
    返回 (A_sum, B_sum)。
    三类“特殊面板”用规范键识别，避免标题改名失效。
    其它面板按 keysA/keysB 求和。
    """
    key = _norm_panel_key(title)

    if key == 'H2STORE_VS_ELEC':
        A_sum = float(df_sub['_h2_store'].sum())
        B_sum = float(df_sub['_elec_h2eq'].sum())

    elif key == 'ELEC_VS_DAC':
        A_sum = float(df_sub['_h2_rate_kgph'].sum())    # kg H2 / h
        B_sum = float(df_sub['_dac_kgco2ph'].sum())     # kg CO2 / h

    elif key == 'ELEC_VS_SYN':
        A_sum = float(df_sub['_h2_rate_kgph'].sum())    # kg H2 / h
        B_sum = float(df_sub['_syn_tph'].sum())         # t / h

    else:
        A_sum = sum(float(df_sub[f'_{k}'].sum()) for k in keysA)
        B_sum = sum(float(df_sub[f'_{k}'].sum()) for k in keysB)

    return A_sum, B_sum

def _panel_percentages_for_cluster(df_cont_one_cluster, panel_specs):
    """
    将 (A_sum, B_sum) 归一化为百分比（A% + B% = 100%）。
    """
    rows = []
    for title, keysA, keysB, labA, labB in panel_specs:
        A_sum, B_sum = _pair_sums(df_cont_one_cluster, title, keysA, keysB)
        total = A_sum + B_sum
        if total <= 0:
            A_pct = B_pct = 0.0
        else:
            A_pct = 100.0 * A_sum / total
            B_pct = 100.0 * B_sum / total
        rows.append((title, labA, labB, A_pct, B_pct))
    out = pd.DataFrame(rows, columns=['panel', 'labA', 'labB', 'A_pct', 'B_pct']).set_index('panel')
    return out

def _wrap_label(s: str, width: int = 10) -> str:
    return textwrap.fill(s, width=width, break_long_words=False, break_on_hyphens=False)

# =========================
# 作图：每个大洲一张（3 个 cluster，横向排布）
# =========================
def plot_continent_3clusters(
    df_cont, scenario, prod, continent, save_dir, panel_specs,
    cluster_gap_rel=0.10, per_cluster_width=4.8, fig_extra_width=1.2, fig_height=4.1,
    group_stride=2.8, bar_frac=0.72, edge_pad=0.60, label_wrap=10, xlabel_pad=5, xrot=0,
):
    plt.close('all')

    n_clusters = len(clusters_order)
    # 轴列与间隔列交替
    width_ratios = []
    for i in range(n_clusters):
        width_ratios.append(1.0)
        if i != n_clusters - 1:
            width_ratios.append(cluster_gap_rel)
    ncols = len(width_ratios)

    fig_w = per_cluster_width * (n_clusters + cluster_gap_rel * (n_clusters - 1)) + fig_extra_width

    fig = plt.figure(figsize=(fig_w, fig_height))
    gs = fig.add_gridspec(1, ncols, width_ratios=width_ratios, wspace=0.0,
                          left=0.06, right=0.995, bottom=0.22, top=0.99)

    panels = [p[0] for p in panel_specs]
    n_pan = len(panels)
    x = np.arange(n_pan) * group_stride
    bar_width = group_stride * bar_frac

    ax_list = []

    for ci, cl in enumerate(clusters_order):
        col_idx = ci * 2
        if ci == 0:
            ax = fig.add_subplot(gs[0, col_idx])
        else:
            ax = fig.add_subplot(gs[0, col_idx], sharey=ax_list[0])
        ax_list.append(ax)

        sub = df_cont[df_cont['cluster_label'] == cl]
        if sub.empty:
            stats = pd.DataFrame({
                'labA': [p[3] for p in panel_specs],
                'labB': [p[4] for p in panel_specs],
                'A_pct': np.zeros(n_pan),
                'B_pct': np.zeros(n_pan),
            }, index=panels)
        else:
            stats = _panel_percentages_for_cluster(sub, panel_specs)

        A_vals = stats.loc[panels, 'A_pct'].values
        B_vals = stats.loc[panels, 'B_pct'].values

        A_colors = [PANEL_COLORS.get(p, ('#777777','#BBBBBB'))[0] for p in panels]
        B_colors = [PANEL_COLORS.get(p, ('#777777','#BBBBBB'))[1] for p in panels]

        ax.bar(x, A_vals, color=A_colors, edgecolor='black', linewidth=0.6, width=bar_width)
        ax.bar(x, B_vals, bottom=A_vals, color=B_colors, edgecolor='black', linewidth=0.6, width=bar_width)

        _apply_ncc_axes_style(ax)
        ax.set_title("")   # 子图不写标题
        ax.set_xlabel('')

        if ci == 0:
            ax.set_ylabel('占比（%）')
        else:
            ax.set_ylabel('')

        xmin = x[0]  - group_stride * edge_pad
        xmax = x[-1] + group_stride * edge_pad
        ax.set_xlim(xmin, xmax)

        ax.set_xticks(x)
        wrapped = [_wrap_label(p, width=label_wrap) for p in panels]
        ax.set_xticklabels(wrapped, rotation=xrot, ha='center' if xrot == 0 else 'right')
        ax.tick_params(axis='x', pad=xlabel_pad)

    os.makedirs(save_dir, exist_ok=True)
    safe_cont = continent.replace(' ', '_')
    outp_png = os.path.join(save_dir, f"{prod}_continent_{safe_cont}_3clusters_row.png")
    outp_pdf = os.path.join(save_dir, f"{prod}_continent_{safe_cont}_3clusters_row.pdf")
    fig.savefig(outp_png, dpi=300, bbox_inches='tight')
    fig.savefig(outp_pdf, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✔ Saved: {outp_png}")

# ============== 情景名规范化（Regional* -> Region*） ==============#
def normalize_scenario(name: str) -> str:
    return re.sub(r'(?i)Regional', 'Region', name)

# =========================
# 主流程
# =========================
def main():
    files = sorted(glob.glob(in_pattern))
    if not files:
        raise FileNotFoundError(f"未找到输入文件：{in_pattern}")

    for fp in files:
        fname = os.path.basename(fp)
        m = fname_re.match(fname)
        if not m:
            print(f"⚠ 跳过无法解析的文件名：{fname}")
            continue

        raw_scenario = m.group('scenario')
        prod_key     = m.group('prod').lower()
        scenario_norm = normalize_scenario(raw_scenario)

        save_dir = os.path.join(out_root, scenario_norm, "ByContinent_3Clusters_Row")
        os.makedirs(save_dir, exist_ok=True)

        df = pd.read_csv(fp)
        if 'continent' not in df.columns or 'cluster_label' not in df.columns:
            print(f"⚠ {fname} 缺少必要列（continent / cluster_label），已跳过。")
            continue

        df = df[df['continent'].isin(continents_order) & (df['cluster_label'] >= 0)].copy()
        if df.empty:
            print(f"⚠ {fname} 过滤后为空，跳过。")
            continue

        df_std = to_common_frame(df, prod_key)
        panel_specs = build_panel_specs(prod_key)  # 氨不含 DAC

        for cont in continents_order:
            sub_cont = df_std[df_std['continent'] == cont]
            if sub_cont.empty:
                print(f"• {fname} 在 {cont} 无数据，跳过该洲。")
                continue

            plot_continent_3clusters(
                sub_cont, scenario_norm, prod_key, cont, save_dir, panel_specs,
                cluster_gap_rel=0.08,     # 轴间距
                per_cluster_width=4.8,
                fig_extra_width=1.2,
                fig_height=4.2,
                group_stride=2.8,         # 组距
                bar_frac=0.72,            # 柱更瘦
                edge_pad=0.60,            # 两端留白
                label_wrap=10,
                xlabel_pad=5,
                xrot=0,                   # 如仍重叠，可设 10~15
            )

if __name__ == "__main__":
    main()


#### 全球绘制

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# NCC 风格：全局样式（中文 + mathtext）
# =========================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "Arial", "Helvetica", "DejaVu Sans"],
    "axes.unicode_minus": False,
    "mathtext.fontset": "dejavusans",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "axes.labelsize": 8.5,
    "axes.titlesize": 9.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 0,
    "ytick.minor.size": 1.5,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "legend.fontsize": 8,
})

# =========================
# 路径与文件名匹配（中位数汇总）
# =========================
base_dir = r"D:\OneDrive\POTSED\03 Cluster"
out_root = os.path.join(base_dir, "MatchPanels")
os.makedirs(out_root, exist_ok=True)

in_pattern = os.path.join(base_dir, "median_points_summary_*.csv")
fname_re = re.compile(r"^median_points_summary_(?P<scenario>.+)\.csv$", flags=re.IGNORECASE)

# =========================
# 常量
# =========================
clusters_order   = [0, 1, 2]  # 三个 cluster
eff_AE, eff_PEM  = 0.65, 0.70
kwh_per_kgH2     = 33.33
HOURS_PER_YEAR   = 8760.0

# =========================
# 面板颜色（与现有配色统一）
# =========================
PANEL_COLORS = {
    '风电/光伏':   ('#4DBBD5', '#E64B35'),
    'AE/PEM':      ('#3C5488', '#F39B7F'),
    '电池/风+光': ('#00A087', '#7E6148'),
    '储能/风+光': ('#8491B4', '#91D1C2'),
    '氢储/电解槽': ('#B09C85', '#EFC000'),
    '电解槽/DAC': ('#2E6F95', '#CC79A7'),
    '电解槽/合成': ('#2F8F7F', '#D89000'),
}

# 统一列名映射（按产品）
COLMAP = {
    "ammonia": {
        "wind": "wind_capacity",
        "pv": "pv_capacity",
        "bat_en": "battery_energy_capacity",
        "bat_st": "battery_storage_capacity",
        "ae": "electrolyzer_capacity_AE",
        "pem": "electrolyzer_capacity_PEM",
        "h2_store": "hydrogen_storage_capacity",
        "syn_th": "ammonia_synthesis_capacity",  # t/h
    },
    "methanol": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
        "syn_tpy": "methanol_synthesis_capacity_tpy",  # t/y
    },
    "ftl": {
        "wind": "wind_capacity_kW",
        "pv": "pv_capacity_kW",
        "bat_en": "battery_energy_capacity_kW",
        "bat_st": "battery_storage_capacity_kWh",
        "ae": "electrolyzer_capacity_AE_kW",
        "pem": "electrolyzer_capacity_PEM_kW",
        "h2_store": "hydrogen_storage_capacity_kgH2",
        "syn_tpy": "FTL_synthesis_capacity_tpy",       # t/y
    },
}

# =========================
# 面板标题（短标题）—— 氨不画 DAC
# =========================
def build_panel_specs(prod_key: str):
    base = [
        ('风电/光伏',   ['wind'],           ['pv'],        '风电',         '光伏'),
        ('AE/PEM',     ['ae'],             ['pem'],       'AE',           'PEM'),
        ('电池/风+光', ['bat_en'],         ['wind','pv'], '电池',         '风+光'),
        ('储能/风+光', ['bat_st'],         ['wind','pv'], '电池储能',     '风+光'),
        ('氢储/电解槽', [],                 [],            '氢存储(等效)', '电解槽(等效)'),
    ]
    if prod_key.lower() == "ammonia":
        return base + [('电解槽/合成', [], [], '电解槽', '合成')]
    else:
        return base + [('电解槽/DAC', [], [], '电解槽', 'DAC'),
                       ('电解槽/合成', [], [], '电解槽', '合成')]

# 标题别名 → 规范键
_PANEL_TITLE_ALIASES = {
    '氢储/电解槽': 'H2STORE_VS_ELEC',
    '电解槽/DAC':  'ELEC_VS_DAC',
    '电解槽/合成':  'ELEC_VS_SYN',
    '氢存储 vs 电解槽': 'H2STORE_VS_ELEC',
}
def _norm_panel_key(title: str) -> str:
    return _PANEL_TITLE_ALIASES.get(title, "")

# =========================
# 工具函数
# =========================
def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = 0.0
    return df

def _to_num(s):
    if isinstance(s, str):
        s = s.replace(',', '').strip()
    return pd.to_numeric(s, errors='coerce')

def to_common_frame(df, prod_key):
    """统一内部字段；计算电解槽产氢速率（kgH2/h）；构造 DAC（kgCO2/h）与合成（t/h）。"""
    prod_key = prod_key.lower()
    if prod_key not in COLMAP:
        raise ValueError(f"未支持的 product: {prod_key}")

    m = COLMAP[prod_key]
    out = df.copy()

    # 基础列保证存在 + 数值化
    base_need = [m[k] for k in ['wind','pv','bat_en','bat_st','ae','pem','h2_store'] if k in m]
    out = ensure_cols(out, base_need)
    for k in ['wind','pv','bat_en','bat_st','ae','pem','h2_store']:
        if k in m and m[k] in out.columns:
            out[m[k]] = out[m[k]].map(_to_num).fillna(0.0)

    out['_wind']    = out[m['wind']]
    out['_pv']      = out[m['pv']]
    out['_bat_en']  = out[m['bat_en']]
    out['_bat_st']  = out[m['bat_st']]
    out['_ae']      = out[m['ae']]
    out['_pem']     = out[m['pem']]

    # “氢储/电解槽”面板的等效处理
    if prod_key == 'ammonia':
        out['_h2_store']  = out[m['h2_store']] * kwh_per_kgH2
        out['_elec_h2eq'] = out['_ae'] * eff_AE + out['_pem'] * eff_PEM
    else:
        out['_h2_store']  = out[m['h2_store']]
        out['_elec_h2eq'] = (out['_ae'] * eff_AE + out['_pem'] * eff_PEM) / kwh_per_kgH2

    # 电解槽产氢速率（kgH2/h）
    out['_h2_rate_kgph'] = (out['_ae'] * eff_AE + out['_pem'] * eff_PEM) / kwh_per_kgH2
    out['_h2_rate_kgph'] = out['_h2_rate_kgph'].fillna(0.0)

    # DAC（kgCO2/h）
    if 'dac_capacity_kgCO2_per_h' in out.columns:
        out['_dac_kgco2ph'] = out['dac_capacity_kgCO2_per_h'].map(_to_num).fillna(0.0)
    else:
        out['_dac_kgco2ph'] = 0.0
        if prod_key != 'ammonia':
            print("⚠ 未发现列 'dac_capacity_kgCO2_per_h'，DAC 作为 0 处理。")

    # 合成（t/h）
    if prod_key == "ammonia":
        if 'ammonia_synthesis_capacity' in out.columns:
            out['_syn_tph'] = out['ammonia_synthesis_capacity'].map(_to_num).fillna(0.0)  # t/h
        else:
            out['_syn_tph'] = 0.0
            print("⚠ 未发现列 'ammonia_synthesis_capacity'，合成能力作为 0 处理。")
    elif prod_key == "methanol":
        col = m.get('syn_tpy', 'methanol_synthesis_capacity_tpy')
        if col in out.columns:
            out['_syn_tph'] = (out[col].map(_to_num)).fillna(0.0)  # t/h
        else:
            out['_syn_tph'] = 0.0
            print("⚠ 未发现列 'methanol_synthesis_capacity_tpy'，合成能力作为 0 处理。")
    else:  # ftl
        col = m.get('syn_tpy', 'FTL_synthesis_capacity_tpy')
        if col in out.columns:
            out['_syn_tph'] = (out[col].map(_to_num)).fillna(0.0)  # t/h
        else:
            out['_syn_tph'] = 0.0
            print("⚠ 未发现列 'FTL_synthesis_capacity_tpy'，合成能力作为 0 处理。")

    return out

def _apply_ncc_axes_style(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='both', direction='in')
    ax.grid(axis='y', linestyle=(0, (3, 3)), color='0.85', linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))

def _pair_sums(df_sub, title, keysA, keysB):
    """返回 (A_sum, B_sum)。特殊面板用规范键识别；其他按 keys 求和。"""
    key = _norm_panel_key(title)

    if key == 'H2STORE_VS_ELEC':
        A_sum = float(df_sub['_h2_store'].sum())
        B_sum = float(df_sub['_elec_h2eq'].sum())

    elif key == 'ELEC_VS_DAC':
        A_sum = float(df_sub['_h2_rate_kgph'].sum())    # kg H2 / h
        B_sum = float(df_sub['_dac_kgco2ph'].sum())     # kg CO2 / h

    elif key == 'ELEC_VS_SYN':
        A_sum = float(df_sub['_h2_rate_kgph'].sum())    # kg H2 / h
        B_sum = float(df_sub['_syn_tph'].sum())         # t / h

    else:
        A_sum = sum(float(df_sub[f'_{k}'].sum()) for k in keysA)
        B_sum = sum(float(df_sub[f'_{k}'].sum()) for k in keysB)
    return A_sum, B_sum

def _panel_percentages_for_cluster(df_one_cluster, panel_specs):
    """把每个面板的 A/B 总量转为百分比。"""
    rows = []
    for title, keysA, keysB, labA, labB in panel_specs:
        A_sum, B_sum = _pair_sums(df_one_cluster, title, keysA, keysB)
        total = A_sum + B_sum
        if total <= 0:
            A_pct = B_pct = 0.0
        else:
            A_pct = 100.0 * A_sum / total
            B_pct = 100.0 * B_sum / total
        rows.append((title, labA, labB, A_pct, B_pct))
    return pd.DataFrame(rows, columns=['panel', 'labA', 'labB', 'A_pct', 'B_pct']).set_index('panel')

def _wrap_label(s: str, width: int = 10) -> str:
    return textwrap.fill(s, width=width, break_long_words=False, break_on_hyphens=False)

# =========================
# 作图：全球一张（3 个 cluster，横向排布）
# =========================
def plot_global_3clusters(
    df_all, scenario, prod, save_dir, panel_specs,
    cluster_gap_rel=0.10, per_cluster_width=4.8, fig_extra_width=1.2, fig_height=4.1,
    group_stride=2.8, bar_frac=0.72, edge_pad=0.60, label_wrap=10, xlabel_pad=5, xrot=0,
):
    plt.close('all')

    n_clusters = len(clusters_order)
    width_ratios = []
    for i in range(n_clusters):
        width_ratios.append(1.0)
        if i != n_clusters - 1:
            width_ratios.append(cluster_gap_rel)
    ncols = len(width_ratios)

    fig_w = per_cluster_width * (n_clusters + cluster_gap_rel * (n_clusters - 1)) + fig_extra_width

    fig = plt.figure(figsize=(fig_w, fig_height))
    gs = fig.add_gridspec(1, ncols, width_ratios=width_ratios, wspace=0.0,
                          left=0.06, right=0.995, bottom=0.22, top=0.99)

    panels = [p[0] for p in panel_specs]
    n_pan = len(panels)
    x = np.arange(n_pan) * group_stride
    bar_width = group_stride * bar_frac

    ax_list = []
    for ci, cl in enumerate(clusters_order):
        col_idx = ci * 2
        if ci == 0:
            ax = fig.add_subplot(gs[0, col_idx])
        else:
            ax = fig.add_subplot(gs[0, col_idx], sharey=ax_list[0])
        ax_list.append(ax)

        sub = df_all[df_all['cluster_label'] == cl]
        if sub.empty:
            stats = pd.DataFrame({
                'labA': [p[3] for p in panel_specs],
                'labB': [p[4] for p in panel_specs],
                'A_pct': np.zeros(n_pan),
                'B_pct': np.zeros(n_pan),
            }, index=panels)
        else:
            stats = _panel_percentages_for_cluster(sub, panel_specs)

        A_vals = stats.loc[panels, 'A_pct'].values
        B_vals = stats.loc[panels, 'B_pct'].values

        A_colors = [PANEL_COLORS.get(p, ('#777777','#BBBBBB'))[0] for p in panels]
        B_colors = [PANEL_COLORS.get(p, ('#777777','#BBBBBB'))[1] for p in panels]

        ax.bar(x, A_vals, color=A_colors, edgecolor='black', linewidth=0.6, width=bar_width)
        ax.bar(x, B_vals, bottom=A_vals, color=B_colors, edgecolor='black', linewidth=0.6, width=bar_width)

        _apply_ncc_axes_style(ax)
        ax.set_title("")   # 子图不写标题
        ax.set_xlabel('')
        if ci == 0:
            ax.set_ylabel('占比（%）')
        else:
            ax.set_ylabel('')

        xmin = x[0]  - group_stride * edge_pad
        xmax = x[-1] + group_stride * edge_pad
        ax.set_xlim(xmin, xmax)

        ax.set_xticks(x)
        wrapped = [_wrap_label(p, width=label_wrap) for p in panels]
        ax.set_xticklabels(wrapped, rotation=xrot, ha='center' if xrot == 0 else 'right')
        ax.tick_params(axis='x', pad=xlabel_pad)

    os.makedirs(save_dir, exist_ok=True)
    outp_png = os.path.join(save_dir, f"{prod}_GLOBAL_3clusters_row.png")
    outp_pdf = os.path.join(save_dir, f"{prod}_GLOBAL_3clusters_row.pdf")
    fig.savefig(outp_png, dpi=300, bbox_inches='tight')
    fig.savefig(outp_pdf, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✔ Saved (Global): {outp_png}")

# ============== 情景名规范化（Regional* -> Region*） ==============#
def normalize_scenario(name: str) -> str:
    return re.sub(r'(?i)Regional', 'Region', name)

# =========================
# 主流程：读取 median_points_summary_*.csv → 每文件按产品画一张“全球图”
# =========================
def main():
    files = sorted(glob.glob(in_pattern))
    if not files:
        raise FileNotFoundError(f"未找到输入文件：{in_pattern}")

    for fp in files:
        fname = os.path.basename(fp)
        m = fname_re.match(fname)
        if not m:
            print(f"⚠ 跳过无法解析的文件名：{fname}")
            continue

        scenario_norm = normalize_scenario(m.group('scenario'))

        # 输出目录：按场景分
        save_root = os.path.join(out_root, f"{scenario_norm}_Median_Global")
        os.makedirs(save_root, exist_ok=True)

        # 读取
        df = pd.read_csv(fp)
        if 'cluster_label' not in df.columns or 'product' not in df.columns:
            print(f"⚠ {fname} 缺少必要列（cluster_label / product），已跳过。")
            continue

        # 仅保留三类簇
        df = df[df['cluster_label'].isin(clusters_order)].copy()
        if df.empty:
            print(f"⚠ {fname} 过滤后为空，跳过。")
            continue

        # 同一文件可能含多个产品 → 分产品各出一张“全球图”
        for prod_key in sorted(df['product'].str.lower().unique()):
            df_prod = df[df['product'].str.lower() == prod_key].copy()
            if df_prod.empty:
                continue

            try:
                df_std = to_common_frame(df_prod, prod_key)
            except Exception as e:
                print(f"⚠ {fname} | product={prod_key} 转换失败：{e}")
                continue

            panel_specs = build_panel_specs(prod_key)
            save_dir = os.path.join(save_root, f"Global_3Clusters_Row_{prod_key}")
            plot_global_3clusters(
                df_std, scenario_norm, prod_key, save_dir, panel_specs,
                cluster_gap_rel=0.08, per_cluster_width=4.8, fig_extra_width=1.2,
                fig_height=4.2, group_stride=2.8, bar_frac=0.72, edge_pad=0.60,
                label_wrap=10, xlabel_pad=5, xrot=0,
            )

if __name__ == "__main__":
    main()


RegionRE情景下绿色燃料技术配置特征的聚类分析
基于K-means聚类算法对全球绿色燃料生产技术配置的分析，本研究识别出三种典型的技术配置模式（聚类0、1、2），代表了不同地理区域和资源禀赋条件下的最优技术路径选择。以下分别对绿氨、绿醇和SAF（合成航空燃料）在2030年、2040年和2050年的技术配置演变特征进行深入分析。

绿氨（Green Ammonia）技术配置特征分析
绿氨生产系统在RegionRE情景下表现出显著的地域分化特征和明显的技术演进趋势。聚类0代表的亚洲高纬度地区（如中亚地区，68.5°N, 45.0°E）展现出以风电为主导的技术配置模式，风电装机容量在2030年达到21.74 MW，而光伏仅为1.70 MW，风光比例约为12.8:1，反映了该地区丰富的风能资源禀赋。电解槽配置以PEM技术为主导（17.26 MW），AE电解槽容量微乎其微（仅0.014 kW），显示出2030年PEM技术的主导地位。到2050年，该聚类的风电容量略降至20.11 MW，但光伏容量增加至2.30 MW，风光比例调整为8.7:1，电解槽总容量提升至18.27 MW，其中PEM技术仍占主导地位，表明随着技术成熟度提升，系统更趋向于技术组合的优化配置。

聚类1对应的非洲与中东地区（17.5°N, 18.25°E）呈现出风光并重的均衡配置特征。2030年风电容量15.32 MW，光伏11.67 MW，风光比例约为1.3:1，反映了该地区风光资源的相对均衡。电解槽配置中PEM（14.82 MW）略高于AE（0.18 kW），但相比聚类0，AE技术的应用比例显著提升。至2050年，该聚类的技术配置发生显著变化：风电容量降至14.53 MW，光伏容量增加至13.85 MW，风光比例进一步收敛至1.05:1；更为重要的是，AE电解槽容量大幅提升至2.32 MW，PEM容量相应调整至12.65 MW，AE技术占比从2030年的1.2%提升至2050年的15.5%，体现了低成本AE技术在成本优化压力下的逐步渗透。

聚类2代表的大洋洲地区（-28.25°S, 117.0°E）展现出独特的光伏主导型配置模式。2030年光伏容量高达22.91 MW，风电仅16.88 MW，风光比例为0.74:1，是三个聚类中唯一光伏占主导的配置。电解槽技术组合相对均衡，PEM（15.47 MW）与AE（0.086 kW）的配比类似于其他聚类。到2050年，该聚类发生了最为剧烈的技术重构：光伏容量激增至50.13 MW，风电大幅降至3.91 MW，风光比例调整为0.078:1，形成了极端的光伏主导模式；同时，AE电解槽容量显著提升至11.21 MW，PEM容量调整至13.84 MW，AE技术占比提升至44.8%，反映了在光照资源极其丰富的地区，低成本AE技术与光伏发电的协同优化效应。

绿醇（Green Methanol）技术配置特征分析
绿醇生产系统相比绿氨展现出更为复杂的技术集成特征，主要体现在CO₂捕获与合成工艺的耦合优化上。聚类0（亚洲地区，68.75°N, 80.0°E）在2030年采用了风电主导的配置模式，风电容量27.20 MW，光伏4.20 MW，风光比例为6.5:1，显著高于绿氨系统的风光配比。电解槽配置几乎完全依赖PEM技术（17.29 MW），AE容量近乎为零（4.16×10⁻⁶ MW），反映了绿醇系统对电解效率和响应速度的更高要求。DAC（直接空气碳捕获）容量为0.113 kg CO₂/h，合成装置容量为12.75 t/y，设备利用率分别达到80.9%和78.4%，显示出较高的系统协调性。到2050年，该聚类的技术配置发生显著调整：风电容量降至25.09 MW，光伏容量大幅降至0.087 MW，风光比例提升至289:1，形成了极端的风电主导模式；电解槽总容量提升至19.10 MW，仍以PEM技术为主，DAC容量微降至2.0×10⁻⁶ kg CO₂/h，但合成装置容量增加至13.11 t/y，系统整体向更高的技术集成度演进。

聚类1（亚洲中纬度地区，47.5°N, 54.0°E）展现出相对均衡的风光配置模式。2030年风电容量26.65 MW，光伏31.86 MW，风光比例为0.84:1，是绿醇系统中风光配置最为均衡的聚类。电解槽技术仍以PEM为主导（16.26 MW），DAC容量显著提升至0.389 kg CO₂/h，合成装置容量为13.22 t/y，设备利用率分别达到85.6%和75.6%，显示出在风光资源相对均衡地区的技术优化特征。到2050年，该聚类的配置进一步优化：风电容量降至22.62 MW，光伏容量增加至38.22 MW，风光比例调整为0.59:1，更加偏向光伏主导；电解槽总容量调整至17.63 MW，其中AE技术开始显著渗透（4.69 MW），占比达到26.6%，DAC容量微降，但合成装置容量略降至11.71 t/y，反映了技术成熟化过程中的系统重新平衡。

聚类2（非洲与中东地区，15.25°N, 24.0°E）呈现出光伏主导的配置特征。2030年光伏容量22.45 MW，风电24.00 MW，风光比例为1.07:1，接近均衡配置。电解槽配置几乎完全依赖PEM技术（14.10 MW），DAC容量极低（3.28×10⁻⁵ kg CO₂/h），合成装置容量为12.07 t/y，设备利用率达到82.9%，显示出在太阳能资源丰富地区的技术特点。到2050年，该聚类发生了最为显著的技术转型：风电容量大幅降至6.72 MW，光伏容量激增至49.43 MW，风光比例调整为0.136:1，形成了极端的光伏主导模式；电解槽配置发生根本性变化，AE技术容量达到14.59 MW，PEM容量降至10.15 MW，AE技术占比高达59.0%，成为主导技术，DAC容量略有提升，合成装置容量降至10.75 t/y，反映了在光照资源极其丰富的地区，低成本技术路径的优势逐步显现。

SAF（合成航空燃料）技术配置特征分析
SAF生产系统作为最为复杂的Power-to-X技术路径，在RegionRE情景下仅在2050年LowAE场景中出现，体现了其技术门槛高、成本敏感性强的特点。聚类0（北美地区，66.75°N, -97.75°W）采用了风电主导的配置模式，风电容量39.14 MW，光伏2.94 MW，风光比例高达13.3:1，显著超过绿氨和绿醇系统的风光配比，反映了SAF系统对稳定电力供应的更高要求。电解槽配置以PEM技术为绝对主导（47.59 MW），AE容量仅1.16 MW，占比2.4%，显示出SAF系统对电解效率的极高要求。DAC容量达到47.09 kg CO₂/h，合成装置容量为12.45 t/y，设备利用率分别达到80.3%和80.3%，显示出极高的系统协调性和技术集成度。

聚类1（亚洲地区，49.5°N, 52.25°E）展现出风电绝对主导的配置特征，风电容量高达88.53 MW，光伏仅4.59 MW，风光比例达到19.3:1，是所有技术路径中风电比例最高的配置。电解槽总容量44.19 MW，其中PEM技术占绝对主导地位（35.58 MW），AE技术容量8.61 MW，占比19.5%，相比聚类0有所提升但仍处于辅助地位。DAC容量显著提升至119.73 kg CO₂/h，合成装置容量为12.00 t/y，设备利用率分别达到83.3%和83.4%，显示出在风能资源极其丰富地区的技术优化配置。

聚类2（非洲与中东地区，15.25°N, 23.5°E）呈现出极端的光伏主导模式，光伏容量高达116.80 MW，风电仅16.02 MW，风光比例为0.137:1，是SAF系统中唯一的光伏主导配置。电解槽配置发生显著变化，AE技术容量达到32.30 MW，PEM容量29.29 MW，AE技术占比高达52.4%，首次在SAF系统中成为主导技术，反映了在光照资源极其丰富、电力成本极低的地区，低成本AE技术的经济优势能够克服其技术劣势。DAC容量相对较低（3.30 kg CO₂/h），合成装置容量为10.84 t/y，设备利用率达到92.2%，显示出在特定资源条件下的技术路径多样性。

技术演进趋势与政策启示
综合分析RegionRE情景下三种绿色燃料的技术配置演变，可以识别出以下关键趋势：首先，AE电解槽技术的渗透呈现出明显的时间梯度和地域分化特征，从2030年的边缘地位逐步演进至2050年的重要角色，特别是在光照资源丰富的地区，AE技术占比可达60%以上；其次，风光配比的优化呈现出明显的资源禀赋依赖性，高纬度地区保持风电主导，低纬度地区逐步向光伏主导转型；再次，技术系统的复杂性与产品复杂性正相关，SAF系统展现出最高的技术集成度和设备协调性要求；最后，地理位置对技术路径选择的影响在长期呈现放大趋势，资源禀赋优势地区的技术特化程度不断提升。这些发现为制定差异化的绿色燃料发展政策、优化全球产业布局提供了重要的科学依据。

In [ ]:
# -*- coding: utf-8 -*-
import os
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ===== 路径 =====
base_dir = r"D:\OneDrive\POTSED\03 Cluster"
out_root = os.path.join(base_dir, "MatchPanels")
os.makedirs(out_root, exist_ok=True)

# ===== 与主图一致的配色 =====
PANEL_COLORS = {
    '风电/光伏':   ('#4DBBD5', '#E64B35'),
    'AE/PEM':      ('#3C5488', '#F39B7F'),
    '电池/风+光': ('#00A087', '#7E6148'),
    '储能/风+光': ('#8491B4', '#91D1C2'),
    '氢储/电解槽': ('#B09C85', '#EFC000'),
    '电解槽/DAC': ('#2E6F95', '#CC79A7'),
    '电解槽/合成': ('#2F8F7F', '#D89000'),
}

def build_items(include_dac=True):
    items = []
    items += [("风电",   PANEL_COLORS['风电/光伏'][0]),
              ("光伏",   PANEL_COLORS['风电/光伏'][1])]
    items += [("AE",     PANEL_COLORS['AE/PEM'][0]),
              ("PEM",    PANEL_COLORS['AE/PEM'][1])]
    items += [("电池",     PANEL_COLORS['电池/风+光'][0]),
              ("风+光(A)", PANEL_COLORS['电池/风+光'][1])]
    items += [("储能",     PANEL_COLORS['储能/风+光'][0]),
              ("风+光(B)", PANEL_COLORS['储能/风+光'][1])]
    items += [("储氢",   PANEL_COLORS['氢储/电解槽'][0]),
              ("电解槽", PANEL_COLORS['氢储/电解槽'][1])]
    if include_dac:
        items += [("电解槽(与DAC)", PANEL_COLORS['电解槽/DAC'][0]),
                  ("DAC",         PANEL_COLORS['电解槽/DAC'][1])]
    items += [("电解槽(与合成)", PANEL_COLORS['电解槽/合成'][0]),
              ("合成",         PANEL_COLORS['电解槽/合成'][1])]
    return items

def save_legend_two_rows(png_path, pdf_path=None, include_dac=True, fontsize=10):
    items = build_items(include_dac)
    handles = [Patch(facecolor=c, edgecolor='black', linewidth=0.8) for _, c in items]
    labels  = [lab for lab, _ in items]

    # —— 强制两行：每行列数 = ceil(总项数 / 2)
    n_total = len(items)
    ncol = math.ceil(n_total / 2)

    # 画布尺寸：按列数自适应
    fig_w = max(6.0, ncol * 1.3)   # 每列约 1.3 英寸
    fig_h = 2.2                    # 两行固定高度更稳

    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Microsoft YaHei", "SimHei", "Arial", "Helvetica", "DejaVu Sans"],
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "legend.fontsize": fontsize,
    })

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.legend(
        handles, labels,
        ncol=ncol, loc='center',
        frameon=False,
        handlelength=1.6, handleheight=0.9,
        columnspacing=1.6, labelspacing=0.8, handletextpad=0.6,
    )
    fig.tight_layout(pad=0)
    fig.savefig(png_path, dpi=300, bbox_inches='tight', transparent=True)
    if pdf_path:
        fig.savefig(pdf_path, bbox_inches='tight', transparent=True)
    plt.close(fig)
    print("✔ 图例已保存：", png_path, ("& " + pdf_path if pdf_path else ""))

if __name__ == "__main__":
    # 甲醇/FTL：包含 DAC
    save_legend_two_rows(
        os.path.join(out_root, "legend_2rows_with_DAC.png"),
        os.path.join(out_root, "legend_2rows_with_DAC.pdf"),
        include_dac=True, fontsize=10
    )
    # 氨：不包含 DAC
    save_legend_two_rows(
        os.path.join(out_root, "legend_2rows_no_DAC.png"),
        os.path.join(out_root, "legend_2rows_no_DAC.pdf"),
        include_dac=False, fontsize=10
    )


### 6.4 结果集成与图像合成

#### 多面板图像合成
为了便于整体对比和汇报展示，将各个聚类的分析面板合成为单一图像：

- **垂直排列**: 按聚类顺序从上到下排列各面板
- **统一宽度**: 调整所有面板至相同宽度以保持视觉一致性  
- **高质量输出**: 保持原始300 DPI分辨率
- **自动间距**: 在面板间添加适当的视觉分隔

#### 最终输出
生成的合成图像包含所有聚类的完整技术配置比较，便于：
- 识别跨聚类的技术选择模式
- 发现大洲间的技术偏好差异
- 支持政策制定和投资决策
- 便于学术交流和成果展示

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
from PIL import Image

# 根目录：之前绘制面板图保存的位置
base_dir = r"D:\OneDrive\POTSED\Cluster\MatchPanels"

# 产物命名：每个情景 & 每个产品 单独输出一张合成图
out_name_tpl = "combined_rank_panels_{scenario}_{prod}.png"

# 匹配单簇面板文件名：<prod>_cluster_<k>_panel.png
fname_re = re.compile(
    r"^(?P<prod>ammonia|methanol|ftl)_cluster_(?P<cl>\d+)_panel\.png$",
    re.IGNORECASE
)
ALL_PRODUCTS = ["ammonia", "methanol", "ftl"]

def find_scenario_dirs(root):
    """递归查找包含 *_cluster_*_panel.png 的情景目录"""
    scen_dirs = set()
    for dirpath, dirnames, filenames in os.walk(root):
        if any(f.endswith("_panel.png") and "_cluster_" in f for f in filenames):
            scen_dirs.add(dirpath)
    return sorted(scen_dirs)

def list_images_for_product(scen_dir, product):
    """列出某情景目录下某产品的面板图，按 cluster 升序排序"""
    files = glob.glob(os.path.join(scen_dir, f"{product}_cluster_*_panel.png"))
    parsed = []
    for fp in files:
        name = os.path.basename(fp)
        m = fname_re.match(name)
        if not m:
            continue
        prod = m.group("prod").lower()
        if prod != product:
            continue
        cl = int(m.group("cl"))
        parsed.append((cl, fp))
    parsed.sort(key=lambda x: x[0])  # by cluster
    return [fp for _, fp in parsed]

def stack_vertical_center(images, out_path):
    """将若干图片竖向拼接；若宽度不同则水平居中"""
    if not images:
        return False

    imgs = [Image.open(fp).convert("RGBA") for fp in images]
    widths, heights = zip(*(im.size for im in imgs))
    max_w = max(widths)
    total_h = sum(heights)

    canvas = Image.new("RGBA", (max_w, total_h), (255, 255, 255, 255))

    y = 0
    for im in imgs:
        x = (max_w - im.width) // 2
        canvas.paste(im, (x, y), im)
        y += im.height

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    canvas.convert("RGB").save(out_path, dpi=(300, 300))
    return True

def main():
    scen_dirs = find_scenario_dirs(base_dir)
    if not scen_dirs:
        print(f"⚠ 未在 {base_dir} 找到任何情景子目录或面板图。")
        return

    for scen_dir in scen_dirs:
        scenario = os.path.basename(scen_dir)
        any_done = False
        for prod in ALL_PRODUCTS:
            imgs = list_images_for_product(scen_dir, prod)
            if not imgs:
                # 该情景无该产品，跳过
                continue
            out_png = os.path.join(scen_dir, out_name_tpl.format(scenario=scenario, prod=prod))
            ok = stack_vertical_center(imgs, out_png)
            if ok:
                any_done = True
                print(f"✔ 拼接完成：{out_png}")
            else:
                print(f"⚠ {scenario} | {prod} 拼接失败（无有效图片）。")
        if not any_done:
            print(f"⚠ {scenario} 没有匹配到任何产品的 *_cluster_*_panel.png，跳过。")

if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image
import matplotlib.image as mpimg

# ================ 调试 ================
VERBOSE = False  # 改成 True 可打印匹配细节

# ================= 路径 =================
base_dir   = r"D:\OneDrive\POTSED\03 Cluster"
map_dir    = os.path.join(base_dir, "maps")
panels_dir = os.path.join(base_dir, "MatchPanels")
heatmap_dir= r"D:\OneDrive\POTSED\8760Supply&Demand"
out_dir    = os.path.join(base_dir, "CombinedFigures")
os.makedirs(out_dir, exist_ok=True)

# ================= 大洲（显示名, safe名） =================
left_continents = [
    ("North America", "North_America"),
    ("South America", "South_America"),
    ("Africa&Middle East", "Africa&Middle_East"),  # 显示名含空格；safe 名保留 & 与 _
]
right_continents = [
    ("Europe", "Europe"),
    ("Asia", "Asia"),
    ("Oceania", "Oceania"),
]
clusters_order = [0, 1, 2, 3]

# ================ 通用规范化/变体 ================
def canonical_key(s: str) -> str:
    """规范化为匹配键：小写，只保留 a-z0-9。去掉空格、下划线、连字符、& 等。"""
    if s is None:
        return ""
    return re.sub(r'[^a-z0-9]+', '', s.lower())

def normalize_scenario_name(scenario: str) -> str:
    """Region/Regional 互换归一到 Region（大小写不敏感）"""
    return re.sub(r'(?i)regional', 'Region', scenario)

def continent_variants(display_name: str, safe_name: str):
    """生成一组常见写法的洲名变体，供匹配使用"""
    cands = set()
    raw = [display_name, safe_name]
    for name in raw:
        if not name:
            continue
        variants = set()
        variants.add(name)
        variants.add(name.replace('_', ' '))
        variants.add(name.replace('_', ''))
        variants.add(name.replace('&', ''))
        variants.add(name.replace('&', 'and'))
        variants.add(name.replace('&', ' and '))
        variants.add(name.replace('&', '_'))
        variants.add(name.replace(' ', '_'))
        variants.add(name.replace(' ', ''))
        # 组合一些常见替换
        variants.add(name.replace('&', '').replace('_', ' '))
        variants.add(name.replace('&', 'and').replace('_', ' '))
        variants.add(name.replace('&', '').replace(' ', '_'))
        for v in list(variants):
            cands.add(v.strip())
    # 返回两份：原始列表（用于尝试构造路径）、以及对应的 canonical 集合（用于扫描匹配）
    return list(cands), set(canonical_key(v) for v in cands if v)

# ================ 文件查找：地图 ================
def find_map_file(scenario, product):
    """鲁棒查找地图：先尝试直连模式，失败则扫描整个 maps 目录做规范化匹配"""
    scen_norm = normalize_scenario_name(scenario)
    scen_try = {
        scen_norm,
        scen_norm.replace('Region', 'Regional'),
        scen_norm.replace('Regional', 'Region')
    }
    prod_try = {product, product.lower(), product.upper(), product.title()}

    # 1) 直接尝试组合
    for s in scen_try:
        for p in prod_try:
            f = os.path.join(map_dir, f"full_grid_cluster_map_{s}_{p}.png")
            if os.path.exists(f):
                if VERBOSE: print("[map] direct hit:", f)
                return f

    # 2) 扫描匹配（更鲁棒）
    target_scen_key = canonical_key(scen_norm)
    target_prod_key = canonical_key(product)
    best = None
    for fp in glob.glob(os.path.join(map_dir, "full_grid_cluster_map_*_*.png")):
        name = os.path.basename(fp)[len("full_grid_cluster_map_"):-4]
        parts = name.split("_")
        if len(parts) < 2:
            continue
        scen_part = "_".join(parts[:-1])
        prod_part = parts[-1]
        if canonical_key(prod_part) != target_prod_key:
            continue
        if canonical_key(normalize_scenario_name(scen_part)) == target_scen_key:
            best = fp
            break
    if best and VERBOSE:
        print("[map] scan hit:", best)
    return best

# ================ 文件查找：面板（ByContinent_4Clusters_Row 优先） ================
def find_panel_image(scenario, product, continent_safe, continent_display=None):
    """鲁棒查找面板图：尝试多变体名与多目录；失败则扫描匹配"""
    scenario = normalize_scenario_name(scenario)
    # 生成洲名变体
    name_variants, canon_set = continent_variants(continent_display or continent_safe, continent_safe)

    # 1) 常见路径+常见洲名写法
    def path_candidates(nm):
        return [
            os.path.join(panels_dir, scenario, "ByContinent_4Clusters_Row",
                         f"{product}_continent_{nm}_4clusters_row.png"),
            os.path.join(panels_dir, scenario, "ByContinent",
                         f"{product}_continent_{nm}_panel.png"),
            os.path.join(panels_dir, scenario,
                         f"{product}_continent_{nm}_panel.png"),
        ]

    for nm in name_variants:
        for path in path_candidates(nm):
            if os.path.exists(path):
                if VERBOSE: print("[panel] direct hit:", path)
                return path

    # 2) 扫描该情景目录，依据 canonical_key 匹配
    base_dirs = [
        os.path.join(panels_dir, scenario, "ByContinent_4Clusters_Row"),
        os.path.join(panels_dir, scenario, "ByContinent"),
        os.path.join(panels_dir, scenario),
    ]
    target_prod_key = canonical_key(product)
    for bd in base_dirs:
        if not os.path.isdir(bd):
            continue
        for fp in glob.glob(os.path.join(bd, "*.png")):
            fname = os.path.basename(fp)
            # 只考虑 product_ 开头的
            if not fname.lower().startswith(product.lower() + "_"):
                continue
            # 提取 continent 部分（两种命名）
            m = re.match(rf"(?i)^{re.escape(product)}_continent_(.+?)_4clusters_row\.png$", fname)
            if not m:
                m = re.match(rf"(?i)^{re.escape(product)}_continent_(.+?)_panel\.png$", fname)
            if not m:
                continue
            cont_part = m.group(1)
            if canonical_key(cont_part) in canon_set:
                if VERBOSE: print("[panel] scan hit:", fp)
                return fp
    if VERBOSE:
        print("[panel] not found:", scenario, product, continent_safe, "variants=", name_variants)
    return None

# ================ 文件查找：热力图 ================
def find_heatmap_images(scenario, product, continent_display, clusters):
    """
    返回与 clusters 一一对应的路径列表（找不到的位置返回 None）。
    - 先尝试使用变体构造的精确路径
    - 若失败，扫描所有情景子目录，用 canonical 匹配 continent 段
    """
    # 生成洲名变体
    name_variants, canon_set = continent_variants(continent_display, continent_display)

    # 汇总所有可能的情景子目录
    scen_dirs = []
    if os.path.isdir(heatmap_dir):
        for sub in os.listdir(heatmap_dir):
            p = os.path.join(heatmap_dir, sub)
            if os.path.isdir(p):
                scen_dirs.append(p)

    out = []
    for cl in clusters:
        # 1) 直接尝试每个变体在所有情景目录的精确路径
        hit = None
        for nm in name_variants:
            filename = f"{product}_{nm}_cluster{cl}_annual_heatmap.png"
            for sdir in scen_dirs:
                cand = os.path.join(sdir, filename)
                if os.path.exists(cand):
                    hit = cand
                    if VERBOSE: print("[heatmap] direct hit:", cand)
                    break
            if hit:
                break

        # 2) 扫描匹配（更鲁棒）
        if not hit:
            target_prefix = f"{product}_".lower()
            target_suffix = f"_cluster{cl}_annual_heatmap.png".lower()
            for sdir in scen_dirs:
                for fp in glob.glob(os.path.join(sdir, "*.png")):
                    fname = os.path.basename(fp)
                    low = fname.lower()
                    if not low.startswith(target_prefix) or not low.endswith(target_suffix):
                        continue
                    # 抽出 continent 段： product_XXXX_clusterX_annual_heatmap.png
                    mid = fname[len(product) + 1: -len(f"_cluster{cl}_annual_heatmap.png")]
                    if canonical_key(mid) in canon_set:
                        hit = fp
                        if VERBOSE: print("[heatmap] scan hit:", fp)
                        break
                if hit:
                    break
        out.append(hit if hit else None)
        if not hit and VERBOSE:
            print("[heatmap] not found:", product, continent_display, f"cluster{cl}", "variants=", list(name_variants))
    return out

# ================ 图像读/显 ================
def load_img(path):
    if path is None or not os.path.exists(path):
        return None
    return np.array(Image.open(path))

# 保持原始长宽比显示
def imshow_keep_aspect(ax, img_array):
    if img_array is not None:
        ax.imshow(img_array)
        h, w = img_array.shape[:2]
        ax.set_box_aspect(h / w)
    else:
        ax.text(0.5, 0.5, "No image", ha="center", va="center", transform=ax.transAxes)
        ax.set_box_aspect(1.0)
    ax.axis("off")

# 轻微裁边（百分比），去掉导出图四周空白；不会改变内容比例
def crop_percent(img, top=0.0, bottom=0.0, left=0.0, right=0.0):
    if img is None:
        return None
    H, W = img.shape[:2]
    y0 = int(H * top); y1 = H - int(H * bottom)
    x0 = int(W * left); x1 = W - int(W * right)
    y0 = max(0, min(y0, H-1)); y1 = max(y0+1, min(y1, H))
    x0 = max(0, min(x0, W-1)); x1 = max(x0+1, min(x1, W))
    return img[y0:y1, x0:x1, ...]

# panel 等宽裁 4 份（再对每份做轻微上下裁边）
def split_panel_into_clusters(panel_img, n_cols=4,
                              left_trim_pct=0.008, right_trim_pct=0.008,
                              vtrim_top=0.01, vtrim_bottom=0.01):
    if panel_img is None:
        return [None] * n_cols
    H, W = panel_img.shape[:2]
    x0 = int(W * left_trim_pct)
    x1 = W - int(W * right_trim_pct)
    W_eff = max(1, x1 - x0)
    cols = []
    for i in range(n_cols):
        xs = x0 + int(i * W_eff / n_cols)
        xe = x0 + int((i + 1) * W_eff / n_cols)
        crop = panel_img[:, xs:xe, ...]
        crop = crop_percent(crop, vtrim_top, vtrim_bottom, 0.0, 0.0)
        cols.append(crop)
    return cols

# ================= 主绘图函数 =================
def create_combined_figure(
    scenario, product,
    # —— 外部（大洲块与块）留白 —— #
    outer_wspace=0.06, outer_hspace=0.04,
    # —— 内部（边框与内容）分边距 —— #
    inner_pad_top=0.012, inner_pad_bottom=0.0005,
    inner_pad_left=0.006, inner_pad_right=0.006,
    # —— 同块内相邻列间距（越小越贴） —— #
    col_gap=0.0,
    # —— 上排(panel)与下排(heatmap)的相对高度 —— #
    panel_heat_ratio=(1.60, 1.60),
    # —— 热力图裁边，去自带空白 —— #
    heat_vtrim_top=0.012, heat_vtrim_bottom=0.010,
    # —— 地区标题距边框的内缩距离（越大离边框越远） —— #
    region_label_offset=0.02,
    # —— 网格宽度分配（左右更宽，子图更大） —— #
    width_ratios=(2.7, 2.7, 2.7),
    figsize=(28, 16)
):
    print(f"\n=== 创建组合图：{scenario} | {product} ===")
    map_file = find_map_file(scenario, product)
    if not map_file:
        print(f"⚠ 未找到地图文件：full_grid_cluster_map_{scenario}_{product.upper()}.png（已做鲁棒匹配仍未命中）")
        return False
    map_img = mpimg.imread(map_file)
    if VERBOSE: print("[map] using:", map_file)

    # 外层 3×3（左｜地图｜右）
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = GridSpec(3, 3, figure=fig,
                  width_ratios=list(width_ratios),
                  height_ratios=[1, 1, 1])

    # 块与块之间的留白
    fig.set_constrained_layout_pads(w_pad=0.012, h_pad=0.012,
                                    wspace=outer_wspace, hspace=outer_hspace)

    # 中央地图（保持比例）
    ax_map = fig.add_subplot(gs[:, 1])
    ax_map.imshow(map_img)
    mh, mw = map_img.shape[:2]
    ax_map.set_box_aspect(mh / mw)
    ax_map.axis("off")
    # 地图标题（如果不需要可注释）
    ax_map.text(0.5, 1.0, f"{scenario} | {product.upper()} - Cluster Distribution",
                transform=ax_map.transAxes, ha="center", va="bottom",
                fontsize=14, fontweight="bold", clip_on=False)

    # —— 每个大洲块：容器轴 + inset_axes —— #
    def draw_block(subplot_spec, continent_display, continent_safe):
        # 1) 外框容器
        ax_frame = fig.add_subplot(subplot_spec)
        ax_frame.set_xticks([]); ax_frame.set_yticks([])
        ax_frame.set_facecolor('none')
        for side in ax_frame.spines.values():
            side.set_visible(True)
            side.set_linewidth(1.2)
            side.set_color("#333333")

        # 2) 内容区矩形（四周分边距）
        pad_l, pad_r = inner_pad_left, inner_pad_right
        pad_t, pad_b = inner_pad_top, inner_pad_bottom
        INNER_W = 1 - pad_l - pad_r
        INNER_H = 1 - pad_b - pad_t

        # 行高
        pr, hr = panel_heat_ratio
        row1_h = INNER_H * pr / (pr + hr)   # panel 行
        row2_h = INNER_H * hr / (pr + hr)   # heatmap 行

        # 列宽：四列等宽
        ncols = 4
        col_w = (INNER_W - (ncols - 1) * col_gap) / ncols

        # 3) 面板图：鲁棒查找 + 等宽切成四列
        panel_path = find_panel_image(scenario, product, continent_safe, continent_display)
        panel_img  = load_img(panel_path)
        if VERBOSE: print("[panel] for", continent_display, "->", panel_path)
        panel_slices = split_panel_into_clusters(
            panel_img, n_cols=4, left_trim_pct=0.006, right_trim_pct=0.006,
            vtrim_top=0.006, vtrim_bottom=0.006
        )

        # 4) 上排：panel 切片（与下排同列同宽）
        for j, img_slice in enumerate(panel_slices):
            x = pad_l + j * (col_w + col_gap)
            ax_p = ax_frame.inset_axes([x, pad_b + row2_h, col_w, row1_h])
            imshow_keep_aspect(ax_p, img_slice)
            ax_p.set_anchor('S')

        # 5) 下排：热力图（同列同宽），鲁棒查找；尽量贴近底部
        heat_paths = find_heatmap_images(scenario, product, continent_display, clusters_order)
        if VERBOSE: print("[heatmap] for", continent_display, "->", heat_paths)
        for j, cl in enumerate(clusters_order):
            x = pad_l + j * (col_w + col_gap)
            ax_h = ax_frame.inset_axes([x, pad_b, col_w, row2_h])
            im_h = crop_percent(load_img(heat_paths[j]),
                                top=heat_vtrim_top, bottom=heat_vtrim_bottom,
                                left=0.0, right=0.0)
            imshow_keep_aspect(ax_h, im_h)
            ax_h.set_anchor('N')

        # 6) 大洲标题（内侧上方）
        ax_frame.text(0.5, 1 - pad_t - region_label_offset, continent_display,
                      transform=ax_frame.transAxes, ha="center", va="top",
                      fontsize=12, fontweight="bold", clip_on=False)

    # 左列三洲
    for r, (cont_disp, cont_safe) in enumerate(left_continents):
        draw_block(gs[r, 0], cont_disp, cont_safe)
    # 右列三洲
    for r, (cont_disp, cont_safe) in enumerate(right_continents):
        draw_block(gs[r, 2], cont_disp, cont_safe)

    out_path = os.path.join(out_dir, f"combined_layout_{scenario}_{product.upper()}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white", edgecolor="none")
    plt.close(fig)
    print(f"✔ 保存组合图：{out_path}")
    return True

# -------------- 批量 & 单测 --------------
def find_scenario_products():
    """扫描 maps 目录推断可用 (scenario, product) 组合，顺便做 Region/Regional 归一"""
    sps = set()
    for fp in glob.glob(os.path.join(map_dir, "full_grid_cluster_map_*_*.png")):
        name = os.path.basename(fp)[len("full_grid_cluster_map_"):-4]
        parts = name.split("_")
        if len(parts) >= 2:
            scenario = "_".join(parts[:-1])
            product = parts[-1]
            sps.add((normalize_scenario_name(scenario), product.lower()))
    return sorted(sps)

def main():
    sps = find_scenario_products()
    if not sps:
        print("⚠ 未找到任何可用的情景-产品组合")
        return
    print(f"找到 {len(sps)} 个情景-产品组合：")
    for sp in sps:
        print("  ", sp)
    ok = 0
    for scenario, product in sps:
        ok += int(create_combined_figure(scenario, product))
    print(f"\n✅ 成功创建 {ok}/{len(sps)} 个组合图")

# -------------- 批量 & 过滤 --------------
def find_scenario_products():
    """扫描 maps 目录推断可用 (scenario, product) 组合，顺便做 Region/Regional 归一"""
    sps = set()
    for fp in glob.glob(os.path.join(map_dir, "full_grid_cluster_map_*_*.png")):
        name = os.path.basename(fp)[len("full_grid_cluster_map_"):-4]
        parts = name.split("_")
        if len(parts) >= 2:
            scenario = "_".join(parts[:-1])
            product = parts[-1]
            sps.add((normalize_scenario_name(scenario), product.lower()))
    return sorted(sps)

# —— 不同产品的参数预设（按需改动）——
PRODUCT_PRESETS = {
    # 针对 ammonia：更紧的上下贴合、更宽的左右面板区、更高的总体高度
    "ammonia": dict(
        outer_wspace=0.06, outer_hspace=0.04,
        inner_pad_top=0.002, inner_pad_bottom=0.002,
        inner_pad_left=0.006, inner_pad_right=0.006,
        col_gap=0.0,
        panel_heat_ratio=(1.40, 1.20),
        heat_vtrim_top=0.000, heat_vtrim_bottom=0.000,
        region_label_offset=0.035,
        width_ratios=(2.9, 2.0, 2.9),
        figsize=(30, 14)
    ),

    # 你也可以给 ftl / methanol 设不同风格；以下是示例（可删）
    "ftl": dict(
        outer_wspace=0.06, outer_hspace=0.04,
        inner_pad_top=0.002, inner_pad_bottom=0.002,
        inner_pad_left=0.006, inner_pad_right=0.006,
        col_gap=0.0,
        panel_heat_ratio=(1.40, 1.50),
        heat_vtrim_top=0.000, heat_vtrim_bottom=0.000,
        region_label_offset=0.035,
        width_ratios=(2.9, 2.0, 2.9),
        figsize=(30, 16.5)
    ),
    "methanol": dict(
        outer_wspace=0.06, outer_hspace=0.04,
        inner_pad_top=0.002, inner_pad_bottom=0.002,
        inner_pad_left=0.006, inner_pad_right=0.006,
        col_gap=0.0,
        panel_heat_ratio=(1.40, 1.70),
        heat_vtrim_top=0.000, heat_vtrim_bottom=0.000,
        region_label_offset=0.035,
        width_ratios=(2.9, 2.0, 2.9),
        figsize=(30, 16.5)
    ),
}

def main(include_products=None, include_scenarios=None, render_kwargs=None):
    """
    include_products: None / iterable[str]  -> 仅渲染这些产品
    include_scenarios: None / iterable[str] -> 仅渲染这些场景（会做 Region/Regional 归一）
    render_kwargs: dict -> 默认绘图参数（会被特定产品预设覆盖）
    """
    sps = find_scenario_products()
    if not sps:
        print("⚠ 未找到任何可用的情景-产品组合")
        return

    # 规范化过滤条件
    prod_set = {p.lower() for p in include_products} if include_products else None
    scen_set = {normalize_scenario_name(s) for s in include_scenarios} if include_scenarios else None

    # 过滤
    if prod_set or scen_set:
        sps = [(sc, pr) for sc, pr in sps
               if (prod_set is None or pr in prod_set)
               and (scen_set is None or normalize_scenario_name(sc) in scen_set)]

    if not sps:
        print("⚠ 过滤后无匹配组合（请检查 include_products / include_scenarios）")
        return

    print(f"将渲染 {len(sps)} 个情景-产品组合：")
    for sp in sps: print("  ", sp)

    ok = 0
    base_kwargs = render_kwargs or {}
    for scenario, product in sps:
        # 合并：默认参数 → 特定产品覆盖
        kwargs = dict(base_kwargs)
        if product in PRODUCT_PRESETS:
            kwargs.update(PRODUCT_PRESETS[product])
        ok += int(create_combined_figure(scenario, product, **kwargs))
    print(f"\n✅ 成功创建 {ok}/{len(sps)} 个组合图")

# -------------- 入口 --------------
if __name__ == "__main__":
    # ========== 用法 1：单个情景 + 单个产品（优先执行） ==========
    TEST_SCENARIO = None          # 例如: "RegionRE_LowAE"
    TEST_PRODUCT  = None          # 例如: "ammonia" / "ftl" / "methanol"

    # ========== 用法 2：批量渲染“若干产品 / 若干场景” ==========
    # 只画某几个产品（对所有可用场景）
    SELECT_PRODUCTS  = ["methanol"]   # None 表示所有产品
    # 只画某些场景（可选）
    SELECT_SCENARIOS = None                 # 例如: ["RegionRE_LowAE"]

    # 默认参数（会被 PRODUCT_PRESETS 中的对应产品覆盖）
    RENDER_KW = dict(
        outer_wspace=0.06, outer_hspace=0.04,
        inner_pad_top=0.002, inner_pad_bottom=0.002,
        inner_pad_left=0.006, inner_pad_right=0.006,
        col_gap=0.0,
        panel_heat_ratio=(1.40, 1.70),
        heat_vtrim_top=0.000, heat_vtrim_bottom=0.000,
        region_label_offset=0.035,
        width_ratios=(2.9, 2.0, 2.9),
        figsize=(30, 18)
    )

    # —— 执行 —— #
    if TEST_SCENARIO and TEST_PRODUCT:
        scen = normalize_scenario_name(TEST_SCENARIO)
        prod = str(TEST_PRODUCT).lower()
        kwargs = dict(RENDER_KW)
        kwargs.update(PRODUCT_PRESETS.get(prod, {}))   # 应用产品预设
        ok = create_combined_figure(scen, prod, **kwargs)
        print(f"\n✅ 单个情景完成：{scen} / {prod} -> {ok}")
    else:
        if isinstance(SELECT_PRODUCTS, str):
            SELECT_PRODUCTS = [SELECT_PRODUCTS]
        main(include_products=SELECT_PRODUCTS,
             include_scenarios=SELECT_SCENARIOS,
             render_kwargs=RENDER_KW)


